# XGBoosted Flash Flood Predictions: Full Day-2 v33 Viewer and Verification

This is the complete v33 Day-1 viewer copied to the Day-2 forecast workflow. It
keeps the original test-case viewer, realtime forecasts, WPC comparison, SHAP,
violin plots, occurrence/contingency tables, categorical metrics, reliability,
Brier score, area-error diagnostics, heatmaps, and centroid/displacement tools.

Day-2 contract:

- Training/realtime input date `D` is the RAP 09Z initialization date.
- Historical viewer `Date=V` is the verification-window start date, while
  `RAP_Init_Date=V-1` preserves the model initialization date.
- Thus viewer case `20240620` is valid 20240620 12Z-20240621 12Z and was
  initialized from RAP on 20240619 at 09Z.
- Predictors span valid offsets 0-48 h (`f03` through `f51`).
- Verification spans `D+1 12Z` through `D+2 12Z`, equivalently `V 12Z`
  through `V+1 12Z` in the historical viewer.
- The four ML members vary by target radius: R40, R60, R75, and R100.
- WPC is one radius-independent Day-2 ERO on the common verification grid.
- Verification retains two separate comparisons from the original viewer:
  probability-like practically-perfect fields, and raw UFVS flood proxies
  expanded to a 40-km radius before scoring.

Run the routing cell first, then proceed through the copied viewer in order.


## Day-2 routing and common verification grid

This cell creates one common Day-2 WPC/PP/UFVS grid. WPC and truth are not duplicated as radius-specific forecasts; only ML members carry an R40/R60/R75/R100 target-radius label. Raw UFVS proxy truth remains separate from practically-perfect truth and is expanded 40 km by the copied verification routines.


In [ ]:
from pathlib import Path
from datetime import datetime, timedelta
import io
import json
import math
import os
import re
import zipfile

import numpy as np
import pandas as pd
from scipy.spatial import cKDTree


# ======================================================================================
# Day-2 forecast/verification contract
# ======================================================================================
#
# Case date V:
#   * event/verification window: V 12Z through V+1 12Z
#   * RAP initialization: V-1 09Z
#   * RAP predictors: f03 through f51 (valid offsets 0 through 48 h)
#   * WPC comparison: one WPC Day-2 ERO for the common verification grid
#   * PP comparison: probability-like practically-perfect fields for that same window
#   * Proxy comparison: raw UFVS flood proxies expanded 40 km by verification routines
#
# WPC and the verifying observations are radius-independent. Only the four ML model
# members carry target-radius labels.

DAY2_VALID_OFFSETS_H = [0, 6, 12, 18, 24, 30, 36, 42, 48]
DAY2_RAP_FILE_FORECAST_HOURS = [3, 9, 15, 21, 27, 33, 39, 45, 51]
DAY2_TARGET_WINDOW_START_OFFSET_H = 24
DAY2_TARGET_WINDOW_H = 24
DAY2_WPC_OUTLOOK_DAY = 2
DAY2_PP_EXPANSION_RADIUS_KM = 40.0
DAY2_PP_SMOOTH_RADIUS_KM = 100.0
DAY2_PROXY_NEAREST_GRID_KM = 25.0
# Keep cases whose archive contains no matching Day-2 polygon set; WPC is NaN for
# that case and is excluded from WPC-only scores rather than being invented as zero.
DAY2_STRICT_WPC = False
DAY2_REUSE_DAY1_PP_SAME_CASE = True
DAY2_BUILD_FORCE = False
DAY2_FORCE_WPC = False
DAY2_FORCE_UFVS = False
DAY2_INCLUDE_REGULAR_FLOOD_LSR = False

DAY2_PROJECT_DIR_DEFAULT = Path(
    os.environ.get(
        "HAZARD_ML_PROJECT_DIR",
        "/home/tyreekfrazier/ISU_Research_LOCAL_RUN/fall_2025_ml_proj",
    )
)
DAY2_VERIFICATION_GRID_NAME = "df_pp_viewer_with_wpc_ero_day2valid.parquet"
DAY2_SOURCE_DAY1_GRID_NAME = "df_pp_viewer_with_wpc_ero_day1.parquet"
DAY2_R40_MASTER_NAME = (
    "pixel_domain_forecasts_rap09z_iem_mrms_ffg_"
    "r40km_singletarget_radiusstats_target_v33day2valid_apcp13p7cv_domain.parquet"
)
DAY2_POINT_MRMS_COLUMN = "Obs_Day2_MRMS_FFG_Exceeded_Point"
DAY2_WPC_IEM_URL = (
    "https://mesonet.agron.iastate.edu/cgi-bin/request/gis/outlooks.py"
)
DAY2_UFVS_BASE_URL = "https://ftp-wpc.ncep.noaa.gov/erickson/FFaIR/UFVS"
DAY2_UFVS_PREFIX_TO_COLUMN = {
    "ST4gFFG": "UFVS_STAGE4_FFG",
    "ST4gARI": "UFVS_STAGE4_ARI",
    "USGS": "UFVS_USGS",
    "LSRFLASH": "UFVS_LSR_FLASH",
    "LSRREG": "UFVS_LSR_REGULAR",
}


def day2_valid_window(case_date):
    """Return the event case window V 12Z through V+1 12Z."""
    date8 = re.search(r"(20\d{6})", str(case_date))
    if date8 is None:
        raise ValueError(f"Could not parse YYYYMMDD from {case_date!r}")
    start = pd.Timestamp(
        datetime.strptime(date8.group(1), "%Y%m%d"), tz="UTC"
    ) + pd.Timedelta(hours=12)
    return start, start + pd.Timedelta(days=1)


def day2_observation_date(case_date):
    """Return V, the Day-1 case date sharing the Day-2 observation window."""
    date8 = re.search(r"(20\d{6})", str(case_date))
    if date8 is None:
        raise ValueError(f"Could not parse YYYYMMDD from {case_date!r}")
    return datetime.strptime(date8.group(1), "%Y%m%d").strftime("%Y%m%d")


def _day2_date8(value):
    match = re.search(r"(20\d{6})", str(value))
    if match is None:
        raise ValueError(f"Could not parse YYYYMMDD from {value!r}")
    return match.group(1)


def day2_viewer_valid_window(viewer_date):
    """Return the historical viewer window V 12Z through V+1 12Z."""
    date8 = _day2_date8(viewer_date)
    start = pd.Timestamp(
        datetime.strptime(date8, "%Y%m%d"), tz="UTC"
    ) + pd.Timedelta(hours=12)
    return start, start + pd.Timedelta(days=1)


def _day2_apply_historical_viewer_date_convention(frame):
    """Keep event-valid Date and retain/derive the preceding RAP initialization."""
    output = frame.copy()
    if "Date" not in output.columns:
        raise KeyError("Historical Day-2 data must contain Date.")
    if "RAP_Init_Date" not in output.columns:
        event_dates = pd.to_datetime(
            output["Date"].astype(str).str[:8],
            format="%Y%m%d",
            errors="coerce",
        )
        if event_dates.isna().any():
            raise ValueError("Historical Day-2 Date contains unparseable values.")
        output["Date"] = event_dates.dt.strftime("%Y%m%d")
        output["RAP_Init_Date"] = (
            event_dates - pd.Timedelta(days=1)
        ).dt.strftime("%Y%m%d")
    else:
        output["Date"] = output["Date"].astype(str).str[:8]
        output["RAP_Init_Date"] = (
            output["RAP_Init_Date"].astype(str).str[:8]
        )
    output["Year"] = output["Date"].str[:4]
    return output


def _day2_latlon_to_xyz(lat, lon):
    lat_r = np.deg2rad(np.asarray(lat, dtype=float))
    lon_r = np.deg2rad(np.asarray(lon, dtype=float))
    return np.column_stack(
        [
            np.cos(lat_r) * np.cos(lon_r),
            np.cos(lat_r) * np.sin(lon_r),
            np.sin(lat_r),
        ]
    )


def _day2_chord_radius(radius_km):
    return 2.0 * np.sin((float(radius_km) / 6371.0) / 2.0)


def _day2_expand_mask(mask, lat, lon, radius_km=40.0):
    raw = np.asarray(mask, dtype=bool)
    if not raw.any() or radius_km is None or float(radius_km) <= 0:
        return raw.copy()
    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    good = np.isfinite(lat) & np.isfinite(lon)
    out = raw.copy()
    if not good.any():
        return out
    xyz = _day2_latlon_to_xyz(lat[good], lon[good])
    tree = cKDTree(xyz)
    good_indices = np.flatnonzero(good)
    event_good = np.flatnonzero(raw[good])
    chord = _day2_chord_radius(radius_km)
    for event_index in event_good:
        neighbors = tree.query_ball_point(xyz[event_index], r=chord)
        out[good_indices[np.asarray(neighbors, dtype=int)]] = True
    return out


def _day2_smooth_expanded_mask(
    expanded_mask,
    lat,
    lon,
    smooth_radius_km=100.0,
    cutoff_sigma=3.0,
    chunk_size=1500,
):
    """Apply the Day-1 viewer's weighted local fractional-coverage PP smoothing."""
    binary = np.asarray(expanded_mask, dtype=np.float32)
    if not np.any(binary > 0):
        return np.zeros(len(binary), dtype=np.float32)
    if smooth_radius_km is None or float(smooth_radius_km) <= 0:
        return np.clip(binary, 0.0, 1.0)

    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    xyz = _day2_latlon_to_xyz(lat, lon)
    tree = cKDTree(xyz)
    sigma_km = float(smooth_radius_km)
    max_chord = _day2_chord_radius(float(cutoff_sigma) * sigma_km)
    out = np.zeros(len(binary), dtype=np.float32)

    for start in range(0, len(binary), int(chunk_size)):
        stop = min(start + int(chunk_size), len(binary))
        neighbor_lists = tree.query_ball_point(xyz[start:stop], r=max_chord)
        for offset, neighbors in enumerate(neighbor_lists):
            if not neighbors:
                continue
            idx = np.asarray(neighbors, dtype=np.int64)
            chord_dist = np.linalg.norm(xyz[idx] - xyz[start + offset], axis=1)
            chord_dist = np.clip(chord_dist, 0.0, 2.0)
            distance_km = 6371.0 * (2.0 * np.arcsin(chord_dist / 2.0))
            weights = np.exp(-0.5 * (distance_km / sigma_km) ** 2)
            denominator = float(np.sum(weights))
            if denominator > 0:
                out[start + offset] = float(
                    np.sum(weights * binary[idx]) / denominator
                )
    return np.clip(out, 0.0, 1.0).astype(np.float32)


def _day2_pp_from_raw_mask(raw_mask, lat, lon):
    expanded = _day2_expand_mask(
        raw_mask,
        lat,
        lon,
        radius_km=DAY2_PP_EXPANSION_RADIUS_KM,
    )
    return _day2_smooth_expanded_mask(
        expanded,
        lat,
        lon,
        smooth_radius_km=DAY2_PP_SMOOTH_RADIUS_KM,
    )


def _day2_parse_ufvs_points(text):
    values = [
        float(value)
        for value in re.findall(r"[-+]?\d+(?:\.\d+)?", str(text))
    ]
    points = []
    for first, second in zip(values[0::2], values[1::2]):
        if 15 <= first <= 60 and -130 <= second <= -60:
            points.append((first, second))
        elif 15 <= second <= 60 and -130 <= first <= -60:
            points.append((second, first))
    if not points:
        return pd.DataFrame(columns=["Lat", "Lon"])
    return (
        pd.DataFrame(points, columns=["Lat", "Lon"])
        .drop_duplicates()
        .reset_index(drop=True)
    )


def _day2_ufvs_windows(case_date):
    obs_date = day2_observation_date(case_date)
    obs_start = datetime.strptime(obs_date, "%Y%m%d")
    obs_end = obs_start + timedelta(days=1)
    return [
        (f"{obs_date}12", f"{obs_end.strftime('%Y%m%d')}12"),
        (f"{obs_date}16", f"{obs_end.strftime('%Y%m%d')}12"),
    ]


def _day2_fetch_ufvs_points(
    case_date,
    prefix,
    cache_dir,
    force=False,
):
    import requests

    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    errors = []
    for start_stamp, end_stamp in _day2_ufvs_windows(case_date):
        cache_path = cache_dir / f"{prefix}_s{start_stamp}_e{end_stamp}.txt"
        url = (
            f"{DAY2_UFVS_BASE_URL}/"
            f"{prefix}_s{start_stamp}_e{end_stamp}.txt"
        )
        try:
            if force or not cache_path.exists() or cache_path.stat().st_size == 0:
                response = requests.get(url, timeout=(10, 60))
                response.raise_for_status()
                cache_path.write_text(response.text, encoding="utf-8")
            points = _day2_parse_ufvs_points(
                cache_path.read_text(encoding="utf-8", errors="replace")
            )
            if len(points):
                return points, f"{start_stamp}-{end_stamp}"
            errors.append(f"{url}: no points")
        except Exception as exc:
            errors.append(f"{url}: {exc}")
    return pd.DataFrame(columns=["Lat", "Lon"]), " | ".join(errors)


def _day2_points_to_grid_mask(points, grid):
    mask = np.zeros(len(grid), dtype=bool)
    if points is None or len(points) == 0 or len(grid) == 0:
        return mask
    grid_xyz = _day2_latlon_to_xyz(
        pd.to_numeric(grid["Lat"], errors="coerce").to_numpy(float),
        pd.to_numeric(grid["Lon"], errors="coerce").to_numpy(float),
    )
    point_xyz = _day2_latlon_to_xyz(
        pd.to_numeric(points["Lat"], errors="coerce").to_numpy(float),
        pd.to_numeric(points["Lon"], errors="coerce").to_numpy(float),
    )
    tree = cKDTree(grid_xyz)
    distance, location = tree.query(point_xyz, k=1)
    keep = np.isfinite(distance) & (
        distance <= _day2_chord_radius(DAY2_PROXY_NEAREST_GRID_KM)
    )
    if keep.any():
        mask[np.unique(location[keep])] = True
    return mask


def _day2_wpc_probability(row):
    text = " ".join(
        str(row.get(column, ""))
        for column in ("CATEGORY", "THRESHOLD", "LABEL", "OUTLOOK", "name")
        if column in row.index
    ).upper()
    if "HIGH" in text:
        return 0.70
    if "MDT" in text or "MOD" in text or "MODERATE" in text:
        return 0.40
    if "SLGT" in text or "SLIGHT" in text:
        return 0.15
    if "MRGL" in text or "MARG" in text or "MARGINAL" in text:
        return 0.05
    return 0.0


def _day2_datetime_column(frame, candidates):
    lookup = {str(column).upper().strip(): column for column in frame.columns}
    for candidate in candidates:
        column = lookup.get(str(candidate).upper().strip())
        if column is None:
            continue
        values = pd.to_datetime(frame[column], errors="coerce", utc=True)
        if values.notna().any():
            return values
    return pd.Series(pd.NaT, index=frame.index, dtype="datetime64[ns, UTC]")


def _day2_filter_wpc_to_valid_window(frame, case_date):
    """Select the latest WPC Day-2 revision for the exact V-to-V+1 window."""
    target_start, target_end = day2_valid_window(case_date)
    issue = _day2_datetime_column(
        frame, ("ISSUE", "VALID", "START", "BEGINTIME")
    )
    expire = _day2_datetime_column(
        frame, ("EXPIRE", "EXPIRATION", "END", "ENDTIME")
    )
    prodiss = _day2_datetime_column(
        frame, ("PRODISS", "PRODUCTISSUANCE", "ISSUANCE")
    )
    tolerance = pd.Timedelta(minutes=2)
    if issue.notna().any() and expire.notna().any():
        selected = (
            issue.sub(target_start).abs().le(tolerance)
            & expire.sub(target_end).abs().le(tolerance)
        )
        reason = "exact_day2_valid_window"
        if not selected.any():
            candidates = (issue < target_end) & (expire > target_start)
            if not candidates.any():
                return frame.iloc[0:0].copy()
            overlap_start = np.maximum(
                issue[candidates].astype("int64"), target_start.value
            )
            overlap_end = np.minimum(
                expire[candidates].astype("int64"), target_end.value
            )
            overlap_hours = (overlap_end - overlap_start) / 3.6e12
            offset_hours = (
                (issue[candidates] - target_start).abs()
                / pd.Timedelta(hours=1)
            ).to_numpy()
            score = np.asarray(overlap_hours) - 0.25 * offset_hours
            best = issue[candidates].index[int(np.nanargmax(score))]
            selected = (issue == issue.loc[best]) & (expire == expire.loc[best])
            reason = "best_day2_overlap"
        latest = pd.NaT
        if prodiss.notna().any() and selected.any():
            latest = prodiss[selected].max()
            selected &= prodiss == latest
        output = frame.loc[selected].copy()
        output.attrs.update(
            {
                "wpc_target_valid_start": str(target_start),
                "wpc_target_valid_end": str(target_end),
                "wpc_selected_valid_start": str(issue.loc[output.index].min()),
                "wpc_selected_valid_end": str(expire.loc[output.index].max()),
                "wpc_selected_prodiss": (
                    str(latest) if pd.notna(latest) else "unknown"
                ),
                "wpc_selected_reason": reason,
            }
        )
        return output
    return frame.iloc[0:0].copy()


def _day2_download_wpc_gdf(case_date, cache_dir, force=False):
    import geopandas as gpd
    import requests

    case_date = _day2_date8(case_date)
    case_start = datetime.strptime(case_date, "%Y%m%d")
    target_start, target_end = day2_valid_window(case_date)
    start_query = (case_start - timedelta(hours=6)).strftime(
        "%Y-%m-%dT%H:%MZ"
    )
    end_query = (case_start + timedelta(days=1, hours=18)).strftime(
        "%Y-%m-%dT%H:%MZ"
    )
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    errors = []
    # The standalone Day-2 verifier already cached the default geometry files.
    # Try that spelling first before any network request for alternate modes.
    for geometry_mode in (None, "cookie", "cutter", "nonoverlap", "layers"):
        suffix = "default" if geometry_mode is None else geometry_mode
        zip_path = cache_dir / (
            f"iem_wpc_ero_day2_{case_date}_valid_v_12_to_v1_12_{suffix}.zip"
        )
        params = {
            "type": "E",
            "d": "2",
            "sts": start_query,
            "ets": end_query,
        }
        if geometry_mode is not None:
            params["geom"] = geometry_mode
        try:
            if force or not zip_path.exists() or zip_path.stat().st_size < 128:
                response = requests.get(
                    DAY2_WPC_IEM_URL, params=params, timeout=(10, 60)
                )
                response.raise_for_status()
                if not response.content.startswith(b"PK"):
                    raise ValueError("IEM response was not a zip archive")
                zip_path.write_bytes(response.content)
            try:
                polygons = gpd.read_file(f"zip://{zip_path}")
            except Exception:
                polygons = gpd.read_file(str(zip_path))
            if polygons is None or polygons.empty:
                raise ValueError("empty WPC shapefile")
            day_lookup = {
                str(column).upper().strip(): column
                for column in polygons.columns
            }
            if "DAY" in day_lookup:
                day_values = pd.to_numeric(
                    polygons[day_lookup["DAY"]], errors="coerce"
                )
                polygons = polygons[day_values == DAY2_WPC_OUTLOOK_DAY].copy()
            polygons = _day2_filter_wpc_to_valid_window(
                polygons, case_date
            )
            if polygons.empty:
                raise ValueError(
                    f"no WPC Day-2 polygons valid {target_start} to {target_end}"
                )
            if polygons.crs is not None:
                polygons = polygons.to_crs(epsg=4326)
            polygons["__WPC_ERO_Risk"] = polygons.apply(
                _day2_wpc_probability, axis=1
            ).astype(float)
            polygons = polygons[polygons["__WPC_ERO_Risk"] > 0].copy()
            if polygons.empty:
                raise ValueError("unrecognized WPC risk categories")
            return polygons
        except Exception as exc:
            errors.append(f"{suffix}: {exc}")
    raise RuntimeError(
        f"WPC Day-2 ERO unavailable for {case_date}: " + " | ".join(errors)
    )


def _day2_rasterize_wpc(polygons, grid):
    output = np.zeros(len(grid), dtype=np.float32)
    if polygons is None or len(polygons) == 0:
        return output
    from shapely.geometry import Point
    from shapely.prepared import prep

    try:
        import shapely

        contains_xy = getattr(shapely, "contains_xy", None)
    except Exception:
        contains_xy = None

    lon = pd.to_numeric(grid["Lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(grid["Lat"], errors="coerce").to_numpy(float)
    finite = np.isfinite(lon) & np.isfinite(lat)
    for _, row in polygons.sort_values("__WPC_ERO_Risk").iterrows():
        geometry = row.geometry
        if geometry is None or geometry.is_empty:
            continue
        min_x, min_y, max_x, max_y = geometry.bounds
        candidates = np.flatnonzero(
            finite
            & (lon >= min_x)
            & (lon <= max_x)
            & (lat >= min_y)
            & (lat <= max_y)
        )
        if not candidates.size:
            continue
        if contains_xy is not None:
            inside = contains_xy(
                geometry, lon[candidates], lat[candidates]
            )
        else:
            prepared = prep(geometry)
            inside = np.asarray(
                [
                    prepared.contains(Point(x, y))
                    or prepared.touches(Point(x, y))
                    for x, y in zip(lon[candidates], lat[candidates])
                ],
                dtype=bool,
            )
        output[candidates[inside]] = np.maximum(
            output[candidates[inside]],
            float(row["__WPC_ERO_Risk"]),
        )
    return output


def _day2_probability_category(values):
    values = np.asarray(values, dtype=float)
    labels = np.full(values.shape, "None", dtype=object)
    labels[values >= 0.05] = "Marginal"
    labels[values >= 0.15] = "Slight"
    labels[values >= 0.40] = "Moderate"
    labels[values >= 0.70] = "High"
    return labels


def _day2_load_day1_pp_same_case(project_dir):
    source = Path(project_dir) / DAY2_SOURCE_DAY1_GRID_NAME
    if not DAY2_REUSE_DAY1_PP_SAME_CASE or not source.exists():
        return pd.DataFrame()
    import pyarrow.parquet as pq
    available = list(pq.ParquetFile(source).schema.names)
    wanted = [
        "Date",
        "Lat",
        "Lon",
        *[
            column
            for column in available
            if column.startswith("PP_")
            or column.startswith("PointProxy_")
            or column.startswith("UFVS_")
        ],
    ]
    same_case = pd.read_parquet(source, columns=list(dict.fromkeys(wanted)))
    observation_dates = pd.to_datetime(
        same_case["Date"].astype(str).str[:8], format="%Y%m%d", errors="coerce"
    )
    same_case["Date"] = observation_dates.dt.strftime("%Y%m%d")
    same_case["__LatKey"] = pd.to_numeric(
        same_case["Lat"], errors="coerce"
    ).round(5)
    same_case["__LonKey"] = pd.to_numeric(
        same_case["Lon"], errors="coerce"
    ).round(5)
    return same_case.drop(columns=["Lat", "Lon"], errors="ignore")


def _day2_build_proxy_and_pp_case(
    case_grid,
    case_date,
    project_dir,
    day1_same_case,
    force=False,
):
    project_dir = Path(project_dir)
    cache_dir = project_dir / "day2_pp_ufvs_case_cache_v33day2valid"
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = cache_dir / f"day2_pp_ufvs_{case_date}.parquet"
    if cache_path.exists() and not force:
        cached = pd.read_parquet(cache_path)
        if len(cached) == len(case_grid):
            return cached

    output = case_grid[
        ["Date", "RAP_Init_Date", "Lat", "Lon", DAY2_POINT_MRMS_COLUMN]
    ].copy()
    output["Date"] = _day2_date8(case_date)
    output["Year"] = output["Date"].str[:4]
    output["__LatKey"] = pd.to_numeric(
        output["Lat"], errors="coerce"
    ).round(5)
    output["__LonKey"] = pd.to_numeric(
        output["Lon"], errors="coerce"
    ).round(5)

    if day1_same_case is not None and len(day1_same_case):
        shifted_case = day1_same_case[
            day1_same_case["Date"].astype(str) == str(case_date)
        ].copy()
        if len(shifted_case):
            shifted_case = shifted_case.drop_duplicates(
                ["Date", "__LatKey", "__LonKey"]
            )
            output = output.merge(
                shifted_case,
                on=["Date", "__LatKey", "__LonKey"],
                how="left",
                suffixes=("", "_shifted_day1"),
            )

    ufvs_cache = project_dir / "day2_ufvs_raw_download_cache_v33day2valid"
    source_status = []
    component_masks = {}
    for prefix, column in DAY2_UFVS_PREFIX_TO_COLUMN.items():
        points, status = _day2_fetch_ufvs_points(
            case_date,
            prefix,
            ufvs_cache,
            force=DAY2_FORCE_UFVS or force,
        )
        component_masks[column] = _day2_points_to_grid_mask(points, output)
        source_status.append(
            f"{prefix}:{len(points)}points:{status}"
        )
        output[column] = component_masks[column].astype(np.int8)
    output["UFVS_LSR_FLOOD"] = output["UFVS_LSR_REGULAR"].astype(np.int8)

    component_columns = [
        "UFVS_STAGE4_FFG",
        "UFVS_STAGE4_ARI",
        "UFVS_USGS",
        "UFVS_LSR_FLASH",
    ]
    if DAY2_INCLUDE_REGULAR_FLOOD_LSR:
        component_columns.append("UFVS_LSR_REGULAR")
    raw_any = output[component_columns].max(axis=1).to_numpy(dtype=np.int8) > 0
    shifted_any_col = "PointProxy_Any flood proxy"
    if shifted_any_col in output.columns:
        shifted_any = (
            pd.to_numeric(output[shifted_any_col], errors="coerce")
            .fillna(0)
            .to_numpy(float)
            > 0
        )
        raw_any |= shifted_any
    output["UFVS_ANY"] = raw_any.astype(np.int8)
    output[shifted_any_col] = raw_any.astype(np.int8)

    lsr_usgs_columns = [
        "UFVS_USGS",
        "UFVS_LSR_FLASH",
    ]
    if DAY2_INCLUDE_REGULAR_FLOOD_LSR:
        lsr_usgs_columns.append("UFVS_LSR_REGULAR")
    raw_lsr_usgs = (
        output[lsr_usgs_columns].max(axis=1).to_numpy(dtype=np.int8) > 0
    )
    shifted_lsr_col = "PointProxy_LSR/USGS only"
    if shifted_lsr_col in output.columns:
        raw_lsr_usgs |= (
            pd.to_numeric(output[shifted_lsr_col], errors="coerce")
            .fillna(0)
            .to_numpy(float)
            > 0
        )
    output[shifted_lsr_col] = raw_lsr_usgs.astype(np.int8)

    raw_mrms = (
        pd.to_numeric(output[DAY2_POINT_MRMS_COLUMN], errors="coerce")
        .fillna(0)
        .to_numpy(float)
        > 0
    )
    output["PointProxy_MRMS > FFG"] = raw_mrms.astype(np.int8)

    lat = pd.to_numeric(output["Lat"], errors="coerce").to_numpy(float)
    lon = pd.to_numeric(output["Lon"], errors="coerce").to_numpy(float)
    pp_inputs = {
        "PP_MRMS > FFG": raw_mrms,
        "PP_LSR/USGS only": raw_lsr_usgs,
        "PP_Any flood proxy": raw_any,
    }
    for column, raw_mask in pp_inputs.items():
        existing = (
            pd.to_numeric(output[column], errors="coerce")
            if column in output.columns
            else pd.Series(np.nan, index=output.index)
        )
        if existing.notna().all():
            output[column] = existing.clip(0, 1).astype(np.float32)
        else:
            output[column] = _day2_pp_from_raw_mask(
                raw_mask, lat, lon
            ).astype(np.float32)

    output["UFVS_Day2_Valid_Start"] = str(day2_valid_window(case_date)[0])
    output["UFVS_Day2_Valid_End"] = str(day2_valid_window(case_date)[1])
    output["UFVS_Source_Status"] = " | ".join(source_status)
    output = output.drop(columns=["__LatKey", "__LonKey"], errors="ignore")
    output.to_parquet(cache_path, index=False)
    return output


def _day2_build_wpc_case(
    case_grid,
    case_date,
    project_dir,
    force=False,
):
    project_dir = Path(project_dir)
    # Reuse the Day-2 caches produced by the standalone verifier when available.
    grid_cache_dir = project_dir / "wpc_ero_day2_cache_v33day2valid"
    shape_cache_dir = grid_cache_dir
    grid_cache_dir.mkdir(parents=True, exist_ok=True)
    cache_path = grid_cache_dir / f"wpc_day2_grid_{case_date}.parquet"
    existing_paths = [cache_path, *sorted(
        grid_cache_dir.glob(f"wpc_day2_grid_{case_date}_*rows.parquet")
    )]
    if not force:
        for existing_path in existing_paths:
            if not existing_path.exists():
                continue
            cached = pd.read_parquet(existing_path)
            if "WPC_Day2_Probability" in cached.columns:
                cached = cached.rename(
                    columns={"WPC_Day2_Probability": "WPC_ERO_Risk"}
                )
            if len(cached) != len(case_grid) or "WPC_ERO_Risk" not in cached:
                continue
            cached_keys = set(zip(
                pd.to_numeric(cached["Lat"], errors="coerce").round(5),
                pd.to_numeric(cached["Lon"], errors="coerce").round(5),
            ))
            grid_keys = set(zip(
                pd.to_numeric(case_grid["Lat"], errors="coerce").round(5),
                pd.to_numeric(case_grid["Lon"], errors="coerce").round(5),
            ))
            if cached_keys != grid_keys:
                continue
            cached["Date"] = _day2_date8(case_date)
            cached["WPC_ERO_Category"] = _day2_probability_category(
                cached["WPC_ERO_Risk"]
            )
            return cached

    polygons = _day2_download_wpc_gdf(
        case_date,
        shape_cache_dir,
        force=DAY2_FORCE_WPC or force,
    )
    risk = _day2_rasterize_wpc(polygons, case_grid)
    output = case_grid[["Date", "Lat", "Lon"]].copy()
    output["Date"] = _day2_date8(case_date)
    output["WPC_ERO_Risk"] = risk.astype(np.float32)
    output["WPC_ERO_Category"] = _day2_probability_category(risk)
    output["WPC_ERO_Source_ProdIssue"] = str(
        polygons.attrs.get("wpc_selected_prodiss", "")
    )
    output["WPC_ERO_Source_ValidStart"] = str(
        polygons.attrs.get("wpc_selected_valid_start", "")
    )
    output["WPC_ERO_Source_ValidEnd"] = str(
        polygons.attrs.get("wpc_selected_valid_end", "")
    )
    output["WPC_ERO_Source_SelectedReason"] = str(
        polygons.attrs.get("wpc_selected_reason", "")
    )
    output["WPC_ERO_Source_MaxRisk"] = float(
        polygons["__WPC_ERO_Risk"].max()
    )
    output["WPC_ERO_Raster_MaxRisk"] = float(np.nanmax(risk))
    output.to_parquet(cache_path, index=False)
    return output


def build_or_load_day2_verification_grid(
    project_dir=DAY2_PROJECT_DIR_DEFAULT,
    test_years=("2024", "2025"),
    force=False,
):
    """Build the common Day-2 PP/UFVS/WPC grid used by every ML radius."""
    project_dir = Path(project_dir)
    output_path = project_dir / DAY2_VERIFICATION_GRID_NAME
    if output_path.exists() and not (force or DAY2_BUILD_FORCE):
        print(f"Loading existing Day-2 verification grid: {output_path}")
        return _day2_apply_historical_viewer_date_convention(
            pd.read_parquet(output_path)
        )

    master_path = project_dir / DAY2_R40_MASTER_NAME
    if not master_path.exists():
        raise FileNotFoundError(
            f"Day-2 R40 master is required as the canonical grid: {master_path}"
        )
    master = pd.read_parquet(
        master_path,
        columns=[
            "Date",
            "RAP_Init_Date",
            "Lat",
            "Lon",
            DAY2_POINT_MRMS_COLUMN,
        ],
    )
    master["Date"] = master["Date"].astype(str).str[:8]
    master["Year"] = master["Date"].str[:4]
    master = master[master["Year"].isin([str(year) for year in test_years])].copy()
    if master.empty:
        raise RuntimeError(
            f"No Day-2 master rows found for test years {list(test_years)}"
        )
    day1_same_case = _day2_load_day1_pp_same_case(project_dir)
    pieces = []
    dates = sorted(master["Date"].dropna().unique().tolist())
    for index, case_date in enumerate(dates, start=1):
        case_grid = master[master["Date"] == case_date].copy().reset_index(drop=True)
        pp_proxy = _day2_build_proxy_and_pp_case(
            case_grid,
            case_date,
            project_dir,
            day1_same_case,
            force=force or DAY2_BUILD_FORCE,
        )
        try:
            wpc = _day2_build_wpc_case(
                case_grid,
                case_date,
                project_dir,
                force=force or DAY2_BUILD_FORCE,
            )
        except Exception:
            if DAY2_STRICT_WPC:
                raise
            wpc = case_grid[["Date", "Lat", "Lon"]].copy()
            wpc["WPC_ERO_Risk"] = np.nan
            wpc["WPC_ERO_Category"] = "Unavailable"
        wpc_columns = [
            column
            for column in wpc.columns
            if column not in {"Date", "Lat", "Lon"}
        ]
        case = pp_proxy.merge(
            wpc[["Date", "Lat", "Lon", *wpc_columns]],
            on=["Date", "Lat", "Lon"],
            how="left",
            validate="one_to_one",
        )
        pieces.append(case)
        print(
            f"Day-2 verification grid {index:03d}/{len(dates):03d}: "
            f"{case_date}, rows={len(case):,}, "
            f"WPC max={case['WPC_ERO_Risk'].max():.2f}, "
            f"UFVS events={int(case['UFVS_ANY'].sum())}"
        )
    output = _day2_apply_historical_viewer_date_convention(
        pd.concat(pieces, ignore_index=True)
    )
    output.to_parquet(output_path, index=False)
    print(f"Saved common Day-2 verification grid: {output_path}")
    return output


def assert_wpc_radius_independent(metric_table):
    """Fail if a verification table incorrectly treats WPC as radius-specific."""
    if metric_table is None or len(metric_table) == 0:
        return True
    source_column = next(
        (
            column
            for column in ("Source", "Method", "Forecast")
            if column in metric_table.columns
        ),
        None,
    )
    if source_column is None:
        return True
    wpc = metric_table[
        metric_table[source_column].astype(str).str.contains(
            "WPC", case=False, na=False
        )
    ].copy()
    if wpc.empty:
        return True
    radius_columns = [
        column
        for column in (
            "Source Radius",
            "ML Target Radius km",
            "ML_Target_Radius_km",
            "radius_km",
            "Radius_km",
            "Radius km",
            "Target Radius km",
        )
        if column in wpc.columns
    ]
    for column in radius_columns:
        values = wpc[column].dropna().astype(str)
        bad = values[~values.str.lower().isin({"wpc", "none", "nan"})]
        if len(bad):
            raise AssertionError(
                f"WPC must be radius-independent, but {column} contains "
                f"{sorted(bad.unique().tolist())}"
            )
    return True


print(
    "Day-2 routing loaded: models use R40/R60/R75/R100 targets; "
    "WPC and the PP/UFVS verification grid are common across radii; "
    "historical Date is the target-valid start date and RAP_Init_Date retains "
    "the RAP 09Z source date; UFVS proxy truth is expanded 40 km by the copied "
    "verification sections."
)


In [ ]:
import os
import re
import json
import glob
import math
import warnings
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output

try:
    import joblib
    
except Exception as e:
    raise RuntimeError("This viewer needs joblib to load the trained model/scaler artifacts.") from e

try:
    import pyarrow.parquet as pq
except Exception as e:
    raise RuntimeError("This viewer needs pyarrow to inspect/read parquet columns safely.") from e

try:
    from scipy.spatial import cKDTree
except Exception as e:
    raise RuntimeError("This viewer needs scipy.spatial.cKDTree for 40-km ROI verification.") from e

try:
    import ipywidgets as widgets
except Exception as e:
    raise RuntimeError("This viewer needs ipywidgets.") from e

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY = True
except Exception:
    HAS_CARTOPY = False


# ======================================================================================
# 0. Settings
# ======================================================================================

# Prefer the project tree that already contains usable v33 prediction caches/artifacts.
# This avoids hard-coding the wrong local-vs-mounted path.
_CANDIDATE_PROJECT_DIRS = [
    # Prefer the explicit local tree when scores tie.
    "/home/tyreekfrazier/ISU_Research_LOCAL_RUN/fall_2025_ml_proj",
    "/home/tyreekfrazier/ISU_Research/fall_2025_ml_proj",
    "/home/tyreekfrazier/ISU_RESEARCH_LOCAL_RUN/fall_2025_ml_proj",
]

def _project_dir_score(p):
    p = os.path.abspath(os.path.expanduser(str(p)))
    score = 0
    if os.path.isdir(p):
        score += 10
    if os.path.exists(os.path.join(p, "df_pp_viewer_with_wpc_ero_day2.parquet")):
        score += 10
    if os.path.isdir(os.path.join(p, "prob_flood_models")):
        score += 10
    pred_dir = os.path.join(p, "v33day2valid_singletarget_radius_sensitivity_viewer_prediction_cache")
    if os.path.isdir(pred_dir):
        score += 5
        score += min(20, len(glob.glob(os.path.join(pred_dir, "*.parquet"))) * 5)
    if glob.glob(os.path.join(p, "pixel_domain_forecasts_rap09z_iem_mrms_ffg_*v33day2valid*.parquet")):
        score += 15
    return score

_PROJECT_DIR_SCORES = [(p, _project_dir_score(p)) for p in _CANDIDATE_PROJECT_DIRS]
def _project_dir_tiebreak(item):
    p, score = item
    # Prefer explicit local mirror when artifact scores tie.
    local_bonus = 1 if "ISU_Research_LOCAL_RUN" in str(p) else 0
    badcase_penalty = -1 if "ISU_RESEARCH_LOCAL_RUN" in str(p) else 0
    return (score, local_bonus, badcase_penalty)

PROJECT_DIR = max(_PROJECT_DIR_SCORES, key=_project_dir_tiebreak)[0]
print("Candidate PROJECT_DIR scores:")
for _p, _s in _PROJECT_DIR_SCORES:
    print(f"  {_s:3d}  {_p}")
print(f"Using PROJECT_DIR = {PROJECT_DIR}")

MODEL_CACHE_DIR = os.path.join(PROJECT_DIR, "prob_flood_models")

PP_WPC_GRID_CACHE = os.path.join(
    PROJECT_DIR,
    "df_pp_viewer_with_wpc_ero_day2.parquet"
)

PP_VIEWER_CACHE = os.path.join(
    PROJECT_DIR,
    "practically_perfect_v33day2valid_interactive_viewer_cache.parquet"
)

RUN_VERSION_TAG = "v33day2valid"
RADIUS_LIST_KM = [40, 60, 75, 100]
TEST_YEARS = ["2024", "2025"]
MODEL_SPECS = [
    {"label": "r40km", "radius_km": 40},
    {"label": "r60km", "radius_km": 60},
    {"label": "r75km", "radius_km": 75},
    {"label": "r100km", "radius_km": 100},
]

def model_label_for_radius(radius_km):
    r = int(round(float(radius_km)))
    return f"r{r}km"

# The v33 driver uses this tag form.
def experiment_tag_for_radius(radius_km):
    r = int(round(float(radius_km)))
    return f"{RUN_VERSION_TAG}_r{r}km_singletarget_radiusstats_mse_apcp13p7cv_domain"

def experiment_tag_for_model(radius_km, model_label=None):
    return experiment_tag_for_radius(radius_km)


def cache_tag_for_model(model_label):
    return None


def _wide_ml_member_metadata(col):
    c = str(col)
    m = re.match(r"^ML_r(\d+)_Prob$", c)
    return (f"r{int(m.group(1))}km", int(m.group(1))) if m else (None, None)

MASTER_PARQUET_TEMPLATE = os.path.join(
    PROJECT_DIR,
    "pixel_domain_forecasts_rap09z_iem_mrms_ffg_{experiment_tag}.parquet"
)

PREDICTION_CACHE_DIR = os.path.join(PROJECT_DIR, "v33day2valid_singletarget_radius_sensitivity_viewer_prediction_cache")
METRIC_CACHE_DIR = os.path.join(PROJECT_DIR, "v33day2valid_singletarget_radius_sensitivity_viewer_metric_cache")
os.makedirs(PREDICTION_CACHE_DIR, exist_ok=True)
os.makedirs(METRIC_CACHE_DIR, exist_ok=True)

# Set True if you retrained models or want to force regeneration.
FORCE_REBUILD_RADIUS_PREDICTIONS = False
FORCE_REBUILD_RADIUS_METRICS = False

FORECAST_COL = "Forecast_Prob"
WPC_COL = "WPC_ERO_Risk"

# Default PP truth fields to use in metrics.
TRUTH_DEFINITIONS = [
    "MRMS > FFG",
    "Any flood proxy",
]

RISK_THRESHOLDS = [
    (0.05, ">5%"),
    (0.15, ">15%"),
    (0.40, ">40%"),
    (0.70, ">70%"),
]
RISK_ORDER = [label for _, label in RISK_THRESHOLDS]

# Verification ROI, not the ML target radius.
VERIFY_ROI_KM = 40.0
EARTH_RADIUS_KM = 6371.0

PREDICT_CHUNK_SIZE = 250_000
PARQUET_READ_ROW_GROUPS = None  # None reads all row groups selected columns.

DEFAULT_EXTENT = [-105.0, -80.5, 30.0, 50.0]

RISK_BOUNDS = [0.00, 0.05, 0.15, 0.40, 0.70, 1.01]
RISK_LABELS = ["<5%", ">5%", ">15%", ">40%", ">70%"]
RISK_COLORS = {
    "<5%": "#FFFFFF",
    ">5%": "#5ED135",
    ">15%": "#F1EE36",
    ">40%": "#D34737",
    ">70%": "#DD4DDD",
}
AGREEMENT_COLORS = {
    "Neither": "#FFFFFF",
    "Hit": "#2ECC71",
    "Miss": "#3498DB",
    "False Alarm": "#E74C3C",
}

# Map settings.
POINT_SIZE_DEFAULT = 8.0
POINT_ALPHA_DEFAULT = 0.90
SHOW_POINTS_BELOW_5_DEFAULT = False

# Metric plot settings.
MAKE_POOLED_METRICS = True


# ======================================================================================
# 1. Pickle compatibility for saved regression wrapper
# ======================================================================================

class ClippedRegressionProbabilityWrapper:
    """
    Compatibility definition kept for older models; v33 binary models load as XGBClassifier directly.

    IMPORTANT: this class intentionally does NOT define __getattr__.
    During pickle/joblib unpickling, __getattr__ can be called before
    base_model is restored, which causes infinite recursion and
    a maximum-recursion-depth error.
    """
    def __init__(self, base_model=None):
        self.base_model = base_model

    def predict(self, X):
        p = self.base_model.predict(X)
        return np.clip(np.asarray(p, dtype=np.float32), 0.0, 1.0)

    def predict_proba(self, X):
        p = self.predict(X)
        return np.column_stack([1.0 - p, p]).astype(np.float32)

    @property
    def feature_importances_(self):
        return getattr(self.base_model, "feature_importances_")

    def get_booster(self):
        return self.base_model.get_booster()


# ======================================================================================
# 2. Generic helpers
# ======================================================================================

def _parquet_columns(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)
    return list(pq.ParquetFile(path).schema.names)


def _read_json(path):
    with open(path, "r") as f:
        return json.load(f)


def _load_feature_names(path):
    obj = _read_json(path)
    if isinstance(obj, list):
        return [str(x) for x in obj]
    if isinstance(obj, dict):
        for key in ["feature_names", "features", "columns"]:
            if key in obj and isinstance(obj[key], list):
                return [str(x) for x in obj[key]]
    raise RuntimeError(f"Could not parse feature names JSON: {path}")


def _safe_float_series(s, fill=None, clip01=False):
    out = pd.to_numeric(s, errors="coerce")
    if fill is not None:
        out = out.fillna(fill)
    if clip01:
        out = out.clip(0, 1)
    return out


def _keyed_for_merge(df):
    out = df.copy()
    out["Date"] = out["Date"].astype(str).str.slice(0, 8)
    out["Lat"] = pd.to_numeric(out["Lat"], errors="coerce")
    out["Lon"] = pd.to_numeric(out["Lon"], errors="coerce")
    out["__LatKey"] = out["Lat"].round(5)
    out["__LonKey"] = out["Lon"].round(5)
    return out


def _clean_safe_name(s):
    return (
        str(s)
        .replace(" ", "_")
        .replace("/", "_")
        .replace(">", "gt")
        .replace("<", "lt")
        .replace("=", "eq")
        .replace("%", "pct")
    )


# ======================================================================================
# 3. Artifact discovery and radius prediction generation
# ======================================================================================

def manifest_candidates_for_radius(radius_km, model_label=None):
    r = int(round(float(radius_km)))
    exp = experiment_tag_for_model(r, model_label=model_label)
    patterns = [
        os.path.join(MODEL_CACHE_DIR, f"active_artifacts_{exp}.json"),
        os.path.join(PROJECT_DIR, f"prob_flood_models_{exp}", f"active_artifacts_{exp}.json"),
        os.path.join(f"/home/tyreekfrazier/ISU_Research_LOCAL_FALLBACK/prob_flood_models_{exp}", f"active_artifacts_{exp}.json"),
    ]
    hits = []
    for p in patterns:
        hits.extend(glob.glob(p))
    return [p for p in hits if os.path.exists(p)]



def target_output_tag_for_radius(radius_km):
    """
    The generated v33 training scripts keep the v28 PIXEL_PARQUET naming pattern:
      pixel_domain_forecasts_..._rXXkm_singletarget_radiusstats_target_v33_apcp13p7cv_domain.parquet

    This is different from the model experiment tag:
      v33_rXXkm_singletarget_radiusstats_mse_apcp13p7cv_domain
    """
    r = int(round(float(radius_km)))
    return f"r{r}km_singletarget_radiusstats_target_{RUN_VERSION_TAG}_apcp13p7cv_domain"


def master_candidates_for_radius(radius_km):
    """
    Return candidate master parquets for one radius.

    The first version of this viewer only checked the model experiment tag, e.g.
      pixel_domain_forecasts_..._v33_r40km_singletarget_radiusstats_mse_...parquet

    but the v33 generated training scripts write the master parquet using the v28
    target-output naming pattern, e.g.
      pixel_domain_forecasts_..._r40km_singletarget_radiusstats_target_v33_...parquet
    """
    r = int(round(float(radius_km)))
    exp = experiment_tag_for_radius(r)
    target_out = target_output_tag_for_radius(r)

    candidates = []

    # Correct v33 generated-script master path.
    candidates.append(os.path.join(
        PROJECT_DIR,
        f"pixel_domain_forecasts_rap09z_iem_mrms_ffg_{target_out}.parquet"
    ))

    # Old/incorrect viewer expectation kept as a fallback.
    candidates.append(MASTER_PARQUET_TEMPLATE.format(experiment_tag=exp))

    # Recursive fallbacks. Restrict to pixel_domain_forecasts master files so we do
    # not accidentally grab daily chunks or metric caches.
    patterns = [
        os.path.join(PROJECT_DIR, f"**/pixel_domain_forecasts_rap09z_iem_mrms_ffg_{target_out}.parquet"),
        os.path.join(PROJECT_DIR, f"**/pixel_domain_forecasts_*{target_out}*.parquet"),
        os.path.join(PROJECT_DIR, f"**/pixel_domain_forecasts_*{exp}*.parquet"),
        os.path.join(PROJECT_DIR, f"**/pixel_domain_forecasts_*r{r}km*singletarget*target_{RUN_VERSION_TAG}*.parquet"),
        os.path.join("/home/tyreekfrazier/ISU_Research_LOCAL_FALLBACK", f"**/pixel_domain_forecasts_*{target_out}*.parquet"),
        os.path.join("/home/tyreekfrazier/ISU_Research_LOCAL_FALLBACK", f"**/pixel_domain_forecasts_*r{r}km*singletarget*target_{RUN_VERSION_TAG}*.parquet"),
    ]

    for pat in patterns:
        candidates.extend(glob.glob(pat, recursive=True))

    # Keep unique existing non-empty candidates, newest first.
    seen = set()
    existing = []
    for p in candidates:
        p = os.path.abspath(str(p))
        if p in seen:
            continue
        seen.add(p)
        if os.path.exists(p) and os.path.getsize(p) > 0:
            existing.append(p)

    return sorted(existing, key=os.path.getmtime, reverse=True)


def find_artifacts_for_radius(radius_km, model_label=None):
    """
    Find model, scaler, feature_names, and master parquet for one radius experiment.
    """
    r = int(round(float(radius_km)))
    model_label = model_label or model_label_for_radius(r)
    exp = experiment_tag_for_model(r, model_label=model_label)
    target_tag = f"r{r}km"
    target_out = target_output_tag_for_radius(r)

    manifest_path = None
    manifest = {}
    cands = manifest_candidates_for_radius(r, model_label=model_label)
    if cands:
        manifest_path = sorted(cands, key=os.path.getmtime)[-1]
        manifest = _read_json(manifest_path)

    master_hits = master_candidates_for_radius(r)
    master_path = master_hits[0] if master_hits else os.path.join(
        PROJECT_DIR,
        f"pixel_domain_forecasts_rap09z_iem_mrms_ffg_{target_out}.parquet"
    )

    model_path = manifest.get("current_model_alias") or manifest.get("model_path")
    scaler_path = manifest.get("current_scaler_alias") or manifest.get("scaler_path")
    features_path = manifest.get("current_feature_names_alias") or manifest.get("feature_names_path")

    model_roots = [
        MODEL_CACHE_DIR,
        os.path.join(PROJECT_DIR, f"prob_flood_models_{exp}"),
        os.path.join("/home/tyreekfrazier/ISU_Research_LOCAL_FALLBACK", f"prob_flood_models_{exp}"),
    ]

    if not model_path or not os.path.exists(model_path):
        hits = []
        for root in model_roots:
            hits += glob.glob(os.path.join(root, f"current_{RUN_VERSION_TAG}_{target_tag}_XGBoost_model.pkl"))
            hits += glob.glob(os.path.join(root, f"rawprob_localoptuna_{exp}_rap09z_iem_mrms_ffg_XGBoost.pkl"))
            hits += glob.glob(os.path.join(root, f"*{exp}*XGBoost*.pkl"))
        hits = [p for p in hits if os.path.exists(p) and os.path.getsize(p) > 0]
        if hits:
            model_path = sorted(hits, key=os.path.getmtime)[-1]

    if not scaler_path or not os.path.exists(scaler_path):
        hits = []
        for root in model_roots:
            hits += glob.glob(os.path.join(root, f"current_{RUN_VERSION_TAG}_{target_tag}_scaler.pkl"))
            hits += glob.glob(os.path.join(root, f"prob_scaler_localoptuna_{exp}.pkl"))
            hits += glob.glob(os.path.join(root, f"*scaler*{exp}*.pkl"))
        hits = [p for p in hits if os.path.exists(p) and os.path.getsize(p) > 0]
        if hits:
            scaler_path = sorted(hits, key=os.path.getmtime)[-1]

    if not features_path or not os.path.exists(features_path):
        hits = []
        for root in model_roots:
            hits += glob.glob(os.path.join(root, f"current_{RUN_VERSION_TAG}_{target_tag}_feature_names.json"))
            hits += glob.glob(os.path.join(root, f"feature_names_localoptuna_{exp}.json"))
            hits += glob.glob(os.path.join(root, f"*feature*{exp}*.json"))
        hits = [p for p in hits if os.path.exists(p) and os.path.getsize(p) > 0]
        if hits:
            features_path = sorted(hits, key=os.path.getmtime)[-1]

    missing = []
    for label, path in [
        ("manifest", manifest_path),
        ("master parquet", master_path),
        ("model", model_path),
        ("scaler", scaler_path),
        ("feature names", features_path),
    ]:
        if not path or not os.path.exists(path):
            missing.append(label)

    if missing:
        checked_master_paths = [
            os.path.join(PROJECT_DIR, f"pixel_domain_forecasts_rap09z_iem_mrms_ffg_{target_out}.parquet"),
            MASTER_PARQUET_TEMPLATE.format(experiment_tag=exp),
            os.path.join(PROJECT_DIR, f"**/pixel_domain_forecasts_*{target_out}*.parquet"),
            os.path.join(PROJECT_DIR, f"**/pixel_domain_forecasts_*{exp}*.parquet"),
        ]
        raise RuntimeError(
            f"Missing artifacts for radius {r} km: {missing}\n"
            f"Expected model experiment tag: {exp}\n"
            f"Expected master target-output tag: {target_out}\n"
            f"Checked model roots: {model_roots}\n"
            f"Checked master patterns:\n  " + "\n  ".join(checked_master_paths)
        )

    return {
        "radius_km": r,
        "model_label": model_label,
        "experiment_tag": exp,
        "target_output_tag": target_out,
        "manifest_path": manifest_path,
        "manifest": manifest,
        "master_path": master_path,
        "model_path": model_path,
        "scaler_path": scaler_path,
        "features_path": features_path,
        "feature_master_paths": {int(k): v for k, v in manifest.get("multiradius_feature_master_parquets", {}).items()},
    }


def prediction_cache_path_for_radius(radius_km, model_label=None):
    r = int(round(float(radius_km)))
    model_label = model_label or model_label_for_radius(r)
    cache_tag = cache_tag_for_model(model_label)
    cache_label = f"{model_label}_{cache_tag}" if cache_tag else model_label
    return os.path.join(
        PREDICTION_CACHE_DIR,
        f"v33day2valid_singletarget_radius_sensitivity_predictions_{cache_label}.parquet"
    )



def _model_positive_class_probability(model, X_scaled):
    """
    Return continuous P(event).

    Important for v33: XGBClassifier.predict(X) returns hard 0/1 class labels,
    not probabilities. Use predict_proba(X)[:, 1] whenever available.
    Older regression wrappers without predict_proba still fall back to predict(X).
    """
    if hasattr(model, "predict_proba"):
        p2 = np.asarray(model.predict_proba(X_scaled))
        if p2.ndim == 2 and p2.shape[1] >= 2:
            return p2[:, 1].astype(np.float32)
        if p2.ndim == 1:
            return p2.astype(np.float32)
        raise RuntimeError(f"Unexpected predict_proba output shape: {p2.shape}")
    return np.asarray(model.predict(X_scaled), dtype=np.float32)

def _parquet_date_rowgroups(path):
    """Index Date values to parquet row groups without loading predictor columns."""
    pf = pq.ParquetFile(path)
    out = defaultdict(list)
    for rg in range(pf.num_row_groups):
        dates = pf.read_row_group(rg, columns=["Date"]).column("Date").to_pandas().astype(str).str[:8]
        for date in dates.unique():
            out[str(date)].append(rg)
    return pf, dict(out)


def _read_parquet_date(pf, rowgroups_by_date, date, columns):
    parts = []
    for rg in rowgroups_by_date.get(str(date), []):
        part = pf.read_row_group(rg, columns=columns).to_pandas()
        part["Date"] = part["Date"].astype(str).str[:8]
        part = part[part["Date"] == str(date)]
        if not part.empty:
            parts.append(part)
    if not parts:
        return pd.DataFrame(columns=columns)
    return pd.concat(parts, ignore_index=True, sort=False)


def _build_multiradius_predictions(art, feature_names):
    """Predict a generic multiradius model from its feature masters."""
    model_label = str(art["model_label"])
    master_paths = {int(k): str(v) for k, v in art.get("feature_master_paths", {}).items()}
    expected_radii = [40, 60, 75]
    missing_paths = [r for r in expected_radii if r not in master_paths or not os.path.exists(master_paths[r])]
    if missing_paths:
        raise RuntimeError(f"{model_label} is missing feature master paths for radii {missing_paths}")

    features_by_radius = {
        r: [c for c in feature_names if re.search(rf"_R{r}km_(Mean|Min|Max|Std)$", str(c))]
        for r in expected_radii
    }
    assigned = [c for r in expected_radii for c in features_by_radius[r]]
    if set(assigned) != set(feature_names) or len(assigned) != len(feature_names):
        unassigned = [c for c in feature_names if c not in set(assigned)]
        raise RuntimeError(f"Could not route all {model_label} features by radius. Unassigned examples: {unassigned[:30]}")

    indexed = {r: _parquet_date_rowgroups(master_paths[r]) for r in expected_radii}
    common_dates = set.intersection(*(set(indexed[r][1]) for r in expected_radii))
    test_dates = sorted(d for d in common_dates if str(d)[:4] in set(TEST_YEARS))
    if not test_dates:
        raise RuntimeError(f"No common {TEST_YEARS} dates found across the {model_label} feature masters.")

    model = joblib.load(art["model_path"])
    scaler = joblib.load(art["scaler_path"])
    output_parts = []
    for date_num, date in enumerate(test_dates, start=1):
        pf40, rg40 = indexed[40]
        base_cols = ["Date", "Lat", "Lon"] + features_by_radius[40]
        frame = _read_parquet_date(pf40, rg40, date, base_cols)
        if frame.empty:
            continue
        frame["Lat"] = pd.to_numeric(frame["Lat"], errors="coerce").astype(np.float32)
        frame["Lon"] = pd.to_numeric(frame["Lon"], errors="coerce").astype(np.float32)
        frame["__LatKey"] = frame["Lat"].round(5)
        frame["__LonKey"] = frame["Lon"].round(5)
        join_keys = ["Date", "__LatKey", "__LonKey"]
        if frame.duplicated(join_keys).any():
            raise RuntimeError(f"Duplicate R40 keys while building {model_label} for {date}")

        for radius in [60, 75]:
            pf, rg_map = indexed[radius]
            extra = _read_parquet_date(pf, rg_map, date, ["Date", "Lat", "Lon"] + features_by_radius[radius])
            extra["Lat"] = pd.to_numeric(extra["Lat"], errors="coerce").astype(np.float32)
            extra["Lon"] = pd.to_numeric(extra["Lon"], errors="coerce").astype(np.float32)
            extra["__LatKey"] = extra["Lat"].round(5)
            extra["__LonKey"] = extra["Lon"].round(5)
            extra = extra[join_keys + features_by_radius[radius]]
            if extra.duplicated(join_keys).any():
                raise RuntimeError(f"Duplicate R{radius} keys while building {model_label} for {date}")
            frame = frame.merge(extra, on=join_keys, how="inner", sort=False, validate="one_to_one")

        if frame.empty:
            continue
        X = frame[feature_names].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32)
        X_scaled = scaler.transform(X).astype(np.float32, copy=False)
        probs = np.clip(_model_positive_class_probability(model, X_scaled), 0.0, 1.0).astype(np.float32)
        out = frame[["Date", "Lat", "Lon"]].copy()
        out["Year"] = out["Date"].str[:4]
        out["ML_Target_Radius_km"] = 100
        out["ML_Model_Label"] = model_label
        out["ML_Forecast_Prob"] = probs
        out["ML_Experiment_Tag"] = art["experiment_tag"]
        out["ML_Model_Path"] = art["model_path"]
        out["ML_Feature_Names_Path"] = art["features_path"]
        output_parts.append(out)
        print(f"  {model_label}: predicted {date_num}/{len(test_dates)} dates ({date}, {len(out):,} rows)")

    if not output_parts:
        raise RuntimeError(f"{model_label} produced no test predictions.")
    return pd.concat(output_parts, ignore_index=True, sort=False)


def _sample_multiradius_feature_frame(art, feature_names, sample_n=50000, random_state=42):
    """Build a bounded SHAP sample from generic multiradius feature masters."""
    model_label = str(art["model_label"])
    master_paths = {int(k): str(v) for k, v in art.get("feature_master_paths", {}).items()}
    radii = [40, 60, 75]
    features_by_radius = {
        r: [c for c in feature_names if re.search(rf"_R{r}km_(Mean|Min|Max|Std)$", str(c))]
        for r in radii
    }
    indexed = {r: _parquet_date_rowgroups(master_paths[r]) for r in radii}
    common_dates = set.intersection(*(set(indexed[r][1]) for r in radii))
    test_dates = sorted(d for d in common_dates if str(d)[:4] in set(TEST_YEARS))
    if not test_dates:
        raise RuntimeError(f"No common test dates found for {model_label} SHAP sampling.")
    per_date = None if not sample_n else max(1, int(math.ceil(int(sample_n) / len(test_dates))))
    rng = np.random.default_rng(int(random_state))
    parts = []
    for date in test_dates:
        frame = _read_parquet_date(*indexed[40], date, ["Date", "Lat", "Lon"] + features_by_radius[40])
        if frame.empty:
            continue
        if per_date and len(frame) > per_date:
            frame = frame.iloc[np.sort(rng.choice(len(frame), size=per_date, replace=False))].copy()
        frame["Lat"] = pd.to_numeric(frame["Lat"], errors="coerce").astype(np.float32)
        frame["Lon"] = pd.to_numeric(frame["Lon"], errors="coerce").astype(np.float32)
        frame["__LatKey"] = frame["Lat"].round(5)
        frame["__LonKey"] = frame["Lon"].round(5)
        keys = ["Date", "__LatKey", "__LonKey"]
        for radius in [60, 75]:
            extra = _read_parquet_date(*indexed[radius], date, ["Date", "Lat", "Lon"] + features_by_radius[radius])
            extra["Lat"] = pd.to_numeric(extra["Lat"], errors="coerce").astype(np.float32)
            extra["Lon"] = pd.to_numeric(extra["Lon"], errors="coerce").astype(np.float32)
            extra["__LatKey"] = extra["Lat"].round(5)
            extra["__LonKey"] = extra["Lon"].round(5)
            frame = frame.merge(extra[keys + features_by_radius[radius]], on=keys, how="inner", sort=False, validate="one_to_one")
        parts.append(frame[["Date", "Lat", "Lon"] + feature_names])
    if not parts:
        raise RuntimeError(f"Could not sample any rows for {model_label}.")
    out = pd.concat(parts, ignore_index=True, sort=False)
    if sample_n and len(out) > int(sample_n):
        out = out.sample(int(sample_n), random_state=int(random_state)).reset_index(drop=True)
    return out


def build_or_load_radius_predictions(radius_km, force=False, model_label=None):
    """
    Build/load per-gridpoint test predictions for a radius experiment.
    Does not pad missing feature columns. Missing model features raise an error.
    """
    r = int(round(float(radius_km)))
    model_label = model_label or model_label_for_radius(r)
    pred_cache = prediction_cache_path_for_radius(r, model_label=model_label)

    if os.path.exists(pred_cache) and not force:
        print(f"Loading cached {model_label} predictions: {pred_cache}")
        out = pd.read_parquet(pred_cache)
        out["Date"] = out["Date"].astype(str).str.slice(0, 8)
        if "ML_Model_Label" not in out.columns:
            out["ML_Model_Label"] = model_label
        return out

    art = find_artifacts_for_radius(r, model_label=model_label)
    print("\n" + "=" * 100)
    print(f"Building predictions for target radius r{r}km")
    print("=" * 100)
    print(f"Experiment tag: {art['experiment_tag']}")
    print(f"Master parquet: {art['master_path']}")
    print(f"Model:          {art['model_path']}")
    print(f"Scaler:         {art['scaler_path']}")
    print(f"Feature names:  {art['features_path']}")

    feature_names = _load_feature_names(art["features_path"])
    if art.get("feature_master_paths"):
        out = _build_multiradius_predictions(art, feature_names)
        out.to_parquet(pred_cache, index=False)
        print(f"Saved {art['model_label']} prediction cache: {pred_cache}")
        return out
    master_cols = _parquet_columns(art["master_path"])

    base_cols = ["Date", "Lat", "Lon"]
    if "Year" in master_cols:
        base_cols.append("Year")

    missing_features = [c for c in feature_names if c not in master_cols]
    if missing_features:
        raise RuntimeError(
            f"Radius r{r}km master parquet is missing {len(missing_features)} model feature columns.\n"
            f"First 50 missing: {missing_features[:50]}\n"
            "No padding/filling is performed here. Rebuild the radius experiment so its master parquet "
            "contains exactly the saved model feature set."
        )

    read_cols = list(dict.fromkeys(base_cols + feature_names))
    df = pd.read_parquet(art["master_path"], columns=read_cols)
    df["Date"] = df["Date"].astype(str).str.slice(0, 8)
    if "Year" not in df.columns:
        df["Year"] = df["Date"].str.slice(0, 4)
    df["Year"] = df["Year"].astype(str)
    df = df[df["Year"].isin(TEST_YEARS)].copy()
    df = df.dropna(subset=["Date", "Lat", "Lon"]).copy()

    if df.empty:
        raise RuntimeError(f"No test rows found for r{r}km after filtering years {TEST_YEARS}.")

    model = joblib.load(art["model_path"])
    scaler = joblib.load(art["scaler_path"])

    n = len(df)
    preds = np.full(n, np.nan, dtype=np.float32)

    print(f"Predicting r{r}km test grid rows: {n:,}")
    for start in range(0, n, int(PREDICT_CHUNK_SIZE)):
        end = min(start + int(PREDICT_CHUNK_SIZE), n)
        X_chunk = df.iloc[start:end][feature_names].copy()
        # Match the v33/v28 training pipeline exactly:
        #   X_train_np_raw = X_train_raw.fillna(0).to_numpy(dtype=np.float32, copy=True)
        #   X_test_np_raw  = X_test_raw.fillna(0).to_numpy(dtype=np.float32, copy=True)
        # The prior viewer only converted +/-inf to NaN and passed those NaNs into the scaler/model,
        # which did NOT match training-time inference and could collapse/spike probabilities.
        # This is not padding missing feature columns; missing columns are still a hard error above.
        X_chunk = X_chunk.replace([np.inf, -np.inf], np.nan).fillna(0.0)
        X_np = X_chunk.to_numpy(dtype=np.float32, copy=True)
        X_scaled = scaler.transform(X_np).astype(np.float32, copy=False)
        p = _model_positive_class_probability(model, X_scaled)
        preds[start:end] = np.clip(np.asarray(p, dtype=np.float32), 0.0, 1.0)
        print(f"  r{r}km predicted rows {start:,}–{end:,}")

    pred_series = pd.Series(preds, name="ML_Forecast_Prob")
    print(
        f"r{r}km prediction summary after training-matched zero fill:\n"
        f"  mean={float(np.nanmean(preds)):.6f}, "
        f"p50={float(np.nanpercentile(preds, 50)):.6f}, "
        f"p90={float(np.nanpercentile(preds, 90)):.6f}, "
        f"p95={float(np.nanpercentile(preds, 95)):.6f}, "
        f"p99={float(np.nanpercentile(preds, 99)):.6f}, "
        f"max={float(np.nanmax(preds)):.6f}"
    )
    for _thr, _lab in RISK_THRESHOLDS:
        print(f"  frac { _lab } = {float(np.nanmean(preds >= _thr)):.6f}")

    out = df[["Date", "Year", "Lat", "Lon"]].copy()
    out["ML_Target_Radius_km"] = int(r)
    out["ML_Model_Label"] = art["model_label"]
    out["ML_Forecast_Prob"] = preds
    out["ML_Experiment_Tag"] = art["experiment_tag"]
    out["ML_Model_Path"] = art["model_path"]
    out["ML_Feature_Names_Path"] = art["features_path"]

    out.to_parquet(pred_cache, index=False)
    print(f"Saved r{r}km prediction cache: {pred_cache}")

    return out


# ======================================================================================
# Day-2 bounded-memory, resumable historical prediction cache builder
# ======================================================================================
#
# The original Day-1 viewer loaded every test-year row and every model feature into
# one pandas dataframe before it began chunking model inference. A Day-2 master has
# about 1.23 million test rows and 820 predictors, so that load can consume many
# gigabytes before PREDICT_CHUNK_SIZE has any effect.
#
# This override streams selected parquet row groups in bounded record batches, saves
# one narrow prediction fragment per source batch, and consolidates those fragments
# into the same prediction-cache schema used by the rest of the viewer. Completed
# fragments are reusable after a kernel crash.

from pathlib import Path
import gc
import hashlib
import pyarrow as pa

DAY2_PREDICT_SOURCE_BATCH_ROWS = int(
    os.environ.get("XGBFFP_DAY2_PREDICT_BATCH_ROWS", "12000")
)
DAY2_PREDICT_CACHE_COLUMNS = [
    "Date",
    "RAP_Init_Date",
    "Year",
    "Lat",
    "Lon",
    "ML_Target_Radius_km",
    "ML_Model_Label",
    "ML_Forecast_Prob",
]


def _day2_prediction_artifact_signature(art, feature_names):
    payload = {
        "experiment_tag": str(art["experiment_tag"]),
        "test_years": sorted(str(year) for year in TEST_YEARS),
        "batch_rows": int(DAY2_PREDICT_SOURCE_BATCH_ROWS),
        "prediction_writer_version": 2,
        "historical_date_convention": (
            "Date=target_valid_start; RAP_Init_Date=RAP09Z"
        ),
        "features": list(map(str, feature_names)),
    }
    for key in ("master_path", "model_path", "scaler_path", "features_path"):
        path = Path(art[key])
        stat = path.stat()
        payload[key] = {
            "path": str(path.resolve()),
            "size": int(stat.st_size),
            "mtime_ns": int(stat.st_mtime_ns),
        }
    encoded = json.dumps(payload, sort_keys=True, separators=(",", ":")).encode()
    return hashlib.sha256(encoded).hexdigest()[:20]


def _day2_prediction_part_is_valid(path):
    path = Path(path)
    if not path.exists() or path.stat().st_size < 100:
        return False
    try:
        parquet = pq.ParquetFile(path)
        return all(
            column in parquet.schema.names
            for column in DAY2_PREDICT_CACHE_COLUMNS
        )
    except Exception:
        return False


def _day2_empty_prediction_frame():
    return pd.DataFrame(
        {
            "Date": pd.Series(dtype="object"),
            "RAP_Init_Date": pd.Series(dtype="object"),
            "Year": pd.Series(dtype="object"),
            "Lat": pd.Series(dtype="float32"),
            "Lon": pd.Series(dtype="float32"),
            "ML_Target_Radius_km": pd.Series(dtype="int16"),
            "ML_Model_Label": pd.Series(dtype="object"),
            "ML_Forecast_Prob": pd.Series(dtype="float32"),
        }
    )


def _day2_read_prediction_cache(path, radius_km, model_label):
    available = set(pq.ParquetFile(path).schema.names)
    columns = [
        column
        for column in DAY2_PREDICT_CACHE_COLUMNS
        if column in available
    ]
    out = pd.read_parquet(path, columns=columns)
    if "RAP_Init_Date" not in out:
        event_dates = pd.to_datetime(
            out["Date"].astype(str).str[:8],
            format="%Y%m%d",
            errors="coerce",
        )
        if event_dates.isna().any():
            raise ValueError(f"Unparseable Date values in prediction cache {path}")
        out["Date"] = event_dates.dt.strftime("%Y%m%d")
        out["RAP_Init_Date"] = (
            event_dates - pd.Timedelta(days=1)
        ).dt.strftime("%Y%m%d")
    else:
        out["Date"] = out["Date"].astype(str).str[:8]
        out["RAP_Init_Date"] = (
            out["RAP_Init_Date"].astype(str).str[:8]
        )
    out["Year"] = out["Date"].str[:4]
    if "ML_Target_Radius_km" not in out:
        out["ML_Target_Radius_km"] = int(radius_km)
    if "ML_Model_Label" not in out:
        out["ML_Model_Label"] = str(model_label)
    out["Lat"] = pd.to_numeric(out["Lat"], errors="coerce").astype(np.float32)
    out["Lon"] = pd.to_numeric(out["Lon"], errors="coerce").astype(np.float32)
    out["ML_Target_Radius_km"] = pd.to_numeric(
        out["ML_Target_Radius_km"], errors="raise"
    ).astype(np.int16)
    out["ML_Forecast_Prob"] = pd.to_numeric(
        out["ML_Forecast_Prob"], errors="coerce"
    ).astype(np.float32)
    return out[DAY2_PREDICT_CACHE_COLUMNS]


def _day2_write_prediction_fragment(
    batch,
    part_path,
    art,
    feature_names,
    model,
    scaler,
    radius_km,
    model_label,
):
    frame = batch.to_pandas()
    frame["Date"] = frame["Date"].astype(str).str[:8]
    if "Year" in frame:
        frame["Year"] = frame["Year"].astype(str).str[:4]
    else:
        frame["Year"] = frame["Date"].str[:4]
    frame = frame[frame["Year"].isin(set(map(str, TEST_YEARS)))].copy()
    frame = frame.dropna(subset=["Date", "Lat", "Lon"])
    if frame.empty:
        _day2_empty_prediction_frame().to_parquet(
            part_path, index=False, compression="zstd"
        )
        return 0

    # Keep only one source batch in memory. float32 input and output match training.
    X = (
        frame[feature_names]
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0.0)
        .to_numpy(dtype=np.float32, copy=True)
    )
    X_scaled = scaler.transform(X).astype(np.float32, copy=False)
    probabilities = np.clip(
        _model_positive_class_probability(model, X_scaled),
        0.0,
        1.0,
    ).astype(np.float32, copy=False)
    event_dates = pd.to_datetime(
        frame["Date"], format="%Y%m%d", errors="raise"
    )
    valid_dates = event_dates.dt.strftime("%Y%m%d").to_numpy()
    if "RAP_Init_Date" in frame:
        rap_init_dates = frame["RAP_Init_Date"].astype(str).str[:8].to_numpy()
    else:
        rap_init_dates = (
            event_dates - pd.Timedelta(days=1)
        ).dt.strftime("%Y%m%d").to_numpy()
    valid_years = np.asarray(
        [str(value)[:4] for value in valid_dates],
        dtype=object,
    )

    output = pd.DataFrame(
        {
            "Date": valid_dates,
            "RAP_Init_Date": rap_init_dates,
            "Year": valid_years,
            "Lat": pd.to_numeric(frame["Lat"], errors="coerce")
            .to_numpy(dtype=np.float32),
            "Lon": pd.to_numeric(frame["Lon"], errors="coerce")
            .to_numpy(dtype=np.float32),
            "ML_Target_Radius_km": np.full(
                len(frame), int(radius_km), dtype=np.int16
            ),
            "ML_Model_Label": np.full(
                len(frame), str(model_label), dtype=object
            ),
            "ML_Forecast_Prob": probabilities,
        }
    )
    output.to_parquet(part_path, index=False, compression="zstd")
    rows = len(output)
    del output, probabilities, valid_years, valid_dates, rap_init_dates
    del X_scaled, X, frame
    gc.collect()
    return rows


def _day2_consolidate_prediction_fragments(part_paths, destination):
    destination = Path(destination)
    temporary = destination.with_suffix(destination.suffix + ".tmp")
    temporary.unlink(missing_ok=True)
    writer = None
    total_rows = 0
    try:
        for part_path in part_paths:
            table = pq.read_table(part_path)
            if table.num_rows == 0:
                continue
            table = table.select(DAY2_PREDICT_CACHE_COLUMNS)
            if writer is None:
                writer = pq.ParquetWriter(
                    temporary,
                    table.schema,
                    compression="zstd",
                )
            writer.write_table(table)
            total_rows += int(table.num_rows)
            del table
        if writer is None or total_rows == 0:
            raise RuntimeError("No test-year prediction rows were produced.")
    finally:
        if writer is not None:
            writer.close()
    os.replace(temporary, destination)
    return total_rows


def _day2_stream_single_radius_predictions(
    art,
    feature_names,
    prediction_cache,
    force=False,
):
    radius_km = int(art["radius_km"])
    model_label = str(art["model_label"])
    master_path = Path(art["master_path"])
    signature = _day2_prediction_artifact_signature(art, feature_names)
    part_dir = Path(str(prediction_cache) + ".parts") / signature
    part_dir.mkdir(parents=True, exist_ok=True)
    manifest_path = part_dir / "manifest.json"
    manifest_path.write_text(
        json.dumps(
            {
                "signature": signature,
                "radius_km": radius_km,
                "model_label": model_label,
                "batch_rows": DAY2_PREDICT_SOURCE_BATCH_ROWS,
                "test_years": list(map(str, TEST_YEARS)),
                "feature_count": len(feature_names),
                "master_path": str(master_path),
            },
            indent=2,
        )
        + "\n"
    )

    parquet = pq.ParquetFile(master_path)
    master_columns = set(parquet.schema.names)
    base_columns = ["Date", "Lat", "Lon"]
    if "RAP_Init_Date" in master_columns:
        base_columns.append("RAP_Init_Date")
    if "Year" in master_columns:
        base_columns.append("Year")
    read_columns = list(
        dict.fromkeys([*base_columns, *feature_names])
    )
    model = joblib.load(art["model_path"])
    scaler = joblib.load(art["scaler_path"])
    part_paths = []
    completed_rows = 0
    reused_parts = 0
    built_parts = 0
    test_years = set(map(str, TEST_YEARS))

    print(
        f"Memory-safe prediction mode: {parquet.num_row_groups} row groups, "
        f"{len(feature_names):,} features, "
        f"{DAY2_PREDICT_SOURCE_BATCH_ROWS:,} source rows per batch."
    )
    print(f"Resumable fragments: {part_dir}")

    for row_group in range(parquet.num_row_groups):
        row_group_dates = (
            parquet.read_row_group(row_group, columns=["Date"])
            .column("Date")
            .to_pandas()
            .astype(str)
            .str[:4]
        )
        if not row_group_dates.isin(test_years).any():
            del row_group_dates
            continue
        del row_group_dates
        for batch_index, batch in enumerate(
            parquet.iter_batches(
                batch_size=DAY2_PREDICT_SOURCE_BATCH_ROWS,
                row_groups=[row_group],
                columns=read_columns,
                use_threads=False,
            )
        ):
            part_path = part_dir / (
                f"rg{row_group:04d}_batch{batch_index:03d}.parquet"
            )
            part_paths.append(part_path)
            if not force and _day2_prediction_part_is_valid(part_path):
                reused_parts += 1
                completed_rows += pq.ParquetFile(part_path).metadata.num_rows
                continue
            if part_path.exists():
                part_path.unlink()
            rows = _day2_write_prediction_fragment(
                batch=batch,
                part_path=part_path,
                art=art,
                feature_names=feature_names,
                model=model,
                scaler=scaler,
                radius_km=radius_km,
                model_label=model_label,
            )
            completed_rows += int(rows)
            built_parts += 1
            if rows:
                print(
                    f"  {model_label}: row group {row_group + 1}/"
                    f"{parquet.num_row_groups}, batch {batch_index + 1}, "
                    f"{rows:,} test rows"
                )
            del batch
            gc.collect()

    missing_parts = [
        str(path) for path in part_paths if not _day2_prediction_part_is_valid(path)
    ]
    if missing_parts:
        raise RuntimeError(
            f"{len(missing_parts)} prediction fragments are missing or invalid; "
            f"first examples: {missing_parts[:5]}"
        )
    print(
        f"Prediction fragments complete: built={built_parts}, "
        f"reused={reused_parts}, test rows={completed_rows:,}"
    )
    total_rows = _day2_consolidate_prediction_fragments(
        part_paths, prediction_cache
    )
    print(
        f"Saved consolidated {model_label} prediction cache: "
        f"{prediction_cache} ({total_rows:,} rows)"
    )
    del model, scaler
    gc.collect()
    return _day2_read_prediction_cache(
        prediction_cache, radius_km, model_label
    )


def build_or_load_radius_predictions(
    radius_km,
    force=False,
    model_label=None,
):
    """Build/load one Day-2 test prediction table without loading its master."""
    radius_km = int(round(float(radius_km)))
    model_label = model_label or model_label_for_radius(radius_km)
    prediction_cache = prediction_cache_path_for_radius(
        radius_km, model_label=model_label
    )
    if os.path.exists(prediction_cache) and not force:
        print(f"Loading cached {model_label} predictions: {prediction_cache}")
        return _day2_read_prediction_cache(
            prediction_cache, radius_km, model_label
        )

    art = find_artifacts_for_radius(
        radius_km, model_label=model_label
    )
    feature_names = _load_feature_names(art["features_path"])
    master_columns = set(_parquet_columns(art["master_path"]))
    missing_features = [
        feature for feature in feature_names if feature not in master_columns
    ]
    if missing_features:
        raise RuntimeError(
            f"{model_label} master is missing {len(missing_features)} saved "
            f"model features. First examples: {missing_features[:50]}"
        )
    if art.get("feature_master_paths"):
        raise RuntimeError(
            "The bounded-memory Day-2 path expects one self-contained master "
            f"per model, but {model_label} advertises multiradius masters."
        )
    return _day2_stream_single_radius_predictions(
        art=art,
        feature_names=feature_names,
        prediction_cache=prediction_cache,
        force=force,
    )


# ======================================================================================
# 4. Load PP/WPC grid and radius predictions
# ======================================================================================

def load_pp_wpc_grid():
    if "df_pp_viewer_wpc" in globals():
        print("Using in-memory df_pp_viewer_wpc.")
        df = globals()["df_pp_viewer_wpc"].copy()
    elif os.path.exists(PP_WPC_GRID_CACHE):
        print(f"Loading PP/WPC grid: {PP_WPC_GRID_CACHE}")
        df = pd.read_parquet(PP_WPC_GRID_CACHE)
    else:
        print("Building the common Day-2 PP/UFVS/WPC verification grid.")
        df = build_or_load_day2_verification_grid(
            project_dir=PROJECT_DIR,
            test_years=TEST_YEARS,
        )

    df["Date"] = df["Date"].astype(str).str.slice(0, 8)
    if "Year" not in df.columns:
        df["Year"] = df["Date"].str.slice(0, 4)
    df["Year"] = df["Year"].astype(str)
    df = df[df["Year"].isin(TEST_YEARS)].copy()
    df["Lat"] = pd.to_numeric(df["Lat"], errors="coerce")
    df["Lon"] = pd.to_numeric(df["Lon"], errors="coerce")

    required = ["Date", "Year", "Lat", "Lon", WPC_COL]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise RuntimeError(f"PP/WPC grid is missing required columns: {missing}")

    # Preserve unavailable WPC cases as NaN so they are omitted from WPC-only scores.
    df[WPC_COL] = pd.to_numeric(df[WPC_COL], errors="coerce").clip(0, 1)

    pp_cols = [c for c in df.columns if c.startswith("PP_") and not c.startswith("PP_ROI_")]
    if not pp_cols:
        raise RuntimeError("PP/WPC grid has no PP_* fields.")

    for c in pp_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce").clip(0, 1)

    keep_cols = [
        "Date", "RAP_Init_Date", "Year", "Lat", "Lon", WPC_COL,
    ]
    proxy_cols = [
        c for c in df.columns
        if (
            c.startswith("UFVS_")
            and not c.startswith("UFVS_Day2_")
            and c != "UFVS_Source_Status"
        )
        or c.startswith("PointProxy_")
        or c.startswith("Obs_Day2_")
    ]
    keep_cols = list(dict.fromkeys(
        [c for c in keep_cols if c in df.columns] + pp_cols + proxy_cols
    ))
    
    out = df[keep_cols].dropna(subset=["Date", "Lat", "Lon"]).copy()
    print(
        f"PP/WPC grid loaded: rows={len(out):,}, cases={out['Date'].nunique():,}, "
        f"PP defs={[c.replace('PP_', '') for c in pp_cols]}"
    )
    return out


df_pp_wpc_base = load_pp_wpc_grid()

truth_defs_available = [c.replace("PP_", "") for c in df_pp_wpc_base.columns if c.startswith("PP_")]
truth_defs_used = [d for d in TRUTH_DEFINITIONS if d in truth_defs_available]
if not truth_defs_used:
    raise RuntimeError(f"Requested truth definitions not found. Available: {truth_defs_available}")

radius_pred_parts = []
for spec in MODEL_SPECS:
    radius = int(spec["radius_km"])
    model_label = str(spec["label"])
    try:
        radius_pred_parts.append(
            build_or_load_radius_predictions(radius, force=FORCE_REBUILD_RADIUS_PREDICTIONS, model_label=model_label)
        )
    except Exception as exc:
        import traceback
        warnings.warn(f"Skipping {model_label} predictions because of error: {exc}")
        traceback.print_exc(limit=3)

if not radius_pred_parts:
    raise RuntimeError("No radius prediction datasets could be loaded/built.")

df_radius_preds_long = pd.concat(radius_pred_parts, ignore_index=True)
df_radius_preds_long["ML_Model_Label"] = df_radius_preds_long["ML_Model_Label"].astype("category")
df_radius_preds_long["ML_Target_Radius_km"] = df_radius_preds_long["ML_Target_Radius_km"].astype(np.int16)
del radius_pred_parts
gc.collect()

# Merge PP/WPC fields onto radius predictions with rounded coordinate keys.
base_keyed = _keyed_for_merge(df_pp_wpc_base)
pred_keyed = _keyed_for_merge(df_radius_preds_long)

merge_cols = ["Date", "__LatKey", "__LonKey"]
base_merge_cols = [c for c in base_keyed.columns if c not in ["Lat", "Lon", "RAP_Init_Date"]]

# Preserve prediction Lat/Lon as plotting coordinates; PP/WPC Lat/Lon should match nearly exactly.
df_radius_viewer = pred_keyed.merge(
    base_keyed[base_merge_cols],
    on=merge_cols,
    how="inner",
    suffixes=("", "_base")
)

df_radius_viewer = df_radius_viewer.drop(columns=["__LatKey", "__LonKey"], errors="ignore")
df_radius_viewer["ML_Target_Radius_km"] = df_radius_viewer["ML_Target_Radius_km"].astype(np.int16)
df_radius_viewer["ML_Model_Label"] = df_radius_viewer["ML_Model_Label"].astype("category")
del base_keyed, pred_keyed
gc.collect()

print(
    f"\nMerged radius viewer dataframe: rows={len(df_radius_viewer):,}, "
    f"radii={sorted(df_radius_viewer['ML_Target_Radius_km'].unique().tolist())}, "
    f"cases={df_radius_viewer['Date'].nunique():,}"
)

if len(df_radius_viewer) == 0:
    raise RuntimeError("Radius predictions did not merge with PP/WPC grid. Check Date/Lat/Lon alignment.")


# ======================================================================================
# 5. Risk/category/map helpers
# ======================================================================================

def risk_category_labels(values):
    v = np.asarray(values, dtype=float)
    labels = np.full(v.shape, "<5%", dtype=object)
    labels[v >= 0.05] = ">5%"
    labels[v >= 0.15] = ">15%"
    labels[v >= 0.40] = ">40%"
    labels[v >= 0.70] = ">70%"
    return labels


def risk_class_ids(values):
    v = np.asarray(values, dtype=float)
    out = np.zeros(v.shape, dtype=np.int16)
    out[v >= 0.05] = 1
    out[v >= 0.15] = 2
    out[v >= 0.40] = 3
    out[v >= 0.70] = 4
    return out


def class_ids_to_labels(class_ids):
    mapping = {0: "<5%", 1: ">5%", 2: ">15%", 3: ">40%", 4: ">70%"}
    return np.array([mapping.get(int(x), "?") for x in class_ids], dtype=object)


def _setup_map_ax(ax, extent, show_states=True, show_countries=True, show_coastline=True):
    if HAS_CARTOPY and hasattr(ax, "set_extent"):
        ax.set_extent(extent, crs=ccrs.PlateCarree())
        if show_coastline:
            ax.coastlines(resolution="50m", linewidth=0.6)
        if show_states:
            ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.35, edgecolor="0.35")
        if show_countries:
            ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=0.45, edgecolor="0.25")
    else:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
    ax.set_xlabel("Longitude")
    ax.set_ylabel("Latitude")


def _scatter_categorical(ax, lon, lat, labels, point_size=8.0, alpha=0.9, show_below_5=False, transform=None):
    labels = np.asarray(labels, dtype=object)
    for lab in RISK_LABELS:
        if lab == "<5%" and not show_below_5:
            continue
        m = labels == lab
        if not np.any(m):
            continue
        kwargs = dict(s=point_size, c=RISK_COLORS[lab], alpha=alpha, label=lab, linewidths=0.0)
        if transform is not None:
            kwargs["transform"] = transform
        ax.scatter(lon[m], lat[m], **kwargs)


def _agreement_labels(forecast_yes, truth_yes):
    f = np.asarray(forecast_yes, dtype=bool)
    y = np.asarray(truth_yes, dtype=bool)
    out = np.full(f.shape, "Neither", dtype=object)
    out[f & y] = "Hit"
    out[(~f) & y] = "Miss"
    out[f & (~y)] = "False Alarm"
    return out


def _scatter_agreement(ax, lon, lat, labels, point_size=8.0, alpha=0.9, transform=None):
    labels = np.asarray(labels, dtype=object)
    for lab in ["Neither", "Hit", "Miss", "False Alarm"]:
        m = labels == lab
        if not np.any(m):
            continue
        if lab == "Neither":
            kwargs = dict(s=max(point_size * 0.6, 1.0), c=AGREEMENT_COLORS[lab], alpha=0.30, label=lab, linewidths=0.0)
        else:
            kwargs = dict(s=point_size, c=AGREEMENT_COLORS[lab], alpha=alpha, label=lab, linewidths=0.0)
        if transform is not None:
            kwargs["transform"] = transform
        ax.scatter(lon[m], lat[m], **kwargs)


def contingency_counts_and_metrics(forecast_yes, truth_yes):
    f = np.asarray(forecast_yes, dtype=bool)
    y = np.asarray(truth_yes, dtype=bool)
    hits = int(np.sum(f & y))
    misses = int(np.sum((~f) & y))
    false_alarms = int(np.sum(f & (~y)))
    correct_negatives = int(np.sum((~f) & (~y)))

    csi_den = hits + misses + false_alarms
    pod_den = hits + misses
    far_den = hits + false_alarms
    bias_den = hits + misses

    return {
        "N Points": int(len(f)),
        "Hits": hits,
        "Misses": misses,
        "False Alarms": false_alarms,
        "Correct Negatives": correct_negatives,
        "CSI": hits / csi_den if csi_den > 0 else np.nan,
        "POD": hits / pod_den if pod_den > 0 else np.nan,
        "FAR": false_alarms / far_den if far_den > 0 else np.nan,
        "Bias": (hits + false_alarms) / bias_den if bias_den > 0 else np.nan,
        "Forecast Area Fraction": float(np.mean(f)) if len(f) else np.nan,
        "Truth Area Fraction": float(np.mean(y)) if len(y) else np.nan,
    }


# ======================================================================================
# 6. Map viewer functions
# ======================================================================================

def selected_case_df(date, radius_km, model_label=None):
    d = str(date)[:8]
    r = int(round(float(radius_km)))
    model_label = model_label or model_label_for_radius(r)
    sub = df_radius_viewer[
        (df_radius_viewer["Date"].astype(str) == d)
        & (df_radius_viewer["ML_Target_Radius_km"].astype(int) == r)
        & (df_radius_viewer["ML_Model_Label"].astype(str) == model_label)
    ].copy()
    if sub.empty:
        raise RuntimeError(f"No viewer rows for date={d}, model={model_label}")
    return sub


def plot_selected_radius_viewer(
    date,
    radius_km,
    model_label=None,
    pp_definition="Any flood proxy",
    risk_threshold=0.05,
    agreement_mode="ML vs PP",
    extent=DEFAULT_EXTENT,
    point_size=POINT_SIZE_DEFAULT,
    alpha=POINT_ALPHA_DEFAULT,
    show_points_below_5=SHOW_POINTS_BELOW_5_DEFAULT,
    show_states=True,
    show_countries=True,
    show_coastline=True,
):
    pp_col = f"PP_{pp_definition}"
    if pp_col not in df_radius_viewer.columns:
        raise RuntimeError(f"Missing PP column: {pp_col}")

    model_label = model_label or model_label_for_radius(radius_km)
    sub = selected_case_df(date, radius_km, model_label=model_label)
    lon = sub["Lon"].to_numpy(dtype=float)
    lat = sub["Lat"].to_numpy(dtype=float)
    ml = sub["ML_Forecast_Prob"].to_numpy(dtype=float)
    wpc = sub[WPC_COL].to_numpy(dtype=float)
    pp = sub[pp_col].to_numpy(dtype=float)

    proj = ccrs.PlateCarree() if HAS_CARTOPY else None
    subplot_kw = {"projection": proj} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), subplot_kw=subplot_kw, constrained_layout=True)
    axes = axes.ravel()
    transform = ccrs.PlateCarree() if HAS_CARTOPY else None

    panels = [
        (f"ML {model_label} target", ml),
        ("WPC ERO Day 2", wpc),
        (f"Practically Perfect: {pp_definition}", pp),
    ]

    for ax, (title, values) in zip(axes[:3], panels):
        _setup_map_ax(ax, extent, show_states=show_states, show_countries=show_countries, show_coastline=show_coastline)
        labs = risk_category_labels(values)
        _scatter_categorical(ax, lon, lat, labs, point_size=point_size, alpha=alpha,
                             show_below_5=show_points_below_5, transform=transform)
        ax.set_title(title)
        ax.legend(loc="lower left", fontsize=8, frameon=True)

    ax = axes[3]
    _setup_map_ax(ax, extent, show_states=show_states, show_countries=show_countries, show_coastline=show_coastline)

    if agreement_mode == "ML vs PP":
        f_yes = ml >= float(risk_threshold)
        y_yes = pp >= float(risk_threshold)
        title = f"Agreement: ML {model_label} vs PP | {risk_threshold:.2f}"
    elif agreement_mode == "WPC ERO vs PP":
        f_yes = wpc >= float(risk_threshold)
        y_yes = pp >= float(risk_threshold)
        title = f"Agreement: WPC ERO vs PP | {risk_threshold:.2f}"
    elif agreement_mode == "ML vs WPC ERO":
        f_yes = ml >= float(risk_threshold)
        y_yes = wpc >= float(risk_threshold)
        title = f"Agreement: ML {model_label} vs WPC ERO | {risk_threshold:.2f}"
    else:
        raise ValueError("agreement_mode must be ML vs PP, WPC ERO vs PP, or ML vs WPC ERO")

    agree = _agreement_labels(f_yes, y_yes)
    _scatter_agreement(ax, lon, lat, agree, point_size=point_size, alpha=alpha, transform=transform)
    ax.set_title(title)
    ax.legend(loc="lower left", fontsize=8, frameon=True)

    fig.suptitle(f"{date} | model {model_label} | target radius r{int(radius_km)}km | PP truth: {pp_definition}", fontsize=15)
    plt.show()

    rows = []
    for name, f, y in [
        ("ML vs PP", ml >= float(risk_threshold), pp >= float(risk_threshold)),
        ("WPC ERO vs PP", wpc >= float(risk_threshold), pp >= float(risk_threshold)),
        ("ML vs WPC ERO", ml >= float(risk_threshold), wpc >= float(risk_threshold)),
        ("Practically Perfect vs PP", pp >= float(risk_threshold), pp >= float(risk_threshold)),
    ]:
        rows.append({"Date": str(date), "Radius km": int(radius_km), "Comparison": name, **contingency_counts_and_metrics(f, y)})

    display(pd.DataFrame(rows).round(4))


def plot_radius_comparison_viewer(
    date,
    pp_definition="Any flood proxy",
    extent=DEFAULT_EXTENT,
    point_size=POINT_SIZE_DEFAULT,
    alpha=POINT_ALPHA_DEFAULT,
    show_points_below_5=SHOW_POINTS_BELOW_5_DEFAULT,
    show_states=True,
    show_countries=True,
    show_coastline=True,
):
    pp_col = f"PP_{pp_definition}"
    if pp_col not in df_radius_viewer.columns:
        raise RuntimeError(f"Missing PP column: {pp_col}")

    d = str(date)[:8]
    models = df_radius_viewer[["ML_Model_Label", "ML_Target_Radius_km"]].drop_duplicates().to_dict("records")

    ncols = 3
    nrows = math.ceil((len(models) + 2) / ncols)
    proj = ccrs.PlateCarree() if HAS_CARTOPY else None
    subplot_kw = {"projection": proj} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 5.0 * nrows), subplot_kw=subplot_kw, constrained_layout=True)
    axes = np.asarray(axes).ravel()
    transform = ccrs.PlateCarree() if HAS_CARTOPY else None

    panel_i = 0
    for model_spec in models:
        r = int(model_spec["ML_Target_Radius_km"])
        model_label = str(model_spec["ML_Model_Label"])
        sub = selected_case_df(d, r, model_label=model_label)
        ax = axes[panel_i]
        _setup_map_ax(ax, extent, show_states=show_states, show_countries=show_countries, show_coastline=show_coastline)
        _scatter_categorical(
            ax,
            sub["Lon"].to_numpy(float),
            sub["Lat"].to_numpy(float),
            risk_category_labels(sub["ML_Forecast_Prob"].to_numpy(float)),
            point_size=point_size,
            alpha=alpha,
            show_below_5=show_points_below_5,
            transform=transform,
        )
        ax.set_title(f"ML target {model_label}")
        panel_i += 1

    # WPC and PP panels from first radius subset, since WPC/PP are duplicated across radii.
    sub0 = selected_case_df(d, int(models[0]["ML_Target_Radius_km"]), model_label=str(models[0]["ML_Model_Label"]))
    for title, values in [
        ("WPC ERO Day 2", sub0[WPC_COL].to_numpy(float)),
        (f"Practically Perfect: {pp_definition}", sub0[pp_col].to_numpy(float)),
    ]:
        ax = axes[panel_i]
        _setup_map_ax(ax, extent, show_states=show_states, show_countries=show_countries, show_coastline=show_coastline)
        _scatter_categorical(
            ax,
            sub0["Lon"].to_numpy(float),
            sub0["Lat"].to_numpy(float),
            risk_category_labels(values),
            point_size=point_size,
            alpha=alpha,
            show_below_5=show_points_below_5,
            transform=transform,
        )
        ax.set_title(title)
        panel_i += 1

    for ax in axes[panel_i:]:
        ax.axis("off")

    handles, labels = axes[0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="lower center", ncols=len(labels), frameon=True)

    fig.suptitle(f"Radius sensitivity map comparison | {d} | PP truth: {pp_definition}", fontsize=15)
    plt.show()


# ======================================================================================
# 7. 40-km ROI verification helpers
# ======================================================================================

def latlon_to_unit_xyz(lat, lon):
    lat_rad = np.deg2rad(np.asarray(lat, dtype=float))
    lon_rad = np.deg2rad(np.asarray(lon, dtype=float))
    cos_lat = np.cos(lat_rad)
    return np.column_stack([
        cos_lat * np.cos(lon_rad),
        cos_lat * np.sin(lon_rad),
        np.sin(lat_rad),
    ]).astype(np.float64)


def km_to_unit_sphere_chord_radius(radius_km):
    angular_radius = float(radius_km) / EARTH_RADIUS_KM
    return 2.0 * np.sin(angular_radius / 2.0)


def expand_binary_mask_radius_km(mask, tree, xyz, radius_km=40.0, chunk_size=1024):
    mask = np.asarray(mask, dtype=bool)
    if mask.size == 0:
        return mask.copy()
    if not mask.any():
        return np.zeros_like(mask, dtype=bool)
    if mask.all():
        return np.ones_like(mask, dtype=bool)

    yes_idx = np.flatnonzero(mask)
    out = np.zeros_like(mask, dtype=bool)
    chord_radius = km_to_unit_sphere_chord_radius(radius_km)

    for start in range(0, len(yes_idx), int(chunk_size)):
        idx_chunk = yes_idx[start:start + int(chunk_size)]
        neighbors = tree.query_ball_point(xyz[idx_chunk], r=chord_radius, return_sorted=False)
        for n in neighbors:
            if len(n) == 0:
                continue
            out[np.asarray(n, dtype=np.int64)] = True
    return out


def expanded_threshold_masks_from_values(values, tree, xyz, radius_km=VERIFY_ROI_KM):
    values = np.asarray(values, dtype=float)
    out = {}
    for threshold, risk_label in RISK_THRESHOLDS:
        raw_yes = np.isfinite(values) & (values >= float(threshold))
        out[risk_label] = expand_binary_mask_radius_km(raw_yes, tree=tree, xyz=xyz, radius_km=radius_km)
    return out


def category_from_expanded_masks(mask_dict):
    n = len(next(iter(mask_dict.values())))
    cat = np.zeros(n, dtype=np.int16)
    cat[mask_dict.get(">5%", np.zeros(n, dtype=bool))] = 1
    cat[mask_dict.get(">15%", np.zeros(n, dtype=bool))] = 2
    cat[mask_dict.get(">40%", np.zeros(n, dtype=bool))] = 3
    cat[mask_dict.get(">70%", np.zeros(n, dtype=bool))] = 4
    return cat


def compute_binary_bs_bss_from_yes(forecast_yes, truth_yes):
    f = np.asarray(forecast_yes, dtype=float)
    y = np.asarray(truth_yes, dtype=float)
    if len(y) == 0:
        return {"Event Frequency": np.nan, "BS Forecast": np.nan, "BS Climatology": np.nan, "BSS": np.nan}
    climo = float(np.mean(y))
    bs_forecast = float(np.mean((f - y) ** 2))
    bs_climo = float(np.mean((climo - y) ** 2))
    bss = np.nan if (not np.isfinite(bs_climo) or bs_climo <= 0) else 1.0 - (bs_forecast / bs_climo)
    return {"Event Frequency": climo, "BS Forecast": bs_forecast, "BS Climatology": bs_climo, "BSS": bss}


def one_hot_from_categories(categories, class_ids=(0, 1, 2, 3, 4)):
    categories = np.asarray(categories, dtype=np.int16)
    classes = np.asarray(class_ids, dtype=np.int16)
    return (categories[:, None] == classes[None, :]).astype(np.float64)


def cumulative_leq_from_categories(categories, class_ids=(0, 1, 2, 3, 4)):
    categories = np.asarray(categories, dtype=np.int16)
    cutoffs = np.asarray(class_ids[:-1], dtype=np.int16)
    return (categories[:, None] <= cutoffs[None, :]).astype(np.float64)


def compute_rps_rpss(forecast_cat, truth_cat, class_ids=(0, 1, 2, 3, 4)):
    forecast_cat = np.asarray(forecast_cat, dtype=np.int16)
    truth_cat = np.asarray(truth_cat, dtype=np.int16)
    if len(truth_cat) == 0:
        return {"RPS Forecast": np.nan, "RPS Climatology": np.nan, "RPSS": np.nan, "Mean Forecast Class": np.nan, "Mean Truth Class": np.nan}

    f_cdf = cumulative_leq_from_categories(forecast_cat, class_ids=class_ids)
    y_cdf = cumulative_leq_from_categories(truth_cat, class_ids=class_ids)
    rps_forecast = float(np.mean(np.sum((f_cdf - y_cdf) ** 2, axis=1)))

    y_oh = one_hot_from_categories(truth_cat, class_ids=class_ids)
    climo_prob = y_oh.mean(axis=0)
    climo_cdf = np.cumsum(climo_prob)[:-1]
    rps_climo = float(np.mean(np.sum((climo_cdf[None, :] - y_cdf) ** 2, axis=1)))

    rpss = np.nan if (not np.isfinite(rps_climo) or rps_climo <= 0) else 1.0 - (rps_forecast / rps_climo)
    return {
        "RPS Forecast": rps_forecast,
        "RPS Climatology": rps_climo,
        "RPSS": rpss,
        "Mean Forecast Class": float(np.mean(forecast_cat)),
        "Mean Truth Class": float(np.mean(truth_cat)),
    }


# ======================================================================================
# 8. Compute/load all radius metrics
# ======================================================================================

def metric_cache_paths():
    model_tags = '_'.join(str(spec['label']) for spec in MODEL_SPECS)
    tag = f"v33day2valid_singletarget_models_{model_tags}_verifyROI{int(VERIFY_ROI_KM)}km"
    return {
        "case": os.path.join(METRIC_CACHE_DIR, f"{tag}_categorical_bss_by_case.parquet"),
        "case_mean": os.path.join(METRIC_CACHE_DIR, f"{tag}_categorical_bss_case_mean.parquet"),
        "pooled": os.path.join(METRIC_CACHE_DIR, f"{tag}_categorical_bss_pooled.parquet"),
        "rpss_case": os.path.join(METRIC_CACHE_DIR, f"{tag}_rpss_by_case.parquet"),
        "rpss_case_mean": os.path.join(METRIC_CACHE_DIR, f"{tag}_rpss_case_mean.parquet"),
        "rpss_pooled": os.path.join(METRIC_CACHE_DIR, f"{tag}_rpss_pooled.parquet"),
    }


def compute_or_load_radius_metrics(force=False):
    paths = metric_cache_paths()
    if (not force) and all(os.path.exists(p) for p in paths.values()):
        print("Loading cached radius metric tables.")
        return {
            "case": pd.read_parquet(paths["case"]),
            "case_mean": pd.read_parquet(paths["case_mean"]),
            "pooled": pd.read_parquet(paths["pooled"]),
            "rpss_case": pd.read_parquet(paths["rpss_case"]),
            "rpss_case_mean": pd.read_parquet(paths["rpss_case_mean"]),
            "rpss_pooled": pd.read_parquet(paths["rpss_pooled"]),
        }

    print("\nComputing radius-sensitivity categorical metrics with 40-km verification ROI...")

    case_rows = []
    pooled_binary_store = defaultdict(lambda: {"forecast": [], "truth": []})
    rpss_case_rows = []
    pooled_category_store = defaultdict(lambda: {"forecast_cat": [], "truth_cat": []})

    # Build one base grid per date from PP/WPC, then align each radius prediction to it.
    base_keyed_all = _keyed_for_merge(df_pp_wpc_base)
    pred_keyed_all = _keyed_for_merge(df_radius_preds_long)

    dates = sorted(base_keyed_all["Date"].astype(str).unique().tolist())
    models = pred_keyed_all[["ML_Model_Label", "ML_Target_Radius_km"]].drop_duplicates().to_dict("records")

    for i, date in enumerate(dates, start=1):
        base = base_keyed_all[base_keyed_all["Date"].astype(str) == str(date)].copy()
        base = base.dropna(subset=["Lat", "Lon", WPC_COL]).copy()
        if base.empty:
            continue

        base = base.sort_values(["__LatKey", "__LonKey"]).reset_index(drop=True)
        lat = base["Lat"].to_numpy(dtype=float)
        lon = base["Lon"].to_numpy(dtype=float)
        xyz = latlon_to_unit_xyz(lat, lon)
        tree = cKDTree(xyz)

        wpc_masks = expanded_threshold_masks_from_values(base[WPC_COL].to_numpy(float), tree, xyz, radius_km=VERIFY_ROI_KM)
        wpc_cat = category_from_expanded_masks(wpc_masks)

        pp_masks_by_truth = {}
        pp_cat_by_truth = {}
        for truth_def in truth_defs_used:
            pp_col = f"PP_{truth_def}"
            pp_masks = expanded_threshold_masks_from_values(base[pp_col].to_numpy(float), tree, xyz, radius_km=VERIFY_ROI_KM)
            pp_masks_by_truth[truth_def] = pp_masks
            pp_cat_by_truth[truth_def] = category_from_expanded_masks(pp_masks)

        # WPC and PP-perfect rows, once per date/truth.
        for truth_def in truth_defs_used:
            pp_masks = pp_masks_by_truth[truth_def]
            pp_cat = pp_cat_by_truth[truth_def]
            for threshold, risk_label in RISK_THRESHOLDS:
                pp_yes = pp_masks[risk_label]
                for method, forecast_yes in [
                    ("WPC ERO", wpc_masks[risk_label]),
                    ("Practically Perfect", pp_yes),
                ]:
                    cont = contingency_counts_and_metrics(forecast_yes, pp_yes)
                    bss = compute_binary_bs_bss_from_yes(forecast_yes, pp_yes)
                    row = {
                        "Date": str(date),
                        "Year": str(date)[:4],
                        "Truth Definition": truth_def,
                        "Method": method,
                        "ML Target Radius km": np.nan,
                        "Risk Area": risk_label,
                        "Threshold": float(threshold),
                        "Verification ROI km": float(VERIFY_ROI_KM),
                        **cont,
                        **bss,
                    }
                    case_rows.append(row)

                    key = (truth_def, method, np.nan, risk_label, float(threshold))
                    pooled_binary_store[key]["forecast"].append(np.asarray(forecast_yes, dtype=bool))
                    pooled_binary_store[key]["truth"].append(np.asarray(pp_yes, dtype=bool))

            for method, forecast_cat in [
                ("WPC ERO", wpc_cat),
                ("Practically Perfect", pp_cat),
            ]:
                rpss = compute_rps_rpss(forecast_cat, pp_cat)
                rpss_case_rows.append({
                    "Date": str(date),
                    "Year": str(date)[:4],
                    "Truth Definition": truth_def,
                    "Method": method,
                    "ML Target Radius km": np.nan,
                    "Verification ROI km": float(VERIFY_ROI_KM),
                    "N Points": int(len(pp_cat)),
                    **rpss,
                })
                key = (truth_def, method, np.nan)
                pooled_category_store[key]["forecast_cat"].append(np.asarray(forecast_cat, dtype=np.int16))
                pooled_category_store[key]["truth_cat"].append(np.asarray(pp_cat, dtype=np.int16))

        # ML rows for each independently trained model.
        for model_spec in models:
            r = int(model_spec["ML_Target_Radius_km"])
            model_label = str(model_spec["ML_Model_Label"])
            pred = pred_keyed_all[
                (pred_keyed_all["Date"].astype(str) == str(date))
                & (pred_keyed_all["ML_Target_Radius_km"].astype(int) == int(r))
                & (pred_keyed_all["ML_Model_Label"].astype(str) == model_label)
            ][["Date", "__LatKey", "__LonKey", "ML_Forecast_Prob"]].copy()

            merged = base[["Date", "__LatKey", "__LonKey"]].merge(
                pred,
                on=["Date", "__LatKey", "__LonKey"],
                how="left",
            )
            if merged["ML_Forecast_Prob"].isna().any():
                nmiss = int(merged["ML_Forecast_Prob"].isna().sum())
                raise RuntimeError(f"Date {date} r{r}km has {nmiss:,} missing ML prediction rows after alignment.")

            ml_masks = expanded_threshold_masks_from_values(merged["ML_Forecast_Prob"].to_numpy(float), tree, xyz, radius_km=VERIFY_ROI_KM)
            ml_cat = category_from_expanded_masks(ml_masks)

            for truth_def in truth_defs_used:
                pp_masks = pp_masks_by_truth[truth_def]
                pp_cat = pp_cat_by_truth[truth_def]

                for threshold, risk_label in RISK_THRESHOLDS:
                    pp_yes = pp_masks[risk_label]
                    ml_yes = ml_masks[risk_label]
                    cont = contingency_counts_and_metrics(ml_yes, pp_yes)
                    bss = compute_binary_bs_bss_from_yes(ml_yes, pp_yes)
                    case_rows.append({
                        "Date": str(date),
                        "Year": str(date)[:4],
                        "Truth Definition": truth_def,
                        "Method": f"ML {model_label}",
                        "ML Target Radius km": int(r),
                        "Risk Area": risk_label,
                        "Threshold": float(threshold),
                        "Verification ROI km": float(VERIFY_ROI_KM),
                        **cont,
                        **bss,
                    })
                    key = (truth_def, f"ML {model_label}", int(r), risk_label, float(threshold))
                    pooled_binary_store[key]["forecast"].append(np.asarray(ml_yes, dtype=bool))
                    pooled_binary_store[key]["truth"].append(np.asarray(pp_yes, dtype=bool))

                rpss = compute_rps_rpss(ml_cat, pp_cat)
                rpss_case_rows.append({
                    "Date": str(date),
                    "Year": str(date)[:4],
                    "Truth Definition": truth_def,
                    "Method": f"ML {model_label}",
                    "ML Target Radius km": int(r),
                    "Verification ROI km": float(VERIFY_ROI_KM),
                    "N Points": int(len(pp_cat)),
                    **rpss,
                })
                key = (truth_def, f"ML {model_label}", int(r))
                pooled_category_store[key]["forecast_cat"].append(np.asarray(ml_cat, dtype=np.int16))
                pooled_category_store[key]["truth_cat"].append(np.asarray(pp_cat, dtype=np.int16))

        print(f"  {i:03d}/{len(dates):03d} {date}: metrics complete")

    df_case = pd.DataFrame(case_rows)
    df_rpss_case = pd.DataFrame(rpss_case_rows)

    pooled_rows = []
    for (truth_def, method, ml_radius, risk_label, threshold), arrays in pooled_binary_store.items():
        f = np.concatenate(arrays["forecast"])
        y = np.concatenate(arrays["truth"])
        cont = contingency_counts_and_metrics(f, y)
        bss = compute_binary_bs_bss_from_yes(f, y)
        pooled_rows.append({
            "Truth Definition": truth_def,
            "Method": method,
            "ML Target Radius km": ml_radius,
            "Risk Area": risk_label,
            "Threshold": float(threshold),
            "Verification ROI km": float(VERIFY_ROI_KM),
            "N Cases": int(len(dates)),
            **cont,
            **bss,
        })
    df_pooled = pd.DataFrame(pooled_rows)

    rpss_pooled_rows = []
    for (truth_def, method, ml_radius), arrays in pooled_category_store.items():
        fcat = np.concatenate(arrays["forecast_cat"])
        ycat = np.concatenate(arrays["truth_cat"])
        rpss = compute_rps_rpss(fcat, ycat)
        rpss_pooled_rows.append({
            "Truth Definition": truth_def,
            "Method": method,
            "ML Target Radius km": ml_radius,
            "Verification ROI km": float(VERIFY_ROI_KM),
            "N Cases": int(len(dates)),
            "N Points": int(len(ycat)),
            **rpss,
        })
    df_rpss_pooled = pd.DataFrame(rpss_pooled_rows)

    def finite_count(x):
        vals = pd.to_numeric(x, errors="coerce")
        return int(np.isfinite(vals).sum())

    df_case_mean = (
        df_case
        .groupby(["Truth Definition", "Method", "ML Target Radius km", "Risk Area", "Threshold", "Verification ROI km"], dropna=False, as_index=False)
        .agg(
            **{
                "N Cases": ("Date", "nunique"),
                "Mean CSI": ("CSI", "mean"),
                "Mean POD": ("POD", "mean"),
                "Mean FAR": ("FAR", "mean"),
                "Mean Bias": ("Bias", "mean"),
                "Mean Forecast Area Fraction": ("Forecast Area Fraction", "mean"),
                "Mean Truth Area Fraction": ("Truth Area Fraction", "mean"),
                "Mean Event Frequency": ("Event Frequency", "mean"),
                "Mean BS Forecast": ("BS Forecast", "mean"),
                "Mean BS Climatology": ("BS Climatology", "mean"),
                "Mean BSS": ("BSS", "mean"),
                "Median BSS": ("BSS", "median"),
                "N Valid BSS Cases": ("BSS", finite_count),
            }
        )
    )

    df_rpss_case_mean = (
        df_rpss_case
        .groupby(["Truth Definition", "Method", "ML Target Radius km", "Verification ROI km"], dropna=False, as_index=False)
        .agg(
            **{
                "N Cases": ("Date", "nunique"),
                "Mean RPS Forecast": ("RPS Forecast", "mean"),
                "Mean RPS Climatology": ("RPS Climatology", "mean"),
                "Mean RPSS": ("RPSS", "mean"),
                "Median RPSS": ("RPSS", "median"),
                "Mean Forecast Class": ("Mean Forecast Class", "mean"),
                "Mean Truth Class": ("Mean Truth Class", "mean"),
                "N Valid RPSS Cases": ("RPSS", finite_count),
            }
        )
    )

    for name, df in [
        ("case", df_case),
        ("case_mean", df_case_mean),
        ("pooled", df_pooled),
        ("rpss_case", df_rpss_case),
        ("rpss_case_mean", df_rpss_case_mean),
        ("rpss_pooled", df_rpss_pooled),
    ]:
        df.to_parquet(paths[name], index=False)
        print(f"Saved metric cache {name}: {paths[name]}")

    return {
        "case": df_case,
        "case_mean": df_case_mean,
        "pooled": df_pooled,
        "rpss_case": df_rpss_case,
        "rpss_case_mean": df_rpss_case_mean,
        "rpss_pooled": df_rpss_pooled,
    }


metric_tables = compute_or_load_radius_metrics(force=FORCE_REBUILD_RADIUS_METRICS)
df_radius_metrics_by_case = metric_tables["case"]
df_radius_metrics_case_mean = metric_tables["case_mean"]
df_radius_metrics_pooled = metric_tables["pooled"]
df_radius_rpss_by_case = metric_tables["rpss_case"]
df_radius_rpss_case_mean = metric_tables["rpss_case_mean"]
df_radius_rpss_pooled = metric_tables["rpss_pooled"]

print("\nMetric tables ready:")
for name, df in [
    ("df_radius_metrics_by_case", df_radius_metrics_by_case),
    ("df_radius_metrics_case_mean", df_radius_metrics_case_mean),
    ("df_radius_metrics_pooled", df_radius_metrics_pooled),
    ("df_radius_rpss_by_case", df_radius_rpss_by_case),
    ("df_radius_rpss_case_mean", df_radius_rpss_case_mean),
    ("df_radius_rpss_pooled", df_radius_rpss_pooled),
]:
    print(f"  {name}: {len(df):,} rows")


# ======================================================================================
# 9. Metric display/plot functions
# ======================================================================================

def _fmt_table(df):
    out = df.copy()
    for c in out.columns:
        if out[c].dtype.kind in "fc":
            out[c] = pd.to_numeric(out[c], errors="coerce").round(4)
    return out


def display_radius_metric_tables(source="case_mean", truth_definition="Any flood proxy"):
    if source == "case_mean":
        df = df_radius_metrics_case_mean.copy()
        cols = [
            "Truth Definition", "Method", "Risk Area", "N Cases",
            "Mean CSI", "Mean POD", "Mean FAR", "Mean Bias",
            "Mean Forecast Area Fraction", "Mean Truth Area Fraction", "Mean BSS", "Median BSS",
        ]
    elif source == "pooled":
        df = df_radius_metrics_pooled.copy()
        cols = [
            "Truth Definition", "Method", "Risk Area", "N Cases", "CSI", "POD", "FAR", "Bias",
            "Forecast Area Fraction", "Truth Area Fraction", "BSS",
        ]
    else:
        raise ValueError("source must be case_mean or pooled")

    sub = df[df["Truth Definition"] == truth_definition].copy()
    display(_fmt_table(sub[cols].sort_values(["Risk Area", "Method"])))

    rpss_df = df_radius_rpss_case_mean if source == "case_mean" else df_radius_rpss_pooled
    rpss_cols = [c for c in [
        "Truth Definition", "Method", "N Cases", "Mean RPSS", "Median RPSS", "RPSS", "Mean Forecast Class", "Mean Truth Class"
    ] if c in rpss_df.columns]
    display(_fmt_table(rpss_df[rpss_df["Truth Definition"] == truth_definition][rpss_cols].sort_values("Method")))


def _method_order(methods):
    order = [f"ML {spec['label']}" for spec in MODEL_SPECS] + ["WPC ERO", "Practically Perfect"]
    return [m for m in order if m in set(methods)] + [m for m in methods if m not in set(order)]


def plot_radius_metric_lines(truth_definition="Any flood proxy", source="case_mean"):
    if source == "case_mean":
        df = df_radius_metrics_case_mean.copy()
        metric_cols = {
            "CSI": "Mean CSI",
            "POD": "Mean POD",
            "FAR": "Mean FAR",
            "Bias": "Mean Bias",
            "BSS": "Mean BSS",
            "Forecast area fraction": "Mean Forecast Area Fraction",
        }
        title_prefix = "Case-mean"
    elif source == "pooled":
        df = df_radius_metrics_pooled.copy()
        metric_cols = {
            "CSI": "CSI",
            "POD": "POD",
            "FAR": "FAR",
            "Bias": "Bias",
            "BSS": "BSS",
            "Forecast area fraction": "Forecast Area Fraction",
        }
        title_prefix = "Pooled"
    else:
        raise ValueError("source must be case_mean or pooled")

    df = df[df["Truth Definition"] == truth_definition].copy()
    methods = _method_order(df["Method"].dropna().astype(str).unique().tolist())
    x = np.arange(len(RISK_ORDER))

    fig, axes = plt.subplots(2, 3, figsize=(20, 10), constrained_layout=True)
    axes = axes.ravel()

    for ax, (title, col) in zip(axes, metric_cols.items()):
        for method in methods:
            sub = df[df["Method"] == method]
            vals = []
            for risk in RISK_ORDER:
                s = sub.loc[sub["Risk Area"] == risk, col]
                vals.append(float(s.iloc[0]) if len(s) else np.nan)
            ax.plot(x, vals, marker="o", linewidth=2.0, label=method)

        ax.axhline(0.0, linestyle="--", linewidth=1.0, alpha=0.55)
        if title == "Bias":
            ax.axhline(1.0, linestyle=":", linewidth=1.0, alpha=0.75)
        ax.set_xticks(x)
        ax.set_xticklabels(RISK_ORDER)
        ax.set_title(title)
        ax.set_xlabel("Risk threshold after 40-km verification ROI")
        ax.grid(alpha=0.3)
        ax.legend(fontsize=8)

    fig.suptitle(f"{title_prefix} radius-sensitivity metrics | Truth: {truth_definition}", fontsize=16)
    plt.show()


def plot_radius_rpss_bars(truth_definition="Any flood proxy", source="case_mean"):
    if source == "case_mean":
        df = df_radius_rpss_case_mean.copy()
        value_col = "Mean RPSS"
        title_prefix = "Case-mean"
    elif source == "pooled":
        df = df_radius_rpss_pooled.copy()
        value_col = "RPSS"
        title_prefix = "Pooled"
    else:
        raise ValueError("source must be case_mean or pooled")

    df = df[df["Truth Definition"] == truth_definition].copy()
    methods = _method_order(df["Method"].dropna().astype(str).unique().tolist())
    vals = []
    for method in methods:
        s = df.loc[df["Method"] == method, value_col]
        vals.append(float(s.iloc[0]) if len(s) else np.nan)

    x = np.arange(len(methods))
    fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    bars = ax.bar(x, vals, width=0.62)
    ax.axhline(0.0, linestyle="--", linewidth=1.0, alpha=0.75)
    ax.axhline(1.0, linestyle=":", linewidth=1.2, alpha=0.85, label="Perfect RPSS = 1")
    ax.set_xticks(x)
    ax.set_xticklabels(methods, rotation=25, ha="right")
    ax.set_ylabel("RPSS")
    ax.set_title(f"{title_prefix} 40-km ROI RPSS | Truth: {truth_definition}")
    ax.grid(axis="y", alpha=0.30)
    ax.legend()

    finite = [v for v in vals if np.isfinite(v)]
    if finite:
        ymin = min(finite + [0.0])
        ymax = max(finite + [1.0])
        pad = max(0.05, 0.10 * (ymax - ymin if ymax > ymin else 1.0))
        ax.set_ylim(ymin - pad, ymax + pad)

    for bar, val in zip(bars, vals):
        if np.isfinite(val):
            ax.annotate(f"{val:.2f}", xy=(bar.get_x() + bar.get_width()/2, val),
                        xytext=(0, 4 if val >= 0 else -14), textcoords="offset points",
                        ha="center", fontsize=9)
    plt.show()


def plot_radius_area_ratio(truth_definition="Any flood proxy", source="case_mean"):
    if source == "case_mean":
        df = df_radius_metrics_case_mean.copy()
        fcol = "Mean Forecast Area Fraction"
        tcol = "Mean Truth Area Fraction"
        title_prefix = "Case-mean"
    else:
        df = df_radius_metrics_pooled.copy()
        fcol = "Forecast Area Fraction"
        tcol = "Truth Area Fraction"
        title_prefix = "Pooled"

    df = df[df["Truth Definition"] == truth_definition].copy()
    methods = _method_order([m for m in df["Method"].dropna().astype(str).unique().tolist() if m != "Practically Perfect"])
    x = np.arange(len(RISK_ORDER))
    fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    for method in methods:
        sub = df[df["Method"] == method]
        vals = []
        for risk in RISK_ORDER:
            s = sub[sub["Risk Area"] == risk]
            if len(s):
                truth_area = float(s[tcol].iloc[0])
                vals.append(float(s[fcol].iloc[0]) / truth_area if truth_area > 0 else np.nan)
            else:
                vals.append(np.nan)
        ax.plot(x, vals, marker="o", linewidth=2, label=method)
    ax.axhline(1.0, linestyle="--", linewidth=1.0, alpha=0.75)
    ax.set_xticks(x)
    ax.set_xticklabels(RISK_ORDER)
    ax.set_xlabel("Risk threshold after 40-km verification ROI")
    ax.set_ylabel("Forecast area fraction / PP area fraction")
    ax.set_title(f"{title_prefix} area-ratio diagnostic | Truth: {truth_definition}")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    plt.show()


# ======================================================================================
# 10. Interactive controls
# ======================================================================================

available_dates = sorted(df_radius_viewer["Date"].dropna().astype(str).unique().tolist())
available_radii = sorted(df_radius_viewer["ML_Target_Radius_km"].dropna().astype(int).unique().tolist())
available_model_specs = df_radius_viewer[["ML_Model_Label", "ML_Target_Radius_km"]].drop_duplicates().to_dict("records")
available_model_radius = {str(x["ML_Model_Label"]): int(x["ML_Target_Radius_km"]) for x in available_model_specs}
available_truth_defs = [d for d in truth_defs_available if f"PP_{d}" in df_radius_viewer.columns]

# Date ranking: prioritize cases with larger Any flood proxy / WPC / ML areas.
def _rank_dates():
    pp_col = "PP_Any flood proxy" if "PP_Any flood proxy" in df_radius_viewer.columns else f"PP_{available_truth_defs[0]}"
    tmp = df_radius_viewer[df_radius_viewer["ML_Target_Radius_km"] == available_radii[0]].copy()
    score = (
        tmp.assign(__score=(tmp[pp_col].fillna(0) + tmp[WPC_COL].fillna(0) + tmp["ML_Forecast_Prob"].fillna(0)))
        .groupby("Date")["__score"].mean()
        .sort_values(ascending=False)
    )
    return score.index.astype(str).tolist()

ranked_dates = _rank_dates()

w_view_mode = widgets.Dropdown(
    options=["Selected radius 4-panel", "All-radius ML comparison", "Metric dashboard"],
    value="Selected radius 4-panel",
    description="View"
)
w_date = widgets.Dropdown(options=ranked_dates, value=ranked_dates[0], description="Date")
w_model = widgets.Dropdown(options=list(available_model_radius), value=list(available_model_radius)[0], description="Model")
w_truth = widgets.Dropdown(options=available_truth_defs, value=("Any flood proxy" if "Any flood proxy" in available_truth_defs else available_truth_defs[0]), description="PP truth")
w_threshold = widgets.Dropdown(options=[(lab, thr) for thr, lab in RISK_THRESHOLDS], value=0.05, description="Threshold")
w_agreement = widgets.Dropdown(options=["ML vs PP", "WPC ERO vs PP", "ML vs WPC ERO"], value="ML vs PP", description="Agree")
w_source = widgets.Dropdown(options=["case_mean", "pooled"], value="case_mean", description="Metrics")

w_lon_min = widgets.FloatText(value=DEFAULT_EXTENT[0], description="Lon min")
w_lon_max = widgets.FloatText(value=DEFAULT_EXTENT[1], description="Lon max")
w_lat_min = widgets.FloatText(value=DEFAULT_EXTENT[2], description="Lat min")
w_lat_max = widgets.FloatText(value=DEFAULT_EXTENT[3], description="Lat max")
w_point_size = widgets.FloatSlider(value=POINT_SIZE_DEFAULT, min=1, max=40, step=1, description="Pt size")
w_alpha = widgets.FloatSlider(value=POINT_ALPHA_DEFAULT, min=0.1, max=1.0, step=0.05, description="Alpha")
w_show_low = widgets.Checkbox(value=SHOW_POINTS_BELOW_5_DEFAULT, description="Show <5%")
w_states = widgets.Checkbox(value=True, description="States")
w_countries = widgets.Checkbox(value=True, description="Countries")
w_coast = widgets.Checkbox(value=True, description="Coast")

btn_plot = widgets.Button(description="Plot", button_style="primary")
out_plot = widgets.Output()


def _on_plot_clicked(_):
    with out_plot:
        clear_output(wait=True)
        extent = [w_lon_min.value, w_lon_max.value, w_lat_min.value, w_lat_max.value]
        if w_view_mode.value == "Selected radius 4-panel":
            plot_selected_radius_viewer(
                date=w_date.value,
                radius_km=available_model_radius[w_model.value],
                model_label=w_model.value,
                pp_definition=w_truth.value,
                risk_threshold=w_threshold.value,
                agreement_mode=w_agreement.value,
                extent=extent,
                point_size=w_point_size.value,
                alpha=w_alpha.value,
                show_points_below_5=w_show_low.value,
                show_states=w_states.value,
                show_countries=w_countries.value,
                show_coastline=w_coast.value,
            )
        elif w_view_mode.value == "All-radius ML comparison":
            plot_radius_comparison_viewer(
                date=w_date.value,
                pp_definition=w_truth.value,
                extent=extent,
                point_size=w_point_size.value,
                alpha=w_alpha.value,
                show_points_below_5=w_show_low.value,
                show_states=w_states.value,
                show_countries=w_countries.value,
                show_coastline=w_coast.value,
            )
        elif w_view_mode.value == "Metric dashboard":
            display_radius_metric_tables(source=w_source.value, truth_definition=w_truth.value)
            plot_radius_metric_lines(truth_definition=w_truth.value, source=w_source.value)
            plot_radius_area_ratio(truth_definition=w_truth.value, source=w_source.value)
            plot_radius_rpss_bars(truth_definition=w_truth.value, source=w_source.value)

btn_plot.on_click(_on_plot_clicked)

controls = widgets.VBox([
    widgets.HBox([w_view_mode, w_date, w_model, w_truth]),
    widgets.HBox([w_threshold, w_agreement, w_source]),
    widgets.HBox([w_lon_min, w_lon_max, w_lat_min, w_lat_max]),
    widgets.HBox([w_point_size, w_alpha, w_show_low, w_states, w_countries, w_coast]),
    btn_plot,
])

display(controls, out_plot)

# Initial plot.
_on_plot_clicked(None)


# ======================================================================================
# 11. Created objects summary
# ======================================================================================

print("\nCreated main objects:")
print("  df_pp_wpc_base")
print("  df_radius_preds_long")
print("  df_radius_viewer")
print("  df_radius_metrics_by_case")
print("  df_radius_metrics_case_mean")
print("  df_radius_metrics_pooled")
print("  df_radius_rpss_by_case")
print("  df_radius_rpss_case_mean")
print("  df_radius_rpss_pooled")
print("\nUseful plotting functions:")
print("  plot_selected_radius_viewer(date, radius_km, pp_definition='Any flood proxy')")
print("  plot_radius_comparison_viewer(date, pp_definition='Any flood proxy')")
print("  display_radius_metric_tables(source='case_mean', truth_definition='Any flood proxy')")
print("  plot_radius_metric_lines(truth_definition='Any flood proxy', source='case_mean')")
print("  plot_radius_area_ratio(truth_definition='Any flood proxy', source='case_mean')")
print("  plot_radius_rpss_bars(truth_definition='Any flood proxy', source='case_mean')")

## Enhanced verification, realtime-style prediction, and SHAP utilities

In [ ]:

# =============================================================================
# Enhanced verification / plotting / SHAP utilities
# =============================================================================
from pathlib import Path

try:
    from scipy.ndimage import gaussian_filter
    HAS_NDIMAGE = True
except Exception:
    HAS_NDIMAGE = False


def _truth_col_from_pp_definition(pp_definition):
    c = str(pp_definition)
    if c.startswith("PP_"):
        return c
    return f"PP_{c}"


def _case_radius_subset(df, date, radius_km):
    d = str(date)[:8]
    r = int(round(float(radius_km)))
    sub = df[(df["Date"].astype(str).str[:8] == d) & (df["ML_Target_Radius_km"].astype(int) == r)].copy()
    if sub.empty:
        raise RuntimeError(f"No rows found for date={d}, radius={r} km.")
    return sub


def _plot_prob_or_smoothed(ax, sub, value_col, title, smooth_sigma_grid=0, point_size=7, alpha=0.85):
    lon = sub["Lon"].to_numpy(float)
    lat = sub["Lat"].to_numpy(float)
    vals = pd.to_numeric(sub[value_col], errors="coerce").to_numpy(float)
    transform = ccrs.PlateCarree() if HAS_CARTOPY else None
    _setup_map_ax(ax, DEFAULT_EXTENT, show_states=True, show_coastline=True)
    ax.set_title(title)
    if smooth_sigma_grid and smooth_sigma_grid > 0:
        if not HAS_NDIMAGE:
            raise RuntimeError("scipy.ndimage is required for smoothing.")
        tmp = pd.DataFrame({"Lat": lat, "Lon": lon, "v": vals}).dropna()
        grid = tmp.pivot_table(index="Lat", columns="Lon", values="v", aggfunc="mean").sort_index()
        arr = grid.to_numpy(float)
        mask = np.isfinite(arr).astype(float)
        arr0 = np.where(np.isfinite(arr), arr, 0.0)
        num = gaussian_filter(arr0, float(smooth_sigma_grid), mode="nearest")
        den = gaussian_filter(mask, float(smooth_sigma_grid), mode="nearest")
        sm = np.where(den > 1e-6, num / den, np.nan)
        X, Y = np.meshgrid(grid.columns.to_numpy(float), grid.index.to_numpy(float))
        levels = [0, 0.05, 0.15, 0.40, 0.70, 1.0]
        if HAS_CARTOPY:
            m = ax.contourf(X, Y, sm, levels=levels, transform=transform, alpha=0.90)
        else:
            m = ax.contourf(X, Y, sm, levels=levels, alpha=0.90)
        return m
    else:
        _scatter_categorical(
            ax, lon, lat, risk_category_labels(vals),
            point_size=point_size, alpha=alpha,
            show_below_5=False, transform=transform
        )
        return None


def plot_case_ml_wpc_pp_proxy(date, radius_km=40, pp_definition="Any flood proxy", proxy_col=None,
                              smooth_sigma_grid=0, point_size=7, alpha=0.85, df=None):
    """Four-panel case map: ML, WPC ERO, practically perfect, and optional proxy/truth column."""
    if df is None:
        df = df_radius_viewer
    sub = _case_radius_subset(df, date, radius_km)
    pp_col = _truth_col_from_pp_definition(pp_definition)
    if pp_col not in sub.columns:
        raise RuntimeError(f"Missing PP column {pp_col}. Available PP columns: {[c for c in sub.columns if c.startswith('PP_')][:20]}")
    proxy_col = proxy_col or pp_col
    if proxy_col not in sub.columns:
        raise RuntimeError(f"Missing proxy/truth column {proxy_col}.")
    proj = ccrs.PlateCarree() if HAS_CARTOPY else None
    subplot_kw = {"projection": proj} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), subplot_kw=subplot_kw, constrained_layout=True)
    axes = axes.ravel()
    _plot_prob_or_smoothed(axes[0], sub, "ML_Forecast_Prob", f"ML probability r{int(radius_km)} km | {str(date)[:8]}", smooth_sigma_grid, point_size, alpha)
    _plot_prob_or_smoothed(axes[1], sub, WPC_COL, "WPC ERO risk", smooth_sigma_grid, point_size, alpha)
    _plot_prob_or_smoothed(axes[2], sub, pp_col, f"Practically Perfect: {pp_definition}", smooth_sigma_grid, point_size, alpha)
    _plot_prob_or_smoothed(axes[3], sub, proxy_col, f"Proxy/truth: {proxy_col}", smooth_sigma_grid, point_size, alpha)
    plt.show()
    return fig


def contingency_metrics_bool(forecast_yes, truth_yes):
    f = np.asarray(forecast_yes, dtype=bool)
    t = np.asarray(truth_yes, dtype=bool)
    h = int(np.sum(f & t)); m = int(np.sum((~f) & t)); fa = int(np.sum(f & (~t)))
    pod = h / (h + m) if (h + m) else np.nan
    far = fa / (h + fa) if (h + fa) else np.nan
    csi = h / (h + m + fa) if (h + m + fa) else np.nan
    bias = (h + fa) / (h + m) if (h + m) else np.nan
    return {"Hits": h, "Misses": m, "FalseAlarms": fa, "POD": pod, "FAR": far, "CSI": csi, "BIAS": bias}


def average_csi_ml_vs_wpc_using_pp(df=None, pp_definition="Any flood proxy", radius_km=40):
    if df is None: df = df_radius_viewer
    pp_col = _truth_col_from_pp_definition(pp_definition)
    rows=[]
    for d, g0 in df[df["ML_Target_Radius_km"].astype(int)==int(radius_km)].groupby("Date"):
        for thr, lab in RISK_THRESHOLDS:
            truth = pd.to_numeric(g0[pp_col], errors="coerce").fillna(0).to_numpy(float) >= thr
            for method, col in [("ML", "ML_Forecast_Prob"), ("WPC", WPC_COL)]:
                fcst = pd.to_numeric(g0[col], errors="coerce").fillna(0).to_numpy(float) >= thr
                m = contingency_metrics_bool(fcst, truth)
                rows.append({"Date": str(d), "Method": method, "Threshold": lab, "ThresholdValue": thr, **m})
    case = pd.DataFrame(rows)
    summary = case.groupby(["Method","Threshold","ThresholdValue"], as_index=False).mean(numeric_only=True)
    display(summary)
    return case, summary


def average_csi_ml_vs_wpc_using_proxy(df=None, truth_col=None, radius_km=40, truth_threshold=0.5):
    if df is None: df = df_radius_viewer
    if truth_col is None:
        candidates = [c for c in df.columns if ("proxy" in c.lower() or "LSR" in c or "USGS" in c or "FFG" in c) and not c.startswith("PP_")]
        raise RuntimeError(f"Set truth_col. Candidate columns include: {candidates[:50]}")
    rows=[]
    for d, g0 in df[df["ML_Target_Radius_km"].astype(int)==int(radius_km)].groupby("Date"):
        truth = pd.to_numeric(g0[truth_col], errors="coerce").fillna(0).to_numpy(float) >= truth_threshold
        for thr, lab in RISK_THRESHOLDS:
            for method, col in [("ML", "ML_Forecast_Prob"), ("WPC", WPC_COL)]:
                fcst = pd.to_numeric(g0[col], errors="coerce").fillna(0).to_numpy(float) >= thr
                m = contingency_metrics_bool(fcst, truth)
                rows.append({"Date": str(d), "Method": method, "Threshold": lab, "ThresholdValue": thr, "TruthCol": truth_col, **m})
    case = pd.DataFrame(rows)
    summary = case.groupby(["Method","Threshold","ThresholdValue","TruthCol"], as_index=False).mean(numeric_only=True)
    display(summary)
    return case, summary


def reliability_table(df=None, forecast_col="ML_Forecast_Prob", truth_col=None, truth_threshold=0.05, bins=None):
    if df is None: df = df_radius_viewer
    if truth_col is None:
        truth_col = _truth_col_from_pp_definition("Any flood proxy")
    if bins is None:
        bins = np.array([0, .001, .005, .01, .02, .05, .10, .15, .25, .40, .70, 1.0])
    p = pd.to_numeric(df[forecast_col], errors="coerce").to_numpy(float)
    y = (pd.to_numeric(df[truth_col], errors="coerce").fillna(0).to_numpy(float) >= truth_threshold).astype(float)
    ok = np.isfinite(p) & np.isfinite(y)
    p=p[ok]; y=y[ok]
    idx = np.digitize(p, bins, right=False) - 1
    rows=[]
    for i in range(len(bins)-1):
        m=idx==i
        if not np.any(m): continue
        rows.append({"BinMin": bins[i], "BinMax": bins[i+1], "Count": int(m.sum()), "MeanForecast": float(np.mean(p[m])), "ObservedFrequency": float(np.mean(y[m]))})
    return pd.DataFrame(rows)


def plot_reliability_diagram(df=None, forecast_col="ML_Forecast_Prob", truth_col=None, truth_threshold=0.05, bins=None):
    rel = reliability_table(df, forecast_col, truth_col, truth_threshold, bins)
    fig, ax = plt.subplots(figsize=(6.5,6))
    ax.plot([0,1],[0,1], linestyle="--", linewidth=1)
    ax.plot(rel["MeanForecast"], rel["ObservedFrequency"], marker="o")
    ax.set_xlabel("Mean forecast probability")
    ax.set_ylabel("Observed relative frequency")
    ax.set_title(f"Reliability: {forecast_col} vs {truth_col or 'PP_Any flood proxy'}")
    ax.grid(True, alpha=0.3)
    plt.show()
    display(rel)
    return rel


def _exclusive_cat(values):
    v = pd.to_numeric(pd.Series(values), errors="coerce").fillna(0).to_numpy(float)
    out=np.full(v.shape, "<5%", dtype=object)
    out[(v>=0.05)&(v<0.15)]="5-15%"
    out[(v>=0.15)&(v<0.40)]="15-40%"
    out[(v>=0.40)&(v<0.70)]="40-70%"
    out[v>=0.70]=">70%"
    return out


def compute_erickson_fractional_coverage(df=None, columns=None, truth_col=None):
    if df is None: df = df_radius_viewer
    if columns is None:
        columns={"ML":"ML_Forecast_Prob", "WPC":"WPC_ERO_Risk"}
    if truth_col is not None:
        columns={**columns, "Truth": truth_col}
    cats=["5-15%","15-40%","40-70%",">70%"]
    rows=[]
    for date,g in df.groupby("Date"):
        denom=len(g)
        for name,col in columns.items():
            if col not in g.columns: continue
            ec=_exclusive_cat(g[col])
            for cat in cats:
                rows.append({"Date":str(date), "Field":name, "Category":cat, "FractionalCoverage":float(np.mean(ec==cat))})
    case=pd.DataFrame(rows)
    summary=case.groupby(["Field","Category"], as_index=False)["FractionalCoverage"].mean()
    display(summary)
    return case, summary


def plot_erickson_fractional_coverage(summary):
    piv=summary.pivot(index="Category", columns="Field", values="FractionalCoverage").loc[["5-15%","15-40%","40-70%",">70%"]]
    ax=piv.plot(kind="bar", figsize=(8,5))
    ax.set_ylabel("Mean fractional coverage")
    ax.set_title("Erickson-style fractional coverage by exclusive risk category")
    plt.xticks(rotation=0)
    plt.tight_layout()
    plt.show()
    return ax


def brier_score_table(df=None, truth_col=None, truth_threshold=0.05, forecast_cols=None):
    if df is None: df = df_radius_viewer
    if truth_col is None: truth_col = _truth_col_from_pp_definition("Any flood proxy")
    if forecast_cols is None: forecast_cols={"ML":"ML_Forecast_Prob", "WPC":"WPC_ERO_Risk"}
    y=(pd.to_numeric(df[truth_col], errors="coerce").fillna(0).to_numpy(float)>=truth_threshold).astype(float)
    clim=float(np.nanmean(y)); bs_clim=float(np.nanmean((clim-y)**2))
    rows=[]
    for name,col in forecast_cols.items():
        p=pd.to_numeric(df[col], errors="coerce").fillna(0).to_numpy(float)
        bs=float(np.nanmean((p-y)**2))
        bss=1-bs/bs_clim if bs_clim>0 else np.nan
        rows.append({"Forecast":name,"TruthCol":truth_col,"TruthThreshold":truth_threshold,"BS":bs,"Climatology":clim,"BS_clim":bs_clim,"BSS":bss})
    out=pd.DataFrame(rows)
    display(out)
    return out


def add_custom_pp_column(df=None, event_col=None, expansion_radius_km=40, out_col=None):
    """Create an expanded binary proxy/PP-like column from a raw event column for each date."""
    if df is None: df = df_radius_viewer
    if event_col is None or event_col not in df.columns:
        candidates=[c for c in df.columns if not c.startswith("PP_") and any(s in c.lower() for s in ["proxy","lsr","usgs","ffg","flood","exceed"])]
        raise RuntimeError(f"Set event_col. Candidate columns include: {candidates[:50]}")
    out=df.copy()
    out_col=out_col or f"CustomPP_{event_col}_ROI{int(expansion_radius_km)}km"
    vals=np.zeros(len(out), dtype=np.float32)
    for d, idx in out.groupby("Date").groups.items():
        sub=out.loc[idx]
        xyz=latlon_to_unit_xyz(sub["Lat"].to_numpy(float), sub["Lon"].to_numpy(float))
        tree=cKDTree(xyz)
        event=pd.to_numeric(sub[event_col], errors="coerce").fillna(0).to_numpy(float)>0
        vals[np.asarray(list(idx))]=expand_binary_mask_radius_km(event, tree, xyz, expansion_radius_km).astype(np.float32)
    out[out_col]=vals
    return out, out_col


def load_shap_matrix(radius_km=40, sample_n=50000, random_state=42):
    art=find_artifacts_for_radius(radius_km, model_label=globals().get("SHAP_MODEL_LABEL"))
    if not os.path.exists(art["master_path"]):
        raise RuntimeError(f"SHAP requires the radius master parquet, but it was not found: {art['master_path']}")
    feats=_load_feature_names(art["features_path"])
    if art.get("feature_master_paths"):
        df=_sample_multiradius_feature_frame(art, feats, sample_n=sample_n, random_state=random_state)
    else:
        cols=["Date","Lat","Lon"]+feats
        df=pd.read_parquet(art["master_path"], columns=cols)
        df["Date"]=df["Date"].astype(str).str[:8]
        df=df[df["Date"].str[:4].isin(TEST_YEARS)].copy()
        if sample_n and len(df)>sample_n:
            df=df.sample(int(sample_n), random_state=random_state)
    X=df[feats].replace([np.inf,-np.inf], np.nan).fillna(0.0).to_numpy(np.float32)
    scaler=joblib.load(art["scaler_path"])
    Xs=scaler.transform(X).astype(np.float32)
    model=joblib.load(art["model_path"])
    return model, Xs, df, feats, art


def plot_shap_waterfall(radius_km=40, row=0, sample_n=50000, max_display=25):
    import shap
    model,Xs,df,feats,art=load_shap_matrix(radius_km, sample_n)
    explainer=shap.TreeExplainer(model)
    sv=explainer(Xs)
    shap.plots.waterfall(sv[int(row)], max_display=max_display)
    return sv, df, feats


def plot_shap_dependence_for_all_predictors(radius_km=40, sample_n=50000, save_dir=None, show=False):
    import shap
    model,Xs,df,feats,art=load_shap_matrix(radius_km, sample_n)
    explainer=shap.TreeExplainer(model)
    sv=explainer(Xs)
    mean_abs=np.abs(sv.values).mean(axis=0)
    order=np.argsort(mean_abs)[::-1]
    imp=pd.DataFrame({"feature":np.array(feats)[order], "mean_abs_shap":mean_abs[order]})
    display(imp.head(50))
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
    for feat in imp["feature"]:
        i=feats.index(feat)
        plt.figure(figsize=(6,4))
        plt.scatter(df[feat], sv.values[:,i], s=4, alpha=0.25)
        plt.xlabel(feat); plt.ylabel("SHAP value"); plt.title(f"SHAP dependence: {feat}")
        plt.tight_layout()
        if save_dir:
            safe=re.sub(r"[^A-Za-z0-9_.-]+","_",feat)[:160]
            plt.savefig(os.path.join(save_dir, f"shap_dependence_{safe}.png"), dpi=150)
        if show:
            plt.show()
        else:
            plt.close()
    return imp, sv


def predict_feature_parquet_with_radius_model(feature_parquet, radius_km=40, output_path=None):
    """Realtime-style predictor: apply trained v33 radius model to a feature parquet that already contains the trained feature columns."""
    art=find_artifacts_for_radius(radius_km)
    feats=_load_feature_names(art["features_path"])
    cols=_parquet_columns(feature_parquet)
    missing=[c for c in feats if c not in cols]
    if missing:
        raise RuntimeError(f"Realtime feature parquet is missing {len(missing)} trained features. First 30: {missing[:30]}")
    base=[c for c in ["Date","Lat","Lon","Year"] if c in cols]
    df=pd.read_parquet(feature_parquet, columns=list(dict.fromkeys(base+feats)))
    X=df[feats].replace([np.inf,-np.inf], np.nan).fillna(0.0).to_numpy(np.float32)
    scaler=joblib.load(art["scaler_path"])
    model=joblib.load(art["model_path"])
    p=_model_positive_class_probability(model, scaler.transform(X).astype(np.float32))
    out=df[[c for c in base if c in df.columns]].copy()
    out["ML_Target_Radius_km"]=int(radius_km)
    out["ML_Forecast_Prob"]=np.clip(p,0,1)
    if output_path:
        out.to_parquet(output_path, index=False)
        print(f"Saved realtime-style predictions: {output_path}")
    return out


## 5. Real-time forecast builder: build features, predict, and merge WPC/PP/UFVS when available

This section is **not** an empty feature-parquet wrapper. It follows the original v28 realtime-viewer idea: import the training feature-builder helpers, build the forecast feature table for a date, check the saved v33 feature list strictly, generate `predict_proba()[:, 1]`, and then merge WPC/PP and UFVS verification columns when those data are available.


In [ ]:

# ======================================================================================
# 5. Full realtime builder for v33 radiusstats models
# ======================================================================================
# This block is intentionally self-contained within this notebook. It does NOT require a
# prebuilt realtime feature parquet. It imports the local generated/training script for the
# requested radius, uses that script's RAP/FFG feature-building workflow, patches observed
# target builders to zeros for forecast-only inference, and then predicts with the saved
# v33 XGBoost model using predict_proba()[:, 1].
# ======================================================================================

from pathlib import Path
from datetime import datetime, timedelta
import importlib.util
import types
import sys
import traceback

REALTIME_CACHE_DIR = Path(PROJECT_DIR) / "v33day2valid_realtime_radiusstats_forecasts"
REALTIME_FEATURE_CACHE_DIR = REALTIME_CACHE_DIR / "features"
REALTIME_PREDICTION_CACHE_DIR = REALTIME_CACHE_DIR / "predictions"
REALTIME_VERIFIED_CACHE_DIR = REALTIME_CACHE_DIR / "verified"
REALTIME_UFVS_CACHE_DIR = REALTIME_CACHE_DIR / "ufvs_raw"
for _p in [REALTIME_CACHE_DIR, REALTIME_FEATURE_CACHE_DIR, REALTIME_PREDICTION_CACHE_DIR, REALTIME_VERIFIED_CACHE_DIR, REALTIME_UFVS_CACHE_DIR]:
    _p.mkdir(parents=True, exist_ok=True)

SCRIPT_DIR = Path(PROJECT_DIR).parent / "mesoanalysis" / "gempak-scripts"
if not SCRIPT_DIR.exists():
    SCRIPT_DIR = Path("/home/tyreekfrazier/ISU_Research_LOCAL_RUN/mesoanalysis/gempak-scripts")

LOCAL_RUN_ROOT = Path("/home/tyreekfrazier/ISU_Research_LOCAL_RUN")
ORIGINAL_ROOT = Path("/home/tyreekfrazier/ISU_Research")

UFVS_BASE_URL = "https://ftp-wpc.ncep.noaa.gov/erickson/FFaIR/UFVS"
UFVS_PREFIX_TO_COL = {
    "ST4gFFG": "UFVS_STAGE4_FFG",
    "ST4gARI": "UFVS_STAGE4_ARI",
    "USGS": "UFVS_USGS",
    "LSRFLASH": "UFVS_LSR_FLASH",
    "LSRREG": "UFVS_LSR_REGULAR",
}


def _date8(value):
    m = re.search(r"(20\d{6})", str(value))
    if not m:
        raise ValueError(f"Could not parse YYYYMMDD date from {value!r}")
    return m.group(1)


def _extent_dict(extent=None):
    if extent is None:
        extent = DEFAULT_EXTENT
    if isinstance(extent, dict):
        return extent
    if len(extent) != 4:
        raise ValueError("extent must be [lon_min, lon_max, lat_min, lat_max]")
    return {"lon_min": float(extent[0]), "lon_max": float(extent[1]), "lat_min": float(extent[2]), "lat_max": float(extent[3])}


def _normalize_lon(lon):
    lon = np.asarray(lon, dtype=float)
    return np.where(lon > 180.0, lon - 360.0, lon)


def install_dummy_ray_for_import_only():
    """Let training scripts import when ray is unavailable; no Ray workers are launched."""
    if "ray" in sys.modules:
        return
    class _DummyRemoteFunction:
        def __init__(self, func): self.func = func
        def remote(self, *a, **k): return self.func(*a, **k)
        def options(self, *a, **k): return self
        def __call__(self, *a, **k): return self.func(*a, **k)
    class _DummyRay(types.ModuleType):
        def __init__(self):
            super().__init__("ray")
            self.ObjectRef = object
        def remote(self, *args, **kwargs):
            if args and callable(args[0]) and len(args) == 1 and not kwargs:
                return _DummyRemoteFunction(args[0])
            return lambda f: _DummyRemoteFunction(f)
        def init(self, *a, **k): return None
        def shutdown(self, *a, **k): return None
        def is_initialized(self): return False
        def get(self, obj): return obj
        def put(self, obj): return obj
        def wait(self, refs, num_returns=1, timeout=None): return list(refs)[:num_returns], list(refs)[num_returns:]
        def cancel(self, *a, **k): return None
        def cluster_resources(self): return {"CPU": 1}
        def available_resources(self): return {"CPU": 1}
    sys.modules["ray"] = _DummyRay()


def _candidate_training_scripts_for_radius(radius_km):
    r = int(round(float(radius_km)))
    patterns = [
        SCRIPT_DIR / "generated_v33_day2_radius_sensitivity_slimmaster_rowsample" / f"hazard_ml_training_v33day2valid_r{r}km_singletarget_radiusstats_MEMSAFE.py",
    ]
    out = []
    for pat in patterns:
        out.extend(sorted(Path(SCRIPT_DIR).glob(str(pat.relative_to(SCRIPT_DIR))) if str(pat).startswith(str(SCRIPT_DIR)) else glob.glob(str(pat))))
    # Normalize and keep unique existing paths.
    seen = set()
    clean = []
    for p in out:
        p = Path(p)
        if p.exists() and p not in seen:
            seen.add(p)
            clean.append(p)
    return clean


def _patch_module_paths_to_local_run(mod):
    """Patch common absolute project-path globals inside imported training modules.

    This is not feature padding; it only prevents the realtime feature builder from writing
    to / reading from the old mounted tree when the local mirror is being used.
    """
    for name, val in list(vars(mod).items()):
        try:
            if isinstance(val, str) and str(ORIGINAL_ROOT) in val:
                setattr(mod, name, val.replace(str(ORIGINAL_ROOT), str(LOCAL_RUN_ROOT)))
            elif isinstance(val, Path) and str(ORIGINAL_ROOT) in str(val):
                setattr(mod, name, Path(str(val).replace(str(ORIGINAL_ROOT), str(LOCAL_RUN_ROOT))))
        except Exception:
            pass
    # Avoid old mount-live guard in local-run mode.
    for flag in ["STRICT_PROJECT_MOUNT_CHECK", "STRICT_PROJECT_MOUNT_CHECK_FOR_DATA_WRITES"]:
        if hasattr(mod, flag):
            try:
                setattr(mod, flag, False)
            except Exception:
                pass
    return mod


_IMPORTED_TRAINING_MODULES = {}


def load_training_module_for_realtime(radius_km=40, explicit_script=None):
    r = int(round(float(radius_km)))
    if explicit_script is not None:
        script = Path(explicit_script)
        if not script.exists():
            raise FileNotFoundError(script)
    else:
        cands = _candidate_training_scripts_for_radius(r)
        if not cands:
            raise FileNotFoundError(
                f"Could not find a generated/local training script for r{r}km under {SCRIPT_DIR}. "
                "The realtime builder needs the same feature-building helpers used by training."
            )
        # Prefer generated v33 rXX script over generic v28 fallback.
        script = cands[0]
    key = str(script.resolve())
    if key in _IMPORTED_TRAINING_MODULES:
        return _IMPORTED_TRAINING_MODULES[key]
    install_dummy_ray_for_import_only()
    mod_name = f"hml_realtime_r{r}_{abs(hash(key))}"
    spec = importlib.util.spec_from_file_location(mod_name, str(script))
    mod = importlib.util.module_from_spec(spec)
    sys.modules[mod_name] = mod
    spec.loader.exec_module(mod)
    mod = _patch_module_paths_to_local_run(mod)
    _IMPORTED_TRAINING_MODULES[key] = mod
    print(f"Loaded realtime feature builder for r{r}km from: {script}")
    return mod


def realtime_feature_cache_path(date, radius_km=40):
    d = _date8(date)
    r = int(round(float(radius_km)))
    return REALTIME_FEATURE_CACHE_DIR / f"realtime_features_v33day2valid_r{r}km_{d}.parquet"


def realtime_prediction_cache_path(date, radius_km=40, model_label=None):
    d = _date8(date)
    r = int(round(float(radius_km)))
    model_label = model_label or model_label_for_radius(r)
    cache_tag = cache_tag_for_model(model_label)
    cache_label = f"{model_label}_{cache_tag}" if cache_tag else model_label
    return REALTIME_PREDICTION_CACHE_DIR / f"realtime_predictions_v33day2valid_{cache_label}_{d}.parquet"


def realtime_verified_cache_path(date, radius_km=40, model_label=None):
    d = _date8(date)
    r = int(round(float(radius_km)))
    model_label = model_label or model_label_for_radius(r)
    return REALTIME_VERIFIED_CACHE_DIR / f"realtime_verified_v33day2valid_{model_label}_{d}.parquet"


def _dummy_hydro_target_dict(mod, n, radius_km):
    r = int(round(float(radius_km)))
    out = {}
    for col in [
        getattr(mod, "TARGET_COLUMN", "Obs_Day2_MRMS_FFG_Exceeded_Point"),
        "Obs_FFG_NumDurationsExceeded",
        getattr(mod, "TRAIN_TARGET_COLUMN", f"Target_Day2_MRMS_FFG_Exceeded_R{r}km"),
        f"Target_Day2_MRMS_FFG_Exceeded_R{r}km",
        f"Obs_Day2_MRMS_FFG_R{r}km_NeighborCount",
        f"Obs_Day2_MRMS_FFG_Exceeded_R{r}km_EventCount",
        f"Obs_Day2_MRMS_FFG_Exceeded_R{r}km_Fraction",
    ]:
        if "Fraction" in col:
            out[col] = np.zeros(n, dtype=np.float32)
        elif "Count" in col or "NumDurations" in col:
            out[col] = np.zeros(n, dtype=np.int16)
        else:
            out[col] = np.zeros(n, dtype=np.int8)
    # Older v28 naming aliases that can still appear in helpers.
    for col in [
        "Target_Day2_MRMS_FFG_Exceeded_R100km",
        "Obs_Day2_MRMS_FFG_R100km_NeighborCount",
        "Obs_Day2_MRMS_FFG_Exceeded_R100km_EventCount",
        "Obs_Day2_MRMS_FFG_Exceeded_R100km_Fraction",
    ]:
        out.setdefault(col, np.zeros(n, dtype=np.float32 if "Fraction" in col else np.int16))
    return out


def build_realtime_features(date, radius_km=40, force_features=False, training_script=None):
    """Build realtime feature table for a date using the training script's feature builder.

    This mirrors the original v28 realtime path: RAP/FFG feature generation is delegated
    to the training script, while observed target/LSR builders are patched to zeros so
    forecast-only inference does not require day-after MRMS/UFVS.
    """
    date = _date8(date)
    r = int(round(float(radius_km)))
    path = realtime_feature_cache_path(date, r)
    if path.exists() and path.stat().st_size > 1024 and not force_features:
        print(f"Using existing realtime feature cache: {path}")
        return pd.read_parquet(path)

    h = load_training_module_for_realtime(r, explicit_script=training_script)
    original = {}

    def _patch(name, replacement):
        if hasattr(h, name):
            original[name] = getattr(h, name)
            setattr(h, name, replacement)

    def dummy_hydro(date_str, lats_1d, lons_1d, *args, **kwargs):
        return _dummy_hydro_target_dict(h, len(lats_1d), r)

    def dummy_lsr(date_str, lats_1d, lons_1d, *args, **kwargs):
        return np.zeros(len(lats_1d), dtype=np.int8)

    _patch("fetch_mrms_ffg_exceedance_target", dummy_hydro)
    _patch("fetch_iem_flash_flood_reports_pixel", dummy_lsr)
    _patch("fetch_iem_flood_reports_pixel", dummy_lsr)

    try:
        print(f"Building realtime features for {date} r{r}km with training helpers...")
        if hasattr(h, "_prepare_domain_vars_for_extraction"):
            domain_vars = h._prepare_domain_vars_for_extraction([date])
        else:
            if not hasattr(h, "ensure_nam_forecast_file") or not hasattr(h, "get_nam_domain_mask"):
                raise RuntimeError("Training module lacks _prepare_domain_vars_for_extraction and ensure_nam_forecast_file/get_nam_domain_mask.")
            sample = h.ensure_nam_forecast_file(date, 0, h.RAP_DIR)
            domain_vars = h.get_nam_domain_mask(sample)

        df = h.process_daily_pixel_data(date_str=date, nam_dir=h.RAP_DIR, domain_vars=domain_vars, is_test_set=True)
        if df is None or len(df) == 0:
            raise RuntimeError(f"Realtime feature builder returned no rows for {date} r{r}km.")
        df = df.copy()
        df["Date"] = date
        if "Year" not in df.columns:
            df["Year"] = date[:4]
    finally:
        for name, fn in original.items():
            setattr(h, name, fn)

    path.parent.mkdir(parents=True, exist_ok=True)
    df.to_parquet(path, index=False)
    print(f"Saved realtime features: {path} rows={len(df):,} cols={len(df.columns):,}")
    return df


def _strict_realtime_model_matrix(df, feature_names, context="realtime"):
    missing = [c for c in feature_names if c not in df.columns]
    if missing:
        raise RuntimeError(
            f"{context} is missing {len(missing)} trained feature columns.\n"
            f"First 80 missing: {missing[:80]}\n"
            "No missing-feature padding is allowed. Rebuild realtime features with the same generated v33 training script used for the model."
        )
    X = df[feature_names].replace([np.inf, -np.inf], np.nan).fillna(0.0).to_numpy(dtype=np.float32, copy=True)
    return X


def _build_realtime_multiradius_features(date, art, feature_names, force_features=False):
    """Build and join generic multiradius realtime feature families."""
    model_label = str(art["model_label"])
    radii = [40, 60, 75]
    features_by_radius = {
        r: [c for c in feature_names if re.search(rf"_R{r}km_(Mean|Min|Max|Std)$", str(c))]
        for r in radii
    }
    frames = {}
    for radius in radii:
        script = SCRIPT_DIR / "generated_v33_day2_radius_sensitivity_slimmaster_rowsample" / f"hazard_ml_training_v33day2valid_r{radius}km_singletarget_radiusstats_MEMSAFE.py"
        if not script.exists():
            raise FileNotFoundError(f"Missing single-radius realtime feature builder for R{radius}: {script}")
        frames[radius] = build_realtime_features(
            date, radius_km=radius, force_features=force_features, training_script=script
        )

    base = _keyed_for_merge(frames[40])
    keys = ["Date", "__LatKey", "__LonKey"]
    missing40 = [c for c in features_by_radius[40] if c not in base.columns]
    if missing40:
        raise RuntimeError(f"R40 realtime builder is missing {len(missing40)} {model_label} features: {missing40[:30]}")
    for radius in [60, 75]:
        extra = _keyed_for_merge(frames[radius])
        missing = [c for c in features_by_radius[radius] if c not in extra.columns]
        if missing:
            raise RuntimeError(f"R{radius} realtime builder is missing {len(missing)} {model_label} features: {missing[:30]}")
        extra = extra[keys + features_by_radius[radius]].drop_duplicates(keys)
        base = base.merge(extra, on=keys, how="inner", sort=False, validate="one_to_one")
    return base.drop(columns=["__LatKey", "__LonKey"], errors="ignore")


def predict_realtime_case(date, radius_km=40, force=False, force_features=False, training_script=None, model_label=None):
    """Build realtime features if needed, predict with the saved v33 radius model, and cache output."""
    date = _date8(date)
    r = int(round(float(radius_km)))
    model_label = model_label or model_label_for_radius(r)
    out_path = realtime_prediction_cache_path(date, r, model_label=model_label)
    if out_path.exists() and out_path.stat().st_size > 1024 and not force:
        print(f"Using existing realtime prediction cache: {out_path}")
        return pd.read_parquet(out_path)

    art = find_artifacts_for_radius(r, model_label=model_label)
    feature_names = _load_feature_names(art["features_path"])
    if art.get("feature_master_paths"):
        df = _build_realtime_multiradius_features(date, art, feature_names, force_features=force_features)
    else:
        df = build_realtime_features(date, radius_km=r, force_features=force_features, training_script=training_script)
    X_raw = _strict_realtime_model_matrix(df, feature_names, context=f"realtime {date} r{r}km")
    scaler = joblib.load(art["scaler_path"])
    model = joblib.load(art["model_path"])
    X_scaled = scaler.transform(X_raw).astype(np.float32, copy=False)
    p = np.clip(_model_positive_class_probability(model, X_scaled), 0.0, 1.0).astype(np.float32)

    keep = [c for c in ["Date", "Year", "Lat", "Lon"] if c in df.columns]
    for extra in ["APCP_RunTotal_0_24h", "Forecast_APCP_24h_Total_to_Guidance_FFG_24h_Ratio"]:
        if extra in df.columns and extra not in keep:
            keep.append(extra)
    out = df[keep].copy()
    out["Date"] = date
    out["Year"] = date[:4]
    out["ML_Target_Radius_km"] = int(r)
    out["ML_Model_Label"] = art.get("model_label", model_label_for_radius(r))
    out["ML_Forecast_Prob"] = p
    out["ML_Experiment_Tag"] = art.get("experiment_tag", experiment_tag_for_radius(r))
    out["ML_Model_Path"] = art["model_path"]
    out["ML_Feature_Names_Path"] = art["features_path"]
    out["Prediction_Created_UTC"] = datetime.utcnow().isoformat() + "Z"

    out_path.parent.mkdir(parents=True, exist_ok=True)
    out.to_parquet(out_path, index=False)
    print(
        f"Saved realtime predictions: {out_path}\n"
        f"  rows={len(out):,} mean={float(np.nanmean(p)):.6f} "
        f"p95={float(np.nanpercentile(p,95)):.6f} p99={float(np.nanpercentile(p,99)):.6f} max={float(np.nanmax(p)):.6f}"
    )
    for thr, lab in RISK_THRESHOLDS:
        print(f"  frac {lab:>4s} = {float(np.nanmean(p >= thr)):.6f}")
    return out


def _merge_grid_by_date_latlon(left, right, columns_to_add=None):
    if right is None or len(right) == 0:
        return left
    a = _keyed_for_merge(left)
    b = _keyed_for_merge(right)
    if columns_to_add is None:
        columns_to_add = [c for c in b.columns if c not in ["__LatKey", "__LonKey"] and c not in a.columns]
    keep = ["Date", "__LatKey", "__LonKey"] + [c for c in columns_to_add if c in b.columns]
    b = b[keep].drop_duplicates(subset=["Date", "__LatKey", "__LonKey"])
    out = a.merge(b, on=["Date", "__LatKey", "__LonKey"], how="left")
    return out.drop(columns=["__LatKey", "__LonKey"], errors="ignore")


def merge_available_wpc_pp_for_realtime(df_pred, date=None):
    """Merge WPC ERO and precomputed PP/proxy columns if the PP/WPC grid has this date."""
    date = _date8(date if date is not None else df_pred["Date"].iloc[0])
    try:
        base = load_pp_wpc_grid()
    except Exception as exc:
        print(f"PP/WPC grid unavailable; skipping WPC/PP merge: {exc}")
        return df_pred.copy()
    base = base.copy()
    base["Date"] = base["Date"].astype(str).str.slice(0, 8)
    base = base[base["Date"] == date].copy()
    if base.empty:
        print(f"No PP/WPC rows available for {date}; realtime forecast will be plotted without WPC/PP verification.")
        return df_pred.copy()
    add_cols = [c for c in base.columns if c == WPC_COL or c.startswith("PP_") or "UFVS" in c or "proxy" in c.lower()]
    if "WPC_ERO_Risk" in base.columns and WPC_COL not in add_cols:
        add_cols.append("WPC_ERO_Risk")
    return _merge_grid_by_date_latlon(df_pred, base, columns_to_add=add_cols)


def _ufvs_window_strings(date):
    date = _date8(date)
    start = datetime.strptime(date, "%Y%m%d") + timedelta(days=1)
    end = start + timedelta(days=1)
    start_date = start.strftime("%Y%m%d")
    end_date = end.strftime("%Y%m%d")
    return [(f"{start_date}12", f"{end_date}12"), (f"{start_date}16", f"{end_date}12")]


def _ufvs_raw_cache_path(prefix, date, start_stamp, end_stamp):
    REALTIME_UFVS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    return REALTIME_UFVS_CACHE_DIR / f"{prefix}_s{start_stamp}_e{end_stamp}.txt"


def parse_ufvs_text_points(text, prefix):
    vals = re.findall(r"[-+]?\d+(?:\.\d+)?", str(text))
    nums = [float(v) for v in vals]
    pts = []
    for a, b in zip(nums[0::2], nums[1::2]):
        if 15.0 <= a <= 60.0 and -130.0 <= b <= -60.0:
            pts.append((a, b))
        elif 15.0 <= b <= 60.0 and -130.0 <= a <= -60.0:
            pts.append((b, a))
    out = pd.DataFrame(pts, columns=["Lat", "Lon"])
    if len(out):
        out = out.drop_duplicates(subset=["Lat", "Lon"]).reset_index(drop=True)
    out["source"] = prefix
    return out


def fetch_ufvs_points(date, prefix, force=False, timeout=6):
    import requests
    last_error = None
    for start_stamp, end_stamp in _ufvs_window_strings(date):
        cache_path = _ufvs_raw_cache_path(prefix, date, start_stamp, end_stamp)
        url = f"{UFVS_BASE_URL}/{prefix}_s{start_stamp}_e{end_stamp}.txt"
        try:
            if cache_path.exists() and cache_path.stat().st_size > 0 and not force:
                text = cache_path.read_text(errors="ignore")
            else:
                print(f"Fetching UFVS {prefix}: {url}", flush=True)
                r = requests.get(url, timeout=timeout)
                if r.status_code != 200:
                    last_error = f"HTTP {r.status_code}: {url}"
                    continue
                text = r.text
                cache_path.write_text(text)
            pts = parse_ufvs_text_points(text, prefix)
            pts["ufvs_file"] = cache_path.name
            return pts
        except Exception as exc:
            last_error = repr(exc)
    print(f"{_date8(date)} {prefix}: no UFVS file found/parsed. Last error: {last_error}", flush=True)
    return pd.DataFrame(columns=["Lat", "Lon", "source", "ufvs_file"])


def filter_points_to_extent(pts, extent=None):
    if pts is None or len(pts) == 0:
        return pd.DataFrame(columns=["Lat", "Lon"])
    ed = _extent_dict(extent)
    lat = pd.to_numeric(pts["Lat"], errors="coerce")
    lon = pd.to_numeric(pts["Lon"], errors="coerce")
    return pts[(lon >= ed["lon_min"]) & (lon <= ed["lon_max"]) & (lat >= ed["lat_min"]) & (lat <= ed["lat_max"])].copy()


def map_points_to_grid_flag(df_grid, pts, max_dist_deg=0.15):
    flags = np.zeros(len(df_grid), dtype=np.int8)
    if pts is None or len(pts) == 0 or len(df_grid) == 0:
        return flags
    grid_xy = df_grid[["Lat", "Lon"]].to_numpy(dtype=np.float64)
    pts_xy = pts[["Lat", "Lon"]].to_numpy(dtype=np.float64)
    ok = np.isfinite(grid_xy).all(axis=1)
    if not ok.any():
        return flags
    tree = cKDTree(grid_xy[ok])
    dist, loc = tree.query(pts_xy, k=1)
    base_idx = np.where(ok)[0]
    good = np.isfinite(dist) & (dist <= float(max_dist_deg))
    if good.any():
        flags[np.unique(base_idx[loc[good]])] = 1
    return flags


def add_ufvs_to_realtime(df, date=None, force_ufvs=False, include_regular_flood_lsr=False, extent=DEFAULT_EXTENT, max_dist_deg=0.15):
    out = df.copy()
    date = _date8(date if date is not None else out["Date"].iloc[0])
    map_df = out.copy()
    ed = _extent_dict(extent)
    map_df = map_df[
        (pd.to_numeric(map_df["Lon"], errors="coerce") >= ed["lon_min"]) &
        (pd.to_numeric(map_df["Lon"], errors="coerce") <= ed["lon_max"]) &
        (pd.to_numeric(map_df["Lat"], errors="coerce") >= ed["lat_min"]) &
        (pd.to_numeric(map_df["Lat"], errors="coerce") <= ed["lat_max"])
    ].copy()
    prefixes = ["ST4gFFG", "ST4gARI", "USGS", "LSRFLASH"] + (["LSRREG"] if include_regular_flood_lsr else [])
    summary = []
    for prefix in prefixes:
        col = UFVS_PREFIX_TO_COL[prefix]
        pts = filter_points_to_extent(fetch_ufvs_points(date, prefix, force=force_ufvs), extent=extent)
        flags = map_points_to_grid_flag(map_df, pts, max_dist_deg=max_dist_deg)
        out[col] = 0
        out.loc[map_df.index, col] = flags.astype(np.int8)
        summary.append({"prefix": prefix, "points_in_extent": len(pts), "mapped_pixels": int(out[col].sum())})
    ufvs_cols = [UFVS_PREFIX_TO_COL[p] for p in prefixes if UFVS_PREFIX_TO_COL[p] in out.columns]
    if ufvs_cols:
        out["UFVS_ANY"] = (out[ufvs_cols].apply(pd.to_numeric, errors="coerce").fillna(0).max(axis=1) > 0).astype(np.int8)
    display(pd.DataFrame(summary))
    return out


def build_predict_verify_realtime_case(
    date,
    radius_km=40,
    force_predict=False,
    force_features=False,
    force_ufvs=False,
    include_ufvs=True,
    include_regular_flood_lsr=False,
    include_wpc_pp=True,
    training_script=None,
):
    """One-call realtime workflow: build features -> predict -> merge WPC/PP -> map UFVS if available."""
    date = _date8(date)
    r = int(round(float(radius_km)))
    pred = predict_realtime_case(date, radius_km=r, force=force_predict, force_features=force_features, training_script=training_script)
    out = pred.copy()
    if include_wpc_pp:
        out = merge_available_wpc_pp_for_realtime(out, date=date)
    if include_ufvs:
        out = add_ufvs_to_realtime(out, date=date, force_ufvs=force_ufvs, include_regular_flood_lsr=include_regular_flood_lsr)
    verified_path = realtime_verified_cache_path(date, r)
    out.to_parquet(verified_path, index=False)
    print(f"Saved realtime merged/verified cache: {verified_path}")
    return out


def _plot_dynamic_realtime_panels(date, radius_km, df, pp_definition="Any flood proxy", proxy_col="UFVS_ANY",
                                  smooth_sigma_grid=0, point_size=7, alpha=0.85):
    """Realtime-safe map plot.

    Unlike the historical four-panel plot, this does not require PP columns.
    Realtime/day-zero cases often have ML probabilities before any PP or UFVS
    verification is available. Panels are added only when their columns exist.
    """
    date = _date8(date)
    r = int(round(float(radius_km)))
    sub = _case_radius_subset(df, date, r)

    panels = []
    if "ML_Forecast_Prob" in sub.columns:
        panels.append((f"ML probability r{r} km | {date}", "ML_Forecast_Prob"))
    else:
        raise RuntimeError("Realtime dataframe is missing ML_Forecast_Prob.")

    if WPC_COL in sub.columns:
        panels.append(("WPC ERO risk", WPC_COL))
    else:
        print(f"No WPC ERO column ({WPC_COL}) available for {date}; plotting without WPC.")

    pp_col = None
    if pp_definition is not None:
        pp_col = _truth_col_from_pp_definition(pp_definition)
        if pp_col in sub.columns:
            panels.append((f"Practically Perfect: {pp_definition}", pp_col))
        else:
            available_pp = [c for c in sub.columns if c.startswith("PP_")][:20]
            print(f"No PP column {pp_col} available for {date}; available PP columns: {available_pp}. Plotting without PP.")

    if proxy_col is not None and proxy_col in sub.columns:
        panels.append((f"UFVS/proxy: {proxy_col}", proxy_col))
    elif "UFVS_ANY" in sub.columns:
        panels.append(("UFVS/proxy: UFVS_ANY", "UFVS_ANY"))
    else:
        proxy_like = [c for c in sub.columns if ("UFVS" in c or "LSR" in c or "USGS" in c or "proxy" in c.lower())][:20]
        print(f"No requested proxy column {proxy_col} available for {date}; proxy-like columns: {proxy_like}. Plotting without proxy.")

    n = len(panels)
    ncols = 2 if n > 1 else 1
    nrows = int(np.ceil(n / ncols))
    proj = ccrs.PlateCarree() if HAS_CARTOPY else None
    subplot_kw = {"projection": proj} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(nrows, ncols, figsize=(8*ncols, 5.5*nrows), subplot_kw=subplot_kw, constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, (title, col) in zip(axes, panels):
        _plot_prob_or_smoothed(ax, sub, col, title, smooth_sigma_grid=smooth_sigma_grid, point_size=point_size, alpha=alpha)
    for ax in axes[len(panels):]:
        ax.set_visible(False)

    plt.show()
    return fig


def plot_realtime_ml_wpc_pp_ufvs(
    date,
    radius_km=40,
    df=None,
    force_predict=False,
    force_features=False,
    force_ufvs=False,
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    smooth_sigma_grid=0,
    include_regular_flood_lsr=False,
):
    """Build/load realtime forecast and plot only fields that are actually available.

    This is intentionally different from the historical case plot, which requires
    a PP/WPC verification grid. Realtime cases may have ML only, ML+WPC, ML+UFVS,
    or ML+WPC+PP+UFVS depending on date and availability.
    """
    date = _date8(date)
    r = int(round(float(radius_km)))
    if df is None:
        path = realtime_verified_cache_path(date, r)
        if path.exists() and not (force_predict or force_features or force_ufvs):
            df = pd.read_parquet(path)
        else:
            df = build_predict_verify_realtime_case(
                date,
                radius_km=r,
                force_predict=force_predict,
                force_features=force_features,
                force_ufvs=force_ufvs,
                include_regular_flood_lsr=include_regular_flood_lsr,
            )

    return _plot_dynamic_realtime_panels(
        date=date,
        radius_km=r,
        df=df,
        pp_definition=pp_definition,
        proxy_col=proxy_col,
        smooth_sigma_grid=smooth_sigma_grid,
    )

print("Full realtime builder loaded.")
print(f"SCRIPT_DIR = {SCRIPT_DIR}")
print("Use build_predict_verify_realtime_case(...) to build features, predict, and merge WPC/PP/UFVS when available.")


## 5b. Realtime verification builder v7

Overrides realtime verification so WPC ERO is fetched/rasterized by the **case 12Z-to-12Z valid window** (`ISSUE`/`EXPIRE`) rather than the latest product issuance in a broad query window. This prevents the following-day Day-2 ERO from being selected. UFVS-derived PP generation remains adjustable by expansion and smoothing radius.


In [ ]:
# ======================================================================================
# 5b. Realtime verification builder v3: WPC fetch + UFVS-derived PP generation
# ======================================================================================
# This cell overrides the realtime verification/plotting functions above.  It keeps the
# existing realtime feature generation and predict_proba() inference path, but fixes the
# verification logic:
#   1. WPC ERO is merged from the existing PP/WPC grid when present, otherwise it is
#      fetched from the IEM WPC/SPC outlook GIS service and rasterized to the ML grid.
#   2. Practically Perfect fields are created from UFVS verification points when available.
#   3. If verification is unavailable, the code prints exactly what was missing and plots
#      ML + any available WPC fields instead of crashing.
#   4. The PP point-expansion radius is user-adjustable; smoothing remains separately
#      adjustable.

from pathlib import Path
from datetime import datetime, timedelta
import tempfile
import zipfile

REALTIME_WPC_CACHE_DIR = Path(PROJECT_DIR) / "realtime_wpc_ero_cache_v33day2valid"
REALTIME_PP_CACHE_DIR = Path(PROJECT_DIR) / "realtime_pp_from_ufvs_cache_v33day2valid"
REALTIME_WPC_CACHE_DIR.mkdir(parents=True, exist_ok=True)
REALTIME_PP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# User-editable defaults.  These can be overridden in the realtime run cell.
REALTIME_PP_EXPANSION_RADIUS_KM = 40.0
REALTIME_PP_SMOOTH_RADIUS_KM = 100.0
REALTIME_PP_SOURCE_PREFIXES = ["ST4gFFG", "ST4gARI", "USGS", "LSRFLASH"]

WPC_IEM_OUTLOOK_URL = "https://mesonet.agron.iastate.edu/cgi-bin/request/gis/outlooks.py"


def _path_obj(p):
    return p if isinstance(p, Path) else Path(str(p))


def _format_realtime_availability(df, date):
    date = _date8(date)
    cols = list(df.columns)
    pp_cols = [c for c in cols if c.startswith("PP_")]
    ufvs_cols = [c for c in cols if c.startswith("UFVS_")]
    print("\nRealtime availability summary")
    print("-" * 72)
    print(f"date: {date}")
    print(f"rows: {len(df):,}")
    print(f"ML_Forecast_Prob: {'yes' if 'ML_Forecast_Prob' in cols else 'NO'}")
    print(f"{WPC_COL}: {'yes' if WPC_COL in cols else 'NO'}")
    print(f"PP columns: {pp_cols if pp_cols else 'none'}")
    print(f"UFVS columns: {ufvs_cols if ufvs_cols else 'none'}")
    print("-" * 72)


def _wpc_probability_from_row(row):
    text = " ".join(str(row.get(c, "")) for c in ["CATEGORY", "THRESHOLD", "LABEL", "OUTLOOK", "name"] if c in row.index).upper()
    # IEM category labels vary by decoder/version; be permissive.
    if "HIGH" in text:
        return 0.70
    if "MDT" in text or "MOD" in text or "MODERATE" in text:
        return 0.40
    if "SLGT" in text or "SLIGHT" in text:
        return 0.15
    if "MRGL" in text or "MARG" in text or "MARGINAL" in text:
        return 0.05
    return 0.0



def _iem_datetime_column(gdf, candidate_names):
    """Return a UTC datetime Series from the first matching IEM DBF time column."""
    cols_upper = {str(c).upper().strip(): c for c in gdf.columns}
    for name in candidate_names:
        key = str(name).upper().strip()
        if key in cols_upper:
            col = cols_upper[key]
            s = pd.to_datetime(gdf[col], errors="coerce", utc=True)
            if s.notna().any():
                return s, col
    return pd.Series(pd.NaT, index=gdf.index, dtype="datetime64[ns, UTC]"), None


def _filter_iem_wpc_ero_to_case_valid_window(gdf, date):
    """Select the WPC ERO whose valid period matches the case 12Z-to-12Z window.

    IEM's outlook shapefile DBF schema uses:
      ISSUE  = outlook beginning valid time UTC
      EXPIRE = outlook ending valid time UTC
      PRODISS = product issuance time UTC

    The previous realtime code selected the latest PRODISS within a broad query window,
    which can incorrectly choose the following Day-2 ERO update.  For a case date
    YYYYMMDD, we instead select the outlook with ISSUE = YYYYMMDD 12Z and
    EXPIRE = YYYYMMDD+1 12Z.  If exact matching is not available, we choose the
    candidate whose valid period overlaps the target window best and whose beginning
    valid time is closest to 12Z on the case date.
    """
    date = _date8(date)
    target_start, target_end = day2_valid_window(date)

    issue, issue_col = _iem_datetime_column(gdf, ["ISSUE", "VALID", "START", "BEGINTIME"])
    expire, expire_col = _iem_datetime_column(gdf, ["EXPIRE", "EXPIRATION", "END", "ENDTIME"])
    prodiss, prodiss_col = _iem_datetime_column(gdf, ["PRODISS", "PRODUCTISSUANCE", "ISSUANCE"])

    if issue.notna().any() and expire.notna().any():
        tol = pd.Timedelta(minutes=2)
        exact = (issue.sub(target_start).abs() <= tol) & (expire.sub(target_end).abs() <= tol)
        if exact.any():
            sel = exact.copy()
            reason = "exact 12Z-to-12Z valid-period match"
        else:
            # Prefer products with the correct 12Z next-day ending time, then choose
            # the beginning valid time closest to the desired 12Z start.  This prevents
            # the latest-product bug that selected products valid on the following day.
            same_end = expire.sub(target_end).abs() <= tol
            candidates = same_end & (issue >= target_start - pd.Timedelta(hours=12)) & (issue < target_end)
            if not candidates.any():
                # Last-resort fallback: any product whose valid period overlaps the case window.
                candidates = (issue < target_end) & (expire > target_start)

            if candidates.any():
                cand_idx = np.where(candidates.to_numpy())[0]
                overlap_start = pd.Series(np.maximum(issue[candidates].view("int64"), target_start.value), index=issue[candidates].index)
                overlap_end = pd.Series(np.minimum(expire[candidates].view("int64"), target_end.value), index=expire[candidates].index)
                overlap_hours = (overlap_end - overlap_start) / 1e9 / 3600.0
                start_offset_hours = (issue[candidates] - target_start).abs() / pd.Timedelta(hours=1)
                score = overlap_hours - 0.25 * start_offset_hours
                best_index = score.idxmax()
                best_issue = issue.loc[best_index]
                best_expire = expire.loc[best_index]
                sel = (issue == best_issue) & (expire == best_expire)
                reason = "best overlapping valid-period fallback"
            else:
                print(
                    f"Warning: no WPC ERO valid-period rows overlap {target_start} to {target_end}; "
                    "falling back to unfiltered rows."
                )
                sel = pd.Series(True, index=gdf.index)
                reason = "unfiltered fallback; no usable ISSUE/EXPIRE overlap"

        # If multiple revisions/corrections exist for the selected valid window, use the
        # latest product issuance within that valid window, not the latest issuance in the
        # entire broad query response.
        if prodiss.notna().any() and sel.any():
            latest_prod = prodiss[sel].max()
            sel = sel & (prodiss == latest_prod)
        else:
            latest_prod = pd.NaT

        out = gdf.loc[sel].copy()
        out.attrs["wpc_target_valid_start"] = str(target_start)
        out.attrs["wpc_target_valid_end"] = str(target_end)
        out.attrs["wpc_selected_reason"] = reason
        if len(out):
            out.attrs["wpc_selected_valid_start"] = str(issue.loc[out.index].min())
            out.attrs["wpc_selected_valid_end"] = str(expire.loc[out.index].max())
            out.attrs["wpc_selected_prodiss"] = str(latest_prod) if pd.notna(latest_prod) else "unknown"
            print(
                "Selected WPC ERO for case window: "
                f"target={target_start} to {target_end}; "
                f"selected={out.attrs['wpc_selected_valid_start']} to {out.attrs['wpc_selected_valid_end']}; "
                f"prodiss={out.attrs['wpc_selected_prodiss']}; reason={reason}; rows={len(out)}"
            )
        return out

    print(
        "Warning: WPC ERO shapefile lacks parseable ISSUE/EXPIRE columns; "
        "falling back to product-issuance selection."
    )
    if prodiss.notna().any():
        latest = prodiss.max()
        gdf = gdf[prodiss == latest].copy()
        gdf.attrs["wpc_selected_prodiss"] = str(latest)
        print(f"Fallback selected latest product issuance: {latest}")
    return gdf

def _download_iem_wpc_ero_gdf(date, force=False):
    """Fetch the Day-2 WPC ERO valid for the case 12Z-to-12Z window.

    IMPORTANT: IEM query parameters select an issuance interval, but the returned DBF
    contains ISSUE/EXPIRE valid times.  We therefore query a broad enough issuance
    interval to include the desired product, then filter the returned polygons by
    ISSUE=case-date 12Z and EXPIRE=next-day 12Z.  We do NOT select the latest PRODISS
    across the whole query window.
    """
    date = _date8(date)
    start_dt = datetime.strptime(date, "%Y%m%d")
    target_start, target_end = day2_valid_window(date)

    # Broad issuance window: includes early-morning product issuances that are valid
    # beginning 12Z on the case date, but filtering below enforces the valid window.
    sts = (start_dt - timedelta(hours=6)).strftime("%Y-%m-%dT%H:%MZ")
    ets = (start_dt + timedelta(days=2, hours=18)).strftime("%Y-%m-%dT%H:%MZ")
    cache_base = REALTIME_WPC_CACHE_DIR / f"iem_wpc_ero_day2_{date}_valid_d1_12_to_d2_12"

    try:
        import requests
        import geopandas as gpd
    except Exception as exc:
        print(f"WPC ERO fetch skipped: requests/geopandas unavailable ({exc}).")
        return None

    # Cookie/cutter geometry gives the highest risk category per location and avoids
    # double-counting layered polygons.  Fall back through alternatives for service quirks.
    geom_candidates = ["cookie", "cutter", "nonoverlap", None, "layers", "layer", "1", "0"]
    last_error = None

    for geom in geom_candidates:
        suffix = "default" if geom is None else str(geom)
        zip_path = cache_base.with_name(cache_base.name + f"_{suffix}.zip")
        params = {"type": "E", "d": "2", "sts": sts, "ets": ets}
        if geom is not None:
            params["geom"] = geom
        try:
            if force or (not zip_path.exists()) or zip_path.stat().st_size < 128:
                print(f"Fetching WPC ERO from IEM: {WPC_IEM_OUTLOOK_URL} params={params}")
                resp = requests.get(WPC_IEM_OUTLOOK_URL, params=params, timeout=45)
                if resp.status_code != 200:
                    last_error = f"HTTP {resp.status_code} for geom={geom}"
                    continue
                content = resp.content
                if not content.startswith(b"PK"):
                    last_error = f"IEM response for geom={geom} was not a zip/shapefile; first bytes={content[:40]!r}"
                    continue
                zip_path.write_bytes(content)

            try:
                gdf = gpd.read_file(f"zip://{zip_path}")
            except Exception:
                gdf = gpd.read_file(str(zip_path))

            if gdf is None or gdf.empty:
                last_error = f"empty shapefile for geom={geom}"
                continue

            cols_upper = {str(c).upper().strip(): c for c in gdf.columns}
            if "TYPE" in cols_upper:
                c = cols_upper["TYPE"]
                gdf = gdf[gdf[c].astype(str).str.upper().str.contains("E", na=False)].copy()
            if "DAY" in cols_upper:
                c = cols_upper["DAY"]
                gdf = gdf[pd.to_numeric(gdf[c], errors="coerce").fillna(-999).astype(int) == 2].copy()
            if gdf.empty:
                last_error = f"no day-2 WPC ERO rows after filtering for geom={geom}"
                continue

            gdf = _filter_iem_wpc_ero_to_case_valid_window(gdf, date)
            if gdf is None or gdf.empty:
                last_error = f"no WPC ERO polygons matched target valid window {target_start} to {target_end} for geom={geom}"
                continue

            if gdf.crs is not None:
                try:
                    gdf = gdf.to_crs(epsg=4326)
                except Exception as exc:
                    print(f"Warning: could not reproject WPC ERO CRS {gdf.crs}: {exc}")
            gdf["__WPC_ERO_Risk"] = gdf.apply(_wpc_probability_from_row, axis=1).astype(float)
            gdf = gdf[gdf["__WPC_ERO_Risk"] > 0].copy()
            if gdf.empty:
                last_error = f"WPC ERO polygons existed but no recognized risk categories for geom={geom}"
                continue

            print(f"Loaded {len(gdf)} valid-window WPC ERO polygons from IEM for {date}.")
            return gdf
        except Exception as exc:
            last_error = repr(exc)
            continue

    print(f"No WPC ERO polygons could be loaded from IEM for {date}. Last error: {last_error}")
    return None

def _rasterize_wpc_gdf_to_grid(gdf, df_grid):
    """Rasterize WPC ERO polygons onto an existing Lat/Lon grid; highest risk wins."""
    out = np.zeros(len(df_grid), dtype=np.float32)
    if gdf is None or len(gdf) == 0 or len(df_grid) == 0:
        return out
    lon = pd.to_numeric(df_grid["Lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(df_grid["Lat"], errors="coerce").to_numpy(float)
    finite = np.isfinite(lon) & np.isfinite(lat)
    if not finite.any():
        return out
    try:
        import shapely
        contains_xy = getattr(shapely, "contains_xy", None)
    except Exception:
        contains_xy = None
    from shapely.geometry import Point
    from shapely.prepared import prep

    for _, row in gdf.sort_values("__WPC_ERO_Risk").iterrows():
        geom = row.geometry
        if geom is None or geom.is_empty:
            continue
        risk = float(row["__WPC_ERO_Risk"])
        minx, miny, maxx, maxy = geom.bounds
        bbox = finite & (lon >= minx) & (lon <= maxx) & (lat >= miny) & (lat <= maxy)
        idx = np.flatnonzero(bbox)
        if idx.size == 0:
            continue
        try:
            if contains_xy is not None:
                inside = contains_xy(geom, lon[idx], lat[idx])
            else:
                pg = prep(geom)
                inside = np.array([pg.contains(Point(x, y)) or pg.touches(Point(x, y)) for x, y in zip(lon[idx], lat[idx])], dtype=bool)
        except Exception:
            pg = prep(geom)
            inside = np.array([pg.contains(Point(x, y)) or pg.touches(Point(x, y)) for x, y in zip(lon[idx], lat[idx])], dtype=bool)
        if np.any(inside):
            out[idx[inside]] = np.maximum(out[idx[inside]], risk)
    return out


def add_wpc_ero_to_realtime_from_iem(df, date=None, force_wpc=False):
    """Add WPC_ERO_Risk to realtime dataframe, fetching/rasterizing from IEM if needed."""
    out = df.copy()
    date = _date8(date if date is not None else out["Date"].iloc[0])
    wpc_cache = REALTIME_WPC_CACHE_DIR / f"wpc_ero_risk_grid_{date}_valid12to12_{len(out)}rows.parquet"
    if wpc_cache.exists() and wpc_cache.stat().st_size > 1024 and not force_wpc:
        tmp = pd.read_parquet(wpc_cache)
        if WPC_COL in tmp.columns and len(tmp) == len(out):
            out[WPC_COL] = pd.to_numeric(tmp[WPC_COL], errors="coerce").fillna(0).to_numpy(np.float32)
            for meta_col in ["WPC_ERO_Target_Valid_Start", "WPC_ERO_Target_Valid_End",
                             "WPC_ERO_Selected_Valid_Start", "WPC_ERO_Selected_Valid_End",
                             "WPC_ERO_Product_Issuance"]:
                if meta_col in tmp.columns:
                    out[meta_col] = tmp[meta_col].astype(str).iloc[0]
            print(f"Loaded cached WPC ERO grid: {wpc_cache}")
            if "WPC_ERO_Selected_Valid_Start" in out.columns:
                print("Cached WPC selected valid window:", out["WPC_ERO_Selected_Valid_Start"].iloc[0], "to", out["WPC_ERO_Selected_Valid_End"].iloc[0])
            return out

    gdf = _download_iem_wpc_ero_gdf(date, force=force_wpc)
    if gdf is None or len(gdf) == 0:
        print(f"WPC ERO unavailable for {date}; realtime plot will proceed without WPC.")
        return out
    out[WPC_COL] = _rasterize_wpc_gdf_to_grid(gdf, out)
    # Keep lightweight metadata columns so stale/wrong valid-window caches are easier to diagnose.
    out["WPC_ERO_Target_Valid_Start"] = str(gdf.attrs.get("wpc_target_valid_start", ""))
    out["WPC_ERO_Target_Valid_End"] = str(gdf.attrs.get("wpc_target_valid_end", ""))
    out["WPC_ERO_Selected_Valid_Start"] = str(gdf.attrs.get("wpc_selected_valid_start", ""))
    out["WPC_ERO_Selected_Valid_End"] = str(gdf.attrs.get("wpc_selected_valid_end", ""))
    out["WPC_ERO_Product_Issuance"] = str(gdf.attrs.get("wpc_selected_prodiss", ""))
    out[["Date", "Lat", "Lon", WPC_COL,
         "WPC_ERO_Target_Valid_Start", "WPC_ERO_Target_Valid_End",
         "WPC_ERO_Selected_Valid_Start", "WPC_ERO_Selected_Valid_End",
         "WPC_ERO_Product_Issuance"]].to_parquet(wpc_cache, index=False)
    print(f"Saved WPC ERO raster cache: {wpc_cache}")
    print(f"WPC risk pixels: >5%={(out[WPC_COL] >= 0.05).sum():,}, >15%={(out[WPC_COL] >= 0.15).sum():,}, >40%={(out[WPC_COL] >= 0.40).sum():,}, >70%={(out[WPC_COL] >= 0.70).sum():,}")
    return out


def _event_mask_from_points(df_grid, pts, max_dist_km=20.0):
    mask = np.zeros(len(df_grid), dtype=bool)
    if pts is None or len(pts) == 0 or len(df_grid) == 0:
        return mask
    grid_xyz = latlon_to_unit_xyz(df_grid["Lat"].to_numpy(float), df_grid["Lon"].to_numpy(float))
    pts_xyz = latlon_to_unit_xyz(pts["Lat"].to_numpy(float), pts["Lon"].to_numpy(float))
    tree = cKDTree(grid_xyz)
    dist, loc = tree.query(pts_xyz, k=1)
    max_chord = km_to_unit_sphere_chord_radius(max_dist_km)
    good = np.isfinite(dist) & (dist <= max_chord)
    if np.any(good):
        mask[np.unique(loc[good])] = True
    return mask


def _smooth_grid_values_by_km(df_grid, values, smooth_radius_km=100.0, chunk_size=1500, cutoff_sigma=3.0):
    """Smooth an expanded binary PP mask on the existing curvilinear grid.

    This intentionally does *not* use nearest-event distance and does *not* preserve
    the expanded core as exactly 1.0.  The previous realtime implementation assigned
    1.0 to every grid point inside the 40-km expansion area and then decayed away from
    that area, which made almost every verified point appear as a High-risk PP area.

    Instead, this approximates a Gaussian convolution/weighted local fraction of the
    expanded binary event mask on the existing RAP-style point grid:

        PP(x_i) = sum_j K(d_ij) * I_expanded(x_j) / sum_j K(d_ij)

    where K is a Gaussian with scale smooth_radius_km.  This behaves much more like
    smoothing a binary observation mask: isolated reports remain lower-amplitude,
    while clustered/areally broad events can reach higher PP categories.
    """
    values = np.asarray(values, dtype=np.float32)
    if values.size == 0 or not np.isfinite(values).any() or np.nanmax(values) <= 0:
        return np.zeros_like(values, dtype=np.float32)
    if smooth_radius_km is None or float(smooth_radius_km) <= 0:
        return np.clip(values, 0.0, 1.0).astype(np.float32)

    sigma_km = float(smooth_radius_km)
    max_km = float(cutoff_sigma) * sigma_km
    max_chord = km_to_unit_sphere_chord_radius(max_km)

    lat = df_grid["Lat"].to_numpy(float)
    lon = df_grid["Lon"].to_numpy(float)
    xyz = latlon_to_unit_xyz(lat, lon)
    tree = cKDTree(xyz)

    binary = np.nan_to_num(values, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)
    binary = (binary > 0).astype(np.float32)

    out = np.zeros(len(binary), dtype=np.float32)

    for start in range(0, len(binary), int(chunk_size)):
        end = min(start + int(chunk_size), len(binary))
        neigh = tree.query_ball_point(xyz[start:end], r=max_chord)
        for ii, idx in enumerate(neigh):
            if not idx:
                out[start + ii] = 0.0
                continue
            idx = np.asarray(idx, dtype=np.int64)
            cd = np.linalg.norm(xyz[idx] - xyz[start + ii], axis=1)
            cd = np.clip(cd, 0.0, 2.0)
            dist_km = EARTH_RADIUS_KM * (2.0 * np.arcsin(cd / 2.0))
            w = np.exp(-0.5 * (dist_km / sigma_km) ** 2).astype(np.float32)
            den = float(np.sum(w))
            out[start + ii] = 0.0 if den <= 0 else float(np.sum(w * binary[idx]) / den)

    return np.clip(out, 0.0, 1.0).astype(np.float32)

def _pp_from_points(df_grid, pts, expansion_radius_km=40.0, smooth_radius_km=100.0, max_nearest_dist_km=20.0):
    if pts is None or len(pts) == 0:
        return np.zeros(len(df_grid), dtype=np.float32), {"raw_points": 0, "nearest_event_pixels": 0, "expanded_pixels": 0, "smoothed": False}
    xyz = latlon_to_unit_xyz(df_grid["Lat"].to_numpy(float), df_grid["Lon"].to_numpy(float))
    tree = cKDTree(xyz)
    event = _event_mask_from_points(df_grid, pts, max_dist_km=max_nearest_dist_km)
    expanded = expand_binary_mask_radius_km(event, tree, xyz, radius_km=float(expansion_radius_km)).astype(np.float32)
    smoothed = _smooth_grid_values_by_km(df_grid, expanded, smooth_radius_km=float(smooth_radius_km))
    smoothed = np.clip(smoothed, 0.0, 1.0).astype(np.float32)
    meta = {
        "raw_points": int(len(pts)),
        "nearest_event_pixels": int(event.sum()),
        "expanded_pixels": int(expanded.sum()),
        "smoothed": bool(smooth_radius_km is not None and float(smooth_radius_km) > 0 and HAS_NDIMAGE),
    }
    return smoothed, meta


def add_ufvs_and_realtime_pp(
    df,
    date=None,
    force_ufvs=False,
    include_regular_flood_lsr=False,
    extent=DEFAULT_EXTENT,
    pp_expansion_radius_km=40.0,
    pp_smooth_radius_km=100.0,
    max_nearest_dist_km=25.0,
):
    """Fetch UFVS points, map raw proxy flags, and build PP fields from the points.

    If no UFVS files are available or no points fall in the domain, the function states that
    PP could not be generated and returns the dataframe unchanged except for zero UFVS flags.
    """
    out = df.copy()
    date = _date8(date if date is not None else out["Date"].iloc[0])
    ed = _extent_dict(extent)
    prefixes = list(REALTIME_PP_SOURCE_PREFIXES)
    if include_regular_flood_lsr and "LSRREG" not in prefixes:
        prefixes.append("LSRREG")

    pp_cache = REALTIME_PP_CACHE_DIR / (
        f"pp_ufvs_{date}_expand{int(round(float(pp_expansion_radius_km)))}km_"
        f"smooth{int(round(float(pp_smooth_radius_km)))}km_{len(out)}rows.parquet"
    )
    if pp_cache.exists() and pp_cache.stat().st_size > 1024 and not force_ufvs:
        cached = pd.read_parquet(pp_cache)
        add_cols = [c for c in cached.columns if c.startswith("UFVS_") or c.startswith("PP_")]
        if add_cols and len(cached) == len(out):
            for c in add_cols:
                out[c] = cached[c].to_numpy()
            print(f"Loaded cached realtime UFVS/PP grid: {pp_cache}")
            return out

    grid_domain = out[
        (pd.to_numeric(out["Lon"], errors="coerce") >= ed["lon_min"]) &
        (pd.to_numeric(out["Lon"], errors="coerce") <= ed["lon_max"]) &
        (pd.to_numeric(out["Lat"], errors="coerce") >= ed["lat_min"]) &
        (pd.to_numeric(out["Lat"], errors="coerce") <= ed["lat_max"])
    ].copy()
    if grid_domain.empty:
        print(f"No grid rows inside requested extent for {date}; cannot map UFVS/PP.")
        return out

    print(f"Starting UFVS/PP processing for {date}: prefixes={prefixes}, rows={len(out):,}, domain_rows={len(grid_domain):,}", flush=True)
    prefix_to_points = {}
    summary = []
    for prefix in prefixes:
        col = UFVS_PREFIX_TO_COL.get(prefix, f"UFVS_{prefix}")
        print(f"  UFVS prefix {prefix}: fetching/parsing...", flush=True)
        pts = fetch_ufvs_points(date, prefix, force=force_ufvs)
        pts = filter_points_to_extent(pts, extent=extent)
        print(f"  UFVS prefix {prefix}: points in extent = {len(pts):,}", flush=True)
        prefix_to_points[prefix] = pts
        flags_domain = _event_mask_from_points(grid_domain, pts, max_dist_km=max_nearest_dist_km).astype(np.int8)
        out[col] = 0
        out.loc[grid_domain.index, col] = flags_domain
        summary.append({"prefix": prefix, "column": col, "points_in_extent": int(len(pts)), "nearest_event_pixels": int(flags_domain.sum())})

    ufvs_cols = [UFVS_PREFIX_TO_COL[p] for p in prefixes if p in UFVS_PREFIX_TO_COL and UFVS_PREFIX_TO_COL[p] in out.columns]
    if ufvs_cols:
        out["UFVS_ANY"] = (out[ufvs_cols].apply(pd.to_numeric, errors="coerce").fillna(0).max(axis=1) > 0).astype(np.int8)

    # Create PP fields from point sets.  Main required field is exactly PP_Any flood proxy.
    point_sets = {
        "PP_Stage IV > FFG": prefix_to_points.get("ST4gFFG", pd.DataFrame(columns=["Lat", "Lon"])),
        "PP_Stage IV ARI": prefix_to_points.get("ST4gARI", pd.DataFrame(columns=["Lat", "Lon"])),
        "PP_USGS": prefix_to_points.get("USGS", pd.DataFrame(columns=["Lat", "Lon"])),
        "PP_Flash LSR": prefix_to_points.get("LSRFLASH", pd.DataFrame(columns=["Lat", "Lon"])),
    }
    if include_regular_flood_lsr:
        point_sets["PP_Flood LSR"] = prefix_to_points.get("LSRREG", pd.DataFrame(columns=["Lat", "Lon"]))

    all_pts = [v for v in point_sets.values() if v is not None and len(v) > 0]
    if all_pts:
        union_pts = pd.concat(all_pts, ignore_index=True).drop_duplicates(subset=["Lat", "Lon"])
    else:
        union_pts = pd.DataFrame(columns=["Lat", "Lon"])
    point_sets["PP_Any flood proxy"] = union_pts

    pp_meta_rows = []
    any_pp_created = False
    for pp_col, pts in point_sets.items():
        print(f"  Building {pp_col} from {0 if pts is None else len(pts):,} point(s)...", flush=True)
        vals_domain, meta = _pp_from_points(
            grid_domain,
            pts,
            expansion_radius_km=pp_expansion_radius_km,
            smooth_radius_km=pp_smooth_radius_km,
            max_nearest_dist_km=max_nearest_dist_km,
        )
        out[pp_col] = 0.0
        out.loc[grid_domain.index, pp_col] = vals_domain.astype(np.float32)
        any_pp_created = any_pp_created or (meta["raw_points"] > 0)
        pp_meta_rows.append({"pp_column": pp_col, **meta})

    display(pd.DataFrame(summary))
    display(pd.DataFrame(pp_meta_rows))

    if not any_pp_created:
        print(
            f"No UFVS verification points were found in the domain for {date}; PP fields could not be created. "
            "The realtime plot will fall back to ML plus any available WPC ERO."
        )
    else:
        print(
            f"Created realtime PP fields for {date} using expansion_radius={pp_expansion_radius_km} km "
            f"and smoothing_radius={pp_smooth_radius_km} km."
        )

    save_cols = ["Date", "Lat", "Lon"] + [c for c in out.columns if c.startswith("UFVS_") or c.startswith("PP_")]
    out[save_cols].to_parquet(pp_cache, index=False)
    print(f"Saved realtime UFVS/PP cache: {pp_cache}")
    return out


def merge_available_wpc_pp_for_realtime(df_pred, date=None, force_wpc=False):
    """Merge precomputed PP/WPC if available, then fetch WPC from IEM if WPC is still missing."""
    date = _date8(date if date is not None else df_pred["Date"].iloc[0])
    out = df_pred.copy()

    # 1. Try existing historical PP/WPC grid first.
    try:
        base = load_pp_wpc_grid()
        base = base.copy()
        base["Date"] = base["Date"].astype(str).str.slice(0, 8)
        base = base[base["Date"] == date].copy()
        if not base.empty:
            add_cols = [c for c in base.columns if c == WPC_COL or c.startswith("PP_") or c.startswith("UFVS_") or "proxy" in c.lower()]
            if "WPC_ERO_Risk" in base.columns and WPC_COL not in add_cols:
                add_cols.append("WPC_ERO_Risk")
            out = _merge_grid_by_date_latlon(out, base, columns_to_add=add_cols)
            print(f"Merged existing PP/WPC grid for {date}: added {[c for c in add_cols if c in out.columns]}")
        else:
            print(f"No precomputed PP/WPC grid rows for {date}; will try realtime WPC and UFVS-derived PP.")
    except Exception as exc:
        print(f"Existing PP/WPC grid unavailable; will try realtime WPC and UFVS-derived PP. Reason: {exc}")

    # 2. If WPC still missing, fetch/rasterize from IEM.
    if WPC_COL not in out.columns or pd.to_numeric(out[WPC_COL], errors="coerce").fillna(0).max() <= 0:
        out = add_wpc_ero_to_realtime_from_iem(out, date=date, force_wpc=force_wpc)
    return out


def build_predict_verify_realtime_case(
    date,
    radius_km=40,
    force_predict=False,
    force_features=False,
    force_ufvs=False,
    force_wpc=False,
    include_ufvs=True,
    include_regular_flood_lsr=False,
    include_wpc_pp=True,
    training_script=None,
    pp_expansion_radius_km=None,
    pp_smooth_radius_km=None,
):
    """One-call realtime workflow: build features -> predict -> WPC -> UFVS -> PP.

    For day-after or older cases, this attempts to build PP from UFVS points.  If no
    verification is available, it explicitly says so and still returns ML plus any WPC.
    """
    date = _date8(date)
    r = int(round(float(radius_km)))
    pp_expansion_radius_km = REALTIME_PP_EXPANSION_RADIUS_KM if pp_expansion_radius_km is None else float(pp_expansion_radius_km)
    pp_smooth_radius_km = REALTIME_PP_SMOOTH_RADIUS_KM if pp_smooth_radius_km is None else float(pp_smooth_radius_km)

    pred = predict_realtime_case(date, radius_km=r, force=force_predict, force_features=force_features, training_script=training_script)
    out = pred.copy()
    if include_wpc_pp:
        out = merge_available_wpc_pp_for_realtime(out, date=date, force_wpc=force_wpc)
    if include_ufvs:
        out = add_ufvs_and_realtime_pp(
            out,
            date=date,
            force_ufvs=force_ufvs,
            include_regular_flood_lsr=include_regular_flood_lsr,
            pp_expansion_radius_km=pp_expansion_radius_km,
            pp_smooth_radius_km=pp_smooth_radius_km,
        )
    _format_realtime_availability(out, date)
    verified_path = realtime_verified_cache_path(date, r)
    out.to_parquet(verified_path, index=False)
    print(f"Saved realtime merged/verified cache: {verified_path}")
    return out


def _plot_dynamic_realtime_panels(
    date,
    radius_km,
    df,
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    smooth_sigma_grid=0,
    point_size=7,
    alpha=0.85,
):
    """Realtime-safe plot: ML required; WPC/PP/UFVS panels added when available."""
    date = _date8(date)
    r = int(round(float(radius_km)))
    sub = _case_radius_subset(df, date, r)

    panels = []
    if "ML_Forecast_Prob" not in sub.columns:
        raise RuntimeError("Realtime dataframe is missing ML_Forecast_Prob.")
    panels.append((f"ML probability r{r} km | {date}", "ML_Forecast_Prob"))

    if WPC_COL in sub.columns and pd.to_numeric(sub[WPC_COL], errors="coerce").fillna(0).max() > 0:
        panels.append(("WPC ERO risk", WPC_COL))
    else:
        print(f"No nonzero {WPC_COL} available for {date}; WPC panel omitted.")

    pp_col = None
    if pp_definition is not None:
        pp_col = _truth_col_from_pp_definition(pp_definition)
        if pp_col in sub.columns and pd.to_numeric(sub[pp_col], errors="coerce").fillna(0).max() > 0:
            panels.append((f"Practically Perfect: {pp_definition}", pp_col))
        else:
            available_pp = [c for c in sub.columns if c.startswith("PP_")]
            print(f"No usable PP column {pp_col} for {date}. Available PP columns: {available_pp}. PP panel omitted.")

    if proxy_col is not None and proxy_col in sub.columns and pd.to_numeric(sub[proxy_col], errors="coerce").fillna(0).max() > 0:
        panels.append((f"UFVS/proxy: {proxy_col}", proxy_col))
    elif "UFVS_ANY" in sub.columns and pd.to_numeric(sub["UFVS_ANY"], errors="coerce").fillna(0).max() > 0:
        panels.append(("UFVS/proxy: UFVS_ANY", "UFVS_ANY"))
    else:
        proxy_like = [c for c in sub.columns if ("UFVS" in c or "LSR" in c or "USGS" in c or "proxy" in c.lower())]
        print(f"No nonzero proxy column available for {date}. Proxy-like columns: {proxy_like}. Proxy panel omitted.")

    n = len(panels)
    ncols = 2 if n > 1 else 1
    nrows = int(np.ceil(n / ncols))
    proj = ccrs.PlateCarree() if HAS_CARTOPY else None
    subplot_kw = {"projection": proj} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(nrows, ncols, figsize=(8 * ncols, 5.5 * nrows), subplot_kw=subplot_kw, constrained_layout=True)
    axes = np.atleast_1d(axes).ravel()

    for ax, (title, col) in zip(axes, panels):
        _plot_prob_or_smoothed(ax, sub, col, title, smooth_sigma_grid=smooth_sigma_grid, point_size=point_size, alpha=alpha)
    for ax in axes[len(panels):]:
        ax.set_visible(False)
    plt.show()
    return fig


def plot_realtime_ml_wpc_pp_ufvs(
    date,
    radius_km=40,
    df=None,
    force_predict=False,
    force_features=False,
    force_ufvs=False,
    force_wpc=False,
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    smooth_sigma_grid=0,
    include_regular_flood_lsr=False,
    pp_expansion_radius_km=None,
    pp_smooth_radius_km=None,
):
    date = _date8(date)
    r = int(round(float(radius_km)))
    if df is None:
        path = realtime_verified_cache_path(date, r)
        if path.exists() and not (force_predict or force_features or force_ufvs or force_wpc):
            df = pd.read_parquet(path)
            # If cached file lacks usable PP for an older case, rebuild verification only.
            pp_col = _truth_col_from_pp_definition(pp_definition)
            if pp_col not in df.columns or pd.to_numeric(df.get(pp_col, pd.Series(0, index=df.index)), errors="coerce").fillna(0).max() <= 0:
                print(f"Cached realtime file lacks usable {pp_col}; attempting to rebuild WPC/UFVS/PP verification.")
                df = build_predict_verify_realtime_case(
                    date,
                    radius_km=r,
                    force_predict=False,
                    force_features=False,
                    force_ufvs=force_ufvs,
                    force_wpc=force_wpc,
                    include_regular_flood_lsr=include_regular_flood_lsr,
                    pp_expansion_radius_km=pp_expansion_radius_km,
                    pp_smooth_radius_km=pp_smooth_radius_km,
                )
        else:
            df = build_predict_verify_realtime_case(
                date,
                radius_km=r,
                force_predict=force_predict,
                force_features=force_features,
                force_ufvs=force_ufvs,
                force_wpc=force_wpc,
                include_regular_flood_lsr=include_regular_flood_lsr,
                pp_expansion_radius_km=pp_expansion_radius_km,
                pp_smooth_radius_km=pp_smooth_radius_km,
            )
    return _plot_dynamic_realtime_panels(
        date=date,
        radius_km=r,
        df=df,
        pp_definition=pp_definition,
        proxy_col=proxy_col,
        smooth_sigma_grid=smooth_sigma_grid,
    )

print("Realtime verification builder v6 loaded: WPC fetch + UFVS-derived PP generation enabled; PP smoothing uses weighted local fractional coverage.")
print("Adjust REALTIME_PP_EXPANSION_RADIUS_KM and REALTIME_PP_SMOOTH_RADIUS_KM in the realtime run cell.")


## 5c. Multi-radius realtime ensemble viewer

This section loads every trained realtime radius member available locally, merges the members onto the same grid, creates radius-ensemble products, and plots the members beside WPC. It also includes a separate verification-oriented panel plot for PP/UFVS when verification is available. The historical viewer and training logic above are left untouched.

In [ ]:

# ======================================================================================
# 5c. Multi-radius realtime ensemble viewer
#
# This overrides only the realtime presentation layer. It keeps the working single-radius
# feature/prediction builders intact, then loads every available radius member, merges the
# predictions onto one grid, creates radius-ensemble products, and plots the available
# members beside WPC. It does not modify the historical viewer or training artifacts.
# ======================================================================================

REALTIME_DEFAULT_MODEL_SPECS = [dict(spec) for spec in MODEL_SPECS]
REALTIME_DEFAULT_RADII_KM = list(dict.fromkeys(int(spec["radius_km"]) for spec in REALTIME_DEFAULT_MODEL_SPECS))
if "_wide_ml_member_metadata" not in globals():
    def _wide_ml_member_metadata(col):
        c = str(col)
        m = re.match(r"^ML_r(\d+)_Prob$", c)
        return (f"r{int(m.group(1))}km", int(m.group(1))) if m else (None, None)


def _model_prob_col(model_label, radius_km):
    return f"ML_r{int(round(float(radius_km)))}_Prob"


def _radius_prob_col(radius_km):
    return _model_prob_col(model_label_for_radius(radius_km), radius_km)


def _multi_radius_cache_path(date, available_models):
    date = _date8(date)
    rr = "_".join(str(x) for x in available_models) if available_models else "none"
    return REALTIME_VERIFIED_CACHE_DIR / f"realtime_verified_v33_multiradius_{rr}_{date}.parquet"


def _probability_matched_mean_from_members(member_matrix):
    """Probability-matched composite from radius-member probability fields.

    member_matrix shape is (n_points, n_members). The PMM preserves the spatial ranking
    of the ensemble mean while matching the one-point distribution to the pooled member
    distribution. This is a PMM-like probability blend, not a new trained model.
    """
    arr = np.asarray(member_matrix, dtype=float)
    if arr.ndim != 2 or arr.shape[0] == 0 or arr.shape[1] == 0:
        return np.full(arr.shape[0] if arr.ndim else 0, np.nan, dtype=np.float32)

    mean = np.nanmean(arr, axis=1)
    pooled = arr[np.isfinite(arr)]
    out = np.full(arr.shape[0], np.nan, dtype=np.float32)
    good = np.isfinite(mean)
    if good.sum() == 0 or pooled.size == 0:
        return out

    pooled = np.clip(pooled.astype(float), 0.0, 1.0)
    order = np.argsort(mean[good], kind="mergesort")
    good_idx = np.flatnonzero(good)
    q = (np.arange(good.sum(), dtype=float) + 0.5) / float(good.sum())
    matched_vals = np.quantile(pooled, q)
    out[good_idx[order]] = np.clip(matched_vals, 0.0, 1.0).astype(np.float32)
    return out


def add_radius_ensemble_products(df, radius_cols=None):
    out = df.copy()
    if radius_cols is None:
        radius_cols = sorted(
            [c for c in out.columns if re.match(r"^ML_r\d+_Prob$", str(c))],
            key=lambda c: int(re.search(r"r(\d+)", c).group(1)),
        )
    radius_cols = [c for c in radius_cols if c in out.columns]
    if not radius_cols:
        print("No ML radius probability columns found; ensemble products not created.")
        return out, []

    arr = out[radius_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    out["ML_Radius_EnsMean"] = np.nanmean(arr, axis=1).astype(np.float32)
    out["ML_Radius_EnsMedian"] = np.nanmedian(arr, axis=1).astype(np.float32)
    out["ML_Radius_EnsMax"] = np.nanmax(arr, axis=1).astype(np.float32)
    out["ML_Radius_EnsSpread"] = np.nanstd(arr, axis=1).astype(np.float32)
    out["ML_Radius_PMM"] = _probability_matched_mean_from_members(arr)

    # Compatibility for older metric/plot cells that expect ML_Forecast_Prob.
    # Use ensemble mean because it is the least surprising aggregate probability.
    out["ML_Forecast_Prob"] = out["ML_Radius_EnsMean"].astype(np.float32)
    out["ML_Target_Radius_km"] = -1
    return out, radius_cols


def print_probability_summary(df, cols, label="probability summary"):
    print("\n" + label)
    print("-" * 72)
    for c in cols:
        if c not in df.columns:
            continue
        p = pd.to_numeric(df[c], errors="coerce").to_numpy(float)
        if not np.isfinite(p).any():
            print(f"{c:24s}: no finite values")
            continue
        msg = (
            f"{c:24s}: mean={np.nanmean(p):.5f} p95={np.nanpercentile(p,95):.5f} "
            f"p99={np.nanpercentile(p,99):.5f} max={np.nanmax(p):.5f}"
        )
        msg += " | " + ", ".join([f">={thr:g}:{np.nanmean(p >= thr):.4f}" for thr in [0.05, 0.15, 0.40, 0.70]])
        print(msg)


def build_predict_verify_realtime_multi_radius(
    date,
    radii=None,
    model_specs=None,
    force_predict=False,
    force_features=False,
    force_ufvs=False,
    force_wpc=False,
    include_ufvs=True,
    include_regular_flood_lsr=False,
    include_wpc_pp=True,
    pp_expansion_radius_km=None,
    pp_smooth_radius_km=None,
    training_script_by_radius=None,
):
    """Build/load realtime predictions for all available radius members and verify once.

    The expensive RAP/FFG feature and model prediction steps are still handled by the
    existing single-radius functions. This wrapper only merges each available radius onto
    the same grid and then adds WPC/PP/UFVS once.
    """
    date = _date8(date)
    if model_specs is None:
        model_specs = REALTIME_DEFAULT_MODEL_SPECS if radii is None else [{"label": model_label_for_radius(r), "radius_km": int(r)} for r in radii]
    requested = [{"label": str(x["label"]), "radius_km": int(x["radius_km"])} for x in model_specs]
    training_script_by_radius = training_script_by_radius or {}

    base = None
    available = []
    missing = []

    for spec in requested:
        r = int(spec["radius_km"])
        model_label = str(spec["label"])
        try:
            pred = predict_realtime_case(
                date,
                radius_km=r,
                force=force_predict,
                force_features=force_features,
                training_script=training_script_by_radius.get(r),
                model_label=model_label,
            )
            pred = pred.copy()
            pred["Date"] = pred["Date"].astype(str).str.slice(0, 8)
            pcol = _model_prob_col(model_label, r)

            keep = [c for c in ["Date", "Year", "Lat", "Lon"] if c in pred.columns]
            if "Year" not in keep:
                pred["Year"] = date[:4]
                keep.append("Year")
            one = pred[keep].copy()
            one[pcol] = pd.to_numeric(pred["ML_Forecast_Prob"], errors="coerce").astype(np.float32)
            safe_label = re.sub(r"[^A-Za-z0-9]+", "_", model_label)
            one[f"ML_{safe_label}_Model_Path"] = pred.get("ML_Model_Path", "")
            one[f"ML_{safe_label}_Feature_Names_Path"] = pred.get("ML_Feature_Names_Path", "")

            if base is None:
                base = one
            else:
                base = _merge_grid_by_date_latlon(
                    base,
                    one,
                    columns_to_add=[pcol, f"ML_{safe_label}_Model_Path", f"ML_{safe_label}_Feature_Names_Path"],
                )
            available.append(model_label)
            print(f"Realtime model member {model_label} loaded: {len(pred):,} rows")
        except Exception as exc:
            missing.append((model_label, repr(exc)))
            print(f"Skipping realtime model {model_label}: {exc}")

    if base is None or not available:
        detail = "\n".join([f"  {label}: {err}" for label, err in missing])
        raise RuntimeError("No realtime radius members could be loaded.\n" + detail)

    base, radius_cols = add_radius_ensemble_products(base)

    if include_wpc_pp:
        base = merge_available_wpc_pp_for_realtime(base, date=date, force_wpc=force_wpc)
    if include_ufvs:
        pp_expansion_radius_km = REALTIME_PP_EXPANSION_RADIUS_KM if pp_expansion_radius_km is None else float(pp_expansion_radius_km)
        pp_smooth_radius_km = REALTIME_PP_SMOOTH_RADIUS_KM if pp_smooth_radius_km is None else float(pp_smooth_radius_km)
        base = add_ufvs_and_realtime_pp(
            base,
            date=date,
            force_ufvs=force_ufvs,
            include_regular_flood_lsr=include_regular_flood_lsr,
            pp_expansion_radius_km=pp_expansion_radius_km,
            pp_smooth_radius_km=pp_smooth_radius_km,
        )

    print("\nRealtime multi-radius availability")
    print("-" * 72)
    print(f"date: {date}")
    print(f"rows: {len(base):,}")
    print(f"requested models: {[x['label'] for x in requested]}")
    print(f"available models: {available}")
    if missing:
        print("missing radii:")
        for label, err in missing:
            print(f"  {label}: {err[:240]}")
    print(f"radius columns: {radius_cols}")
    print(f"WPC available: {WPC_COL in base.columns and pd.to_numeric(base.get(WPC_COL), errors='coerce').fillna(0).max() > 0}")
    print(f"PP columns: {[c for c in base.columns if c.startswith('PP_')]}")
    print(f"UFVS columns: {[c for c in base.columns if c.startswith('UFVS_')]}")

    summary_cols = radius_cols + ["ML_Radius_EnsMean", "ML_Radius_PMM", "ML_Radius_EnsMax"]
    if WPC_COL in base.columns:
        summary_cols.append(WPC_COL)
    print_probability_summary(base, summary_cols, label="Realtime multi-radius probability summary")

    cache_path = _multi_radius_cache_path(date, available)
    base.to_parquet(cache_path, index=False)
    print(f"Saved realtime multi-radius verified cache: {cache_path}")
    return base


def _plot_columns_as_risk_panels(
    df,
    panels,
    title_prefix="",
    smooth_sigma_grid=0,
    point_size=7,
    alpha=0.85,
    ncols=3,
    show=True,
):
    panels = [(t, c) for t, c in panels if c in df.columns]
    if not panels:
        raise RuntimeError("No requested panel columns are available to plot.")
    ncols = max(1, int(ncols))
    nrows = int(np.ceil(len(panels) / ncols))
    proj = ccrs.PlateCarree() if HAS_CARTOPY else None
    subplot_kw = {"projection": proj} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(7.6 * ncols, 5.4 * nrows),
        subplot_kw=subplot_kw,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()
    for ax, (title, col) in zip(axes, panels):
        full_title = f"{title_prefix}{title}" if title_prefix else title
        _plot_prob_or_smoothed(
            ax,
            df,
            col,
            full_title,
            smooth_sigma_grid=smooth_sigma_grid,
            point_size=point_size,
            alpha=alpha,
        )
    for ax in axes[len(panels):]:
        ax.set_visible(False)
    if show:
        plt.show()
        plt.close(fig)
        return None
    return fig


def plot_realtime_multi_radius_forecast_panels(
    date,
    df=None,
    radii=None,
    model_specs=None,
    include_ensemble=True,
    include_wpc=True,
    smooth_sigma_grid=0,
    point_size=7,
    alpha=0.85,
    ncols=3,
):
    """Forecast-focused realtime panel plot: radius members + ensemble products + WPC."""
    date = _date8(date)
    if df is None:
        df = build_predict_verify_realtime_multi_radius(date, radii=radii, model_specs=model_specs, include_ufvs=False, include_wpc_pp=True)
    sub = df[df["Date"].astype(str).str.slice(0, 8) == date].copy()
    if sub.empty:
        raise RuntimeError(f"No realtime rows for {date}.")

    panels = []
    if model_specs is None:
        model_specs = REALTIME_DEFAULT_MODEL_SPECS if radii is None else [{"label": model_label_for_radius(r), "radius_km": int(r)} for r in radii]
    for spec in model_specs:
        r = int(spec["radius_km"])
        model_label = str(spec["label"])
        c = _model_prob_col(model_label, r)
        if c in sub.columns and pd.to_numeric(sub[c], errors="coerce").notna().any():
            panels.append((f"ML {model_label}", c))
    if include_ensemble:
        for title, c in [
            ("Radius ensemble mean", "ML_Radius_EnsMean"),
            ("Radius PMM-like blend", "ML_Radius_PMM"),
            ("Radius ensemble max", "ML_Radius_EnsMax"),
            ("Radius spread", "ML_Radius_EnsSpread"),
        ]:
            if c in sub.columns:
                panels.append((title, c))
    if include_wpc and WPC_COL in sub.columns and pd.to_numeric(sub[WPC_COL], errors="coerce").fillna(0).max() > 0:
        panels.append(("WPC ERO", WPC_COL))
    elif include_wpc:
        print(f"No nonzero {WPC_COL} available for {date}; WPC panel omitted.")

    return _plot_columns_as_risk_panels(
        sub,
        panels,
        title_prefix=f"{date} | ",
        smooth_sigma_grid=smooth_sigma_grid,
        point_size=point_size,
        alpha=alpha,
        ncols=ncols,
        show=True,
    )


def plot_realtime_multi_radius_verification_panels(
    date,
    df=None,
    radii=None,
    model_specs=None,
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    ml_field="ML_Radius_PMM",
    smooth_sigma_grid=0,
    point_size=7,
    alpha=0.85,
    ncols=3,
):
    """Post-event panel plot: chosen ML aggregate/member + WPC + PP + UFVS/proxy."""
    date = _date8(date)
    if df is None:
        df = build_predict_verify_realtime_multi_radius(date, radii=radii, model_specs=model_specs, include_ufvs=True, include_wpc_pp=True)
    sub = df[df["Date"].astype(str).str.slice(0, 8) == date].copy()
    if sub.empty:
        raise RuntimeError(f"No realtime rows for {date}.")

    panels = []
    if ml_field in sub.columns:
        panels.append((f"ML: {ml_field}", ml_field))
    elif "ML_Radius_EnsMean" in sub.columns:
        panels.append(("ML: radius ensemble mean", "ML_Radius_EnsMean"))
    else:
        # Fall back to the smallest available radius member.
        radius_cols = sorted([c for c in sub.columns if re.match(r"^ML_r\d+_Prob$", c)], key=lambda c: int(re.search(r"r(\d+)", c).group(1)))
        if radius_cols:
            panels.append((f"ML: {radius_cols[0]}", radius_cols[0]))

    if WPC_COL in sub.columns and pd.to_numeric(sub[WPC_COL], errors="coerce").fillna(0).max() > 0:
        panels.append(("WPC ERO", WPC_COL))

    pp_col = _truth_col_from_pp_definition(pp_definition) if pp_definition else None
    if pp_col and pp_col in sub.columns and pd.to_numeric(sub[pp_col], errors="coerce").fillna(0).max() > 0:
        panels.append((f"Practically Perfect: {pp_definition}", pp_col))
    elif pp_col:
        print(f"No usable PP column {pp_col}; PP panel omitted. Available PP columns: {[c for c in sub.columns if c.startswith('PP_')]}")

    if proxy_col and proxy_col in sub.columns and pd.to_numeric(sub[proxy_col], errors="coerce").fillna(0).max() > 0:
        panels.append((f"UFVS/proxy: {proxy_col}", proxy_col))
    elif "UFVS_ANY" in sub.columns and pd.to_numeric(sub["UFVS_ANY"], errors="coerce").fillna(0).max() > 0:
        panels.append(("UFVS/proxy: UFVS_ANY", "UFVS_ANY"))
    else:
        print("No nonzero UFVS/proxy column available; proxy panel omitted.")

    return _plot_columns_as_risk_panels(
        sub,
        panels,
        title_prefix=f"{date} | ",
        smooth_sigma_grid=smooth_sigma_grid,
        point_size=point_size,
        alpha=alpha,
        ncols=ncols,
        show=True,
    )

print("Multi-radius realtime ensemble tools loaded.")
print("Use build_predict_verify_realtime_multi_radius(...) and plot_realtime_multi_radius_forecast_panels(...).")


In [ ]:
# ======================================================================================
# TARGETED FIX 1:
# Realtime giant-panel plot:
#   - remove PMM from realtime ensemble products
#   - show all ML radius members + ensemble mean/max/spread + WPC + PP + UFVS in one figure
#   - use risk colors for probabilities/risks
#   - use a separate diverging cmap + colorbar for spread
# ======================================================================================

import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

try:
    import cartopy.crs as ccrs
    HAS_CARTOPY_LOCAL = True
except Exception:
    HAS_CARTOPY_LOCAL = False


# -----------------------------
# User controls
# -----------------------------
REALTIME_GIANT_FIGSIZE_PER_PANEL = (6.3, 4.8)
REALTIME_GIANT_NCOLS = 4
REALTIME_GIANT_POINT_SIZE = 7
REALTIME_GIANT_ALPHA = 0.85
REALTIME_GIANT_SMOOTH_SIGMA_GRID = 0

# Spread should not use risk colors.
REALTIME_SPREAD_CMAP = "coolwarm"     # diverging; neutral near the center
REALTIME_SPREAD_VCENTER_MODE = "mid"  # "mid", "median", or numeric value
REALTIME_SPREAD_POINT_SIZE = 8
REALTIME_SPREAD_ALPHA = 0.90

# Choose which PP/UFVS columns to show.
# Use "all" to show every PP_* / UFVS_* column that exists.
REALTIME_GIANT_PP_COLUMNS = "all"
REALTIME_GIANT_UFVS_COLUMNS = "all"

# Set to False if you do not want to save the giant figure.
REALTIME_GIANT_SAVE_FIG = True
REALTIME_GIANT_DPI = 220
REALTIME_GIANT_OUTDIR = Path(PROJECT_DIR) / "realtime_giant_figures_v33day2valid_no_pmm"


def add_radius_ensemble_products(df, radius_cols=None):
    """
    Override previous version:
      - keeps ensemble mean, median, max, spread
      - removes PMM
      - ML_Forecast_Prob compatibility column = ensemble mean
    """
    out = df.copy()

    if radius_cols is None:
        radius_cols = sorted(
            [c for c in out.columns if _wide_ml_member_metadata(c)[0] is not None],
            key=lambda c: (_wide_ml_member_metadata(c)[1], _wide_ml_member_metadata(c)[0]),
        )

    radius_cols = [c for c in radius_cols if c in out.columns]
    if not radius_cols:
        print("No ML radius probability columns found; ensemble products not created.")
        return out, []

    arr = out[radius_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)

    out["ML_Radius_EnsMean"] = np.nanmean(arr, axis=1).astype(np.float32)
    out["ML_Radius_EnsMedian"] = np.nanmedian(arr, axis=1).astype(np.float32)
    out["ML_Radius_EnsMax"] = np.nanmax(arr, axis=1).astype(np.float32)
    out["ML_Radius_EnsSpread"] = np.nanstd(arr, axis=1).astype(np.float32)

    # Remove PMM if an older cached dataframe already has it.
    if "ML_Radius_PMM" in out.columns:
        out = out.drop(columns=["ML_Radius_PMM"])

    # Compatibility for older metric/plot cells that expect ML_Forecast_Prob.
    out["ML_Forecast_Prob"] = out["ML_Radius_EnsMean"].astype(np.float32)
    out["ML_Target_Radius_km"] = -1

    return out, radius_cols


def _pretty_realtime_panel_label(col):
    label_map = {
        "ML_Radius_EnsMean": "ML Radius Ensemble Mean",
        "ML_Radius_EnsMedian": "ML Radius Ensemble Median",
        "ML_Radius_EnsMax": "ML Radius Ensemble Max",
        "ML_Radius_EnsSpread": "ML Radius Ensemble Spread",
        "WPC_ERO_Risk": "WPC ERO",
        "UFVS_ANY": "Any Flood Proxy",
        "UFVS_STAGE4_FFG": "Stage IV > FFG",
        "UFVS_STAGE4_ARI": "Stage IV ARI",
        "UFVS_USGS": "USGS Streamflow",
        "UFVS_LSR_FLASH": "Flash Flood LSR",
        "UFVS_LSR_REGULAR": "Flood LSR",
        "UFVS_LSR_FLOOD": "Flood LSR",
        "PP_Any flood proxy": "Practically Perfect: Any Flood Proxy",
        "PP_MRMS > FFG": "Practically Perfect: MRMS > FFG",
        "PP_Stage IV > FFG": "Practically Perfect: Stage IV > FFG",
        "PP_Stage IV ARI": "Practically Perfect: Stage IV ARI",
        "PP_LSR/USGS only": "Practically Perfect: LSR/USGS Only",
        "PP_USGS": "Practically Perfect: USGS",
        "PP_Flash LSR": "Practically Perfect: Flash Flood LSR",
    }

    if col in label_map:
        return label_map[col]

    m = re.match(r"ML_r(\d+)_Prob", str(col))
    if m:
        return f"ML r{m.group(1)} km"

    if str(col).startswith("PP_"):
        return "Practically Perfect: " + str(col).replace("PP_", "").replace("_", " ")

    if str(col).startswith("UFVS_"):
        return str(col).replace("UFVS_", "").replace("_", " ").title()

    return str(col).replace("_", " ")


def _available_requested_columns(df, requested):
    if requested == "all":
        return None
    if requested is None:
        return []
    return [c for c in requested if c in df.columns]


def _setup_realtime_axis(ax, title):
    _setup_map_ax(ax, DEFAULT_EXTENT, show_states=True, show_coastline=True)
    ax.set_title(title)


def _plot_spread_panel_with_colorbar(
    ax,
    df,
    col="ML_Radius_EnsSpread",
    title="ML Radius Ensemble Spread",
    cmap=REALTIME_SPREAD_CMAP,
    point_size=REALTIME_SPREAD_POINT_SIZE,
    alpha=REALTIME_SPREAD_ALPHA,
):
    _setup_realtime_axis(ax, title)

    lon = pd.to_numeric(df["Lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(df["Lat"], errors="coerce").to_numpy(float)
    vals = pd.to_numeric(df[col], errors="coerce").to_numpy(float)

    good = np.isfinite(lon) & np.isfinite(lat) & np.isfinite(vals)
    if not np.any(good):
        ax.text(0.5, 0.5, f"No finite {col}", transform=ax.transAxes, ha="center", va="center")
        return None

    vmin = float(np.nanmin(vals[good]))
    vmax = float(np.nanmax(vals[good]))

    if np.isclose(vmin, vmax):
        vcenter = vmin
        norm = None
    else:
        if REALTIME_SPREAD_VCENTER_MODE == "median":
            vcenter = float(np.nanmedian(vals[good]))
        elif isinstance(REALTIME_SPREAD_VCENTER_MODE, (int, float)):
            vcenter = float(REALTIME_SPREAD_VCENTER_MODE)
        else:
            vcenter = 0.5 * (vmin + vmax)

        # Keep vcenter strictly inside range for TwoSlopeNorm.
        vcenter = min(max(vcenter, vmin + 1e-8), vmax - 1e-8)
        norm = TwoSlopeNorm(vmin=vmin, vcenter=vcenter, vmax=vmax)

    transform = ccrs.PlateCarree() if HAS_CARTOPY else None
    if HAS_CARTOPY:
        sc = ax.scatter(
            lon[good], lat[good],
            c=vals[good],
            s=point_size,
            alpha=alpha,
            cmap=cmap,
            norm=norm,
            transform=transform,
            linewidths=0,
        )
    else:
        sc = ax.scatter(
            lon[good], lat[good],
            c=vals[good],
            s=point_size,
            alpha=alpha,
            cmap=cmap,
            norm=norm,
            linewidths=0,
        )

    cbar = plt.colorbar(sc, ax=ax, shrink=0.78, pad=0.02)
    cbar.set_label("Ensemble spread (probability std. dev.)")
    return sc


def _plot_risk_panel(ax, df, col, title, smooth_sigma_grid=0, point_size=7, alpha=0.85):
    if col not in df.columns:
        _setup_realtime_axis(ax, title)
        ax.text(0.5, 0.5, f"Missing column:\n{col}", transform=ax.transAxes, ha="center", va="center")
        return None

    vals = pd.to_numeric(df[col], errors="coerce")
    if vals.notna().sum() == 0:
        _setup_realtime_axis(ax, title)
        ax.text(0.5, 0.5, f"No finite data:\n{col}", transform=ax.transAxes, ha="center", va="center")
        return None

    return _plot_prob_or_smoothed(
        ax,
        df,
        col,
        title,
        smooth_sigma_grid=smooth_sigma_grid,
        point_size=point_size,
        alpha=alpha,
    )


def plot_realtime_giant_all_members_wpc_pp_ufvs(
    date=None,
    df=None,
    radii=None,
    pp_columns=REALTIME_GIANT_PP_COLUMNS,
    ufvs_columns=REALTIME_GIANT_UFVS_COLUMNS,
    ncols=REALTIME_GIANT_NCOLS,
    figsize_per_panel=REALTIME_GIANT_FIGSIZE_PER_PANEL,
    smooth_sigma_grid=REALTIME_GIANT_SMOOTH_SIGMA_GRID,
    point_size=REALTIME_GIANT_POINT_SIZE,
    alpha=REALTIME_GIANT_ALPHA,
    save=REALTIME_GIANT_SAVE_FIG,
    outdir=REALTIME_GIANT_OUTDIR,
):
    """
    Giant realtime figure with:
      - all available ML radius members
      - ensemble mean
      - ensemble max
      - ensemble spread with separate diverging colorbar
      - WPC ERO
      - PP columns
      - UFVS columns

    PMM is intentionally omitted.
    """
    if df is None:
        if "df_realtime_viewer" not in globals():
            raise RuntimeError("df_realtime_viewer does not exist. Run the realtime build cell first.")
        df = df_realtime_viewer.copy()
    else:
        df = df.copy()

    if date is None:
        if "REALTIME_DATE" in globals():
            date = REALTIME_DATE
        elif "Date" in df.columns:
            date = str(df["Date"].iloc[0])[:8]
        else:
            date = "realtime"

    date8 = _date8(date) if "_date8" in globals() else str(date)[:8]
    if "Date" in df.columns:
        sub = df[df["Date"].astype(str).str[:8] == date8].copy()
        if sub.empty:
            print(f"No rows matched date {date8}; plotting all rows in dataframe.")
            sub = df.copy()
    else:
        sub = df.copy()

    # Ensure no PMM column is carried into plotting from old caches.
    if "ML_Radius_PMM" in sub.columns:
        sub = sub.drop(columns=["ML_Radius_PMM"])

    if radii is None:
        radius_cols = sorted(
            [c for c in sub.columns if re.match(r"^ML_r\d+_Prob$", str(c))],
            key=lambda c: int(re.search(r"r(\d+)", c).group(1)),
        )
    else:
        radius_cols = [_radius_prob_col(r) for r in radii if _radius_prob_col(r) in sub.columns]

    panels = []
    for c in radius_cols:
        panels.append((_pretty_realtime_panel_label(c), c, "risk"))

    for c in ["ML_Radius_EnsMean", "ML_Radius_EnsMax"]:
        if c in sub.columns:
            panels.append((_pretty_realtime_panel_label(c), c, "risk"))

    if "ML_Radius_EnsSpread" in sub.columns:
        panels.append((_pretty_realtime_panel_label("ML_Radius_EnsSpread"), "ML_Radius_EnsSpread", "spread"))

    if WPC_COL in sub.columns:
        panels.append((_pretty_realtime_panel_label(WPC_COL), WPC_COL, "risk"))

    if pp_columns == "all":
        pp_cols = [c for c in sub.columns if str(c).startswith("PP_")]
    else:
        pp_cols = _available_requested_columns(sub, pp_columns)
    pp_cols = [c for c in pp_cols if pd.to_numeric(sub[c], errors="coerce").fillna(0).max() > 0]
    for c in pp_cols:
        panels.append((_pretty_realtime_panel_label(c), c, "risk"))

    if ufvs_columns == "all":
        ufvs_cols = [c for c in sub.columns if str(c).startswith("UFVS_")]
    else:
        ufvs_cols = _available_requested_columns(sub, ufvs_columns)
    ufvs_cols = [c for c in ufvs_cols if pd.to_numeric(sub[c], errors="coerce").fillna(0).max() > 0]
    for c in ufvs_cols:
        panels.append((_pretty_realtime_panel_label(c), c, "risk"))

    if not panels:
        raise RuntimeError("No columns available for giant realtime panel plot.")

    ncols = max(1, int(ncols))
    nrows = int(math.ceil(len(panels) / ncols))

    proj = ccrs.PlateCarree() if HAS_CARTOPY else None
    subplot_kw = {"projection": proj} if HAS_CARTOPY else {}

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=(figsize_per_panel[0] * ncols, figsize_per_panel[1] * nrows),
        subplot_kw=subplot_kw,
        constrained_layout=True,
    )
    axes = np.atleast_1d(axes).ravel()

    for ax, (title, col, kind) in zip(axes, panels):
        if kind == "spread":
            _plot_spread_panel_with_colorbar(ax, sub, col=col, title=f"{date8} | {title}")
        else:
            _plot_risk_panel(
                ax,
                sub,
                col,
                title=f"{date8} | {title}",
                smooth_sigma_grid=smooth_sigma_grid,
                point_size=point_size,
                alpha=alpha,
            )

    for ax in axes[len(panels):]:
        ax.set_visible(False)

    if save:
        outdir = Path(outdir)
        outdir.mkdir(parents=True, exist_ok=True)
        outpath = outdir / f"realtime_giant_members_wpc_pp_ufvs_{date8}.png"
        fig.savefig(outpath, dpi=int(REALTIME_GIANT_DPI), bbox_inches="tight")
        print("Saved:", outpath)

    plt.show()
    return fig


# Optional: re-add ensemble products without PMM to existing realtime dataframe, then plot.
if "df_realtime_viewer" in globals():
    df_realtime_viewer, _radius_cols_no_pmm = add_radius_ensemble_products(df_realtime_viewer)
    plot_realtime_giant_all_members_wpc_pp_ufvs(
        date=REALTIME_DATE if "REALTIME_DATE" in globals() else None,
        df=df_realtime_viewer,
        radii=REALTIME_RADII_KM if "REALTIME_RADII_KM" in globals() else None,
    )
else:
    print("Loaded plotting functions. Run your realtime build cell first, then call plot_realtime_giant_all_members_wpc_pp_ufvs(...).")

In [ ]:
# ======================================================================================
# PAPER-READY REALTIME PROBABILITY MAP FIGURE
#
# This replaces the realtime viewer build + plotting workflow.
#
# It:
#   1. Rebuilds df_realtime_viewer
#   2. Computes only ML ensemble mean
#   3. Plots:
#        Row 1: ML r40, ML r60, ML r75, ML r100
#        Row 2: ML ensemble mean, WPC ERO, Practically Perfect Any, UFVS Any
#
# It removes:
#   - Radius spread
#   - Ensemble max
#   - PMM
#
# Formatting:
#   - Font size 50 everywhere
#   - One shared row legend on the far right
#   - a), b), c), ... panel labels in upper-left of each subplot
#   - Only plots probability/risk values >= 5%
#   - Saves PNG and PDF
# ======================================================================================

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY_RT_PAPER = True
except Exception:
    HAS_CARTOPY_RT_PAPER = False


# ======================================================================================
# REALTIME BUILD CONTROLS
# ======================================================================================

REALTIME_DATE = "20260702"

REALTIME_MODEL_SPECS = [dict(spec) for spec in MODEL_SPECS]

FORCE_REALTIME_PREDICT = False
FORCE_REALTIME_FEATURES = False
FORCE_REALTIME_WPC = False
FORCE_REALTIME_UFVS = False

INCLUDE_REGULAR_FLOOD_LSR = False

REALTIME_PP_EXPANSION_RADIUS_KM = 40.0
REALTIME_PP_SMOOTH_RADIUS_KM = 100.0


# ======================================================================================
# PAPER FIGURE CONTROLS
# ======================================================================================

PAPER_RT_FONT_SIZE = 50
PAPER_RT_PANEL_LABEL_SIZE = 50
PAPER_RT_LEGEND_SIZE = 42
PAPER_RT_TITLE_SIZE = 50

PAPER_RT_POINT_SIZE = 32
PAPER_RT_ALPHA = 0.88

PAPER_RT_UFVS_BACKGROUND_SIZE = 8
PAPER_RT_UFVS_BACKGROUND_ALPHA = 0.10
PAPER_RT_UFVS_EVENT_SIZE = 46
PAPER_RT_UFVS_EVENT_ALPHA = 0.95

PAPER_RT_FIGSIZE = (64, 30)
PAPER_RT_DPI = 300
PAPER_RT_SAVE_FIG = True

if "PROJECT_DIR" in globals():
    PAPER_RT_OUTDIR = Path(PROJECT_DIR) / "paper_realtime_probability_maps_v33day2valid"
else:
    PAPER_RT_OUTDIR = Path.cwd() / "paper_realtime_probability_maps_v33day2valid"

# Probability/risk bins
PAPER_RT_MIN_PROB = 0.05

PAPER_RT_RISK_BOUNDS = [0.05, 0.15, 0.40, 0.70, 1.01]
PAPER_RT_RISK_COLORS = [
    "#76E34E",  # 5-15%
    "#FFFB00",  # 15-40%
    "#DE3E2C",  # 40-70%
    "#FF5BFF",  # >=70%
]
PAPER_RT_RISK_LABELS = [
    "5–15%",
    "15–40%",
    "40–70%",
    "≥70%",
]

PAPER_RT_CMAP = ListedColormap(PAPER_RT_RISK_COLORS)
PAPER_RT_NORM = BoundaryNorm(PAPER_RT_RISK_BOUNDS, PAPER_RT_CMAP.N, clip=True)

PAPER_RT_PANEL_TITLES = {
    "ML_r40_Prob": "ML r40 km",
    "ML_r60_Prob": "ML r60 km",
    "ML_r75_Prob": "ML r75 km",
    "ML_r100_Prob": "ML r100 km",
    "ML_Radius_EnsMean": "ML Ensemble Mean",
    "WPC_ERO_Risk": "WPC ERO",
    "PP_ANY": "Practically Perfect",
    "UFVS_ANY": "UFVS Any Flood Proxy",
}


# ======================================================================================
# FONT SETTINGS
# ======================================================================================

plt.rcParams.update({
    "font.size": PAPER_RT_FONT_SIZE,
    "axes.titlesize": PAPER_RT_TITLE_SIZE,
    "axes.labelsize": PAPER_RT_FONT_SIZE,
    "xtick.labelsize": PAPER_RT_FONT_SIZE,
    "ytick.labelsize": PAPER_RT_FONT_SIZE,
    "legend.fontsize": PAPER_RT_LEGEND_SIZE,
    "figure.titlesize": PAPER_RT_FONT_SIZE,
})


# ======================================================================================
# BUILD df_realtime_viewer
# ======================================================================================

PAPER_RT_REQUIRED_ML_COLS = [
    "ML_r40_Prob",
    "ML_r60_Prob",
    "ML_r75_Prob",
    "ML_r100_Prob",
]


def _rt_ml_prob_columns_present(df):
    """Return all ML probability-like columns for diagnostics."""
    return [
        c for c in df.columns
        if str(c).startswith("ML_") and str(c).endswith("_Prob")
    ]


def _rt_radius_prob_cols(df):
    """
    Return the four paper-figure ML members exactly once and in plot order.
    """
    return [c for c in PAPER_RT_REQUIRED_ML_COLS if c in df.columns]


def _rt_recompute_paper_ensemble_mean_only(df):
    """
    Keep only products needed for paper figure:
      - ML_Radius_EnsMean

    Removes:
      - ML_Radius_PMM
      - ML_Radius_EnsMax
      - ML_Radius_EnsSpread
    """
    out = df.copy()

    drop_cols = [
        c for c in ["ML_Radius_PMM", "ML_Radius_EnsMax", "ML_Radius_EnsSpread"]
        if c in out.columns
    ]
    if drop_cols:
        out = out.drop(columns=drop_cols)

    member_cols = _rt_radius_prob_cols(out)

    missing_members = [c for c in PAPER_RT_REQUIRED_ML_COLS if c not in out.columns]
    if missing_members:
        available = _rt_ml_prob_columns_present(out)
        raise RuntimeError(
            "Cannot create the four-member ML ensemble mean because these "
            f"columns are missing: {missing_members}. "
            f"Available ML probability columns are: {available}"
        )

    arr = out[member_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    out["ML_Radius_EnsMean"] = np.nanmean(arr, axis=1).astype(np.float32)

    # Compatibility if some existing helpers expect this
    out["ML_Forecast_Prob"] = out["ML_Radius_EnsMean"].astype(np.float32)

    return out, member_cols


if "build_predict_verify_realtime_multi_radius" not in globals():
    raise RuntimeError(
        "build_predict_verify_realtime_multi_radius is not defined. "
        "Run the realtime helper-definition cells above this point first."
    )

df_realtime_viewer = build_predict_verify_realtime_multi_radius(
    REALTIME_DATE,
    model_specs=REALTIME_MODEL_SPECS,
    force_predict=FORCE_REALTIME_PREDICT,
    force_features=FORCE_REALTIME_FEATURES,
    force_wpc=FORCE_REALTIME_WPC,
    force_ufvs=FORCE_REALTIME_UFVS,
    include_ufvs=True,
    include_wpc_pp=True,
    include_regular_flood_lsr=INCLUDE_REGULAR_FLOOD_LSR,
    pp_expansion_radius_km=REALTIME_PP_EXPANSION_RADIUS_KM,
    pp_smooth_radius_km=REALTIME_PP_SMOOTH_RADIUS_KM,
)

df_realtime_viewer, _rt_member_cols = _rt_recompute_paper_ensemble_mean_only(df_realtime_viewer)

print("df_realtime_viewer created")
print("Rows:", len(df_realtime_viewer))
print("ML member columns:", _rt_member_cols)
print("Paper ensemble columns:", [c for c in ["ML_Radius_EnsMean"] if c in df_realtime_viewer.columns])
print("WPC columns:", [c for c in df_realtime_viewer.columns if c == "WPC_ERO_Risk"])
print("PP Any columns:", [c for c in df_realtime_viewer.columns if str(c).startswith("PP_") and "any" in str(c).lower()])
print("UFVS Any columns:", [c for c in df_realtime_viewer.columns if c == "UFVS_ANY"])


# ======================================================================================
# PAPER PLOT HELPERS
# ======================================================================================

def _paper_rt_date8(x):
    if x is None:
        return None
    return str(x)[:10].replace("-", "")[:8]


def _paper_rt_find_pp_any_col(df):
    candidates = [
        "PP_Any flood proxy",
        "PP_Any Flood Proxy",
        "PP_ANY",
        "PP_UFVS_ANY",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    any_cols = [
        c for c in df.columns
        if str(c).startswith("PP_") and "any" in str(c).lower()
    ]

    return any_cols[0] if any_cols else None


def _paper_rt_normalize_probability_values(series):
    """
    Converts common probability/risk encodings to 0-1 probabilities.

    Handles:
      - already 0-1 probabilities
      - percent values like 5, 15, 40, 70
      - categorical integers 0, 1, 2, 3, 4
      - risk strings such as Marginal/Slight/Moderate/High
    """
    s = pd.Series(series)

    vals = pd.to_numeric(s, errors="coerce").to_numpy(float)

    if np.isfinite(vals).any():
        finite = vals[np.isfinite(vals)]
        vmax = float(np.nanmax(finite))
        unique = set(np.unique(finite).astype(float).tolist())

        # WPC-style integer categories
        if vmax <= 4.0 and unique.issubset({0.0, 1.0, 2.0, 3.0, 4.0}):
            mapped = np.full(vals.shape, np.nan, dtype=float)
            mapped[vals == 0] = 0.0
            mapped[vals == 1] = 0.05
            mapped[vals == 2] = 0.15
            mapped[vals == 3] = 0.40
            mapped[vals == 4] = 0.70
            return mapped

        # Percent values
        if vmax > 1.0:
            return vals / 100.0

        return vals

    lower = s.astype(str).str.lower().str.strip()
    mapped = np.full(len(s), np.nan, dtype=float)

    mapped[lower.str.contains("marginal", na=False)] = 0.05
    mapped[lower.str.contains("slight", na=False)] = 0.15
    mapped[lower.str.contains("moderate", na=False)] = 0.40
    mapped[lower.str.contains("high", na=False)] = 0.70

    return mapped


def _paper_rt_get_extent(df):
    if "DEFAULT_EXTENT" in globals():
        return DEFAULT_EXTENT

    lon = pd.to_numeric(df["Lon"], errors="coerce")
    lat = pd.to_numeric(df["Lat"], errors="coerce")

    return [
        float(lon.quantile(0.01)),
        float(lon.quantile(0.99)),
        float(lat.quantile(0.01)),
        float(lat.quantile(0.99)),
    ]


def _paper_rt_setup_map_axis(ax, df):
    extent = _paper_rt_get_extent(df)

    if HAS_CARTOPY_RT_PAPER:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

        try:
            ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=1.8)
        except Exception:
            try:
                ax.coastlines(resolution="50m", linewidth=1.8)
            except Exception:
                pass

        try:
            ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=1.4)
        except Exception:
            pass

        try:
            ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=1.0, edgecolor="0.35")
        except Exception:
            pass

    else:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.grid(True, alpha=0.20)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(
        labelbottom=False,
        labelleft=False,
        bottom=False,
        left=False,
    )


def _paper_rt_add_panel_label(ax, label):
    ax.text(
        0.025,
        0.965,
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=PAPER_RT_PANEL_LABEL_SIZE,
        fontweight="bold",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.84,
            boxstyle="round,pad=0.15",
        ),
        zorder=100,
    )


def _paper_rt_plot_probability_panel(ax, df, col, title, panel_label):
    _paper_rt_setup_map_axis(ax, df)

    lon = pd.to_numeric(df["Lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(df["Lat"], errors="coerce").to_numpy(float)
    vals = _paper_rt_normalize_probability_values(df[col])

    good = (
        np.isfinite(lon)
        & np.isfinite(lat)
        & np.isfinite(vals)
        & (vals >= PAPER_RT_MIN_PROB)
    )

    if np.any(good):
        scatter_kwargs = dict(
            c=np.clip(vals[good], PAPER_RT_MIN_PROB, 1.0),
            s=PAPER_RT_POINT_SIZE,
            alpha=PAPER_RT_ALPHA,
            cmap=PAPER_RT_CMAP,
            norm=PAPER_RT_NORM,
            linewidths=0,
            rasterized=True,
        )

        if HAS_CARTOPY_RT_PAPER:
            scatter_kwargs["transform"] = ccrs.PlateCarree()

        ax.scatter(lon[good], lat[good], **scatter_kwargs)

    else:
        ax.text(
            0.5,
            0.5,
            "No values ≥5%",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=38,
        )

    ax.set_title(title, pad=18)
    _paper_rt_add_panel_label(ax, panel_label)


def _paper_rt_plot_ufvs_panel(ax, df, col, title, panel_label):
    _paper_rt_setup_map_axis(ax, df)

    lon = pd.to_numeric(df["Lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(df["Lat"], errors="coerce").to_numpy(float)
    vals = pd.to_numeric(df[col], errors="coerce").fillna(0).to_numpy(float)

    good = np.isfinite(lon) & np.isfinite(lat)
    event = good & (vals > 0)

    bg_kwargs = dict(
        s=PAPER_RT_UFVS_BACKGROUND_SIZE,
        c="0.78",
        alpha=PAPER_RT_UFVS_BACKGROUND_ALPHA,
        linewidths=0,
        rasterized=True,
    )

    ev_kwargs = dict(
        s=PAPER_RT_UFVS_EVENT_SIZE,
        c="black",
        alpha=PAPER_RT_UFVS_EVENT_ALPHA,
        linewidths=0,
        rasterized=True,
    )

    if HAS_CARTOPY_RT_PAPER:
        bg_kwargs["transform"] = ccrs.PlateCarree()
        ev_kwargs["transform"] = ccrs.PlateCarree()

    if np.any(good):
        ax.scatter(lon[good], lat[good], **bg_kwargs)

    if np.any(event):
        ax.scatter(lon[event], lat[event], **ev_kwargs)
    else:
        ax.text(
            0.5,
            0.5,
            "No UFVS events",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=38,
        )

    ax.set_title(title, pad=18)
    _paper_rt_add_panel_label(ax, panel_label)


def _paper_rt_risk_legend_handles():
    return [
        Patch(
            facecolor=color,
            edgecolor="black",
            linewidth=1.5,
            label=label,
        )
        for color, label in zip(PAPER_RT_RISK_COLORS, PAPER_RT_RISK_LABELS)
    ]


def _paper_rt_draw_row1_legend(ax):
    ax.axis("off")

    ax.legend(
        handles=_paper_rt_risk_legend_handles(),
        title="Probability",
        loc="center left",
        frameon=True,
        framealpha=0.96,
        fontsize=PAPER_RT_LEGEND_SIZE,
        title_fontsize=PAPER_RT_LEGEND_SIZE,
        borderpad=0.8,
        labelspacing=0.9,
        handlelength=1.5,
        handleheight=1.2,
    )


def _paper_rt_draw_row2_legend(ax):
    ax.axis("off")

    risk_legend = ax.legend(
        handles=_paper_rt_risk_legend_handles(),
        title="Probability",
        loc="upper left",
        bbox_to_anchor=(0.0, 0.98),
        frameon=True,
        framealpha=0.96,
        fontsize=PAPER_RT_LEGEND_SIZE,
        title_fontsize=PAPER_RT_LEGEND_SIZE,
        borderpad=0.8,
        labelspacing=0.9,
        handlelength=1.5,
        handleheight=1.2,
    )
    ax.add_artist(risk_legend)

    ufvs_handle = Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        color="black",
        markerfacecolor="black",
        markeredgecolor="black",
        markersize=24,
        label="UFVS event",
    )

    ax.legend(
        handles=[ufvs_handle],
        title="Observed\nproxy",
        loc="lower left",
        bbox_to_anchor=(0.0, 0.03),
        frameon=True,
        framealpha=0.96,
        fontsize=PAPER_RT_LEGEND_SIZE,
        title_fontsize=PAPER_RT_LEGEND_SIZE,
        borderpad=0.8,
        labelspacing=0.9,
    )


def plot_paper_realtime_probability_maps(df, date):
    date8 = _paper_rt_date8(date)

    if "Date" in df.columns and date8 is not None:
        sub = df[df["Date"].astype(str).str[:8] == date8].copy()
        if sub.empty:
            print(f"No rows matched date {date8}; plotting all rows.")
            sub = df.copy()
    else:
        sub = df.copy()

    sub, member_cols = _rt_recompute_paper_ensemble_mean_only(sub)

    row1_cols = list(PAPER_RT_REQUIRED_ML_COLS)

    missing_row1 = [c for c in row1_cols if c not in sub.columns]
    if missing_row1:
        raise RuntimeError(f"Missing required ML member columns: {missing_row1}")

    pp_any_col = _paper_rt_find_pp_any_col(sub)
    if pp_any_col is None:
        raise RuntimeError("No PP Any Flood Proxy column found.")

    row2_items = [
        ("prob", "ML_Radius_EnsMean", PAPER_RT_PANEL_TITLES["ML_Radius_EnsMean"]),
        ("prob", "WPC_ERO_Risk", PAPER_RT_PANEL_TITLES["WPC_ERO_Risk"]),
        ("prob", pp_any_col, PAPER_RT_PANEL_TITLES["PP_ANY"]),
        ("ufvs", "UFVS_ANY", PAPER_RT_PANEL_TITLES["UFVS_ANY"]),
    ]

    missing_row2 = [col for kind, col, title in row2_items if col not in sub.columns]
    if missing_row2:
        raise RuntimeError(f"Missing required row-2 columns: {missing_row2}")

    projection = ccrs.PlateCarree() if HAS_CARTOPY_RT_PAPER else None

    fig = plt.figure(figsize=PAPER_RT_FIGSIZE)

    gs = fig.add_gridspec(
        nrows=2,
        ncols=5,
        width_ratios=[1, 1, 1, 1, 0.40],
        height_ratios=[1, 1],
        wspace=0.055,
        hspace=0.18,
    )

    row1_axes = []
    for col in range(4):
        if HAS_CARTOPY_RT_PAPER:
            ax = fig.add_subplot(gs[0, col], projection=projection)
        else:
            ax = fig.add_subplot(gs[0, col])
        row1_axes.append(ax)

    row2_axes = []
    for col in range(4):
        if HAS_CARTOPY_RT_PAPER:
            ax = fig.add_subplot(gs[1, col], projection=projection)
        else:
            ax = fig.add_subplot(gs[1, col])
        row2_axes.append(ax)

    legend_ax_row1 = fig.add_subplot(gs[0, 4])
    legend_ax_row2 = fig.add_subplot(gs[1, 4])

    panel_labels = [f"{letter})" for letter in string.ascii_lowercase]

    # Row 1: individual ML radii
    for i, col in enumerate(row1_cols):
        _paper_rt_plot_probability_panel(
            row1_axes[i],
            sub,
            col=col,
            title=PAPER_RT_PANEL_TITLES[col],
            panel_label=panel_labels[i],
        )

    # Row 2: ensemble mean, WPC, PP, UFVS
    for j, (kind, col, title) in enumerate(row2_items):
        ax = row2_axes[j]
        panel_label = panel_labels[len(row1_cols) + j]

        if kind == "ufvs":
            _paper_rt_plot_ufvs_panel(
                ax,
                sub,
                col=col,
                title=title,
                panel_label=panel_label,
            )
        else:
            _paper_rt_plot_probability_panel(
                ax,
                sub,
                col=col,
                title=title,
                panel_label=panel_label,
            )

    # One shared legend per row on the far right
    _paper_rt_draw_row1_legend(legend_ax_row1)
    _paper_rt_draw_row2_legend(legend_ax_row2)

    if PAPER_RT_SAVE_FIG:
        PAPER_RT_OUTDIR.mkdir(parents=True, exist_ok=True)

        png_path = PAPER_RT_OUTDIR / f"paper_realtime_probability_maps_{date8}.png"
        pdf_path = PAPER_RT_OUTDIR / f"paper_realtime_probability_maps_{date8}.pdf"

        fig.savefig(png_path, dpi=int(PAPER_RT_DPI), bbox_inches="tight")
        fig.savefig(pdf_path, dpi=int(PAPER_RT_DPI), bbox_inches="tight")

        print("Saved:", png_path)
        print("Saved:", pdf_path)

    plt.show()
    return fig


fig_paper_realtime_probability_maps = plot_paper_realtime_probability_maps(
    df=df_realtime_viewer,
    date=REALTIME_DATE,
)

### 5cb. ML/WPC PP Risk Confusion Matrix

In [ ]:
# ======================================================================================
# PAPER-READY REALTIME PROBABILITY MAP FIGURE
#
# This replaces the realtime viewer build + plotting workflow.
#
# It:
#   1. Rebuilds df_realtime_viewer
#   2. Computes only ML ensemble mean
#   3. Plots:
#        Row 1: ML r40, ML r60, ML r75, ML r100
#        Row 2: ML ensemble mean, WPC ERO, Practically Perfect Any, UFVS Any
#
# It removes:
#   - Radius spread
#   - Ensemble max
#   - PMM
#
# Formatting:
#   - Font size 50 everywhere
#   - One shared row legend on the far right
#   - a), b), c), ... panel labels in upper-left of each subplot
#   - Only plots probability/risk values >= 5%
#   - Saves PNG and PDF
# ======================================================================================

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY_RT_PAPER = True
except Exception:
    HAS_CARTOPY_RT_PAPER = False


# ======================================================================================
# REALTIME BUILD CONTROLS
# ======================================================================================

REALTIME_DATE = "20260709"

REALTIME_MODEL_SPECS = [dict(spec) for spec in MODEL_SPECS]

FORCE_REALTIME_PREDICT = False
FORCE_REALTIME_FEATURES = False
FORCE_REALTIME_WPC = False
FORCE_REALTIME_UFVS = False

INCLUDE_REGULAR_FLOOD_LSR = False

REALTIME_PP_EXPANSION_RADIUS_KM = 40.0
REALTIME_PP_SMOOTH_RADIUS_KM = 100.0


# ======================================================================================
# PAPER FIGURE CONTROLS
# ======================================================================================

PAPER_RT_FONT_SIZE = 50
PAPER_RT_PANEL_LABEL_SIZE = 50
PAPER_RT_LEGEND_SIZE = 42
PAPER_RT_TITLE_SIZE = 50

PAPER_RT_POINT_SIZE = 32
PAPER_RT_ALPHA = 0.88

PAPER_RT_UFVS_BACKGROUND_SIZE = 8
PAPER_RT_UFVS_BACKGROUND_ALPHA = 0.10
PAPER_RT_UFVS_EVENT_SIZE = 46
PAPER_RT_UFVS_EVENT_ALPHA = 0.95

PAPER_RT_FIGSIZE = (64, 30)
PAPER_RT_DPI = 300
PAPER_RT_SAVE_FIG = True

if "PROJECT_DIR" in globals():
    PAPER_RT_OUTDIR = Path(PROJECT_DIR) / "paper_realtime_probability_maps_v33day2valid"
else:
    PAPER_RT_OUTDIR = Path.cwd() / "paper_realtime_probability_maps_v33day2valid"

# Probability/risk bins
PAPER_RT_MIN_PROB = 0.05

PAPER_RT_RISK_BOUNDS = [0.05, 0.15, 0.40, 0.70, 1.01]
PAPER_RT_RISK_COLORS = [
    "#76E34E",  # 5-15%
    "#FFFB00",  # 15-40%
    "#DE3E2C",  # 40-70%
    "#FF5BFF",  # >=70%
]
PAPER_RT_RISK_LABELS = [
    "5–15%",
    "15–40%",
    "40–70%",
    "≥70%",
]

PAPER_RT_CMAP = ListedColormap(PAPER_RT_RISK_COLORS)
PAPER_RT_NORM = BoundaryNorm(PAPER_RT_RISK_BOUNDS, PAPER_RT_CMAP.N, clip=True)

PAPER_RT_PANEL_TITLES = {
    "ML_r40_Prob": "ML r40 km",
    "ML_r60_Prob": "ML r60 km",
    "ML_r75_Prob": "ML r75 km",
    "ML_r100_Prob": "ML r100 km",
    "ML_Radius_EnsMean": "ML Ensemble Mean",
    "WPC_ERO_Risk": "WPC ERO",
    "PP_ANY": "Practically Perfect",
    "UFVS_ANY": "UFVS Any Flood Proxy",
}


# ======================================================================================
# FONT SETTINGS
# ======================================================================================

plt.rcParams.update({
    "font.size": PAPER_RT_FONT_SIZE,
    "axes.titlesize": PAPER_RT_TITLE_SIZE,
    "axes.labelsize": PAPER_RT_FONT_SIZE,
    "xtick.labelsize": PAPER_RT_FONT_SIZE,
    "ytick.labelsize": PAPER_RT_FONT_SIZE,
    "legend.fontsize": PAPER_RT_LEGEND_SIZE,
    "figure.titlesize": PAPER_RT_FONT_SIZE,
})


# ======================================================================================
# BUILD df_realtime_viewer
# ======================================================================================

PAPER_RT_REQUIRED_ML_COLS = [
    "ML_r40_Prob",
    "ML_r60_Prob",
    "ML_r75_Prob",
    "ML_r100_Prob",
]


def _rt_ml_prob_columns_present(df):
    """Return all ML probability-like columns for diagnostics."""
    return [
        c for c in df.columns
        if str(c).startswith("ML_") and str(c).endswith("_Prob")
    ]


def _rt_radius_prob_cols(df):
    """
    Return the four paper-figure ML members exactly once and in plot order.
    """
    return [c for c in PAPER_RT_REQUIRED_ML_COLS if c in df.columns]


def _rt_recompute_paper_ensemble_mean_only(df):
    """
    Keep only products needed for paper figure:
      - ML_Radius_EnsMean

    Removes:
      - ML_Radius_PMM
      - ML_Radius_EnsMax
      - ML_Radius_EnsSpread
    """
    out = df.copy()

    drop_cols = [
        c for c in ["ML_Radius_PMM", "ML_Radius_EnsMax", "ML_Radius_EnsSpread"]
        if c in out.columns
    ]
    if drop_cols:
        out = out.drop(columns=drop_cols)

    member_cols = _rt_radius_prob_cols(out)

    missing_members = [c for c in PAPER_RT_REQUIRED_ML_COLS if c not in out.columns]
    if missing_members:
        available = _rt_ml_prob_columns_present(out)
        raise RuntimeError(
            "Cannot create the four-member ML ensemble mean because these "
            f"columns are missing: {missing_members}. "
            f"Available ML probability columns are: {available}"
        )

    arr = out[member_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    out["ML_Radius_EnsMean"] = np.nanmean(arr, axis=1).astype(np.float32)

    # Compatibility if some existing helpers expect this
    out["ML_Forecast_Prob"] = out["ML_Radius_EnsMean"].astype(np.float32)

    return out, member_cols


if "build_predict_verify_realtime_multi_radius" not in globals():
    raise RuntimeError(
        "build_predict_verify_realtime_multi_radius is not defined. "
        "Run the realtime helper-definition cells above this point first."
    )

df_realtime_viewer = build_predict_verify_realtime_multi_radius(
    REALTIME_DATE,
    model_specs=REALTIME_MODEL_SPECS,
    force_predict=FORCE_REALTIME_PREDICT,
    force_features=FORCE_REALTIME_FEATURES,
    force_wpc=FORCE_REALTIME_WPC,
    force_ufvs=FORCE_REALTIME_UFVS,
    include_ufvs=True,
    include_wpc_pp=True,
    include_regular_flood_lsr=INCLUDE_REGULAR_FLOOD_LSR,
    pp_expansion_radius_km=REALTIME_PP_EXPANSION_RADIUS_KM,
    pp_smooth_radius_km=REALTIME_PP_SMOOTH_RADIUS_KM,
)

df_realtime_viewer, _rt_member_cols = _rt_recompute_paper_ensemble_mean_only(df_realtime_viewer)

print("df_realtime_viewer created")
print("Rows:", len(df_realtime_viewer))
print("ML member columns:", _rt_member_cols)
print("Paper ensemble columns:", [c for c in ["ML_Radius_EnsMean"] if c in df_realtime_viewer.columns])
print("WPC columns:", [c for c in df_realtime_viewer.columns if c == "WPC_ERO_Risk"])
print("PP Any columns:", [c for c in df_realtime_viewer.columns if str(c).startswith("PP_") and "any" in str(c).lower()])
print("UFVS Any columns:", [c for c in df_realtime_viewer.columns if c == "UFVS_ANY"])


# ======================================================================================
# PAPER PLOT HELPERS
# ======================================================================================

def _paper_rt_date8(x):
    if x is None:
        return None
    return str(x)[:10].replace("-", "")[:8]


def _paper_rt_find_pp_any_col(df):
    candidates = [
        "PP_Any flood proxy",
        "PP_Any Flood Proxy",
        "PP_ANY",
        "PP_UFVS_ANY",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    any_cols = [
        c for c in df.columns
        if str(c).startswith("PP_") and "any" in str(c).lower()
    ]

    return any_cols[0] if any_cols else None


def _paper_rt_normalize_probability_values(series):
    """
    Converts common probability/risk encodings to 0-1 probabilities.

    Handles:
      - already 0-1 probabilities
      - percent values like 5, 15, 40, 70
      - categorical integers 0, 1, 2, 3, 4
      - risk strings such as Marginal/Slight/Moderate/High
    """
    s = pd.Series(series)

    vals = pd.to_numeric(s, errors="coerce").to_numpy(float)

    if np.isfinite(vals).any():
        finite = vals[np.isfinite(vals)]
        vmax = float(np.nanmax(finite))
        unique = set(np.unique(finite).astype(float).tolist())

        # WPC-style integer categories
        if vmax <= 4.0 and unique.issubset({0.0, 1.0, 2.0, 3.0, 4.0}):
            mapped = np.full(vals.shape, np.nan, dtype=float)
            mapped[vals == 0] = 0.0
            mapped[vals == 1] = 0.05
            mapped[vals == 2] = 0.15
            mapped[vals == 3] = 0.40
            mapped[vals == 4] = 0.70
            return mapped

        # Percent values
        if vmax > 1.0:
            return vals / 100.0

        return vals

    lower = s.astype(str).str.lower().str.strip()
    mapped = np.full(len(s), np.nan, dtype=float)

    mapped[lower.str.contains("marginal", na=False)] = 0.05
    mapped[lower.str.contains("slight", na=False)] = 0.15
    mapped[lower.str.contains("moderate", na=False)] = 0.40
    mapped[lower.str.contains("high", na=False)] = 0.70

    return mapped


def _paper_rt_get_extent(df):
    if "DEFAULT_EXTENT" in globals():
        return DEFAULT_EXTENT

    lon = pd.to_numeric(df["Lon"], errors="coerce")
    lat = pd.to_numeric(df["Lat"], errors="coerce")

    return [
        float(lon.quantile(0.01)),
        float(lon.quantile(0.99)),
        float(lat.quantile(0.01)),
        float(lat.quantile(0.99)),
    ]


def _paper_rt_setup_map_axis(ax, df):
    extent = _paper_rt_get_extent(df)

    if HAS_CARTOPY_RT_PAPER:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

        try:
            ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=1.8)
        except Exception:
            try:
                ax.coastlines(resolution="50m", linewidth=1.8)
            except Exception:
                pass

        try:
            ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=1.4)
        except Exception:
            pass

        try:
            ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=1.0, edgecolor="0.35")
        except Exception:
            pass

    else:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.grid(True, alpha=0.20)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(
        labelbottom=False,
        labelleft=False,
        bottom=False,
        left=False,
    )


def _paper_rt_add_panel_label(ax, label):
    ax.text(
        0.025,
        0.965,
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=PAPER_RT_PANEL_LABEL_SIZE,
        fontweight="bold",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.84,
            boxstyle="round,pad=0.15",
        ),
        zorder=100,
    )


def _paper_rt_plot_probability_panel(ax, df, col, title, panel_label):
    _paper_rt_setup_map_axis(ax, df)

    lon = pd.to_numeric(df["Lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(df["Lat"], errors="coerce").to_numpy(float)
    vals = _paper_rt_normalize_probability_values(df[col])

    good = (
        np.isfinite(lon)
        & np.isfinite(lat)
        & np.isfinite(vals)
        & (vals >= PAPER_RT_MIN_PROB)
    )

    if np.any(good):
        scatter_kwargs = dict(
            c=np.clip(vals[good], PAPER_RT_MIN_PROB, 1.0),
            s=PAPER_RT_POINT_SIZE,
            alpha=PAPER_RT_ALPHA,
            cmap=PAPER_RT_CMAP,
            norm=PAPER_RT_NORM,
            linewidths=0,
            rasterized=True,
        )

        if HAS_CARTOPY_RT_PAPER:
            scatter_kwargs["transform"] = ccrs.PlateCarree()

        ax.scatter(lon[good], lat[good], **scatter_kwargs)

    else:
        ax.text(
            0.5,
            0.5,
            "No values ≥5%",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=38,
        )

    ax.set_title(title, pad=18)
    _paper_rt_add_panel_label(ax, panel_label)


def _paper_rt_plot_ufvs_panel(ax, df, col, title, panel_label):
    _paper_rt_setup_map_axis(ax, df)

    lon = pd.to_numeric(df["Lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(df["Lat"], errors="coerce").to_numpy(float)
    vals = pd.to_numeric(df[col], errors="coerce").fillna(0).to_numpy(float)

    good = np.isfinite(lon) & np.isfinite(lat)
    event = good & (vals > 0)

    bg_kwargs = dict(
        s=PAPER_RT_UFVS_BACKGROUND_SIZE,
        c="0.78",
        alpha=PAPER_RT_UFVS_BACKGROUND_ALPHA,
        linewidths=0,
        rasterized=True,
    )

    ev_kwargs = dict(
        s=PAPER_RT_UFVS_EVENT_SIZE,
        c="black",
        alpha=PAPER_RT_UFVS_EVENT_ALPHA,
        linewidths=0,
        rasterized=True,
    )

    if HAS_CARTOPY_RT_PAPER:
        bg_kwargs["transform"] = ccrs.PlateCarree()
        ev_kwargs["transform"] = ccrs.PlateCarree()

    if np.any(good):
        ax.scatter(lon[good], lat[good], **bg_kwargs)

    if np.any(event):
        ax.scatter(lon[event], lat[event], **ev_kwargs)
    else:
        ax.text(
            0.5,
            0.5,
            "No UFVS events",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=38,
        )

    ax.set_title(title, pad=18)
    _paper_rt_add_panel_label(ax, panel_label)


def _paper_rt_risk_legend_handles():
    return [
        Patch(
            facecolor=color,
            edgecolor="black",
            linewidth=1.5,
            label=label,
        )
        for color, label in zip(PAPER_RT_RISK_COLORS, PAPER_RT_RISK_LABELS)
    ]


def _paper_rt_draw_row1_legend(ax):
    ax.axis("off")

    ax.legend(
        handles=_paper_rt_risk_legend_handles(),
        title="Probability",
        loc="center left",
        frameon=True,
        framealpha=0.96,
        fontsize=PAPER_RT_LEGEND_SIZE,
        title_fontsize=PAPER_RT_LEGEND_SIZE,
        borderpad=0.8,
        labelspacing=0.9,
        handlelength=1.5,
        handleheight=1.2,
    )


def _paper_rt_draw_row2_legend(ax):
    ax.axis("off")

    risk_legend = ax.legend(
        handles=_paper_rt_risk_legend_handles(),
        title="Probability",
        loc="upper left",
        bbox_to_anchor=(0.0, 0.98),
        frameon=True,
        framealpha=0.96,
        fontsize=PAPER_RT_LEGEND_SIZE,
        title_fontsize=PAPER_RT_LEGEND_SIZE,
        borderpad=0.8,
        labelspacing=0.9,
        handlelength=1.5,
        handleheight=1.2,
    )
    ax.add_artist(risk_legend)

    ufvs_handle = Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        color="black",
        markerfacecolor="black",
        markeredgecolor="black",
        markersize=24,
        label="UFVS event",
    )

    ax.legend(
        handles=[ufvs_handle],
        title="Observed\nproxy",
        loc="lower left",
        bbox_to_anchor=(0.0, 0.03),
        frameon=True,
        framealpha=0.96,
        fontsize=PAPER_RT_LEGEND_SIZE,
        title_fontsize=PAPER_RT_LEGEND_SIZE,
        borderpad=0.8,
        labelspacing=0.9,
    )


def plot_paper_realtime_probability_maps(df, date):
    date8 = _paper_rt_date8(date)

    if "Date" in df.columns and date8 is not None:
        sub = df[df["Date"].astype(str).str[:8] == date8].copy()
        if sub.empty:
            print(f"No rows matched date {date8}; plotting all rows.")
            sub = df.copy()
    else:
        sub = df.copy()

    sub, member_cols = _rt_recompute_paper_ensemble_mean_only(sub)

    row1_cols = list(PAPER_RT_REQUIRED_ML_COLS)

    missing_row1 = [c for c in row1_cols if c not in sub.columns]
    if missing_row1:
        raise RuntimeError(f"Missing required ML member columns: {missing_row1}")

    pp_any_col = _paper_rt_find_pp_any_col(sub)
    if pp_any_col is None:
        raise RuntimeError("No PP Any Flood Proxy column found.")

    row2_items = [
        ("prob", "ML_Radius_EnsMean", PAPER_RT_PANEL_TITLES["ML_Radius_EnsMean"]),
        ("prob", "WPC_ERO_Risk", PAPER_RT_PANEL_TITLES["WPC_ERO_Risk"]),
        ("prob", pp_any_col, PAPER_RT_PANEL_TITLES["PP_ANY"]),
        ("ufvs", "UFVS_ANY", PAPER_RT_PANEL_TITLES["UFVS_ANY"]),
    ]

    missing_row2 = [col for kind, col, title in row2_items if col not in sub.columns]
    if missing_row2:
        raise RuntimeError(f"Missing required row-2 columns: {missing_row2}")

    projection = ccrs.PlateCarree() if HAS_CARTOPY_RT_PAPER else None

    fig = plt.figure(figsize=PAPER_RT_FIGSIZE)

    gs = fig.add_gridspec(
        nrows=2,
        ncols=5,
        width_ratios=[1, 1, 1, 1, 0.40],
        height_ratios=[1, 1],
        wspace=0.055,
        hspace=0.18,
    )

    row1_axes = []
    for col in range(4):
        if HAS_CARTOPY_RT_PAPER:
            ax = fig.add_subplot(gs[0, col], projection=projection)
        else:
            ax = fig.add_subplot(gs[0, col])
        row1_axes.append(ax)

    row2_axes = []
    for col in range(4):
        if HAS_CARTOPY_RT_PAPER:
            ax = fig.add_subplot(gs[1, col], projection=projection)
        else:
            ax = fig.add_subplot(gs[1, col])
        row2_axes.append(ax)

    legend_ax_row1 = fig.add_subplot(gs[0, 4])
    legend_ax_row2 = fig.add_subplot(gs[1, 4])

    panel_labels = [f"{letter})" for letter in string.ascii_lowercase]

    # Row 1: individual ML radii
    for i, col in enumerate(row1_cols):
        _paper_rt_plot_probability_panel(
            row1_axes[i],
            sub,
            col=col,
            title=PAPER_RT_PANEL_TITLES[col],
            panel_label=panel_labels[i],
        )

    # Row 2: ensemble mean, WPC, PP, UFVS
    for j, (kind, col, title) in enumerate(row2_items):
        ax = row2_axes[j]
        panel_label = panel_labels[len(row1_cols) + j]

        if kind == "ufvs":
            _paper_rt_plot_ufvs_panel(
                ax,
                sub,
                col=col,
                title=title,
                panel_label=panel_label,
            )
        else:
            _paper_rt_plot_probability_panel(
                ax,
                sub,
                col=col,
                title=title,
                panel_label=panel_label,
            )

    # One shared legend per row on the far right
    _paper_rt_draw_row1_legend(legend_ax_row1)
    _paper_rt_draw_row2_legend(legend_ax_row2)

    if PAPER_RT_SAVE_FIG:
        PAPER_RT_OUTDIR.mkdir(parents=True, exist_ok=True)

        png_path = PAPER_RT_OUTDIR / f"paper_realtime_probability_maps_{date8}.png"
        pdf_path = PAPER_RT_OUTDIR / f"paper_realtime_probability_maps_{date8}.pdf"

        fig.savefig(png_path, dpi=int(PAPER_RT_DPI), bbox_inches="tight")
        fig.savefig(pdf_path, dpi=int(PAPER_RT_DPI), bbox_inches="tight")

        print("Saved:", png_path)
        print("Saved:", pdf_path)

    plt.show()
    return fig


fig_paper_realtime_probability_maps = plot_paper_realtime_probability_maps(
    df=df_realtime_viewer,
    date=REALTIME_DATE,
)

In [ ]:
# ======================================================================================
# FULL REPLACEMENT: CASE-LEVEL RISK-AREA COUNTS AND PP CONFUSION MATRICES
#
# Use this AFTER running the block that creates ML probabilities.
#
# Important:
#   This cell does NOT reselect the original WPC/PP dataframe unless df_risk_counts does
#   not exist. It preserves/uses the df_risk_counts dataframe that should now contain:
#
#       ML_r40_Prob
#       ML_r60_Prob
#       ML_r75_Prob
#       ML_r100_Prob
#       ML_Radius_EnsMean
#       WPC_ERO_Risk
#       PP_Any flood proxy
#
# It evaluates case-level occurrence of:
#   Slight-or-greater:   >= 15%
#   Moderate-or-greater: >= 40%
#   High:                >= 70%
#
# Reference field:
#   Practically Perfect Any Flood Proxy
#
# Outputs:
#   risk_case_summary
#   matrix_tables[(risk_label, source_name)]
# ======================================================================================

import re
import numpy as np
import pandas as pd
from pathlib import Path

# --------------------------------------------------------------------------------------
# Controls
# --------------------------------------------------------------------------------------
CASE_RISK_THRESHOLDS = [
    ("Slight-or-greater", 0.15),
    ("Moderate-or-greater", 0.40),
    ("High", 0.70),
]

CASE_RISK_MODEL_SPECS = [dict(spec) for spec in MODEL_SPECS]
CASE_RISK_INCLUDE_ENSEMBLE_MEAN = True
CASE_RISK_INCLUDE_WPC = True

if "PROJECT_DIR" in globals():
    CASE_RISK_OUT_CSV = Path(PROJECT_DIR) / "case_level_risk_area_counts_pp_confusion.csv"
else:
    CASE_RISK_OUT_CSV = Path.cwd() / "case_level_risk_area_counts_pp_confusion.csv"


# --------------------------------------------------------------------------------------
# Dataframe source
# --------------------------------------------------------------------------------------
def _get_existing_case_risk_dataframe():
    """
    Prefer the dataframe already created by the ML probability prediction block.
    Do NOT overwrite it with the old base dataframe unless it does not exist.
    """
    if "df_risk_counts" in globals():
        print("Using existing df_risk_counts from previous ML-probability block.")
        return df_risk_counts.copy()

    if "df_case_level_ml_probs" in globals():
        print("Using df_case_level_ml_probs.")
        return df_case_level_ml_probs.copy()

    raise RuntimeError(
        "df_risk_counts does not exist. Run the ML-probability creation block first, "
        "then run this confusion-matrix block."
    )


df_case_conf = _get_existing_case_risk_dataframe()

if "Date" not in df_case_conf.columns:
    if "ValidDate" in df_case_conf.columns:
        df_case_conf["Date"] = df_case_conf["ValidDate"].astype(str).str[:8]
    else:
        raise RuntimeError("Dataframe needs Date or ValidDate.")

df_case_conf["Date"] = df_case_conf["Date"].astype(str).str[:8]


# --------------------------------------------------------------------------------------
# If needed, merge cached ML probability files from the earlier prediction block
# --------------------------------------------------------------------------------------
def _case_conf_ml_cols(df):
    return sorted(
        [c for c in df.columns if re.match(r"^ML_r\d+_Prob$", str(c))],
        key=lambda c: int(re.search(r"r(\d+)", c).group(1)),
    )


def _try_merge_cached_ml_probs(df):
    """
    If the dataframe still lacks ML_rXX_Prob columns, try to merge the cached predictions
    saved by the previous ML-probability creation block.
    """
    out = df.copy()

    existing = _case_conf_ml_cols(out)
    if existing:
        print("ML probability columns already present:", existing)
        return out

    if "CASE_RISK_ML_CACHE_DIR" not in globals():
        print("No ML columns found and CASE_RISK_ML_CACHE_DIR is not defined.")
        return out

    cache_dir = Path(CASE_RISK_ML_CACHE_DIR)
    if not cache_dir.exists():
        print("No ML columns found and cache directory does not exist:", cache_dir)
        return out

    print("No ML_rXX_Prob columns found in dataframe. Trying to merge cached ML probabilities from:")
    print(cache_dir)

    required_keys = ["Date", "Lat", "Lon"]
    missing_keys = [c for c in required_keys if c not in out.columns]
    if missing_keys:
        raise RuntimeError(f"Cannot merge cached ML probabilities; missing key columns: {missing_keys}")

    for spec in CASE_RISK_MODEL_SPECS:
        r = int(spec["radius_km"])
        model_label = str(spec["label"])
        prob_col = _model_prob_col(model_label, r)

        if prob_col in out.columns:
            continue

        candidates = sorted(cache_dir.glob(f"*{prob_col}*.parquet"))

        if not candidates:
            print(f"  No cache file found for {prob_col}")
            continue

        cache_path = candidates[0]
        pred = pd.read_parquet(cache_path).copy()

        if "Date" not in pred.columns or "Lat" not in pred.columns or "Lon" not in pred.columns or prob_col not in pred.columns:
            print(f"  Cache file for {prob_col} is missing required columns: {cache_path}")
            continue

        pred["Date"] = pred["Date"].astype(str).str[:8]
        pred = pred[["Date", "Lat", "Lon", prob_col]].drop_duplicates(subset=["Date", "Lat", "Lon"])

        out = out.merge(pred, on=["Date", "Lat", "Lon"], how="left")

        filled = out[prob_col].notna().sum()
        print(f"  Merged {prob_col}: {filled:,}/{len(out):,} rows from {cache_path}")

    return out


df_case_conf = _try_merge_cached_ml_probs(df_case_conf)


# --------------------------------------------------------------------------------------
# Add ensemble mean from ML columns
# --------------------------------------------------------------------------------------
ml_member_cols = _case_conf_ml_cols(df_case_conf)

if ml_member_cols:
    arr = df_case_conf[ml_member_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    df_case_conf["ML_Radius_EnsMean"] = np.nanmean(arr, axis=1).astype(np.float32)
    print("Added/updated ML_Radius_EnsMean from:", ml_member_cols)
else:
    print("WARNING: No ML_rXX_Prob columns found after cache merge attempt.")

print("\nColumns containing ML:")
print([c for c in df_case_conf.columns if "ML" in str(c)][:100])

print("\nColumns containing WPC:")
print([c for c in df_case_conf.columns if "WPC" in str(c)][:100])

print("\nColumns containing PP:")
print([c for c in df_case_conf.columns if str(c).startswith("PP_")][:100])


# --------------------------------------------------------------------------------------
# Probability normalization
# --------------------------------------------------------------------------------------
def _normalize_case_probability_values(series):
    """
    Convert common risk/probability encodings to 0-1 probabilities.

    Handles:
      - continuous 0-1 probabilities
      - percent values like 5, 15, 40, 70
      - WPC integer categories 0, 1, 2, 3, 4
      - strings like Marginal, Slight, Moderate, High
    """
    s = pd.Series(series)
    vals = pd.to_numeric(s, errors="coerce").to_numpy(float)

    if np.isfinite(vals).any():
        finite = vals[np.isfinite(vals)]
        vmax = float(np.nanmax(finite))
        unique = set(np.unique(finite).astype(float).tolist())

        # WPC-style integer categories
        if vmax <= 4.0 and unique.issubset({0.0, 1.0, 2.0, 3.0, 4.0}):
            out = np.full(vals.shape, np.nan, dtype=float)
            out[vals == 0] = 0.00
            out[vals == 1] = 0.05
            out[vals == 2] = 0.15
            out[vals == 3] = 0.40
            out[vals == 4] = 0.70
            return out

        # Percent values
        if vmax > 1.0:
            return vals / 100.0

        return vals

    lower = s.astype(str).str.lower().str.strip()
    out = np.full(len(s), np.nan, dtype=float)

    out[lower.str.contains("marginal", na=False)] = 0.05
    out[lower.str.contains("slight", na=False)] = 0.15
    out[lower.str.contains("moderate", na=False)] = 0.40
    out[lower.str.contains("high", na=False)] = 0.70

    return out


# --------------------------------------------------------------------------------------
# Find PP and forecast source columns
# --------------------------------------------------------------------------------------
def _find_pp_any_col_for_case_conf(df):
    candidates = [
        "PP_Any flood proxy",
        "PP_Any Flood Proxy",
        "PP_ANY",
        "PP_UFVS_ANY",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    any_cols = [
        c for c in df.columns
        if str(c).startswith("PP_") and "any" in str(c).lower()
    ]

    if any_cols:
        return any_cols[0]

    raise RuntimeError(
        "No Practically Perfect Any Flood Proxy column found. "
        f"Available PP columns: {[c for c in df.columns if str(c).startswith('PP_')]}"
    )


def _case_conf_radius_sort_key(col):
    m = re.search(r"ML_r(\d+)_Prob", str(col))
    return int(m.group(1)) if m else 9999


def _forecast_sources_for_case_conf(df):
    sources = []

    for spec in CASE_RISK_MODEL_SPECS:
        model_label = str(spec["label"])
        c = _model_prob_col(model_label, spec["radius_km"])
        if c in df.columns:
            sources.append((f"ML {model_label}", c))

    if CASE_RISK_INCLUDE_ENSEMBLE_MEAN and "ML_Radius_EnsMean" in df.columns:
        sources.append(("ML Ensemble Mean", "ML_Radius_EnsMean"))

    if CASE_RISK_INCLUDE_WPC and "WPC_ERO_Risk" in df.columns:
        sources.append(("WPC ERO", "WPC_ERO_Risk"))

    print("\nForecast sources that will be evaluated:")
    for name, col in sources:
        nfinite = np.isfinite(_normalize_case_probability_values(df[col])).sum()
        print(f"  {name}: {col} | finite values={nfinite:,}/{len(df):,}")

    if not sources:
        raise RuntimeError("No forecast sources found. Expected ML_rXX_Prob and/or WPC_ERO_Risk.")

    return sources


pp_col = _find_pp_any_col_for_case_conf(df_case_conf)
sources = _forecast_sources_for_case_conf(df_case_conf)

print("\nPP reference column:", pp_col)


# --------------------------------------------------------------------------------------
# Case-level yes/no flags
# --------------------------------------------------------------------------------------
def _case_has_any_area(df, col, threshold):
    dates = df["Date"].astype(str).str[:8].to_numpy()
    vals = _normalize_case_probability_values(df[col])
    valid = np.isfinite(vals)

    rows = []

    for d in sorted(pd.Series(dates).dropna().astype(str).unique().tolist()):
        m = dates == d

        rows.append({
            "Date": d,
            "Has Area": bool(np.any(valid[m] & (vals[m] >= float(threshold)))),
            "Max Value": float(np.nanmax(vals[m])) if np.any(np.isfinite(vals[m])) else np.nan,
            "N Grid Points": int(np.sum(m)),
            "N Exceeding": int(np.sum(valid[m] & (vals[m] >= float(threshold)))),
        })

    return pd.DataFrame(rows)


def _case_confusion_against_pp(source_case, pp_case):
    merged = pp_case.rename(columns={"Has Area": "PP Has Area"}).merge(
        source_case.rename(columns={"Has Area": "Forecast Has Area"}),
        on="Date",
        how="inner",
        suffixes=("_PP", "_Forecast"),
    )

    pp_yes = merged["PP Has Area"].astype(bool).to_numpy()
    fcst_yes = merged["Forecast Has Area"].astype(bool).to_numpy()

    tn = int(np.sum(~pp_yes & ~fcst_yes))
    fp = int(np.sum(~pp_yes & fcst_yes))
    fn = int(np.sum(pp_yes & ~fcst_yes))
    tp = int(np.sum(pp_yes & fcst_yes))

    return merged, {
        "N Paired Cases": int(len(merged)),
        "PP Cases With Area": int(np.sum(pp_yes)),
        "Forecast Cases With Area": int(np.sum(fcst_yes)),
        "Both PP and Forecast": tp,
        "PP Only": fn,
        "Forecast Only": fp,
        "Neither": tn,
        "TP": tp,
        "FP": fp,
        "FN": fn,
        "TN": tn,
    }


# --------------------------------------------------------------------------------------
# Main computation
# --------------------------------------------------------------------------------------
summary_rows = []
matrix_tables = {}
case_flag_tables = {}

for risk_label, threshold in CASE_RISK_THRESHOLDS:
    print(f"\n=== {risk_label} >= {threshold:.0%} ===")

    pp_case = _case_has_any_area(df_case_conf, pp_col, threshold)
    case_flag_tables[(risk_label, "PP")] = pp_case

    risk_rows = []

    for source_name, source_col in sources:
        source_case = _case_has_any_area(df_case_conf, source_col, threshold)
        case_flag_tables[(risk_label, source_name)] = source_case

        paired_flags, cm = _case_confusion_against_pp(source_case, pp_case)

        row = {
            "Risk threshold": risk_label,
            "Threshold": threshold,
            "Forecast source": source_name,
            "PP cases": cm["PP Cases With Area"],
            "Forecast cases": cm["Forecast Cases With Area"],
            "Both PP and forecast": cm["Both PP and Forecast"],
            "PP only": cm["PP Only"],
            "Forecast only": cm["Forecast Only"],
            "Neither": cm["Neither"],
            "Total paired cases": cm["N Paired Cases"],
            "TP": cm["TP"],
            "FP": cm["FP"],
            "FN": cm["FN"],
            "TN": cm["TN"],
        }

        summary_rows.append(row)
        risk_rows.append(row)

        matrix_tables[(risk_label, source_name)] = pd.DataFrame(
            [
                [cm["TN"], cm["FP"]],
                [cm["FN"], cm["TP"]],
            ],
            index=["PP no", "PP yes"],
            columns=["Forecast no", "Forecast yes"],
        )

    display(pd.DataFrame(risk_rows))


risk_case_summary = pd.DataFrame(summary_rows)

print("\nCombined summary:")
display(risk_case_summary)

print("\nAvailable confusion matrix keys:")
print(list(matrix_tables.keys()))

risk_case_summary.to_csv(CASE_RISK_OUT_CSV, index=False)
print("\nSaved:", CASE_RISK_OUT_CSV)

# Keep this as the active dataframe for future use
df_risk_counts = df_case_conf.copy()

### 5cc. Categorical risk area by method

Compute case-level and all-case categorical risk areas for ML r40/r60/r75/r100, WPC ERO, and Practically Perfect. Areas use an existing cell-area column when available; otherwise the constant projected-grid cell area is estimated from the median nearest-neighbor spacing and reported in square kilometers.

In [ ]:
# Categorical risk-area tables for the four ML radii, WPC ERO, and Practically Perfect
from pathlib import Path
import re
import numpy as np
import pandas as pd
from scipy.spatial import cKDTree

RISK_AREA_THRESHOLDS = [(0.05, " >5%"), (0.15, " >15%"), (0.40, " >40%"), (0.70, " >70%")]
RISK_AREA_OUTDIR = Path(PROJECT_DIR) / "v33day2valid_categorical_risk_area_tables"
RISK_AREA_OUTDIR.mkdir(parents=True, exist_ok=True)

if "df_risk_counts" in globals():
    _risk_area_df = df_risk_counts.copy()
elif "df_case_level_ml_probs" in globals():
    _risk_area_df = df_case_level_ml_probs.copy()
else:
    raise RuntimeError("Run the ML radius-probability cell before this risk-area section.")

_risk_area_df["Date"] = _risk_area_df["Date"].astype(str).str[:8]
_method_cols = {}
for _spec in MODEL_SPECS:
    _radius = int(_spec["radius_km"])
    _label = str(_spec["label"])
    _col = _model_prob_col(_label, _radius)
    if _col in _risk_area_df.columns:
        _method_cols[f"ML {_label}"] = _col
if "WPC_ERO_Risk" in _risk_area_df.columns:
    _method_cols["WPC ERO"] = "WPC_ERO_Risk"
_pp_candidates = ["PP_Any flood proxy", "PP_Any Flood Proxy"]
_pp_col = next((c for c in _pp_candidates if c in _risk_area_df.columns), None)
if _pp_col is not None:
    _method_cols["Practically Perfect"] = _pp_col
if not _method_cols:
    raise RuntimeError("No ML/WPC/Practically Perfect probability columns were found.")

_area_col = next((c for c in ["Grid_Cell_Area_km2", "Cell_Area_km2", "cell_area_km2"] if c in _risk_area_df.columns), None)
if _area_col is None:
    _first_date = sorted(_risk_area_df["Date"].dropna().unique())[0]
    _grid = _risk_area_df.loc[_risk_area_df["Date"] == _first_date, ["Lat", "Lon"]].drop_duplicates().dropna()
    _lat = np.deg2rad(pd.to_numeric(_grid["Lat"], errors="coerce").to_numpy(float))
    _lon = np.deg2rad(pd.to_numeric(_grid["Lon"], errors="coerce").to_numpy(float))
    _xyz = np.column_stack([np.cos(_lat) * np.cos(_lon), np.cos(_lat) * np.sin(_lon), np.sin(_lat)])
    _chord = cKDTree(_xyz).query(_xyz, k=2)[0][:, 1]
    _spacing_km = float(np.nanmedian(2.0 * 6371.0 * np.arcsin(np.clip(_chord / 2.0, 0.0, 1.0))))
    _constant_cell_area_km2 = _spacing_km ** 2
    print(f"Estimated projected-grid spacing: {_spacing_km:.3f} km; cell area: {_constant_cell_area_km2:.3f} km^2")
else:
    _constant_cell_area_km2 = np.nan
    print(f"Using existing grid-cell area column: {_area_col}")

_risk_area_rows = []
for _date, _case in _risk_area_df.groupby("Date", sort=True):
    _weights = (pd.to_numeric(_case[_area_col], errors="coerce").to_numpy(float) if _area_col else np.full(len(_case), _constant_cell_area_km2))
    for _method, _col in _method_cols.items():
        _values = pd.to_numeric(_case[_col], errors="coerce").to_numpy(float)
        _valid = np.isfinite(_values) & np.isfinite(_weights) & (_weights > 0)
        _domain_area = float(np.sum(_weights[_valid]))
        for _threshold, _label in RISK_AREA_THRESHOLDS:
            _yes = _valid & (_values >= _threshold)
            _area = float(np.sum(_weights[_yes]))
            _risk_area_rows.append({
                "Date": _date, "Method": _method, "Threshold": _threshold,
                "Threshold Label": _label.strip(), "Risk Area km^2": _area,
                "Risk Area Fraction": (_area / _domain_area if _domain_area > 0 else np.nan),
                "N Points": int(np.sum(_yes)),
            })

case_level_categorical_risk_area = pd.DataFrame(_risk_area_rows)
summary_categorical_risk_area = (
    case_level_categorical_risk_area.groupby(["Method", "Threshold", "Threshold Label"], as_index=False)
    .agg(**{
        "Mean Area km^2": ("Risk Area km^2", "mean"), "Median Area km^2": ("Risk Area km^2", "median"),
        "P25 km^2": ("Risk Area km^2", lambda x: x.quantile(0.25)), "P75 km^2": ("Risk Area km^2", lambda x: x.quantile(0.75)),
        "Min km^2": ("Risk Area km^2", "min"), "Max km^2": ("Risk Area km^2", "max"),
    })
)
_case_csv = RISK_AREA_OUTDIR / "case_level_categorical_risk_area_by_method.csv"
_summary_csv = RISK_AREA_OUTDIR / "summary_categorical_risk_area_by_method.csv"
case_level_categorical_risk_area.to_csv(_case_csv, index=False)
summary_categorical_risk_area.to_csv(_summary_csv, index=False)
display(case_level_categorical_risk_area)
display(summary_categorical_risk_area)
print("Saved:", _case_csv)
print("Saved:", _summary_csv)

### 5d. Run multi-radius realtime forecast / verification

Edit the date and controls below. The cell will load/build every available model member from `REALTIME_DEFAULT_MODEL_SPECS` (`r40km`, `r60km`, `r75km`, and `r100km` Day-2 targets), create ensemble mean/PMM/spread products, then show a forecast-comparison figure beside WPC and a separate verification figure when PP/UFVS are available. It automatically skips models that have not been trained yet.

In [ ]:
# ======================================================================================
# PAPER-READY REALTIME PROBABILITY MAP FIGURE
#
# This replaces the realtime viewer build + plotting workflow.
#
# It:
#   1. Rebuilds df_realtime_viewer
#   2. Computes only ML ensemble mean
#   3. Plots:
#        Row 1: ML r40, ML r60, ML r75, ML r100
#        Row 2: ML ensemble mean, WPC ERO, Practically Perfect Any, UFVS Any
#
# It removes:
#   - Radius spread
#   - Ensemble max
#   - PMM
#
# Formatting:
#   - Font size 50 everywhere
#   - One shared row legend on the far right
#   - a), b), c), ... panel labels in upper-left of each subplot
#   - Only plots probability/risk values >= 5%
#   - Saves PNG and PDF
# ======================================================================================

import re
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import ListedColormap, BoundaryNorm
from matplotlib.patches import Patch
from matplotlib.lines import Line2D

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY_RT_PAPER = True
except Exception:
    HAS_CARTOPY_RT_PAPER = False


# ======================================================================================
# REALTIME BUILD CONTROLS
# ======================================================================================

REALTIME_DATE = "20260702"

REALTIME_MODEL_SPECS = [dict(spec) for spec in MODEL_SPECS]

FORCE_REALTIME_PREDICT = False
FORCE_REALTIME_FEATURES = False
FORCE_REALTIME_WPC = False
FORCE_REALTIME_UFVS = True

INCLUDE_REGULAR_FLOOD_LSR = False

REALTIME_PP_EXPANSION_RADIUS_KM = 40.0
REALTIME_PP_SMOOTH_RADIUS_KM = 100.0


# ======================================================================================
# PAPER FIGURE CONTROLS
# ======================================================================================

PAPER_RT_FONT_SIZE = 50
PAPER_RT_PANEL_LABEL_SIZE = 50
PAPER_RT_LEGEND_SIZE = 42
PAPER_RT_TITLE_SIZE = 50

PAPER_RT_POINT_SIZE = 32
PAPER_RT_ALPHA = 0.88

PAPER_RT_UFVS_BACKGROUND_SIZE = 8
PAPER_RT_UFVS_BACKGROUND_ALPHA = 0.10
PAPER_RT_UFVS_EVENT_SIZE = 46
PAPER_RT_UFVS_EVENT_ALPHA = 0.95

PAPER_RT_FIGSIZE = (64, 30)
PAPER_RT_DPI = 300
PAPER_RT_SAVE_FIG = True

if "PROJECT_DIR" in globals():
    PAPER_RT_OUTDIR = Path(PROJECT_DIR) / "paper_realtime_probability_maps_v33day2valid"
else:
    PAPER_RT_OUTDIR = Path.cwd() / "paper_realtime_probability_maps_v33day2valid"

# Probability/risk bins
PAPER_RT_MIN_PROB = 0.05

PAPER_RT_RISK_BOUNDS = [0.05, 0.15, 0.40, 0.70, 1.01]
PAPER_RT_RISK_COLORS = [
    "#3EE700",  # 5-15%
    "#FFFB00",  # 15-40%
    "#E74C3C",  # 40-70%
    "#FF00FF",  # >=70%
]
PAPER_RT_RISK_LABELS = [
    "5–15%",
    "15–40%",
    "40–70%",
    "≥70%",
]

PAPER_RT_CMAP = ListedColormap(PAPER_RT_RISK_COLORS)
PAPER_RT_NORM = BoundaryNorm(PAPER_RT_RISK_BOUNDS, PAPER_RT_CMAP.N, clip=True)

PAPER_RT_PANEL_TITLES = {
    "ML_r40_Prob": "ML r40 km",
    "ML_r60_Prob": "ML r60 km",
    "ML_r75_Prob": "ML r75 km",
    "ML_r100_Prob": "ML r100 km",
    "ML_Radius_EnsMean": "ML Ensemble Mean",
    "WPC_ERO_Risk": "WPC ERO",
    "PP_ANY": "Practically Perfect",
    "UFVS_ANY": "UFVS Any Flood Proxy",
}


# ======================================================================================
# FONT SETTINGS
# ======================================================================================

plt.rcParams.update({
    "font.size": PAPER_RT_FONT_SIZE,
    "axes.titlesize": PAPER_RT_TITLE_SIZE,
    "axes.labelsize": PAPER_RT_FONT_SIZE,
    "xtick.labelsize": PAPER_RT_FONT_SIZE,
    "ytick.labelsize": PAPER_RT_FONT_SIZE,
    "legend.fontsize": PAPER_RT_LEGEND_SIZE,
    "figure.titlesize": PAPER_RT_FONT_SIZE,
})


# ======================================================================================
# BUILD df_realtime_viewer
# ======================================================================================

def _rt_radius_prob_cols(df):
    return sorted(
        [c for c in df.columns if _wide_ml_member_metadata(c)[0] is not None],
        key=lambda c: (_wide_ml_member_metadata(c)[1], _wide_ml_member_metadata(c)[0]),
    )


def _rt_recompute_paper_ensemble_mean_only(df):
    """
    Keep only products needed for paper figure:
      - ML_Radius_EnsMean

    Removes:
      - ML_Radius_PMM
      - ML_Radius_EnsMax
      - ML_Radius_EnsSpread
    """
    out = df.copy()

    drop_cols = [
        c for c in ["ML_Radius_PMM", "ML_Radius_EnsMax", "ML_Radius_EnsSpread"]
        if c in out.columns
    ]
    if drop_cols:
        out = out.drop(columns=drop_cols)

    member_cols = _rt_radius_prob_cols(out)

    if not member_cols:
        raise RuntimeError("No ML_rXX_Prob columns found. Cannot create ensemble mean.")

    arr = out[member_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
    out["ML_Radius_EnsMean"] = np.nanmean(arr, axis=1).astype(np.float32)

    # Compatibility if some existing helpers expect this
    out["ML_Forecast_Prob"] = out["ML_Radius_EnsMean"].astype(np.float32)

    return out, member_cols


if "build_predict_verify_realtime_multi_radius" not in globals():
    raise RuntimeError(
        "build_predict_verify_realtime_multi_radius is not defined. "
        "Run the realtime helper-definition cells above this point first."
    )

df_realtime_viewer = build_predict_verify_realtime_multi_radius(
    REALTIME_DATE,
    model_specs=REALTIME_MODEL_SPECS,
    force_predict=FORCE_REALTIME_PREDICT,
    force_features=FORCE_REALTIME_FEATURES,
    force_wpc=FORCE_REALTIME_WPC,
    force_ufvs=FORCE_REALTIME_UFVS,
    include_ufvs=True,
    include_wpc_pp=True,
    include_regular_flood_lsr=INCLUDE_REGULAR_FLOOD_LSR,
    pp_expansion_radius_km=REALTIME_PP_EXPANSION_RADIUS_KM,
    pp_smooth_radius_km=REALTIME_PP_SMOOTH_RADIUS_KM,
)

df_realtime_viewer, _rt_member_cols = _rt_recompute_paper_ensemble_mean_only(df_realtime_viewer)

print("df_realtime_viewer created")
print("Rows:", len(df_realtime_viewer))
print("ML member columns:", _rt_member_cols)
print("Paper ensemble columns:", [c for c in ["ML_Radius_EnsMean"] if c in df_realtime_viewer.columns])
print("WPC columns:", [c for c in df_realtime_viewer.columns if c == "WPC_ERO_Risk"])
print("PP Any columns:", [c for c in df_realtime_viewer.columns if str(c).startswith("PP_") and "any" in str(c).lower()])
print("UFVS Any columns:", [c for c in df_realtime_viewer.columns if c == "UFVS_ANY"])


# ======================================================================================
# PAPER PLOT HELPERS
# ======================================================================================

def _paper_rt_date8(x):
    if x is None:
        return None
    return str(x)[:10].replace("-", "")[:8]


def _paper_rt_find_pp_any_col(df):
    candidates = [
        "PP_Any flood proxy",
        "PP_Any Flood Proxy",
        "PP_ANY",
        "PP_UFVS_ANY",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    any_cols = [
        c for c in df.columns
        if str(c).startswith("PP_") and "any" in str(c).lower()
    ]

    return any_cols[0] if any_cols else None


def _paper_rt_normalize_probability_values(series):
    """
    Converts common probability/risk encodings to 0-1 probabilities.

    Handles:
      - already 0-1 probabilities
      - percent values like 5, 15, 40, 70
      - categorical integers 0, 1, 2, 3, 4
      - risk strings such as Marginal/Slight/Moderate/High
    """
    s = pd.Series(series)

    vals = pd.to_numeric(s, errors="coerce").to_numpy(float)

    if np.isfinite(vals).any():
        finite = vals[np.isfinite(vals)]
        vmax = float(np.nanmax(finite))
        unique = set(np.unique(finite).astype(float).tolist())

        # WPC-style integer categories
        if vmax <= 4.0 and unique.issubset({0.0, 1.0, 2.0, 3.0, 4.0}):
            mapped = np.full(vals.shape, np.nan, dtype=float)
            mapped[vals == 0] = 0.0
            mapped[vals == 1] = 0.05
            mapped[vals == 2] = 0.15
            mapped[vals == 3] = 0.40
            mapped[vals == 4] = 0.70
            return mapped

        # Percent values
        if vmax > 1.0:
            return vals / 100.0

        return vals

    lower = s.astype(str).str.lower().str.strip()
    mapped = np.full(len(s), np.nan, dtype=float)

    mapped[lower.str.contains("marginal", na=False)] = 0.05
    mapped[lower.str.contains("slight", na=False)] = 0.15
    mapped[lower.str.contains("moderate", na=False)] = 0.40
    mapped[lower.str.contains("high", na=False)] = 0.70

    return mapped


def _paper_rt_get_extent(df):
    if "DEFAULT_EXTENT" in globals():
        return DEFAULT_EXTENT

    lon = pd.to_numeric(df["Lon"], errors="coerce")
    lat = pd.to_numeric(df["Lat"], errors="coerce")

    return [
        float(lon.quantile(0.01)),
        float(lon.quantile(0.99)),
        float(lat.quantile(0.01)),
        float(lat.quantile(0.99)),
    ]


def _paper_rt_setup_map_axis(ax, df):
    extent = _paper_rt_get_extent(df)

    if HAS_CARTOPY_RT_PAPER:
        ax.set_extent(extent, crs=ccrs.PlateCarree())

        try:
            ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=1.8)
        except Exception:
            try:
                ax.coastlines(resolution="50m", linewidth=1.8)
            except Exception:
                pass

        try:
            ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=1.4)
        except Exception:
            pass

        try:
            ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=1.0, edgecolor="0.35")
        except Exception:
            pass

    else:
        ax.set_xlim(extent[0], extent[1])
        ax.set_ylim(extent[2], extent[3])
        ax.grid(True, alpha=0.20)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(
        labelbottom=False,
        labelleft=False,
        bottom=False,
        left=False,
    )


def _paper_rt_add_panel_label(ax, label):
    ax.text(
        0.025,
        0.965,
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=PAPER_RT_PANEL_LABEL_SIZE,
        fontweight="bold",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.84,
            boxstyle="round,pad=0.15",
        ),
        zorder=100,
    )


def _paper_rt_plot_probability_panel(ax, df, col, title, panel_label):
    _paper_rt_setup_map_axis(ax, df)

    lon = pd.to_numeric(df["Lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(df["Lat"], errors="coerce").to_numpy(float)
    vals = _paper_rt_normalize_probability_values(df[col])

    good = (
        np.isfinite(lon)
        & np.isfinite(lat)
        & np.isfinite(vals)
        & (vals >= PAPER_RT_MIN_PROB)
    )

    if np.any(good):
        scatter_kwargs = dict(
            c=np.clip(vals[good], PAPER_RT_MIN_PROB, 1.0),
            s=PAPER_RT_POINT_SIZE,
            alpha=PAPER_RT_ALPHA,
            cmap=PAPER_RT_CMAP,
            norm=PAPER_RT_NORM,
            linewidths=0,
            rasterized=True,
        )

        if HAS_CARTOPY_RT_PAPER:
            scatter_kwargs["transform"] = ccrs.PlateCarree()

        ax.scatter(lon[good], lat[good], **scatter_kwargs)

    else:
        ax.text(
            0.5,
            0.5,
            "No values ≥5%",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=38,
        )

    ax.set_title(title, pad=18)
    _paper_rt_add_panel_label(ax, panel_label)


def _paper_rt_plot_ufvs_panel(ax, df, col, title, panel_label):
    _paper_rt_setup_map_axis(ax, df)

    lon = pd.to_numeric(df["Lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(df["Lat"], errors="coerce").to_numpy(float)
    vals = pd.to_numeric(df[col], errors="coerce").fillna(0).to_numpy(float)

    good = np.isfinite(lon) & np.isfinite(lat)
    event = good & (vals > 0)

    bg_kwargs = dict(
        s=PAPER_RT_UFVS_BACKGROUND_SIZE,
        c="0.78",
        alpha=PAPER_RT_UFVS_BACKGROUND_ALPHA,
        linewidths=0,
        rasterized=True,
    )

    ev_kwargs = dict(
        s=PAPER_RT_UFVS_EVENT_SIZE,
        c="black",
        alpha=PAPER_RT_UFVS_EVENT_ALPHA,
        linewidths=0,
        rasterized=True,
    )

    if HAS_CARTOPY_RT_PAPER:
        bg_kwargs["transform"] = ccrs.PlateCarree()
        ev_kwargs["transform"] = ccrs.PlateCarree()

    if np.any(good):
        ax.scatter(lon[good], lat[good], **bg_kwargs)

    if np.any(event):
        ax.scatter(lon[event], lat[event], **ev_kwargs)
    else:
        ax.text(
            0.5,
            0.5,
            "No UFVS events",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=38,
        )

    ax.set_title(title, pad=18)
    _paper_rt_add_panel_label(ax, panel_label)


def _paper_rt_risk_legend_handles():
    return [
        Patch(
            facecolor=color,
            edgecolor="black",
            linewidth=1.5,
            label=label,
        )
        for color, label in zip(PAPER_RT_RISK_COLORS, PAPER_RT_RISK_LABELS)
    ]


def _paper_rt_draw_row1_legend(ax):
    ax.axis("off")

    ax.legend(
        handles=_paper_rt_risk_legend_handles(),
        title="Probability",
        loc="center left",
        frameon=True,
        framealpha=0.96,
        fontsize=PAPER_RT_LEGEND_SIZE,
        title_fontsize=PAPER_RT_LEGEND_SIZE,
        borderpad=0.8,
        labelspacing=0.9,
        handlelength=1.5,
        handleheight=1.2,
    )


def _paper_rt_draw_row2_legend(ax):
    ax.axis("off")

    risk_legend = ax.legend(
        handles=_paper_rt_risk_legend_handles(),
        title="Probability",
        loc="upper left",
        bbox_to_anchor=(0.0, 0.98),
        frameon=True,
        framealpha=0.96,
        fontsize=PAPER_RT_LEGEND_SIZE,
        title_fontsize=PAPER_RT_LEGEND_SIZE,
        borderpad=0.8,
        labelspacing=0.9,
        handlelength=1.5,
        handleheight=1.2,
    )
    ax.add_artist(risk_legend)

    ufvs_handle = Line2D(
        [0],
        [0],
        marker="o",
        linestyle="None",
        color="black",
        markerfacecolor="black",
        markeredgecolor="black",
        markersize=24,
        label="UFVS event",
    )

    ax.legend(
        handles=[ufvs_handle],
        title="Observed\nproxy",
        loc="lower left",
        bbox_to_anchor=(0.0, 0.03),
        frameon=True,
        framealpha=0.96,
        fontsize=PAPER_RT_LEGEND_SIZE,
        title_fontsize=PAPER_RT_LEGEND_SIZE,
        borderpad=0.8,
        labelspacing=0.9,
    )


def plot_paper_realtime_probability_maps(df, date):
    date8 = _paper_rt_date8(date)

    if "Date" in df.columns and date8 is not None:
        sub = df[df["Date"].astype(str).str[:8] == date8].copy()
        if sub.empty:
            print(f"No rows matched date {date8}; plotting all rows.")
            sub = df.copy()
    else:
        sub = df.copy()

    sub, member_cols = _rt_recompute_paper_ensemble_mean_only(sub)

    row1_cols = [
        "ML_r40_Prob",
        "ML_r60_Prob",
        "ML_r75_Prob",
        "ML_r100_Prob",
    ]

    missing_row1 = [c for c in row1_cols if c not in sub.columns]
    if missing_row1:
        raise RuntimeError(f"Missing required ML member columns: {missing_row1}")

    pp_any_col = _paper_rt_find_pp_any_col(sub)
    if pp_any_col is None:
        raise RuntimeError("No PP Any Flood Proxy column found.")

    row2_items = [
        ("prob", "ML_Radius_EnsMean", PAPER_RT_PANEL_TITLES["ML_Radius_EnsMean"]),
        ("prob", "WPC_ERO_Risk", PAPER_RT_PANEL_TITLES["WPC_ERO_Risk"]),
        ("prob", pp_any_col, PAPER_RT_PANEL_TITLES["PP_ANY"]),
        ("ufvs", "UFVS_ANY", PAPER_RT_PANEL_TITLES["UFVS_ANY"]),
    ]

    missing_row2 = [col for kind, col, title in row2_items if col not in sub.columns]
    if missing_row2:
        raise RuntimeError(f"Missing required row-2 columns: {missing_row2}")

    projection = ccrs.PlateCarree() if HAS_CARTOPY_RT_PAPER else None

    fig = plt.figure(figsize=PAPER_RT_FIGSIZE)

    gs = fig.add_gridspec(
        nrows=2,
        ncols=5,
        width_ratios=[1, 1, 1, 1, 0.40],
        height_ratios=[1, 1],
        wspace=0.055,
        hspace=0.18,
    )

    axes = []
    for row in range(2):
        for col in range(4):
            if HAS_CARTOPY_RT_PAPER:
                ax = fig.add_subplot(gs[row, col], projection=projection)
            else:
                ax = fig.add_subplot(gs[row, col])
            axes.append(ax)

    legend_ax_row1 = fig.add_subplot(gs[0, 4])
    legend_ax_row2 = fig.add_subplot(gs[1, 4])

    panel_labels = [f"{letter})" for letter in string.ascii_lowercase]

    # Row 1: individual ML radii
    for i, col in enumerate(row1_cols):
        _paper_rt_plot_probability_panel(
            axes[i],
            sub,
            col=col,
            title=PAPER_RT_PANEL_TITLES[col],
            panel_label=panel_labels[i],
        )

    # Row 2: ensemble mean, WPC, PP, UFVS
    for j, (kind, col, title) in enumerate(row2_items):
        ax = axes[4 + j]
        panel_label = panel_labels[4 + j]

        if kind == "ufvs":
            _paper_rt_plot_ufvs_panel(
                ax,
                sub,
                col=col,
                title=title,
                panel_label=panel_label,
            )
        else:
            _paper_rt_plot_probability_panel(
                ax,
                sub,
                col=col,
                title=title,
                panel_label=panel_label,
            )

    # One shared legend per row on the far right
    _paper_rt_draw_row1_legend(legend_ax_row1)
    _paper_rt_draw_row2_legend(legend_ax_row2)

    if PAPER_RT_SAVE_FIG:
        PAPER_RT_OUTDIR.mkdir(parents=True, exist_ok=True)

        png_path = PAPER_RT_OUTDIR / f"paper_realtime_probability_maps_{date8}.png"
        pdf_path = PAPER_RT_OUTDIR / f"paper_realtime_probability_maps_{date8}.pdf"

        fig.savefig(png_path, dpi=int(PAPER_RT_DPI), bbox_inches="tight")
        fig.savefig(pdf_path, dpi=int(PAPER_RT_DPI), bbox_inches="tight")

        print("Saved:", png_path)
        print("Saved:", pdf_path)

    plt.show()
    return fig


fig_paper_realtime_probability_maps = plot_paper_realtime_probability_maps(
    df=df_realtime_viewer,
    date=REALTIME_DATE,
)

## 5e. Categorical CSI / flood-proxy fractional coverage / predictor browser

These cells are integrated into the notebook so you do not need to paste helper code manually. They work with the historical dataframe (`df_radius_viewer`) and, after the realtime cell has run, with the realtime dataframe (`df_realtime_viewer`).

In [ ]:
# ======================================================================================
# Integrated categorical verification, flood-proxy fractional coverage, and predictor browser
# Multi-radius capable; proxy truth requires raw UFVS columns
# ======================================================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import ipywidgets as widgets
    from IPython.display import display
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False

CATEGORICAL_THRESHOLDS = [
    (0.05, ">5%"),
    (0.15, ">15%"),
    (0.40, ">40%"),
    (0.70, ">70%"),
]

EXCLUSIVE_RISK_BINS = [
    (0.05, 0.15, "5–15%"),
    (0.15, 0.40, "15–40%"),
    (0.40, 0.70, "40–70%"),
    (0.70, 1.01, "≥70%"),
]

DEFAULT_UFVS_PROXY_CANDIDATES = [
    "UFVS_ANY",
    "UFVS_STAGE4_FFG",
    "UFVS_STAGE4_ARI",
    "UFVS_USGS",
    "UFVS_LSR_FLASH",
    "UFVS_LSR_REGULAR",
    "UFVS_LSR_FLOOD",
]

DEFAULT_PP_PROXY_FALLBACKS = [
    "PP_Any flood proxy",
    "PP_LSR/USGS only",
    "PP_MRMS > FFG",
    "PP_Stage IV > FFG",
    "PP_Stage IV ARI",
    "PP_USGS",
    "PP_Flash LSR",
]


def _date8_local(date):
    return str(date)[:8].replace("-", "")


def _truth_col_from_pp_definition_integrated(pp_definition):
    c = str(pp_definition)
    return c if c.startswith("PP_") else f"PP_{c}"


def _wide_ml_member_metadata(col):
    c = str(col)
    m = re.match(r"^ML_r(\d+)_Prob$", c)
    return (f"r{int(m.group(1))}km", int(m.group(1))) if m else (None, None)


def _available_radii_from_df(df, requested=None):
    """Return available radius members in a historical long df or realtime wide df."""
    if requested is not None:
        req = [int(round(float(r))) for r in requested]
    else:
        req = None

    radii = set()
    if "ML_Target_Radius_km" in df.columns:
        vals = pd.to_numeric(df["ML_Target_Radius_km"], errors="coerce").dropna().astype(int).unique().tolist()
        for v in vals:
            if v >= 0:
                radii.add(int(v))
    for c in df.columns:
        _label, _radius = _wide_ml_member_metadata(c)
        if _radius is not None:
            radii.add(int(_radius))
    out = sorted(radii)
    if req is not None:
        out = [r for r in req if r in out]
    return out


def _forecast_cols_for_evaluation(df, radius_km=None, include_members=True, include_ensembles=True, include_wpc=True):
    """Return label->column mapping for one evaluation subset.

    Historical df_radius_viewer is long-format: one ML_Forecast_Prob column plus ML_Target_Radius_km.
    Realtime df_realtime_viewer is wide-format: ML_r40_Prob, ML_r60_Prob, etc.
    """
    out = {}
    r = None if radius_km is None else int(round(float(radius_km)))

    if include_members:
        # Realtime wide members first.
        member_cols = sorted(
            [c for c in df.columns if _wide_ml_member_metadata(c)[0] is not None],
            key=lambda c: (_wide_ml_member_metadata(c)[1], _wide_ml_member_metadata(c)[0]),
        )
        for c in member_cols:
            member_label, rr = _wide_ml_member_metadata(c)
            if r is None or rr == r:
                out[f"ML {member_label}"] = c

    # Historical long-format ML field for this radius.
    if "ML_Forecast_Prob" in df.columns and not any(k.startswith("ML r") for k in out):
        if r is not None:
            labels = df["ML_Model_Label"].dropna().astype(str).unique().tolist() if "ML_Model_Label" in df.columns else [model_label_for_radius(r)]
            out[f"ML {labels[0]}"] = "ML_Forecast_Prob"
        else:
            out["ML"] = "ML_Forecast_Prob"

    if include_ensembles:
        for label, col in [
            ("ML ens mean", "ML_Radius_EnsMean"),
            ("ML ens median", "ML_Radius_EnsMedian"),
            ("ML ens max", "ML_Radius_EnsMax"),
            ("ML PMM", "ML_Radius_PMM"),
            ("ML spread", "ML_Radius_EnsSpread"),
        ]:
            if col in df.columns:
                out[label] = col

    if include_wpc and "WPC_ERO_Risk" in df.columns:
        out["WPC ERO"] = "WPC_ERO_Risk"

    return out


def _subset_for_eval_radius(df, radius_km=None):
    """Subset historical long-format data by radius; leave realtime wide data unchanged."""
    if radius_km is None or "ML_Target_Radius_km" not in df.columns:
        return df.copy()
    r = int(round(float(radius_km)))
    rad = pd.to_numeric(df["ML_Target_Radius_km"], errors="coerce")
    # If aggregate realtime rows are marked -1 and no radius-specific rows exist, do not filter away.
    if (rad == -1).any() and not (rad == r).any():
        return df.copy()
    out = df[rad == r].copy()
    return out


def _resolve_proxy_column(df, proxy_col="UFVS_ANY", allow_pp_fallback=True):
    """Resolve a proxy/truth column robustly.

    Historical df_radius_viewer usually does not have raw UFVS_* columns; it may only have PP_* fields.
    In that case, this function falls back to PP_Any flood proxy unless explicitly disabled.
    """
    if proxy_col is not None and proxy_col in df.columns:
        return proxy_col, "ufvs_or_selected"

    for c in DEFAULT_UFVS_PROXY_CANDIDATES:
        if c in df.columns:
            print(f"Using available UFVS proxy column: {c}")
            return c, "ufvs_or_selected"

    if allow_pp_fallback:
        # Try exact requested PP-style column if the user passed a PP name/definition.
        if proxy_col is not None:
            maybe_pp = _truth_col_from_pp_definition_integrated(proxy_col)
            if maybe_pp in df.columns:
                print(f"Raw UFVS proxy column {proxy_col!r} not found; using PP fallback column {maybe_pp!r}.")
                return maybe_pp, "pp_fallback"
        for c in DEFAULT_PP_PROXY_FALLBACKS:
            if c in df.columns:
                print(
                    "Raw UFVS proxy columns are not available in this dataframe; "
                    f"using {c!r} as a proxy-like fallback. Use realtime verified df for raw UFVS_* columns."
                )
                return c, "pp_fallback"

    raise RuntimeError(
        "No usable proxy column found. Available UFVS columns: "
        f"{[c for c in df.columns if str(c).startswith('UFVS_')]}. "
        "Available PP columns: "
        f"{[c for c in df.columns if str(c).startswith('PP_')]}"
    )


def _binary_from_col(df, col, threshold=0.0, mode=">"):
    if col not in df.columns:
        raise RuntimeError(f"Column {col!r} not found.")
    vals = pd.to_numeric(df[col], errors="coerce").to_numpy(float)
    if mode == ">=":
        return np.isfinite(vals) & (vals >= float(threshold))
    return np.isfinite(vals) & (vals > float(threshold))


def _contingency_stats(fcst_yes, truth_yes):
    fcst_yes = np.asarray(fcst_yes, dtype=bool)
    truth_yes = np.asarray(truth_yes, dtype=bool)
    n = min(fcst_yes.size, truth_yes.size)
    fcst_yes = fcst_yes[:n]
    truth_yes = truth_yes[:n]
    H = int(np.sum(fcst_yes & truth_yes))
    M = int(np.sum((~fcst_yes) & truth_yes))
    F = int(np.sum(fcst_yes & (~truth_yes)))
    C = int(np.sum((~fcst_yes) & (~truth_yes)))
    csi = H / (H + M + F) if (H + M + F) > 0 else np.nan
    pod = H / (H + M) if (H + M) > 0 else np.nan
    far = F / (H + F) if (H + F) > 0 else np.nan
    bias = (H + F) / (H + M) if (H + M) > 0 else np.nan
    return {
        "N Points": n,
        "Hits": H,
        "Misses": M,
        "False Alarms": F,
        "Correct Negatives": C,
        "CSI": csi,
        "POD": pod,
        "FAR": far,
        "Bias": bias,
        "Forecast Area Fraction": float(np.mean(fcst_yes)) if n else np.nan,
        "Truth Area Fraction": float(np.mean(truth_yes)) if n else np.nan,
    }


# Backward-compatible alias for older cells/snippets.
# Keep the real _contingency_stats implementation above; expose the older helper name too.
_contingency_from_binary = _contingency_stats


def _source_radius_label(source_label, fallback_radius="aggregate"):
    m = re.search(r"r(\d+)", str(source_label))
    return int(m.group(1)) if m else fallback_radius


def compute_categorical_area_skill(
    df,
    forecast_cols=None,
    radii=None,
    radius_km=None,
    truth_mode="pp",              # "pp" or "proxy"
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    proxy_truth_threshold=0.0,
    thresholds=CATEGORICAL_THRESHOLDS,
    include_members=True,
    include_ensembles=True,
    include_wpc=True,
    allow_pp_proxy_fallback=False,
):
    """Compute categorical CSI/POD/FAR/Bias by cumulative threshold.

    Multi-radius behavior:
      * Historical long dataframe: loops over available ML_Target_Radius_km values.
      * Realtime wide dataframe: evaluates all ML_rXX_Prob member columns plus ensemble/WPC columns.

    For PP truth, threshold t is compared with PP >= t.
    For proxy truth, raw UFVS proxies use proxy > proxy_truth_threshold.
    If raw UFVS columns are absent and allow_pp_proxy_fallback=False, PP_Any flood proxy is used as a fallback.
    """
    truth_mode = str(truth_mode).lower()
    if radius_km is not None and radii is None:
        radii = [radius_km]

    # Historical long-format: loop radii if present.
    if "ML_Target_Radius_km" in df.columns and not any(re.match(r"^ML_r\d+_Prob$", str(c)) for c in df.columns):
        eval_radii = _available_radii_from_df(df, requested=radii)
        if not eval_radii:
            eval_radii = [radius_km] if radius_km is not None else [None]
    else:
        eval_radii = [None] if radii is None else [None]  # wide realtime columns already contain all radii

    case_rows = []
    pooled_rows = []

    for r in eval_radii:
        sub = _subset_for_eval_radius(df, radius_km=r)
        if sub.empty:
            print(f"Skipping radius {r}: no rows.")
            continue

        fcst_cols = forecast_cols
        if fcst_cols is None:
            fcst_cols = _forecast_cols_for_evaluation(
                sub, radius_km=r, include_members=include_members,
                include_ensembles=include_ensembles, include_wpc=include_wpc,
            )
        fcst_cols = {k: v for k, v in fcst_cols.items() if v in sub.columns}
        if not fcst_cols:
            print(f"Skipping radius {r}: no forecast columns.")
            continue

        if truth_mode == "pp":
            truth_col = _resolve_pp_truth_column_for_eval(sub, pp_definition) if "_resolve_pp_truth_column_for_eval" in globals() else _truth_col_from_pp_definition_integrated(pp_definition)
            if truth_col is None or truth_col not in sub.columns:
                print(
                    f"Skipping radius {r}: could not resolve PP definition {pp_definition!r}. "
                    f"Available PP columns: {[c for c in sub.columns if str(c).startswith('PP_')]}"
                )
                continue
            truth_kind = "pp"
        elif truth_mode == "proxy":
            truth_col, truth_kind = _resolve_proxy_column(sub, proxy_col=proxy_col, allow_pp_fallback=allow_pp_proxy_fallback)
        else:
            raise RuntimeError("truth_mode must be 'pp' or 'proxy'.")

        date_series = sub["Date"].astype(str).str[:8] if "Date" in sub.columns else pd.Series("ALL", index=sub.index)

        for thr, thr_label in thresholds:
            if truth_mode == "pp" or truth_kind == "pp_fallback":
                pooled_truth = _binary_from_col(sub, truth_col, threshold=thr if truth_mode == "pp" else proxy_truth_threshold, mode=">=")
            else:
                pooled_truth = _binary_from_col(sub, truth_col, threshold=proxy_truth_threshold, mode=">")

            for source_label, fcst_col in fcst_cols.items():
                vals = pd.to_numeric(sub[fcst_col], errors="coerce").to_numpy(float)
                pooled_fcst = np.isfinite(vals) & (vals >= float(thr))
                source_radius = _source_radius_label(source_label, fallback_radius=(r if r is not None else "aggregate"))
                row = _contingency_stats(pooled_fcst, pooled_truth)
                row.update({
                    "Source": source_label,
                    "Forecast Column": fcst_col,
                    "Truth Mode": truth_mode,
                    "Truth Column": truth_col,
                    "Truth Kind": truth_kind,
                    "Radius km": source_radius,
                    "Eval Radius km": r if r is not None else "all/aggregate",
                    "Threshold": thr_label,
                    "Threshold Value": float(thr),
                    "Aggregation": "pooled",
                })
                pooled_rows.append(row)

                for date, idx in sub.groupby(date_series).groups.items():
                    g = sub.loc[idx]
                    if truth_mode == "pp" or truth_kind == "pp_fallback":
                        truth = _binary_from_col(g, truth_col, threshold=thr if truth_mode == "pp" else proxy_truth_threshold, mode=">=")
                    else:
                        truth = _binary_from_col(g, truth_col, threshold=proxy_truth_threshold, mode=">")
                    vals_g = pd.to_numeric(g[fcst_col], errors="coerce").to_numpy(float)
                    fcst = np.isfinite(vals_g) & (vals_g >= float(thr))
                    crow = _contingency_stats(fcst, truth)
                    crow.update({
                        "Date": str(date),
                        "Source": source_label,
                        "Forecast Column": fcst_col,
                        "Truth Mode": truth_mode,
                        "Truth Column": truth_col,
                        "Truth Kind": truth_kind,
                        "Radius km": source_radius,
                        "Eval Radius km": r if r is not None else "all/aggregate",
                        "Threshold": thr_label,
                        "Threshold Value": float(thr),
                    })
                    case_rows.append(crow)

    case_table = pd.DataFrame(case_rows)
    pooled_table = pd.DataFrame(pooled_rows)
    if case_table.empty:
        print("No categorical metrics could be computed. Check forecast/truth columns above.")
        return case_table, pd.DataFrame(), pooled_table

    metric_cols = [
        "CSI", "POD", "FAR", "Bias", "Forecast Area Fraction", "Truth Area Fraction",
        "Hits", "Misses", "False Alarms", "Correct Negatives", "N Points",
    ]
    case_mean = (
        case_table.groupby(["Radius km", "Truth Mode", "Truth Column", "Truth Kind", "Threshold", "Source"], as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )
    return case_table, case_mean, pooled_table


def plot_categorical_skill_lines(summary_df, metric="CSI", title=None, include_wpc=True):
    """Plot all radii/sources on the same threshold-vs-metric plot."""
    if summary_df is None or summary_df.empty:
        print("No data to plot.")
        return None
    threshold_order = [lab for _, lab in CATEGORICAL_THRESHOLDS]
    fig, ax = plt.subplots(figsize=(10, 5.8))

    data = summary_df.copy()
    data["Threshold"] = pd.Categorical(data["Threshold"], categories=threshold_order, ordered=True)
    data = data.sort_values(["Source", "Threshold"])

    for src, g in data.groupby("Source", sort=False):
        if (not include_wpc) and str(src).lower().startswith("wpc"):
            continue
        gg = g.drop_duplicates("Threshold").set_index("Threshold").reindex(threshold_order)
        y = gg[metric].to_numpy(float)
        ax.plot(threshold_order, y, marker="o", linewidth=2, label=str(src))

    ax.set_xlabel("Forecast threshold")
    ax.set_ylabel(metric)
    ax.set_title(title or f"{metric} by categorical threshold")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=9)
    plt.show()
    plt.close(fig)
    return None


def plot_categorical_skill_bars(summary_df, metric="CSI", title=None):
    """Grouped bars retained for single-radius/small-source comparisons."""
    if summary_df is None or summary_df.empty:
        print("No data to plot.")
        return None
    threshold_order = [lab for _, lab in CATEGORICAL_THRESHOLDS]
    sources = list(summary_df["Source"].dropna().unique())
    pivot = summary_df.pivot_table(index="Threshold", columns="Source", values=metric, aggfunc="mean").reindex(threshold_order)
    x = np.arange(len(pivot.index))
    width = min(0.8 / max(len(sources), 1), 0.25)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for i, src in enumerate(sources):
        vals = pivot[src].to_numpy(float) if src in pivot.columns else np.full(len(x), np.nan)
        ax.bar(x + (i - (len(sources)-1)/2)*width, vals, width, label=src)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index.tolist())
    ax.set_ylabel(metric)
    ax.set_xlabel("Forecast threshold")
    ax.set_title(title or f"{metric} by categorical threshold")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(ncol=min(3, len(sources)))
    plt.show()
    plt.close(fig)
    return None


def compute_flood_proxy_fractional_coverage_by_category(
    df,
    forecast_cols=None,
    radii=None,
    radius_km=None,
    proxy_col="UFVS_ANY",
    bins=EXCLUSIVE_RISK_BINS,
    include_members=True,
    include_ensembles=True,
    include_wpc=True,
    allow_pp_proxy_fallback=False,
):
    """Compute fractional proxy coverage within each exclusive forecast risk bin.

    The primary fractional-coverage metric is conditional on the *forecast bin area*:

        Fraction Of Bin Covered By Proxy = count(proxy & forecast_bin) / count(forecast_bin)

    This is the Erickson-style interpretation needed for categorical areas: among all grid
    points forecast in 5-15%, 15-40%, 40-70%, or >=70%, what fraction was covered by the
    selected proxy/truth field?

    The table also reports:
      - Forecast Area Fraction: count(forecast_bin) / total grid count
      - Proxy Capture Fraction: count(proxy & forecast_bin) / total proxy count

    Proxy Capture Fraction is a secondary diagnostic and is *not* the fractional coverage
    plotted by default.
    """
    if radius_km is not None and radii is None:
        radii = [radius_km]

    if "ML_Target_Radius_km" in df.columns and not any(re.match(r"^ML_r\d+_Prob$", str(c)) for c in df.columns):
        eval_radii = _available_radii_from_df(df, requested=radii)
        if not eval_radii:
            eval_radii = [radius_km] if radius_km is not None else [None]
    else:
        eval_radii = [None]

    rows = []
    for r in eval_radii:
        sub = _subset_for_eval_radius(df, radius_km=r)
        if sub.empty:
            print(f"Skipping radius {r}: no rows.")
            continue
        fcst_cols = forecast_cols
        if fcst_cols is None:
            fcst_cols = _forecast_cols_for_evaluation(
                sub, radius_km=r, include_members=include_members,
                include_ensembles=include_ensembles, include_wpc=include_wpc,
            )
        fcst_cols = {k: v for k, v in fcst_cols.items() if v in sub.columns}
        if not fcst_cols:
            print(f"Skipping radius {r}: no forecast columns.")
            continue

        truth_col, truth_kind = _resolve_proxy_column(sub, proxy_col=proxy_col, allow_pp_fallback=allow_pp_proxy_fallback)
        # For raw UFVS/MRMS proxy fields, any value > 0 is treated as event coverage.
        # PP fallback is normally disabled; if explicitly enabled, PP >= 0.05 is treated as the event mask.
        if truth_kind == "pp_fallback":
            truth_all = _binary_from_col(sub, truth_col, threshold=0.05, mode=">=")
        else:
            truth_all = _binary_from_col(sub, truth_col, threshold=0.0, mode=">")

        date_series = sub["Date"].astype(str).str[:8] if "Date" in sub.columns else pd.Series("ALL", index=sub.index)
        total_proxy_count_all = int(np.sum(truth_all))
        total_grid_count_all = int(len(truth_all))

        for source_label, fcst_col in fcst_cols.items():
            fcst_vals_all = pd.to_numeric(sub[fcst_col], errors="coerce").to_numpy(float)
            source_radius = _source_radius_label(source_label, fallback_radius=(r if r is not None else "aggregate"))
            for lo, hi, bin_label in bins:
                in_bin_all = np.isfinite(fcst_vals_all) & (fcst_vals_all >= float(lo)) & (fcst_vals_all < float(hi))
                n_bin_all = int(np.sum(in_bin_all))
                n_overlap_all = int(np.sum(truth_all & in_bin_all))
                rows.append({
                    "Date": "POOLED",
                    "Source": source_label,
                    "Forecast Column": fcst_col,
                    "Proxy Column": truth_col,
                    "Truth Kind": truth_kind,
                    "Radius km": source_radius,
                    "Eval Radius km": r if r is not None else "all/aggregate",
                    "Risk Bin": bin_label,
                    "Risk Lo": float(lo),
                    "Risk Hi": float(hi),
                    "N In Bin": n_bin_all,
                    "N Proxy In Bin": n_overlap_all,
                    "N Proxy Total": total_proxy_count_all,
                    "N Grid Total": total_grid_count_all,
                    "Forecast Area Fraction": float(n_bin_all / total_grid_count_all) if total_grid_count_all > 0 else np.nan,
                    "Fraction Of Bin Covered By Proxy": float(n_overlap_all / n_bin_all) if n_bin_all > 0 else np.nan,
                    "Proxy Conditional Frequency": float(n_overlap_all / n_bin_all) if n_bin_all > 0 else np.nan,
                    "Proxy Capture Fraction": float(n_overlap_all / total_proxy_count_all) if total_proxy_count_all > 0 else np.nan,
                    # Backward-compatible old name. This is NOT the default fractional coverage denominator.
                    "Fraction Of Proxy Covered By Bin": float(n_overlap_all / total_proxy_count_all) if total_proxy_count_all > 0 else np.nan,
                    "Aggregation": "pooled",
                })

            for date, idx in sub.groupby(date_series).groups.items():
                g = sub.loc[idx]
                if truth_kind == "pp_fallback":
                    truth = _binary_from_col(g, truth_col, threshold=0.05, mode=">=")
                else:
                    truth = _binary_from_col(g, truth_col, threshold=0.0, mode=">")
                total_truth = int(np.sum(truth))
                total_grid = int(len(truth))
                vals = pd.to_numeric(g[fcst_col], errors="coerce").to_numpy(float)
                for lo, hi, bin_label in bins:
                    in_bin = np.isfinite(vals) & (vals >= float(lo)) & (vals < float(hi))
                    n_bin = int(np.sum(in_bin))
                    n_overlap = int(np.sum(truth & in_bin))
                    rows.append({
                        "Date": str(date),
                        "Source": source_label,
                        "Forecast Column": fcst_col,
                        "Proxy Column": truth_col,
                        "Truth Kind": truth_kind,
                        "Radius km": source_radius,
                        "Eval Radius km": r if r is not None else "all/aggregate",
                        "Risk Bin": bin_label,
                        "Risk Lo": float(lo),
                        "Risk Hi": float(hi),
                        "N In Bin": n_bin,
                        "N Proxy In Bin": n_overlap,
                        "N Proxy Total": total_truth,
                        "N Grid Total": total_grid,
                        "Forecast Area Fraction": float(n_bin / total_grid) if total_grid > 0 else np.nan,
                        "Fraction Of Bin Covered By Proxy": float(n_overlap / n_bin) if n_bin > 0 else np.nan,
                        "Proxy Conditional Frequency": float(n_overlap / n_bin) if n_bin > 0 else np.nan,
                        "Proxy Capture Fraction": float(n_overlap / total_truth) if total_truth > 0 else np.nan,
                        # Backward-compatible old name. This is NOT the default fractional coverage denominator.
                        "Fraction Of Proxy Covered By Bin": float(n_overlap / total_truth) if total_truth > 0 else np.nan,
                        "Aggregation": "case",
                    })

    case_table = pd.DataFrame([r for r in rows if r.get("Aggregation") == "case"])
    if case_table.empty:
        print("No fractional coverage rows could be computed.")
        return case_table, pd.DataFrame()
    metric_cols = [
        "N In Bin", "N Proxy In Bin", "N Proxy Total", "N Grid Total",
        "Forecast Area Fraction",
        "Fraction Of Bin Covered By Proxy",
        "Proxy Conditional Frequency",
        "Proxy Capture Fraction",
        "Fraction Of Proxy Covered By Bin",
    ]
    summary = (
        case_table.groupby(["Radius km", "Source", "Proxy Column", "Truth Kind", "Risk Bin"], as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )
    return case_table, summary


def plot_fractional_coverage_lines(summary_df, value_col="Fraction Of Bin Covered By Proxy", title=None):
    if summary_df is None or summary_df.empty:
        print("No data to plot.")
        return None
    if value_col not in summary_df.columns:
        print(f"Column {value_col!r} not found. Available columns: {list(summary_df.columns)}")
        return None
    bin_order = [lab for _, _, lab in EXCLUSIVE_RISK_BINS]
    fig, ax = plt.subplots(figsize=(10, 5.8))
    data = summary_df.copy()
    data["Risk Bin"] = pd.Categorical(data["Risk Bin"], categories=bin_order, ordered=True)
    data = data.sort_values(["Source", "Risk Bin"])
    for src, g in data.groupby("Source", sort=False):
        gg = g.drop_duplicates("Risk Bin").set_index("Risk Bin").reindex(bin_order)
        ax.plot(bin_order, gg[value_col].to_numpy(float), marker="o", linewidth=2, label=str(src))
    ax.set_xlabel("Exclusive forecast risk category")
    if value_col == "Fraction Of Bin Covered By Proxy":
        ax.set_ylabel("Proxy-covered fraction of forecast bin")
    elif value_col == "Proxy Capture Fraction":
        ax.set_ylabel("Fraction of proxy area captured by bin")
    else:
        ax.set_ylabel(value_col)
    ax.set_title(title or f"{value_col} by forecast risk category")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=9)
    plt.show()
    plt.close(fig)
    return None



def plot_fractional_coverage_bars(summary_df, value_col="Fraction Of Proxy Covered By Bin", title=None):
    if summary_df is None or summary_df.empty:
        print("No data to plot.")
        return None
    bin_order = [lab for _, _, lab in EXCLUSIVE_RISK_BINS]
    sources = list(summary_df["Source"].dropna().unique())
    pivot = summary_df.pivot_table(index="Risk Bin", columns="Source", values=value_col, aggfunc="mean").reindex(bin_order)
    x = np.arange(len(pivot.index))
    width = min(0.8 / max(len(sources), 1), 0.25)
    fig, ax = plt.subplots(figsize=(10, 5.5))
    for i, src in enumerate(sources):
        vals = pivot[src].to_numpy(float) if src in pivot.columns else np.full(len(x), np.nan)
        ax.bar(x + (i - (len(sources)-1)/2)*width, vals, width, label=src)
    ax.set_xticks(x)
    ax.set_xticklabels(pivot.index.tolist())
    ax.set_ylabel(value_col)
    ax.set_xlabel("Exclusive forecast risk category")
    ax.set_title(title or f"{value_col} by forecast risk category")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(ncol=min(3, len(sources)))
    plt.show()
    plt.close(fig)
    return None


# --------------------------------------------------------------------------------------
# Predictor side-by-side browser
# --------------------------------------------------------------------------------------

def _setup_predictor_browser_map(ax, title=None):
    if "_setup_map_ax" in globals():
        _setup_map_ax(ax, DEFAULT_EXTENT, show_states=True, show_countries=True, show_coastline=True)
    else:
        ax.set_xlim(DEFAULT_EXTENT[0], DEFAULT_EXTENT[1])
        ax.set_ylim(DEFAULT_EXTENT[2], DEFAULT_EXTENT[3])
    if title:
        ax.set_title(title)


def _plot_continuous_scatter(ax, lon, lat, vals, title, cmap="viridis", vmin=None, vmax=None, point_size=7, alpha=0.85):
    transform = ccrs.PlateCarree() if HAS_CARTOPY else None
    _setup_predictor_browser_map(ax, title=title)
    kwargs = dict(c=vals, s=point_size, alpha=alpha, cmap=cmap, vmin=vmin, vmax=vmax)
    if HAS_CARTOPY:
        sc = ax.scatter(lon, lat, transform=transform, **kwargs)
    else:
        sc = ax.scatter(lon, lat, **kwargs)
    plt.colorbar(sc, ax=ax, shrink=0.85)
    return sc


def _load_feature_names_for_radius(radius_km):
    art = find_artifacts_for_radius(int(round(float(radius_km))))
    feats = _load_feature_names(art["features_path"])
    return art, feats


def _load_predictor_source_for_browser(date, radius_km=40, source="historical", predictor_cols=None):
    date = _date8_local(date)
    r = int(round(float(radius_km)))
    art, feats = _load_feature_names_for_radius(r)
    if predictor_cols is None:
        predictor_cols = feats
    predictor_cols = [c for c in predictor_cols if c in feats]
    base_cols = ["Date", "Lat", "Lon"]

    source = str(source).lower()
    if source.startswith("real"):
        fpath = realtime_feature_cache_path(date, r)
        if not Path(fpath).exists():
            raise RuntimeError(f"Realtime feature cache not found: {fpath}. Run the realtime builder with FORCE_REALTIME_FEATURES=True first.")
        cols_avail = _parquet_columns(str(fpath)) if "_parquet_columns" in globals() else list(pd.read_parquet(fpath, columns=[]).columns)
        read_cols = [c for c in base_cols + predictor_cols if c in cols_avail]
        dfp = pd.read_parquet(fpath, columns=read_cols)
    else:
        fpath = art["master_path"]
        cols_avail = _parquet_columns(str(fpath)) if "_parquet_columns" in globals() else None
        read_cols = [c for c in base_cols + predictor_cols if (cols_avail is None or c in cols_avail)]
        dfp = pd.read_parquet(fpath, columns=read_cols)
        dfp["Date"] = dfp["Date"].astype(str).str[:8]
        dfp = dfp[dfp["Date"] == date].copy()
    return dfp, [c for c in predictor_cols if c in dfp.columns]


def _viewer_subset_for_predictor_browser(date, radius_km=40, source="historical", ml_field=None):
    date = _date8_local(date)
    r = int(round(float(radius_km)))
    source = str(source).lower()
    if source.startswith("real"):
        if "df_realtime_viewer" not in globals():
            raise RuntimeError("df_realtime_viewer not found. Run the realtime cell first.")
        dfv = df_realtime_viewer.copy()
        if ml_field is None:
            cand = f"ML_r{r}_Prob"
            ml_field = cand if cand in dfv.columns else ("ML_Radius_PMM" if "ML_Radius_PMM" in dfv.columns else "ML_Radius_EnsMean")
    else:
        if "df_radius_viewer" not in globals():
            raise RuntimeError("df_radius_viewer not found. Run the historical viewer load cell first.")
        dfv = df_radius_viewer.copy()
        if "ML_Target_Radius_km" in dfv.columns:
            dfv = dfv[pd.to_numeric(dfv["ML_Target_Radius_km"], errors="coerce") == r].copy()
        ml_field = ml_field or "ML_Forecast_Prob"
    dfv["Date"] = dfv["Date"].astype(str).str[:8]
    sub = dfv[dfv["Date"] == date].copy()
    if sub.empty:
        raise RuntimeError(f"No viewer rows found for date={date}, radius={r}, source={source}.")
    if ml_field not in sub.columns:
        raise RuntimeError(f"ML field {ml_field} not found. Available ML-like columns: {[c for c in sub.columns if str(c).startswith('ML_') or str(c).startswith('ML_r')][:50]}")
    return sub[["Date", "Lat", "Lon", ml_field]].copy(), ml_field


def plot_ml_next_to_predictor(date, radius_km=40, predictor_name=None, source="historical", ml_field=None, point_size=7, alpha=0.85):
    r = int(round(float(radius_km)))
    art, feats = _load_feature_names_for_radius(r)
    if predictor_name is None:
        predictor_name = feats[0]
    if predictor_name not in feats:
        raise RuntimeError(f"Predictor {predictor_name!r} is not in the saved feature list for r{r}.")

    pred_df, found = _load_predictor_source_for_browser(date, r, source=source, predictor_cols=[predictor_name])
    if predictor_name not in found:
        raise RuntimeError(f"Predictor {predictor_name!r} not found in {source} predictor source for date={date}, r={r}.")
    ml_df, ml_field = _viewer_subset_for_predictor_browser(date, r, source=source, ml_field=ml_field)
    merged = ml_df.merge(pred_df[["Date", "Lat", "Lon", predictor_name]], on=["Date", "Lat", "Lon"], how="left")

    lon = pd.to_numeric(merged["Lon"], errors="coerce").to_numpy(float)
    lat = pd.to_numeric(merged["Lat"], errors="coerce").to_numpy(float)
    pred_vals = pd.to_numeric(merged[predictor_name], errors="coerce").to_numpy(float)

    subplot_kw = {"projection": ccrs.PlateCarree()} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(1, 2, figsize=(15.5, 5.8), subplot_kw=subplot_kw, constrained_layout=True)

    _plot_prob_or_smoothed(
        axes[0], merged.rename(columns={ml_field: "__ML_FIELD__"}), "__ML_FIELD__",
        f"ML forecast: {ml_field} | {date} | r{r}",
        smooth_sigma_grid=0, point_size=point_size, alpha=alpha,
    )

    good = np.isfinite(pred_vals)
    vmin = np.nanpercentile(pred_vals[good], 2) if good.any() else None
    vmax = np.nanpercentile(pred_vals[good], 98) if good.any() else None
    if vmin is not None and vmax is not None and np.isclose(vmin, vmax):
        vmin, vmax = None, None
    _plot_continuous_scatter(
        axes[1], lon, lat, pred_vals,
        title=predictor_name,
        cmap="viridis", vmin=vmin, vmax=vmax, point_size=point_size, alpha=alpha,
    )
    plt.show()
    plt.close(fig)
    return None


def launch_predictor_browser(initial_date=None, initial_radius=40, source="historical"):
    if not HAS_WIDGETS:
        raise RuntimeError("ipywidgets is not available in this notebook environment.")

    source_options = ["historical", "realtime"]
    source_dd = widgets.Dropdown(options=source_options, value=source if source in source_options else "historical", description="Source")
    radius_dd = widgets.Dropdown(options=[40, 60, 75, 100], value=int(initial_radius), description="Radius")

    def _dates_for(src, rr):
        if src == "realtime" and "df_realtime_viewer" in globals():
            return sorted(df_realtime_viewer["Date"].astype(str).str[:8].unique().tolist())
        if "df_radius_viewer" in globals():
            g = df_radius_viewer.copy()
            if "ML_Target_Radius_km" in g.columns:
                g = g[pd.to_numeric(g["ML_Target_Radius_km"], errors="coerce") == int(rr)]
            return sorted(g["Date"].astype(str).str[:8].unique().tolist())
        return []

    dates0 = _dates_for(source_dd.value, radius_dd.value)
    date_dd = widgets.Dropdown(options=dates0, value=(initial_date if initial_date in dates0 else (dates0[0] if dates0 else None)), description="Date")

    _, feats0 = _load_feature_names_for_radius(radius_dd.value)
    feat_slider = widgets.IntSlider(value=0, min=0, max=max(len(feats0)-1, 0), step=1, description="Feature #")
    feat_label = widgets.HTML(value=f"<b>{feats0[0] if feats0 else 'None'}</b>")
    ml_field_text = widgets.Text(value="", description="ML field", placeholder="blank = default")
    out = widgets.Output()

    def _refresh_controls(*args):
        rr = int(radius_dd.value)
        dts = _dates_for(source_dd.value, rr)
        date_dd.options = dts
        if dts and date_dd.value not in dts:
            date_dd.value = dts[0]
        _, feats = _load_feature_names_for_radius(rr)
        feat_slider.max = max(len(feats)-1, 0)
        if feat_slider.value > feat_slider.max:
            feat_slider.value = 0
        feat_label.value = f"<b>{feats[feat_slider.value] if feats else 'None'}</b>"

    def _plot(*args):
        _refresh_controls()
        rr = int(radius_dd.value)
        _, feats = _load_feature_names_for_radius(rr)
        if not feats or date_dd.value is None:
            return
        feat = feats[int(feat_slider.value)]
        ml_field = ml_field_text.value.strip() or None
        with out:
            out.clear_output(wait=True)
            plot_ml_next_to_predictor(
                date=date_dd.value,
                radius_km=rr,
                predictor_name=feat,
                source=source_dd.value,
                ml_field=ml_field,
            )

    for w in [source_dd, radius_dd, date_dd, feat_slider, ml_field_text]:
        w.observe(lambda change: _plot(), names="value")

    display(widgets.VBox([widgets.HBox([source_dd, date_dd, radius_dd]), feat_slider, feat_label, ml_field_text]), out)
    _plot()


# ======================================================================================
# v3 overrides: auto-select realtime multi-radius dataframe and robustly expose all members
# ======================================================================================


def _wide_radius_member_cols(df):
    """Return {label: col} for all wide-format radius member probability columns."""
    out = {}
    patterns = [
        r"^ML_r(?P<r>\d+)_Prob$",
        r"^ML_R(?P<r>\d+)_Prob$",
        r"^ML_(?P<r>\d+)km_Prob$",
        r"^ML_r(?P<r>\d+)km_Prob$",
    ]
    for c in df.columns:
        sc = str(c)
        for pat in patterns:
            m = re.match(pat, sc)
            if m:
                rr = int(m.group("r"))
                out[f"ML r{rr}km"] = c
                break
    return dict(sorted(out.items(), key=lambda kv: int(re.search(r"r(\d+)", kv[0]).group(1))))


def _long_available_radii(df):
    if "ML_Target_Radius_km" not in df.columns:
        return []
    vals = pd.to_numeric(df["ML_Target_Radius_km"], errors="coerce").dropna().astype(int).unique().tolist()
    return sorted([int(v) for v in vals if int(v) >= 0])


def choose_evaluation_dataframe(prefer_realtime=True):
    """Prefer realtime multi-radius df when it exists; otherwise use historical df_radius_viewer."""
    if prefer_realtime and "df_realtime_viewer" in globals():
        try:
            df = globals()["df_realtime_viewer"]
            if isinstance(df, pd.DataFrame) and (len(_wide_radius_member_cols(df)) > 0):
                return df, "realtime multi-radius df_realtime_viewer"
        except Exception:
            pass
    if "df_radius_viewer" in globals():
        df = globals()["df_radius_viewer"]
        if isinstance(df, pd.DataFrame):
            return df, "historical long-format df_radius_viewer"
    raise RuntimeError("No evaluation dataframe found. Run the historical viewer load cell or the multi-radius realtime cell first.")


def print_available_forecasts_for_eval(df, label="evaluation dataframe"):
    wide = _wide_radius_member_cols(df)
    long = _long_available_radii(df)
    ens = [c for c in ["ML_Radius_EnsMean", "ML_Radius_EnsMedian", "ML_Radius_EnsMax", "ML_Radius_PMM", "ML_Radius_EnsSpread"] if c in df.columns]
    print(f"Available forecasts in {label}:")
    print(f"  wide radius members: {list(wide.keys())}")
    print(f"  long-format radii: {long}")
    print(f"  ensemble columns: {ens}")
    print(f"  WPC_ERO_Risk: {'yes' if 'WPC_ERO_Risk' in df.columns else 'no'}")
    print(f"  PP columns: {[c for c in df.columns if str(c).startswith('PP_')][:20]}")
    print(f"  UFVS columns: {[c for c in df.columns if str(c).startswith('UFVS_')][:20]}")


def _forecast_cols_for_evaluation(df, radius_km=None, include_members=True, include_ensembles=True, include_wpc=True):
    """Return label->column mapping for historical-long or realtime-wide evaluation.

    This override is intentionally permissive and prints all available radius members through
    print_available_forecasts_for_eval(...). Historical long-format data is evaluated one
    radius at a time by compute_categorical_area_skill(...); realtime wide data contains one
    column per radius member.
    """
    out = {}
    r = None if radius_km is None else int(round(float(radius_km)))

    if include_members:
        wide = _wide_radius_member_cols(df)
        for label, col in wide.items():
            rr = int(re.search(r"r(\d+)", label).group(1))
            if r is None or rr == r:
                out[label] = col

    # Historical long-format ML field. Only add this when no wide member was found for this subset.
    if include_members and "ML_Forecast_Prob" in df.columns and not any(str(k).startswith("ML r") for k in out):
        if r is not None:
            labels = df["ML_Model_Label"].dropna().astype(str).unique().tolist() if "ML_Model_Label" in df.columns else [model_label_for_radius(r)]
            out[f"ML {labels[0]}"] = "ML_Forecast_Prob"
        else:
            # If long-format radii exist and no explicit r was provided, do not collapse them into one line.
            # compute_categorical_area_skill will loop each radius and call us with r.
            if not _long_available_radii(df):
                out["ML"] = "ML_Forecast_Prob"

    if include_ensembles:
        for label, col in [
            ("ML ens mean", "ML_Radius_EnsMean"),
            ("ML ens median", "ML_Radius_EnsMedian"),
            ("ML ens max", "ML_Radius_EnsMax"),
            ("ML PMM", "ML_Radius_PMM"),
            ("ML spread", "ML_Radius_EnsSpread"),
        ]:
            if col in df.columns:
                out[label] = col

    if include_wpc and "WPC_ERO_Risk" in df.columns:
        out["WPC ERO"] = "WPC_ERO_Risk"

    return out


def _source_sort_key(label):
    s = str(label)
    m = re.search(r"ML r(\d+)", s)
    if m:
        return (0, int(m.group(1)), s)
    order = {"ML ens mean": 100, "ML ens median": 101, "ML ens max": 102, "ML PMM": 103, "ML spread": 104, "WPC ERO": 200}
    return (1, order.get(s, 999), s)


def plot_categorical_skill_lines(summary_df, metric="CSI", title=None, include_wpc=True):
    """Plot all radius members / ensembles / WPC on one threshold-vs-metric plot."""
    if summary_df is None or summary_df.empty:
        print("No data to plot.")
        return None
    threshold_order = [lab for _, lab in CATEGORICAL_THRESHOLDS]
    fig, ax = plt.subplots(figsize=(11, 6))
    data = summary_df.copy()
    data["Threshold"] = pd.Categorical(data["Threshold"], categories=threshold_order, ordered=True)
    sources = sorted(data["Source"].dropna().unique().tolist(), key=_source_sort_key)
    print(f"Plotting {metric} sources: {sources}")
    for src in sources:
        if (not include_wpc) and str(src).lower().startswith("wpc"):
            continue
        g = data[data["Source"] == src].copy()
        # If WPC is repeated for multiple eval radii, average duplicates at each threshold.
        gg = g.groupby("Threshold", observed=False)[metric].mean().reindex(threshold_order)
        ax.plot(threshold_order, gg.to_numpy(float), marker="o", linewidth=2, label=str(src))
    ax.set_xlabel("Forecast threshold")
    ax.set_ylabel(metric)
    ax.set_title(title or f"{metric} by categorical threshold")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=9)
    plt.show()
    plt.close(fig)
    return None


def plot_fractional_coverage_lines(summary_df, value_col="Fraction Of Proxy Covered By Bin", title=None):
    """Plot all radius members / ensembles / WPC on one exclusive-bin fractional coverage plot."""
    if summary_df is None or summary_df.empty:
        print("No data to plot.")
        return None
    bin_order = [lab for _, _, lab in EXCLUSIVE_RISK_BINS]
    fig, ax = plt.subplots(figsize=(11, 6))
    data = summary_df.copy()
    data["Risk Bin"] = pd.Categorical(data["Risk Bin"], categories=bin_order, ordered=True)
    sources = sorted(data["Source"].dropna().unique().tolist(), key=_source_sort_key)
    print(f"Plotting {value_col} sources: {sources}")
    for src in sources:
        g = data[data["Source"] == src].copy()
        gg = g.groupby("Risk Bin", observed=False)[value_col].mean().reindex(bin_order)
        ax.plot(bin_order, gg.to_numpy(float), marker="o", linewidth=2, label=str(src))
    ax.set_xlabel("Exclusive forecast risk category")
    ax.set_ylabel(value_col)
    ax.set_title(title or f"{value_col} by forecast risk category")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=9)
    plt.show()
    plt.close(fig)
    return None

# ======================================================================================
# V4 OVERRIDES: all-radius threshold-exceedance verification, reliability, and Brier score
# ======================================================================================
# These override/extend the integrated metric helpers above.  CSI/POD/FAR/Bias use
# cumulative thresholds (>5%, >15%, >40%, >70%), not exclusive bins.  Fractional-coverage
# diagnostics may still use exclusive bins because those are category-area diagnostics.

THRESHOLD_EXCEEDANCE_BINS = CATEGORICAL_THRESHOLDS  # [(0.05, '>5%'), ...]


def _has_wide_radius_members(df):
    return any(_wide_ml_member_metadata(c)[0] is not None for c in df.columns)


def _iter_eval_sources(
    df,
    radii=None,
    include_members=True,
    include_ensembles=True,
    include_wpc=True,
):
    """Yield (source_label, forecast_col, sub_df, source_radius).

    Supports both:
      * historical long format: ML_Forecast_Prob + ML_Target_Radius_km
      * realtime wide format: ML_r40_Prob, ML_r60_Prob, ...

    WPC is yielded once, not once per radius, because it is not radius-dependent.
    """
    if radii is not None:
        requested = [int(round(float(r))) for r in radii]
    else:
        requested = None

    # Wide realtime dataframe: one column per radius member.
    if _has_wide_radius_members(df):
        if include_members:
            for c in sorted([c for c in df.columns if _wide_ml_member_metadata(c)[0] is not None], key=lambda x: (_wide_ml_member_metadata(x)[1], _wide_ml_member_metadata(x)[0])):
                member_label, rr = _wide_ml_member_metadata(c)
                if requested is None or rr in requested:
                    yield f"ML {member_label}", c, df.copy(), rr
        if include_ensembles:
            for label, col in [
                ("ML ens mean", "ML_Radius_EnsMean"),
                ("ML ens median", "ML_Radius_EnsMedian"),
                ("ML ens max", "ML_Radius_EnsMax"),
                ("ML PMM", "ML_Radius_PMM"),
                ("ML spread", "ML_Radius_EnsSpread"),
            ]:
                if col in df.columns:
                    yield label, col, df.copy(), "aggregate"
        if include_wpc and "WPC_ERO_Risk" in df.columns:
            wpc = df[pd.to_numeric(df["WPC_ERO_Risk"], errors="coerce").notna()].copy()
            if not wpc.empty:
                yield "WPC ERO", "WPC_ERO_Risk", wpc, "wpc"
        return

    # Historical long dataframe: one ML_Forecast_Prob column, subset by target radius.
    if "ML_Target_Radius_km" in df.columns and "ML_Forecast_Prob" in df.columns:
        available = _available_radii_from_df(df, requested=requested)
        if include_members:
            model_rows = df[["ML_Model_Label", "ML_Target_Radius_km"]].drop_duplicates().to_dict("records") if "ML_Model_Label" in df.columns else [{"ML_Model_Label": model_label_for_radius(rr), "ML_Target_Radius_km": rr} for rr in available]
            for model_row in model_rows:
                rr = int(model_row["ML_Target_Radius_km"])
                member_label = str(model_row["ML_Model_Label"])
                if requested is not None and rr not in requested:
                    continue
                mask = pd.to_numeric(df["ML_Target_Radius_km"], errors="coerce").astype("Int64") == rr
                if "ML_Model_Label" in df.columns:
                    mask &= df["ML_Model_Label"].astype(str) == member_label
                sub = df[mask].copy()
                if not sub.empty:
                    yield f"ML {member_label}", "ML_Forecast_Prob", sub, rr
        if include_wpc and "WPC_ERO_Risk" in df.columns:
            # WPC is independent of the ML radius, so use the first available radius subset to avoid duplicate WPC lines.
            if available:
                rr = available[0]
                sub = df[pd.to_numeric(df["ML_Target_Radius_km"], errors="coerce").astype("Int64") == int(rr)].copy()
            else:
                sub = df.copy()
            sub = sub[pd.to_numeric(sub["WPC_ERO_Risk"], errors="coerce").notna()].copy()
            if not sub.empty:
                yield "WPC ERO", "WPC_ERO_Risk", sub, "wpc"
        return

    # Generic fallback.
    if include_members and "ML_Forecast_Prob" in df.columns:
        yield "ML", "ML_Forecast_Prob", df.copy(), "ml"
    if include_wpc and "WPC_ERO_Risk" in df.columns:
        wpc = df[pd.to_numeric(df["WPC_ERO_Risk"], errors="coerce").notna()].copy()
        if not wpc.empty:
            yield "WPC ERO", "WPC_ERO_Risk", wpc, "wpc"


def _resolve_truth_for_subset(
    sub,
    truth_mode="pp",
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    allow_pp_proxy_fallback=False,
    proxy_truth_threshold=0.0,
    pp_event_threshold=0.05,
):
    """Return truth_col, truth_kind, and a binary truth vector for reliability/BS.

    For CSI at threshold t, callers should threshold PP at t.  For BS/reliability,
    this returns an event/non-event truth using pp_event_threshold for PP-like fields.
    """
    truth_mode = str(truth_mode).lower()
    if truth_mode == "pp":
        truth_col = _truth_col_from_pp_definition_integrated(pp_definition)
        if truth_col not in sub.columns:
            raise RuntimeError(f"Missing PP truth column {truth_col}. Available PP columns: {[c for c in sub.columns if str(c).startswith('PP_')]}")
        truth_kind = "pp"
        y = _binary_from_col(sub, truth_col, threshold=pp_event_threshold, mode=">=")
        return truth_col, truth_kind, y
    elif truth_mode == "proxy":
        truth_col, truth_kind = _resolve_proxy_column(sub, proxy_col=proxy_col, allow_pp_fallback=allow_pp_proxy_fallback)
        if truth_kind == "pp_fallback":
            print(f"Raw proxy {proxy_col!r} not available for this dataframe/source; using {truth_col!r} with threshold >= {pp_event_threshold}.")
            y = _binary_from_col(sub, truth_col, threshold=pp_event_threshold, mode=">=")
        else:
            y = _binary_from_col(sub, truth_col, threshold=proxy_truth_threshold, mode=">")
        return truth_col, truth_kind, y
    else:
        raise RuntimeError("truth_mode must be 'pp' or 'proxy'.")


def compute_categorical_area_skill(
    df,
    forecast_cols=None,
    radii=None,
    radius_km=None,
    truth_mode="pp",
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    proxy_truth_threshold=0.0,
    thresholds=THRESHOLD_EXCEEDANCE_BINS,
    include_members=True,
    include_ensembles=True,
    include_wpc=True,
    allow_pp_proxy_fallback=False,
):
    """Compute CSI/POD/FAR/Bias using cumulative threshold exceedances.

    Thresholds are >5%, >15%, >40%, >70% style exceedances, not exclusive 5--15% bins.
    All available radius members are included in one output table/plot.
    """
    if radius_km is not None and radii is None:
        radii = [radius_km]

    if forecast_cols is not None:
        # Explicit forecast columns: evaluate all on the full df.
        sources = [(label, col, df.copy(), "explicit") for label, col in forecast_cols.items() if col in df.columns]
    else:
        sources = list(_iter_eval_sources(df, radii=radii, include_members=include_members, include_ensembles=include_ensembles, include_wpc=include_wpc))

    case_rows = []
    pooled_rows = []
    for source_label, fcst_col, sub, source_radius in sources:
        if sub.empty or fcst_col not in sub.columns:
            continue

        if str(truth_mode).lower() == "pp":
            truth_col = _truth_col_from_pp_definition_integrated(pp_definition)
            if truth_col not in sub.columns:
                print(f"Skipping {source_label}: missing PP truth column {truth_col}.")
                continue
            truth_kind = "pp"
        else:
            try:
                truth_col, truth_kind = _resolve_proxy_column(sub, proxy_col=proxy_col, allow_pp_fallback=allow_pp_proxy_fallback)
            except Exception as exc:
                print(f"Skipping {source_label}: {exc}")
                continue

        fcst_vals_all = pd.to_numeric(sub[fcst_col], errors="coerce").to_numpy(float)
        date_series = sub["Date"].astype(str).str[:8] if "Date" in sub.columns else pd.Series("ALL", index=sub.index)

        for thr, thr_label in thresholds:
            fcst_yes_all = np.isfinite(fcst_vals_all) & (fcst_vals_all >= float(thr))
            if str(truth_mode).lower() == "pp" or truth_kind == "pp_fallback":
                truth_yes_all = _binary_from_col(sub, truth_col, threshold=float(thr), mode=">=")
            else:
                truth_yes_all = _binary_from_col(sub, truth_col, threshold=proxy_truth_threshold, mode=">")

            pooled = _contingency_stats(fcst_yes_all, truth_yes_all)
            pooled.update({
                "Date": "POOLED",
                "Source": source_label,
                "Forecast Column": fcst_col,
                "Source Radius": source_radius,
                "Truth Mode": truth_mode,
                "Truth Column": truth_col,
                "Truth Kind": truth_kind,
                "Threshold": thr_label,
                "Threshold Value": float(thr),
            })
            pooled_rows.append(pooled)

            for d in sorted(date_series.dropna().unique()):
                m = (date_series == d).to_numpy()
                if not np.any(m):
                    continue
                stats = _contingency_stats(fcst_yes_all[m], truth_yes_all[m])
                stats.update({
                    "Date": str(d),
                    "Source": source_label,
                    "Forecast Column": fcst_col,
                    "Source Radius": source_radius,
                    "Truth Mode": truth_mode,
                    "Truth Column": truth_col,
                    "Truth Kind": truth_kind,
                    "Threshold": thr_label,
                    "Threshold Value": float(thr),
                })
                case_rows.append(stats)

    case_table = pd.DataFrame(case_rows)
    pooled_table = pd.DataFrame(pooled_rows)
    if case_table.empty:
        return case_table, case_table.copy(), pooled_table

    metric_cols = [
        "CSI", "POD", "FAR", "Bias",
        "Forecast Area Fraction", "Truth Area Fraction",
        "Hits", "Misses", "False Alarms", "Correct Negatives", "N Points",
    ]
    case_mean = (
        case_table.groupby(["Source", "Forecast Column", "Source Radius", "Truth Mode", "Truth Column", "Truth Kind", "Threshold", "Threshold Value"], as_index=False)[metric_cols]
        .mean(numeric_only=True)
        .sort_values(["Threshold Value", "Source"], key=lambda x: x.map(_source_sort_key) if x.name == "Source" else x)
    )
    return case_table, case_mean, pooled_table


def plot_categorical_skill_lines(summary_df, metric="CSI", title=None, include_wpc=True):
    """Plot all sources/radius tests on the same cumulative-threshold plot."""
    if summary_df is None or summary_df.empty:
        print("No data to plot.")
        return None
    threshold_order = [lab for _, lab in THRESHOLD_EXCEEDANCE_BINS]
    fig, ax = plt.subplots(figsize=(11, 6.2))
    data = summary_df.copy()
    data["Threshold"] = pd.Categorical(data["Threshold"], categories=threshold_order, ordered=True)
    data = data.sort_values(["Source", "Threshold"], key=lambda x: x.map(_source_sort_key) if x.name == "Source" else x)
    for src, g in data.groupby("Source", sort=False):
        if (not include_wpc) and str(src).lower().startswith("wpc"):
            continue
        gg = g.drop_duplicates("Threshold").set_index("Threshold").reindex(threshold_order)
        ax.plot(threshold_order, gg[metric].to_numpy(float), marker="o", linewidth=2, label=str(src))
    ax.set_xlabel("Cumulative forecast threshold")
    ax.set_ylabel(metric)
    ax.set_title(title or f"{metric} by threshold exceedance")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=9)
    plt.show()
    plt.close(fig)
    return None


def compute_reliability_all_sources(
    df,
    radii=None,
    truth_mode="pp",
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    pp_event_threshold=0.05,
    proxy_truth_threshold=0.0,
    bins=np.linspace(0.0, 1.0, 11),
    include_members=True,
    include_ensembles=True,
    include_wpc=True,
    allow_pp_proxy_fallback=False,
):
    """Compute reliability tables for all available radius tests/sources."""
    rows = []
    for source_label, fcst_col, sub, source_radius in _iter_eval_sources(
        df, radii=radii, include_members=include_members, include_ensembles=include_ensembles, include_wpc=include_wpc
    ):
        try:
            truth_col, truth_kind, y = _resolve_truth_for_subset(
                sub, truth_mode=truth_mode, pp_definition=pp_definition, proxy_col=proxy_col,
                allow_pp_proxy_fallback=allow_pp_proxy_fallback, proxy_truth_threshold=proxy_truth_threshold,
                pp_event_threshold=pp_event_threshold,
            )
        except Exception as exc:
            print(f"Skipping reliability for {source_label}: {exc}")
            continue
        p = pd.to_numeric(sub[fcst_col], errors="coerce").to_numpy(float)
        good = np.isfinite(p) & np.isfinite(y.astype(float))
        p = p[good]
        y = y[good]
        if len(p) == 0:
            continue
        # Bin convention: [lo, hi), final includes hi.
        for i in range(len(bins) - 1):
            lo, hi = float(bins[i]), float(bins[i+1])
            if i == len(bins) - 2:
                m = (p >= lo) & (p <= hi)
            else:
                m = (p >= lo) & (p < hi)
            if not np.any(m):
                rows.append({
                    "Source": source_label, "Forecast Column": fcst_col, "Source Radius": source_radius,
                    "Truth Mode": truth_mode, "Truth Column": truth_col, "Truth Kind": truth_kind,
                    "Bin Low": lo, "Bin High": hi, "Bin Center": (lo + hi) / 2,
                    "N": 0, "Mean Forecast": np.nan, "Observed Frequency": np.nan,
                })
                continue
            rows.append({
                "Source": source_label, "Forecast Column": fcst_col, "Source Radius": source_radius,
                "Truth Mode": truth_mode, "Truth Column": truth_col, "Truth Kind": truth_kind,
                "Bin Low": lo, "Bin High": hi, "Bin Center": (lo + hi) / 2,
                "N": int(np.sum(m)),
                "Mean Forecast": float(np.mean(p[m])),
                "Observed Frequency": float(np.mean(y[m])),
            })
    return pd.DataFrame(rows)


def plot_reliability_all_sources(rel_df, min_bin_n=1, title=None):
    if rel_df is None or rel_df.empty:
        print("No reliability data to plot.")
        return None
    fig, ax = plt.subplots(figsize=(7.8, 7.0))
    ax.plot([0, 1], [0, 1], color="0.3", linestyle="--", linewidth=1.5, label="Perfect reliability")
    data = rel_df[rel_df["N"] >= int(min_bin_n)].copy()
    data = data.sort_values(["Source", "Bin Center"], key=lambda x: x.map(_source_sort_key) if x.name == "Source" else x)
    for src, g in data.groupby("Source", sort=False):
        ax.plot(g["Mean Forecast"], g["Observed Frequency"], marker="o", linewidth=2, label=str(src))
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.set_xlabel("Mean forecast probability")
    ax.set_ylabel("Observed frequency")
    ax.set_title(title or "Reliability diagram: all available radius tests")
    ax.grid(True, alpha=0.3)
    ax.legend(ncol=1, fontsize=8)
    plt.show()
    plt.close(fig)
    return None


def compute_brier_scores_all_sources(
    df,
    radii=None,
    truth_mode="pp",
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    pp_event_threshold=0.05,
    proxy_truth_threshold=0.0,
    include_members=True,
    include_ensembles=True,
    include_wpc=True,
    allow_pp_proxy_fallback=False,
):
    """Compute Brier Score and BSS for all radius tests/sources."""
    pooled_rows = []
    case_rows = []
    for source_label, fcst_col, sub, source_radius in _iter_eval_sources(
        df, radii=radii, include_members=include_members, include_ensembles=include_ensembles, include_wpc=include_wpc
    ):
        try:
            truth_col, truth_kind, y = _resolve_truth_for_subset(
                sub, truth_mode=truth_mode, pp_definition=pp_definition, proxy_col=proxy_col,
                allow_pp_proxy_fallback=allow_pp_proxy_fallback, proxy_truth_threshold=proxy_truth_threshold,
                pp_event_threshold=pp_event_threshold,
            )
        except Exception as exc:
            print(f"Skipping Brier score for {source_label}: {exc}")
            continue
        p = pd.to_numeric(sub[fcst_col], errors="coerce").to_numpy(float)
        good = np.isfinite(p) & np.isfinite(y.astype(float))
        p = np.clip(p[good], 0, 1)
        y = y[good].astype(float)
        if len(p) == 0:
            continue
        clim = float(np.mean(y))
        bs = float(np.mean((p - y) ** 2))
        bs_ref = float(np.mean((clim - y) ** 2))
        bss = float(1.0 - bs / bs_ref) if bs_ref > 0 else np.nan
        pooled_rows.append({
            "Source": source_label, "Forecast Column": fcst_col, "Source Radius": source_radius,
            "Truth Mode": truth_mode, "Truth Column": truth_col, "Truth Kind": truth_kind,
            "N": len(p), "Climatology": clim, "Brier Score": bs, "Reference BS": bs_ref, "BSS": bss,
        })
        date_series = sub.loc[good, "Date"].astype(str).str[:8] if "Date" in sub.columns else pd.Series("ALL", index=np.arange(len(p)))
        # Need aligned arrays after good mask.
        for d in sorted(pd.Series(date_series).dropna().unique()):
            m = (pd.Series(date_series).to_numpy() == d)
            if not np.any(m):
                continue
            yy = y[m]
            pp = p[m]
            bs_d = float(np.mean((pp - yy) ** 2))
            bsref_d = float(np.mean((clim - yy) ** 2))
            bss_d = float(1.0 - bs_d / bsref_d) if bsref_d > 0 else np.nan
            case_rows.append({
                "Date": str(d), "Source": source_label, "Forecast Column": fcst_col, "Source Radius": source_radius,
                "Truth Mode": truth_mode, "Truth Column": truth_col, "Truth Kind": truth_kind,
                "N": int(np.sum(m)), "Climatology": clim, "Brier Score": bs_d, "Reference BS": bsref_d, "BSS": bss_d,
            })
    case_table = pd.DataFrame(case_rows)
    pooled_table = pd.DataFrame(pooled_rows)
    if case_table.empty:
        return case_table, case_table.copy(), pooled_table
    summary = (
        case_table.groupby(["Source", "Forecast Column", "Source Radius", "Truth Mode", "Truth Column", "Truth Kind"], as_index=False)[["Brier Score", "Reference BS", "BSS", "N", "Climatology"]]
        .mean(numeric_only=True)
        .sort_values("Source", key=lambda x: x.map(_source_sort_key))
    )
    return case_table, summary, pooled_table


def plot_brier_scores_all_sources(summary_df, metric="Brier Score", title=None):
    if summary_df is None or summary_df.empty:
        print("No Brier-score data to plot.")
        return None
    data = summary_df.copy().sort_values("Source", key=lambda x: x.map(_source_sort_key))
    fig, ax = plt.subplots(figsize=(11, 5.8))
    ax.bar(data["Source"].astype(str), data[metric].to_numpy(float))
    ax.set_ylabel(metric)
    ax.set_xlabel("Forecast source / radius member")
    ax.set_title(title or f"{metric}: all available radius tests")
    ax.grid(True, axis="y", alpha=0.3)
    ax.tick_params(axis="x", rotation=35)
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    return None

print("V4 all-radius threshold-exceedance metrics loaded: CSI/POD/FAR/Bias, reliability, and Brier/BSS.")


# ======================================================================================
# V7 STRICT RAW-PROXY OVERRIDES
# ======================================================================================
# Conceptual rule:
#   PP truth uses PP_* columns.
#   Proxy truth uses raw UFVS_* columns only.
#   There is no PP proxy fallback, because PP is derived from the proxies.
#
# If a historical dataframe is missing UFVS_* columns, the code tries to find a local
# parquet with raw UFVS_* columns and merge them by Date/Lat/Lon. If it cannot find one,
# proxy-truth metrics raise/skip with a clear message instead of substituting PP.

import os as _os
import glob as _glob

_RAW_UFVS_PROXY_TABLE_CACHE = {}


def _possible_project_dirs_for_raw_proxy_search():
    out = []
    if "PROJECT_DIR" in globals() and PROJECT_DIR:
        out.append(str(PROJECT_DIR))
    for p in [
        "/home/tyreekfrazier/ISU_Research_LOCAL_RUN/fall_2025_ml_proj",
        "/home/tyreekfrazier/ISU_Research/fall_2025_ml_proj",
        "/home/tyreekfrazier/ISU_RESEARCH_LOCAL_RUN/fall_2025_ml_proj",
    ]:
        if p not in out:
            out.append(p)
    return [p for p in out if _os.path.isdir(p)]


def _parquet_schema_columns(path):
    try:
        import pyarrow.parquet as pq
        return list(pq.read_schema(path).names)
    except Exception:
        try:
            return list(pd.read_parquet(path, engine="pyarrow").columns)
        except Exception:
            return []


def _raw_ufvs_candidate_score(path, cols):
    name = _os.path.basename(str(path)).lower()
    score = 0
    if "ufvs" in name:
        score += 60
    if "verified" in name:
        score += 35
    if "viewer" in name:
        score += 25
    if "pp" in name:
        score += 10
    if "wpc" in name:
        score += 5
    score += 3 * len([c for c in cols if str(c).startswith("UFVS_")])
    # Prefer Date/Lat/Lon gridded files over point-only products.
    if {"Date", "Lat", "Lon"}.issubset(set(cols)):
        score += 50
    return score


def _find_raw_ufvs_proxy_parquet():
    patterns = [
        "**/*ufvs*.parquet",
        "**/*UFVS*.parquet",
        "**/*verified*.parquet",
        "**/*viewer*.parquet",
        "**/*proxy*.parquet",
        "**/*pp*.parquet",
    ]
    candidates = []
    seen = set()
    for root in _possible_project_dirs_for_raw_proxy_search():
        for pat in patterns:
            for path in _glob.glob(_os.path.join(root, pat), recursive=True):
                if path in seen or not _os.path.isfile(path):
                    continue
                seen.add(path)
                cols = _parquet_schema_columns(path)
                ufvs_cols = [c for c in cols if str(c).startswith("UFVS_")]
                if not ufvs_cols:
                    continue
                if not {"Date", "Lat", "Lon"}.issubset(set(cols)):
                    continue
                candidates.append((_raw_ufvs_candidate_score(path, cols), path, cols, ufvs_cols))
    if not candidates:
        return None, [], []
    candidates.sort(key=lambda x: (x[0], x[1]), reverse=True)
    score, path, cols, ufvs_cols = candidates[0]
    return path, cols, ufvs_cols


def _load_raw_ufvs_proxy_table():
    cache_key = tuple(_possible_project_dirs_for_raw_proxy_search())
    if cache_key in _RAW_UFVS_PROXY_TABLE_CACHE:
        return _RAW_UFVS_PROXY_TABLE_CACHE[cache_key]
    path, cols, ufvs_cols = _find_raw_ufvs_proxy_parquet()
    if path is None:
        _RAW_UFVS_PROXY_TABLE_CACHE[cache_key] = None
        return None
    read_cols = ["Date", "Lat", "Lon"] + list(ufvs_cols)
    print(f"Loading raw UFVS proxy columns from: {path}")
    print(f"Raw UFVS columns: {ufvs_cols}")
    raw = pd.read_parquet(path, columns=read_cols)
    raw = raw.copy()
    raw["__DateKey"] = raw["Date"].astype(str).str[:8].str.replace("-", "", regex=False)
    raw["__LatKey"] = pd.to_numeric(raw["Lat"], errors="coerce").round(5)
    raw["__LonKey"] = pd.to_numeric(raw["Lon"], errors="coerce").round(5)
    # If the source has duplicate rows per grid/date, collapse to binary/maximum proxy value.
    for c in ufvs_cols:
        raw[c] = pd.to_numeric(raw[c], errors="coerce").fillna(0)
    raw = raw.groupby(["__DateKey", "__LatKey", "__LonKey"], as_index=False)[ufvs_cols].max()
    _RAW_UFVS_PROXY_TABLE_CACHE[cache_key] = raw
    return raw


def _merge_raw_ufvs_proxy_columns_inplace(df):
    """Add UFVS_* columns to df in-place by merging a local raw-UFVS parquet if needed."""
    existing = [c for c in df.columns if str(c).startswith("UFVS_")]
    if existing:
        return existing
    if not {"Date", "Lat", "Lon"}.issubset(set(df.columns)):
        raise RuntimeError(
            "Proxy-truth verification requires raw UFVS_* columns or Date/Lat/Lon keys to merge them. "
            f"This dataframe has columns: {list(df.columns)[:30]}..."
        )
    raw = _load_raw_ufvs_proxy_table()
    if raw is None or raw.empty:
        raise RuntimeError(
            "Proxy-truth verification requires raw UFVS_* columns, but none are present in this dataframe "
            "and no local Date/Lat/Lon parquet containing UFVS_* columns was found. "
            "This is not allowed to fall back to PP_* because PP is derived from those proxies. "
            "Find/merge the raw UFVS verification grid first."
        )
    ufvs_cols = [c for c in raw.columns if str(c).startswith("UFVS_")]
    key = pd.DataFrame({
        "__DateKey": df["Date"].astype(str).str[:8].str.replace("-", "", regex=False),
        "__LatKey": pd.to_numeric(df["Lat"], errors="coerce").round(5),
        "__LonKey": pd.to_numeric(df["Lon"], errors="coerce").round(5),
    }, index=df.index)
    merged = key.merge(raw, on=["__DateKey", "__LatKey", "__LonKey"], how="left")
    for c in ufvs_cols:
        df[c] = pd.to_numeric(merged[c], errors="coerce").fillna(0).to_numpy()
    print(f"Merged raw UFVS proxy columns into evaluation dataframe: {ufvs_cols}")
    return ufvs_cols


def _resolve_proxy_column(df, proxy_col="UFVS_ANY", allow_pp_fallback=False):
    """Resolve raw proxy truth columns only. PP fallback is intentionally disabled."""
    ufvs_cols = [c for c in df.columns if str(c).startswith("UFVS_")]
    if not ufvs_cols:
        ufvs_cols = _merge_raw_ufvs_proxy_columns_inplace(df)
    if proxy_col is not None and proxy_col in df.columns:
        return proxy_col, "raw_ufvs"
    for c in DEFAULT_UFVS_PROXY_CANDIDATES:
        if c in df.columns:
            print(f"Requested proxy {proxy_col!r} not found; using available raw UFVS proxy column {c!r}.")
            return c, "raw_ufvs"
    raise RuntimeError(
        f"Requested raw proxy column {proxy_col!r} not found. Available raw UFVS columns: "
        f"{[c for c in df.columns if str(c).startswith('UFVS_')]}. "
        "PP_* columns are not accepted for proxy-truth verification."
    )


def _resolve_truth_for_subset(
    sub,
    truth_mode="pp",
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    allow_pp_proxy_fallback=False,
    proxy_truth_threshold=0.0,
    pp_event_threshold=0.05,
):
    """Return truth_col, truth_kind, and a binary truth vector for reliability/BS."""
    truth_mode = str(truth_mode).lower()
    if truth_mode == "pp":
        truth_col = _truth_col_from_pp_definition_integrated(pp_definition)
        if truth_col not in sub.columns:
            raise RuntimeError(
                f"Missing PP truth column {truth_col}. Available PP columns: "
                f"{[c for c in sub.columns if str(c).startswith('PP_')]}"
            )
        y = _binary_from_col(sub, truth_col, threshold=pp_event_threshold, mode=">=")
        return truth_col, "pp", y
    if truth_mode == "proxy":
        truth_col, truth_kind = _resolve_proxy_column(sub, proxy_col=proxy_col, allow_pp_fallback=False)
        y = _binary_from_col(sub, truth_col, threshold=proxy_truth_threshold, mode=">")
        return truth_col, truth_kind, y
    raise RuntimeError("truth_mode must be 'pp' or 'proxy'.")

print("V7 strict raw-proxy verification loaded: proxy-truth metrics require UFVS_* columns; PP fallback is disabled.")


# ======================================================================================
# V8 ARCHIVE-BACKED RAW UFVS PROXY RECONSTRUCTION
# ======================================================================================
# If a historical evaluation dataframe has PP_* fields but no raw UFVS_* columns, that means
# the raw proxy columns were dropped before this viewer stage.  The proxy data still exist
# upstream, because the PP fields were created from those proxies.  For proxy-truth CSI/BS/
# reliability/fractional-coverage diagnostics, rebuild the missing UFVS_* columns from the
# UFVS archive text files instead of falling back to PP or skipping.

import hashlib as _hashlib
from pathlib import Path as _Path

if "FORCE_HISTORICAL_UFVS_ARCHIVE_FETCH" not in globals():
    FORCE_HISTORICAL_UFVS_ARCHIVE_FETCH = False
if "HISTORICAL_UFVS_INCLUDE_REGULAR_FLOOD_LSR" not in globals():
    HISTORICAL_UFVS_INCLUDE_REGULAR_FLOOD_LSR = False
if "HISTORICAL_UFVS_MAX_NEAREST_DIST_KM" not in globals():
    # Raw UFVS proxy flags are mapped to the nearest model grid point.  This is deliberately
    # not the PP expansion radius; PP expansion/smoothing belongs to PP_* truth, not raw UFVS_* truth.
    HISTORICAL_UFVS_MAX_NEAREST_DIST_KM = 25.0
if "HISTORICAL_UFVS_EXTENT" not in globals():
    HISTORICAL_UFVS_EXTENT = DEFAULT_EXTENT if "DEFAULT_EXTENT" in globals() else (-105.0, -80.5, 30.0, 50.0)

_HISTORICAL_UFVS_GRID_CACHE_MEM = {}


def _historical_ufvs_cache_dir():
    roots = []
    if "PROJECT_DIR" in globals() and PROJECT_DIR:
        roots.append(_Path(str(PROJECT_DIR)))
    roots.extend([
        _Path("/home/tyreekfrazier/ISU_Research_LOCAL_RUN/fall_2025_ml_proj"),
        _Path("/home/tyreekfrazier/ISU_Research/fall_2025_ml_proj"),
        _Path.cwd(),
    ])
    root = next((p for p in roots if p.exists()), roots[-1])
    d = root / "historical_ufvs_grid_cache_v33day2valid"
    d.mkdir(parents=True, exist_ok=True)
    return d


def _date8_safe(x):
    if "_date8" in globals():
        return _date8(x)
    return str(x)[:10].replace("-", "")[:8]


def _ufvs_prefixes_for_proxy_rebuild(include_regular_flood_lsr=None):
    if include_regular_flood_lsr is None:
        include_regular_flood_lsr = bool(globals().get("HISTORICAL_UFVS_INCLUDE_REGULAR_FLOOD_LSR", False))
    prefixes = ["ST4gFFG", "ST4gARI", "USGS", "LSRFLASH"]
    if include_regular_flood_lsr:
        prefixes.append("LSRREG")
    return prefixes


def _grid_signature_for_ufvs_cache(df_grid):
    if not {"Lat", "Lon"}.issubset(set(df_grid.columns)):
        return f"nogrid_n{len(df_grid)}"
    xy = df_grid[["Lat", "Lon"]].copy()
    xy["Lat"] = pd.to_numeric(xy["Lat"], errors="coerce").round(5)
    xy["Lon"] = pd.to_numeric(xy["Lon"], errors="coerce").round(5)
    # Include row order in the signature because a cache can be assigned directly after Date/Lat/Lon merge.
    hv = pd.util.hash_pandas_object(xy, index=False).to_numpy(dtype=np.uint64)
    return _hashlib.md5(hv.tobytes()).hexdigest()[:12]


def _keys_for_date_lat_lon(df):
    return pd.DataFrame({
        "__DateKey": df["Date"].astype(str).str[:10].str.replace("-", "", regex=False).str[:8],
        "__LatKey": pd.to_numeric(df["Lat"], errors="coerce").round(5),
        "__LonKey": pd.to_numeric(df["Lon"], errors="coerce").round(5),
    }, index=df.index)


def _build_raw_ufvs_grid_for_one_date_from_archive(
    date,
    df_grid,
    force_fetch=None,
    include_regular_flood_lsr=None,
    max_nearest_dist_km=None,
    extent=None,
):
    """Fetch UFVS archive text files for one date and map raw proxy flags onto df_grid."""
    if force_fetch is None:
        force_fetch = bool(globals().get("FORCE_HISTORICAL_UFVS_ARCHIVE_FETCH", False))
    if max_nearest_dist_km is None:
        max_nearest_dist_km = float(globals().get("HISTORICAL_UFVS_MAX_NEAREST_DIST_KM", 25.0))
    if extent is None:
        extent = globals().get("HISTORICAL_UFVS_EXTENT", globals().get("DEFAULT_EXTENT", (-105.0, -80.5, 30.0, 50.0)))

    date8 = _date8_safe(date)
    prefixes = _ufvs_prefixes_for_proxy_rebuild(include_regular_flood_lsr=include_regular_flood_lsr)
    sig = _grid_signature_for_ufvs_cache(df_grid)
    cache_path = _historical_ufvs_cache_dir() / (
        f"raw_ufvs_grid_{date8}_n{len(df_grid)}_{sig}_d{int(round(float(max_nearest_dist_km)))}km.parquet"
    )

    mem_key = (str(cache_path), bool(force_fetch))
    if (not force_fetch) and str(cache_path) in _HISTORICAL_UFVS_GRID_CACHE_MEM:
        return _HISTORICAL_UFVS_GRID_CACHE_MEM[str(cache_path)].copy()
    if cache_path.exists() and cache_path.stat().st_size > 512 and not force_fetch:
        raw = pd.read_parquet(cache_path)
        _HISTORICAL_UFVS_GRID_CACHE_MEM[str(cache_path)] = raw.copy()
        return raw

    if "fetch_ufvs_points" not in globals():
        raise RuntimeError(
            "Cannot rebuild raw UFVS proxy columns because fetch_ufvs_points(...) is not defined. "
            "Run the realtime/UFVS helper-definition cells before the evaluation cells."
        )
    if "filter_points_to_extent" not in globals():
        raise RuntimeError("Cannot rebuild raw UFVS columns because filter_points_to_extent(...) is not defined.")
    if "_event_mask_from_points" not in globals():
        raise RuntimeError("Cannot rebuild raw UFVS columns because _event_mask_from_points(...) is not defined.")

    raw = df_grid[["Date", "Lat", "Lon"]].copy()
    raw["Date"] = date8
    summary = []

    for prefix in prefixes:
        col = UFVS_PREFIX_TO_COL.get(prefix, f"UFVS_{prefix}") if "UFVS_PREFIX_TO_COL" in globals() else f"UFVS_{prefix}"
        try:
            pts = fetch_ufvs_points(date8, prefix, force=force_fetch)
            pts = filter_points_to_extent(pts, extent=extent)
            flags = _event_mask_from_points(raw, pts, max_dist_km=float(max_nearest_dist_km)).astype(np.int8)
            raw[col] = flags
            summary.append({
                "Date": date8,
                "prefix": prefix,
                "column": col,
                "points_in_extent": int(len(pts)),
                "mapped_pixels": int(np.sum(flags)),
            })
        except Exception as exc:
            raw[col] = np.zeros(len(raw), dtype=np.int8)
            summary.append({
                "Date": date8,
                "prefix": prefix,
                "column": col,
                "points_in_extent": 0,
                "mapped_pixels": 0,
                "error": repr(exc),
            })
            print(f"UFVS archive rebuild warning for {date8} {prefix}: {exc}")

    ufvs_cols = [c for c in raw.columns if str(c).startswith("UFVS_")]
    if ufvs_cols:
        raw["UFVS_ANY"] = (raw[ufvs_cols].apply(pd.to_numeric, errors="coerce").fillna(0).max(axis=1) > 0).astype(np.int8)

    try:
        raw.to_parquet(cache_path, index=False)
    except Exception as exc:
        print(f"Could not write historical UFVS cache {cache_path}: {exc}")

    print(
        f"Rebuilt raw UFVS proxy grid from archive for {date8}: "
        f"{', '.join([c for c in raw.columns if str(c).startswith('UFVS_')])}"
    )
    if summary:
        try:
            display(pd.DataFrame(summary))
        except Exception:
            print(pd.DataFrame(summary).to_string(index=False))

    _HISTORICAL_UFVS_GRID_CACHE_MEM[str(cache_path)] = raw.copy()
    return raw


def _merge_existing_raw_ufvs_proxy_parquet_if_available(df):
    """Try a local Date/Lat/Lon + UFVS_* parquet first. Return UFVS cols or [] if none found."""
    try:
        raw = _load_raw_ufvs_proxy_table()
    except Exception as exc:
        print(f"Local raw-UFVS parquet search failed; will rebuild from archive. Reason: {exc}")
        raw = None
    if raw is None or raw.empty:
        return []
    ufvs_cols = [c for c in raw.columns if str(c).startswith("UFVS_")]
    if not ufvs_cols:
        return []
    key = _keys_for_date_lat_lon(df)
    merged = key.merge(raw, on=["__DateKey", "__LatKey", "__LonKey"], how="left")
    # Only accept the parquet if it supplies non-null columns for at least some rows/dates.
    available_nonnull = False
    for c in ufvs_cols:
        vals = pd.to_numeric(merged[c], errors="coerce") if c in merged.columns else pd.Series(np.nan, index=merged.index)
        if vals.notna().any():
            available_nonnull = True
        df[c] = vals.fillna(0).to_numpy(dtype=np.int8)
    if available_nonnull:
        print(f"Merged raw UFVS proxy columns from local parquet: {ufvs_cols}")
        return ufvs_cols
    # If a candidate parquet had the columns but did not align to this grid, remove them and rebuild.
    for c in ufvs_cols:
        if c in df.columns:
            df.drop(columns=[c], inplace=True)
    return []


def _merge_raw_ufvs_proxy_columns_inplace(df):
    """Ensure df has raw UFVS_* columns. If missing, rebuild them from the UFVS archive by date."""
    existing = [c for c in df.columns if str(c).startswith("UFVS_")]
    if existing:
        return existing
    if not {"Date", "Lat", "Lon"}.issubset(set(df.columns)):
        raise RuntimeError(
            "Proxy-truth verification requires raw UFVS_* columns or Date/Lat/Lon keys so they can be rebuilt "
            f"from the UFVS archive. This dataframe has columns: {list(df.columns)[:30]}..."
        )

    # First try to merge a prebuilt local raw-UFVS grid if one exists.
    local_cols = _merge_existing_raw_ufvs_proxy_parquet_if_available(df)
    if local_cols:
        return local_cols

    # If no prebuilt grid exists, rebuild from the archive for every date represented in df.
    date_keys = df["Date"].astype(str).str[:10].str.replace("-", "", regex=False).str[:8]
    unique_dates = sorted(pd.Series(date_keys).dropna().unique().tolist())
    if not unique_dates:
        raise RuntimeError("Cannot rebuild UFVS proxy columns because no valid Date values were found.")

    print(
        "Raw UFVS columns are missing from this evaluation dataframe. "
        f"Rebuilding them from the UFVS archive for {len(unique_dates)} date(s): "
        f"{unique_dates[:5]}{' ...' if len(unique_dates) > 5 else ''}"
    )

    all_ufvs_cols = set()
    for date8 in unique_dates:
        idx = np.where(date_keys.to_numpy() == date8)[0]
        if len(idx) == 0:
            continue
        sub = df.iloc[idx][["Date", "Lat", "Lon"]].copy()
        raw = _build_raw_ufvs_grid_for_one_date_from_archive(date8, sub)
        raw_cols = [c for c in raw.columns if str(c).startswith("UFVS_")]
        all_ufvs_cols.update(raw_cols)

        left = _keys_for_date_lat_lon(sub).reset_index(drop=True)
        right = pd.concat([_keys_for_date_lat_lon(raw).reset_index(drop=True), raw[raw_cols].reset_index(drop=True)], axis=1)
        right = right.groupby(["__DateKey", "__LatKey", "__LonKey"], as_index=False)[raw_cols].max()
        merged = left.merge(right, on=["__DateKey", "__LatKey", "__LonKey"], how="left")
        for c in raw_cols:
            if c not in df.columns:
                df[c] = np.zeros(len(df), dtype=np.int8)
            df.iloc[idx, df.columns.get_loc(c)] = pd.to_numeric(merged[c], errors="coerce").fillna(0).to_numpy(dtype=np.int8)

    out_cols = sorted([c for c in df.columns if str(c).startswith("UFVS_")])
    if not out_cols:
        raise RuntimeError(
            "Tried to rebuild raw UFVS proxy columns from the archive, but no UFVS_* columns were created. "
            "Check UFVS archive access and prefix/date naming."
        )
    if "UFVS_ANY" not in df.columns:
        component_cols = [c for c in out_cols if c != "UFVS_ANY"]
        df["UFVS_ANY"] = (df[component_cols].apply(pd.to_numeric, errors="coerce").fillna(0).max(axis=1) > 0).astype(np.int8)
        out_cols = sorted([c for c in df.columns if str(c).startswith("UFVS_")])
    print(f"Raw UFVS proxy columns available for evaluation: {out_cols}")
    return out_cols


def _resolve_proxy_column(df, proxy_col="UFVS_ANY", allow_pp_fallback=False):
    """Resolve raw proxy truth columns only. If missing, rebuild from the UFVS archive."""
    ufvs_cols = [c for c in df.columns if str(c).startswith("UFVS_")]
    if not ufvs_cols:
        ufvs_cols = _merge_raw_ufvs_proxy_columns_inplace(df)
    if proxy_col is not None and proxy_col in df.columns:
        return proxy_col, "raw_ufvs"
    for c in DEFAULT_UFVS_PROXY_CANDIDATES:
        if c in df.columns:
            print(f"Requested proxy {proxy_col!r} not found; using available raw UFVS proxy column {c!r}.")
            return c, "raw_ufvs"
    raise RuntimeError(
        f"Requested raw proxy column {proxy_col!r} not found after archive rebuild. Available raw UFVS columns: "
        f"{[c for c in df.columns if str(c).startswith('UFVS_')]}."
    )

print(
    "V8 raw-proxy rebuild loaded: if UFVS_* columns are missing, proxy-truth metrics rebuild them "
    "from the UFVS archive by Date/Lat/Lon instead of falling back to PP_* or skipping."
)


### 5e-1. Truth/proxy option registry and MRMS/UFVS proxy materialization

This cell makes the raw proxy options explicit, enables aliases like `STAGE4_FFG`, `FLASH_LSR`, `FLOOD_LSR`, and `MRMS_FFG`, and rebuilds missing UFVS proxy columns from the archive on demand.

In [ ]:
# ======================================================================================
# V9 TRUTH/PROXY OPTION REGISTRY + MRMS TARGET + FULL UFVS PROXY MATERIALIZATION
# ======================================================================================
# Conceptual rule used below:
#   * PP truth uses PP_* probability-like fields.
#   * UFVS proxy truth uses raw UFVS_* event flags rebuilt from the UFVS archive when missing.
#   * MRMS_FFG is not a UFVS product. It is loaded from the model master parquet target columns.
#
# Usable strings for *_PROXY_COL include:
#   UFVS_ANY, STAGE4_FFG, STAGE4_ARI, USGS, FLASH_LSR, FLOOD_LSR,
#   MRMS_FFG, MRMS_FFG_TARGET, MRMS_FFG_POINT
# ======================================================================================

# Include regular/flood LSR by default when rebuilding UFVS proxies from the archive.
HISTORICAL_UFVS_INCLUDE_REGULAR_FLOOD_LSR = True

# Canonical raw proxy choices.  These are the strings you can put in PROXY_TRUTH_COL,
# FRAC_PROXY_COL, RELIABILITY_PROXY_COL, CSI_PROXY_COL, or BRIER_PROXY_COL.
RAW_PROXY_OPTIONS = [
    {
        "option": "UFVS_ANY",
        "canonical_column": "UFVS_ANY",
        "source": "UFVS archive",
        "description": "Union of Stage IV > FFG, Stage IV ARI, USGS, flash LSR, and flood LSR proxies.",
    },
    {
        "option": "STAGE4_FFG",
        "canonical_column": "UFVS_STAGE4_FFG",
        "source": "UFVS archive: ST4gFFG",
        "description": "Stage IV precipitation exceeding FFG.",
    },
    {
        "option": "STAGE4_ARI",
        "canonical_column": "UFVS_STAGE4_ARI",
        "source": "UFVS archive: ST4gARI",
        "description": "Stage IV average recurrence interval proxy.",
    },
    {
        "option": "USGS",
        "canonical_column": "UFVS_USGS",
        "source": "UFVS archive: USGS",
        "description": "USGS streamflow flood proxy points.",
    },
    {
        "option": "FLASH_LSR",
        "canonical_column": "UFVS_LSR_FLASH",
        "source": "UFVS archive: LSRFLASH",
        "description": "Flash-flood local storm reports.",
    },
    {
        "option": "FLOOD_LSR",
        "canonical_column": "UFVS_LSR_FLOOD",
        "source": "UFVS archive: LSRREG/regular flood LSRs",
        "description": "Regular flood local storm reports. Internally mirrored from UFVS_LSR_REGULAR when that is the parsed column name.",
    },
    {
        "option": "MRMS_FFG",
        "canonical_column": "MRMS_FFG",
        "source": "model master parquet",
        "description": "Radius-specific ML target: any MRMS QPE > FFG within the active target radius.",
    },
    {
        "option": "MRMS_FFG_TARGET",
        "canonical_column": "MRMS_FFG",
        "source": "model master parquet",
        "description": "Alias for MRMS_FFG; radius-specific ML target.",
    },
    {
        "option": "MRMS_FFG_POINT",
        "canonical_column": "MRMS_FFG_POINT",
        "source": "model master parquet",
        "description": "Pointwise MRMS QPE > FFG event flag, not radius-expanded.",
    },
]


def _norm_proxy_name(x):
    return re.sub(r"[^a-z0-9]+", "", str(x).strip().lower())


_PROXY_ALIAS_TO_CANONICAL = {
    "ufvsany": "UFVS_ANY",
    "any": "UFVS_ANY",
    "anyfloodproxy": "UFVS_ANY",
    "floodproxy": "UFVS_ANY",
    "all": "UFVS_ANY",

    "stage4ffg": "UFVS_STAGE4_FFG",
    "stageivffg": "UFVS_STAGE4_FFG",
    "st4gffg": "UFVS_STAGE4_FFG",
    "ufvsstage4ffg": "UFVS_STAGE4_FFG",
    "ufvsstageivffg": "UFVS_STAGE4_FFG",

    "stage4ari": "UFVS_STAGE4_ARI",
    "stageivari": "UFVS_STAGE4_ARI",
    "st4gari": "UFVS_STAGE4_ARI",
    "ufvsstage4ari": "UFVS_STAGE4_ARI",
    "ufvsstageivari": "UFVS_STAGE4_ARI",

    "usgs": "UFVS_USGS",
    "ufvsusgs": "UFVS_USGS",

    "flashlsr": "UFVS_LSR_FLASH",
    "lsrflash": "UFVS_LSR_FLASH",
    "flashfloodlsr": "UFVS_LSR_FLASH",
    "ufvslsrflash": "UFVS_LSR_FLASH",

    "floodlsr": "UFVS_LSR_FLOOD",
    "regularfloodlsr": "UFVS_LSR_FLOOD",
    "regularlsr": "UFVS_LSR_FLOOD",
    "lsrflood": "UFVS_LSR_FLOOD",
    "lsrregular": "UFVS_LSR_FLOOD",
    "lsrreg": "UFVS_LSR_FLOOD",
    "ufvslsrflood": "UFVS_LSR_FLOOD",
    "ufvslsrregular": "UFVS_LSR_FLOOD",

    "mrms": "MRMS_FFG",
    "mrmsffg": "MRMS_FFG",
    "mrmsgtffg": "MRMS_FFG",
    "mrmsexceedffg": "MRMS_FFG",
    "mrmsexceedsffg": "MRMS_FFG",
    "target": "MRMS_FFG",
    "modeltarget": "MRMS_FFG",
    "mrmstarget": "MRMS_FFG",
    "mrmsffgtarget": "MRMS_FFG",

    "mrmsffgpoint": "MRMS_FFG_POINT",
    "mrmspoint": "MRMS_FFG_POINT",
    "pointmrmsffg": "MRMS_FFG_POINT",
    "pointwise": "MRMS_FFG_POINT",
    "pointwisetarget": "MRMS_FFG_POINT",
}


def _canonical_proxy_column(proxy_col):
    if proxy_col is None:
        return None
    if proxy_col in globals().get("DEFAULT_UFVS_PROXY_CANDIDATES", []):
        return proxy_col
    s = str(proxy_col).strip()
    if s in {row["canonical_column"] for row in RAW_PROXY_OPTIONS}:
        return s
    return _PROXY_ALIAS_TO_CANONICAL.get(_norm_proxy_name(s), s)


def _proxy_options_dataframe(df=None, radii=None):
    rows = []
    cols_now = set(df.columns) if df is not None else set()
    for opt in RAW_PROXY_OPTIONS:
        c = opt["canonical_column"]
        rows.append({
            **opt,
            "currently_in_dataframe": bool(c in cols_now),
        })
    if df is not None:
        for c in sorted([c for c in df.columns if str(c).startswith("UFVS_") or str(c).startswith("PP_") or str(c).startswith("MRMS_FFG")]):
            if c not in {r["canonical_column"] for r in rows}:
                rows.append({
                    "option": c,
                    "canonical_column": c,
                    "source": "current dataframe",
                    "description": "Column already present in the selected evaluation dataframe.",
                    "currently_in_dataframe": True,
                })
    return pd.DataFrame(rows)


def display_available_truth_options(df=None, prefer_realtime=False, radii=None, attempt_rebuild=False):
    """Show the valid strings for PP and raw-proxy truth selections."""
    if df is None:
        df, label = choose_evaluation_dataframe(prefer_realtime=prefer_realtime)
    else:
        label = "provided dataframe"
    print(f"Truth/proxy options for {label}")

    pp_cols = sorted([c for c in df.columns if str(c).startswith("PP_")])
    if pp_cols:
        print("\nPP truth options for *_PP_DEFINITION:")
        for c in pp_cols:
            print(f"  {c.replace('PP_', '')!r}  (column {c})")
    else:
        print("\nNo PP_* columns currently in this dataframe.")

    if attempt_rebuild:
        try:
            _ensure_raw_ufvs_proxy_columns_inplace(df, requested_col="UFVS_ANY", ensure_all=True)
        except Exception as exc:
            print(f"UFVS option rebuild check failed: {exc}")
        try:
            _merge_mrms_ffg_proxy_column_inplace(df, proxy_col="MRMS_FFG", radii=radii)
        except Exception as exc:
            print(f"MRMS_FFG option check failed: {exc}")

    print("\nRaw proxy truth options for *_PROXY_COL:")
    opts = _proxy_options_dataframe(df=df, radii=radii)
    try:
        display(opts)
    except Exception:
        print(opts.to_string(index=False))
    print("\nExamples:")
    print("  PROXY_TRUTH_COL = 'UFVS_ANY'")
    print("  PROXY_TRUTH_COL = 'STAGE4_FFG'")
    print("  PROXY_TRUTH_COL = 'STAGE4_ARI'")
    print("  PROXY_TRUTH_COL = 'USGS'")
    print("  PROXY_TRUTH_COL = 'FLASH_LSR'")
    print("  PROXY_TRUTH_COL = 'FLOOD_LSR'")
    print("  PROXY_TRUTH_COL = 'MRMS_FFG'        # radius-specific model target")
    print("  PROXY_TRUTH_COL = 'MRMS_FFG_POINT'  # pointwise MRMS>FFG")
    return opts


# Preserve v8 implementations so the v9 functions can delegate when useful.
if "_merge_raw_ufvs_proxy_columns_inplace_v8" not in globals() and "_merge_raw_ufvs_proxy_columns_inplace" in globals():
    _merge_raw_ufvs_proxy_columns_inplace_v8 = _merge_raw_ufvs_proxy_columns_inplace
if "_resolve_proxy_column_v8" not in globals() and "_resolve_proxy_column" in globals():
    _resolve_proxy_column_v8 = _resolve_proxy_column


def _ufvs_desired_columns(include_regular_flood_lsr=True):
    cols = ["UFVS_STAGE4_FFG", "UFVS_STAGE4_ARI", "UFVS_USGS", "UFVS_LSR_FLASH"]
    if include_regular_flood_lsr:
        cols.extend(["UFVS_LSR_REGULAR", "UFVS_LSR_FLOOD"])
    cols.append("UFVS_ANY")
    return list(dict.fromkeys(cols))


def _make_lsr_flood_aliases_inplace(df):
    if "UFVS_LSR_FLOOD" not in df.columns and "UFVS_LSR_REGULAR" in df.columns:
        df["UFVS_LSR_FLOOD"] = pd.to_numeric(df["UFVS_LSR_REGULAR"], errors="coerce").fillna(0).astype(np.int8)
    if "UFVS_LSR_REGULAR" not in df.columns and "UFVS_LSR_FLOOD" in df.columns:
        df["UFVS_LSR_REGULAR"] = pd.to_numeric(df["UFVS_LSR_FLOOD"], errors="coerce").fillna(0).astype(np.int8)
    return df


def _recompute_ufvs_any_inplace(df):
    component_cols = [
        c for c in ["UFVS_STAGE4_FFG", "UFVS_STAGE4_ARI", "UFVS_USGS", "UFVS_LSR_FLASH", "UFVS_LSR_REGULAR", "UFVS_LSR_FLOOD"]
        if c in df.columns
    ]
    if component_cols:
        df["UFVS_ANY"] = (df[component_cols].apply(pd.to_numeric, errors="coerce").fillna(0).max(axis=1) > 0).astype(np.int8)
    return df


def _ensure_raw_ufvs_proxy_columns_inplace(df, requested_col=None, ensure_all=False):
    """Ensure raw UFVS proxy columns exist, rebuilding from archive by date if needed."""
    requested_col = _canonical_proxy_column(requested_col)
    if requested_col in {"MRMS_FFG", "MRMS_FFG_POINT"}:
        return [c for c in df.columns if str(c).startswith("UFVS_")]

    _make_lsr_flood_aliases_inplace(df)
    _recompute_ufvs_any_inplace(df)

    desired = _ufvs_desired_columns(include_regular_flood_lsr=True)
    if requested_col and str(requested_col).startswith("UFVS_"):
        desired = list(dict.fromkeys(desired + [requested_col]))

    existing = [c for c in desired if c in df.columns]
    missing = [c for c in desired if c not in df.columns]
    if requested_col == "UFVS_ANY" and "UFVS_ANY" in df.columns and not ensure_all:
        return sorted([c for c in df.columns if str(c).startswith("UFVS_")])
    if requested_col and requested_col in df.columns and not ensure_all:
        return sorted([c for c in df.columns if str(c).startswith("UFVS_")])
    if not missing and (not requested_col or requested_col in df.columns):
        return sorted([c for c in df.columns if str(c).startswith("UFVS_")])

    if not {"Date", "Lat", "Lon"}.issubset(set(df.columns)):
        raise RuntimeError(
            "Raw UFVS proxies are missing and cannot be rebuilt because this dataframe lacks Date/Lat/Lon. "
            f"Available columns: {list(df.columns)[:40]}"
        )

    # Try any prebuilt local raw-UFVS grid first.
    try:
        local_cols = _merge_existing_raw_ufvs_proxy_parquet_if_available(df)
        if local_cols:
            _make_lsr_flood_aliases_inplace(df)
            _recompute_ufvs_any_inplace(df)
    except Exception as exc:
        print(f"Local UFVS parquet merge failed; rebuilding from archive. Reason: {exc}")

    _make_lsr_flood_aliases_inplace(df)
    _recompute_ufvs_any_inplace(df)
    missing_after_local = [c for c in desired if c not in df.columns]
    if not missing_after_local and (not requested_col or requested_col in df.columns):
        return sorted([c for c in df.columns if str(c).startswith("UFVS_")])

    date_keys = df["Date"].astype(str).str[:10].str.replace("-", "", regex=False).str[:8]
    unique_dates = sorted(pd.Series(date_keys).dropna().unique().tolist())
    print(
        "Raw UFVS proxy columns are missing/incomplete. "
        f"Rebuilding from UFVS archive for {len(unique_dates)} date(s). "
        f"Missing examples: {missing_after_local[:10]}"
    )

    for date8 in unique_dates:
        row_index = df.index[date_keys.to_numpy() == date8]
        if len(row_index) == 0:
            continue
        sub_grid = df.loc[row_index, ["Date", "Lat", "Lon"]].copy()
        # Force rebuild if the older cache did not include flood LSR columns and they are needed.
        force_for_date = bool("UFVS_LSR_FLOOD" in missing_after_local or "UFVS_LSR_REGULAR" in missing_after_local)
        raw = _build_raw_ufvs_grid_for_one_date_from_archive(
            date8,
            sub_grid,
            include_regular_flood_lsr=True,
            force_fetch=bool(globals().get("FORCE_HISTORICAL_UFVS_ARCHIVE_FETCH", False)) or force_for_date,
        )
        _make_lsr_flood_aliases_inplace(raw)
        _recompute_ufvs_any_inplace(raw)
        raw_cols = [c for c in raw.columns if str(c).startswith("UFVS_")]
        left = _keys_for_date_lat_lon(sub_grid).reset_index(drop=True)
        right = pd.concat([_keys_for_date_lat_lon(raw).reset_index(drop=True), raw[raw_cols].reset_index(drop=True)], axis=1)
        right = right.groupby(["__DateKey", "__LatKey", "__LonKey"], as_index=False)[raw_cols].max()
        merged = left.merge(right, on=["__DateKey", "__LatKey", "__LonKey"], how="left")
        for c in raw_cols:
            if c not in df.columns:
                df[c] = np.zeros(len(df), dtype=np.int8)
            df.loc[row_index, c] = pd.to_numeric(merged[c], errors="coerce").fillna(0).to_numpy(dtype=np.int8)

    _make_lsr_flood_aliases_inplace(df)
    _recompute_ufvs_any_inplace(df)
    out_cols = sorted([c for c in df.columns if str(c).startswith("UFVS_")])
    if requested_col and requested_col not in df.columns:
        raise RuntimeError(
            f"Requested UFVS proxy {requested_col!r} could not be created from the archive. "
            f"Available UFVS columns now: {out_cols}"
        )
    print(f"Raw UFVS proxy columns available for evaluation: {out_cols}")
    return out_cols


_MRMS_PROXY_TABLE_CACHE = {}


def _mrms_target_candidates(radius_km=None, pointwise=False):
    if pointwise:
        return [
            "Obs_Day2_MRMS_FFG_Exceeded_Point",
            "Target_Day2_MRMS_FFG_Exceeded_Point",
            "Obs_MRMS_FFG_Exceeded_Point",
            "MRMS_FFG_Exceeded_Point",
            "Target_MRMS_FFG_Exceeded_Point",
        ]
    if radius_km is None:
        return []
    r = int(round(float(radius_km)))
    return [
        f"Target_Day2_MRMS_FFG_Exceeded_R{r}km",
        f"Obs_Day2_MRMS_FFG_Exceeded_R{r}km",
        f"Obs_Day2_MRMS_FFG_Exceeded_R{r}km_Fraction",
        f"Target_MRMS_FFG_Exceeded_R{r}km",
        f"Obs_MRMS_FFG_Exceeded_R{r}km",
        f"MRMS_FFG_Exceeded_R{r}km",
        f"Obs_MRMS_FFG_Exceeded_R{r}km_Fraction",
    ]


def _find_existing_col_case_insensitive(cols, candidates):
    lookup = {str(c).lower(): c for c in cols}
    for cand in candidates:
        if str(cand).lower() in lookup:
            return lookup[str(cand).lower()]
    return None


def _load_mrms_proxy_table_for_radius(radius_km, pointwise=False):
    r = int(round(float(radius_km)))
    art = find_artifacts_for_radius(r)
    path = art.get("master_path")
    if not path or not os.path.exists(path):
        raise RuntimeError(f"Could not find master parquet for MRMS_FFG radius r{r}: {path}")
    cols = _parquet_schema_columns(path) if "_parquet_schema_columns" in globals() else _parquet_columns(path)
    target_col = _find_existing_col_case_insensitive(cols, _mrms_target_candidates(r, pointwise=pointwise))
    if target_col is None:
        raise RuntimeError(
            f"Could not find MRMS/FFG target column for r{r} in {path}. "
            f"Tried {_mrms_target_candidates(r, pointwise=pointwise)}. "
            f"Available target-like columns: {[c for c in cols if 'MRMS_FFG' in str(c) or 'FFG_Exceeded' in str(c)][:80]}"
        )
    cache_key = (str(path), str(target_col))
    if cache_key in _MRMS_PROXY_TABLE_CACHE:
        return _MRMS_PROXY_TABLE_CACHE[cache_key].copy(), target_col
    read_cols = ["Date", "Lat", "Lon", target_col]
    print(f"Loading MRMS/FFG truth from r{r} master parquet: {target_col}")
    tab = pd.read_parquet(path, columns=read_cols)
    tab = tab.copy()
    tab["__DateKey"] = tab["Date"].astype(str).str[:10].str.replace("-", "", regex=False).str[:8]
    tab["__LatKey"] = pd.to_numeric(tab["Lat"], errors="coerce").round(5)
    tab["__LonKey"] = pd.to_numeric(tab["Lon"], errors="coerce").round(5)
    tab[target_col] = pd.to_numeric(tab[target_col], errors="coerce").fillna(0)
    tab = tab.groupby(["__DateKey", "__LatKey", "__LonKey"], as_index=False)[target_col].max()
    _MRMS_PROXY_TABLE_CACHE[cache_key] = tab.copy()
    return tab, target_col


def _merge_mrms_ffg_proxy_column_inplace(df, proxy_col="MRMS_FFG", radii=None):
    """Create MRMS_FFG or MRMS_FFG_POINT in df from existing columns or the radius master parquet."""
    canonical = _canonical_proxy_column(proxy_col)
    out_col = "MRMS_FFG_POINT" if canonical == "MRMS_FFG_POINT" else "MRMS_FFG"
    pointwise = (out_col == "MRMS_FFG_POINT")
    if out_col in df.columns:
        return out_col

    # First use any matching column already present in this dataframe.
    if pointwise:
        existing_col = _find_existing_col_case_insensitive(df.columns, _mrms_target_candidates(pointwise=True))
        if existing_col is not None:
            df[out_col] = (pd.to_numeric(df[existing_col], errors="coerce").fillna(0) > 0).astype(np.int8)
            print(f"Created {out_col} from existing dataframe column {existing_col}.")
            return out_col
    else:
        # Historical long-format can contain one row radius at a time.  Try all radius-specific targets already present.
        existing_radius_cols = []
        for rr in (_available_radii_from_df(df, requested=radii) or ([40, 60, 75, 100] if radii is None else radii)):
            c = _find_existing_col_case_insensitive(df.columns, _mrms_target_candidates(rr, pointwise=False))
            if c is not None:
                existing_radius_cols.append((int(rr), c))
        if existing_radius_cols:
            df[out_col] = np.zeros(len(df), dtype=np.int8)
            if "ML_Target_Radius_km" in df.columns:
                rad = pd.to_numeric(df["ML_Target_Radius_km"], errors="coerce")
                for rr, c in existing_radius_cols:
                    m = (rad == int(rr)).to_numpy()
                    if m.any():
                        df.loc[df.index[m], out_col] = (pd.to_numeric(df.loc[df.index[m], c], errors="coerce").fillna(0) > 0).astype(np.int8)
            else:
                # Wide/aggregate df: use the first available target column.
                rr, c = existing_radius_cols[0]
                df[out_col] = (pd.to_numeric(df[c], errors="coerce").fillna(0) > 0).astype(np.int8)
            print(f"Created {out_col} from existing dataframe target columns: {existing_radius_cols}")
            return out_col

    if not {"Date", "Lat", "Lon"}.issubset(set(df.columns)):
        raise RuntimeError(f"Cannot create {out_col}; dataframe lacks Date/Lat/Lon keys.")

    if radii is None:
        if "ML_Target_Radius_km" in df.columns:
            radii_to_use = _available_radii_from_df(df)
        else:
            radii_to_use = _available_radii_from_df(df) or [40]
    else:
        radii_to_use = [int(round(float(r))) for r in radii]
    if not radii_to_use:
        radii_to_use = [40]

    df[out_col] = np.zeros(len(df), dtype=np.int8)
    date_keys = df["Date"].astype(str).str[:10].str.replace("-", "", regex=False).str[:8]
    base_keys_all = _keys_for_date_lat_lon(df)

    if "ML_Target_Radius_km" in df.columns and not pointwise:
        rad_series = pd.to_numeric(df["ML_Target_Radius_km"], errors="coerce")
    else:
        rad_series = pd.Series(np.nan, index=df.index)

    total_matched = 0
    for rr in radii_to_use:
        if pointwise:
            # Pointwise target is the same column for every radius master; use first available master once.
            mask_idx = df.index
        elif "ML_Target_Radius_km" in df.columns:
            mask_idx = df.index[(rad_series == int(rr)).to_numpy()]
        else:
            # Wide dataframe: no row radius.  Use the requested/first radius target for all rows.
            mask_idx = df.index
        if len(mask_idx) == 0:
            continue
        tab, target_col = _load_mrms_proxy_table_for_radius(rr, pointwise=pointwise)
        # Avoid merging dates that are not represented in this subset.
        needed_dates = set(date_keys.loc[mask_idx].dropna().unique().tolist())
        tab_use = tab[tab["__DateKey"].isin(needed_dates)].copy()
        left = base_keys_all.loc[mask_idx].reset_index(drop=True)
        merged = left.merge(tab_use, on=["__DateKey", "__LatKey", "__LonKey"], how="left")
        vals = pd.to_numeric(merged[target_col], errors="coerce")
        matched = int(vals.notna().sum())
        total_matched += matched
        df.loc[mask_idx, out_col] = (vals.fillna(0).to_numpy(float) > 0).astype(np.int8)
        print(
            f"Merged {out_col} from {target_col} for r{rr}: "
            f"matched_rows={matched:,}/{len(mask_idx):,}, positives={int(df.loc[mask_idx, out_col].sum()):,}"
        )
        if pointwise:
            break

    if total_matched == 0:
        raise RuntimeError(
            f"No rows matched when creating {out_col} from the MRMS/FFG master parquet. "
            "Check Date/Lat/Lon precision and that the master parquet covers these dates."
        )
    return out_col


# Override raw UFVS merge to ensure the full proxy set is materialized, not only whichever
# subset happens to exist in the current viewer dataframe.
def _merge_raw_ufvs_proxy_columns_inplace(df):
    return _ensure_raw_ufvs_proxy_columns_inplace(df, requested_col="UFVS_ANY", ensure_all=True)


# Override proxy resolver.  This is used by CSI, BS/BSS, reliability, and fractional coverage.
def _resolve_proxy_column(df, proxy_col="UFVS_ANY", allow_pp_fallback=False):
    canonical = _canonical_proxy_column(proxy_col)
    if canonical is None:
        canonical = "UFVS_ANY"
    if canonical in {"MRMS_FFG", "MRMS_FFG_POINT"}:
        out_col = _merge_mrms_ffg_proxy_column_inplace(df, proxy_col=canonical)
        return out_col, "mrms_target" if out_col == "MRMS_FFG" else "mrms_point"
    if canonical.startswith("UFVS_"):
        _ensure_raw_ufvs_proxy_columns_inplace(df, requested_col=canonical, ensure_all=(canonical == "UFVS_ANY"))
        if canonical in df.columns:
            return canonical, "raw_ufvs"
    # Exact user-provided column still allowed if it exists and is not PP_*.
    if proxy_col is not None and str(proxy_col) in df.columns and not str(proxy_col).startswith("PP_"):
        return str(proxy_col), "selected_raw"
    raise RuntimeError(
        f"Requested proxy {proxy_col!r} resolved to {canonical!r}, but that column is unavailable.\n"
        f"Available raw proxy columns: {[c for c in df.columns if str(c).startswith('UFVS_') or str(c).startswith('MRMS_FFG')]}\n"
        "Run display_available_truth_options(...) for the valid option strings. PP_* columns are not accepted for proxy-truth verification."
    )


# Override the 4-panel case plot so aliases like MRMS_FFG, STAGE4_FFG, FLOOD_LSR work there too.
def plot_case_ml_wpc_pp_proxy(date, radius_km=40, pp_definition="Any flood proxy", proxy_col=None,
                              smooth_sigma_grid=0, point_size=7, alpha=0.85, df=None):
    """Four-panel case map: ML, WPC ERO, practically perfect, and selected proxy/truth column.

    proxy_col accepts aliases such as UFVS_ANY, STAGE4_FFG, STAGE4_ARI, USGS,
    FLASH_LSR, FLOOD_LSR, MRMS_FFG, and MRMS_FFG_POINT.
    """
    if df is None:
        df = df_radius_viewer
    sub = _case_radius_subset(df, date, radius_km).copy()
    pp_col = _truth_col_from_pp_definition(pp_definition)
    if pp_col not in sub.columns:
        raise RuntimeError(f"Missing PP column {pp_col}. Available PP columns: {[c for c in sub.columns if c.startswith('PP_')][:20]}")
    if proxy_col is None:
        resolved_proxy_col = pp_col
    else:
        resolved_proxy_col, truth_kind = _resolve_proxy_column(sub, proxy_col=proxy_col, allow_pp_fallback=False)
        print(f"Resolved proxy_col={proxy_col!r} -> {resolved_proxy_col!r} ({truth_kind})")
    if resolved_proxy_col not in sub.columns:
        raise RuntimeError(f"Missing proxy/truth column {resolved_proxy_col} after resolution.")
    proj = ccrs.PlateCarree() if HAS_CARTOPY else None
    subplot_kw = {"projection": proj} if HAS_CARTOPY else {}
    fig, axes = plt.subplots(2, 2, figsize=(16, 11), subplot_kw=subplot_kw, constrained_layout=True)
    axes = axes.ravel()
    _plot_prob_or_smoothed(axes[0], sub, "ML_Forecast_Prob", f"ML probability r{int(radius_km)} km | {str(date)[:8]}", smooth_sigma_grid, point_size, alpha)
    _plot_prob_or_smoothed(axes[1], sub, WPC_COL, "WPC ERO risk", smooth_sigma_grid, point_size, alpha)
    _plot_prob_or_smoothed(axes[2], sub, pp_col, f"Practically Perfect: {pp_definition}", smooth_sigma_grid, point_size, alpha)
    _plot_prob_or_smoothed(axes[3], sub, resolved_proxy_col, f"Proxy/truth: {resolved_proxy_col}", smooth_sigma_grid, point_size, alpha)
    plt.show()
    plt.close(fig)
    return None


print("V9 truth/proxy registry loaded.")
print("Use display_available_truth_options(prefer_realtime=False, radii=[40,60,75,100]) to list valid PP/proxy settings.")


# ======================================================================================
# V10 PP-DEFINITION ALIAS FIX
# ======================================================================================
# CAT_PP_DEFINITION is for PP_* fields, but common strings like "MRMS_FFG" should
# resolve to the PP column that actually exists, e.g. PP_MRMS > FFG.  Raw target/proxy
# verification still belongs in *_PROXY_COL, but this prevents a misleading "No data to plot"
# when the user selects a natural alias for a PP field.

_PP_DEFINITION_ALIAS_CANDIDATES = {
    # Historical PP definitions
    "mrms": ["PP_MRMS > FFG", "PP_MRMS_FFG", "PP_Stage IV > FFG"],
    "mrmsffg": ["PP_MRMS > FFG", "PP_MRMS_FFG", "PP_Stage IV > FFG"],
    "mrmsgtffg": ["PP_MRMS > FFG", "PP_MRMS_FFG", "PP_Stage IV > FFG"],
    "mrmsexceedffg": ["PP_MRMS > FFG", "PP_MRMS_FFG", "PP_Stage IV > FFG"],
    "mrmsexceedsffg": ["PP_MRMS > FFG", "PP_MRMS_FFG", "PP_Stage IV > FFG"],
    "mrmstarget": ["PP_MRMS > FFG", "PP_MRMS_FFG", "PP_Stage IV > FFG"],

    # Generic/union PP definitions
    "any": ["PP_Any flood proxy"],
    "ufvsany": ["PP_Any flood proxy"],
    "anyfloodproxy": ["PP_Any flood proxy"],
    "floodproxy": ["PP_Any flood proxy"],
    "allproxies": ["PP_Any flood proxy"],

    # LSR/USGS historical PP definition
    "lsrusgs": ["PP_LSR/USGS only", "PP_USGS", "PP_Flash LSR"],
    "usgslsr": ["PP_LSR/USGS only", "PP_USGS", "PP_Flash LSR"],
    "lsrandusgs": ["PP_LSR/USGS only", "PP_USGS", "PP_Flash LSR"],
    "usgs": ["PP_USGS", "PP_LSR/USGS only"],
    "flashlsr": ["PP_Flash LSR", "PP_LSR/USGS only"],
    "lsrflash": ["PP_Flash LSR", "PP_LSR/USGS only"],
    "floodlsr": ["PP_Flood LSR", "PP_LSR/USGS only"],

    # Realtime PP definitions created directly from UFVS sources
    "stage4ffg": ["PP_Stage IV > FFG", "PP_MRMS > FFG"],
    "stageivffg": ["PP_Stage IV > FFG", "PP_MRMS > FFG"],
    "st4gffg": ["PP_Stage IV > FFG", "PP_MRMS > FFG"],
    "stage4ari": ["PP_Stage IV ARI"],
    "stageivari": ["PP_Stage IV ARI"],
    "st4gari": ["PP_Stage IV ARI"],
}


def _pp_definition_candidates(pp_definition):
    """Return ordered PP column candidates for a user-facing PP definition string."""
    s = str(pp_definition).strip()
    norm = _norm_proxy_name(s) if "_norm_proxy_name" in globals() else re.sub(r"[^a-z0-9]+", "", s.lower())
    candidates = []

    if s.startswith("PP_"):
        candidates.append(s)
    else:
        candidates.append(f"PP_{s}")

    # Raw/proxy option aliases are accepted here only as PP aliases.  They do not
    # change truth_mode; they merely resolve to an existing PP_* field.
    candidates.extend(_PP_DEFINITION_ALIAS_CANDIDATES.get(norm, []))

    # Also try canonical raw proxy option names as aliases to the most likely PP field.
    canonical = _canonical_proxy_column(s) if "_canonical_proxy_column" in globals() else None
    if canonical == "MRMS_FFG":
        candidates.extend(["PP_MRMS > FFG", "PP_MRMS_FFG", "PP_Stage IV > FFG"])
    elif canonical == "UFVS_ANY":
        candidates.append("PP_Any flood proxy")
    elif canonical == "UFVS_STAGE4_FFG":
        candidates.extend(["PP_Stage IV > FFG", "PP_MRMS > FFG"])
    elif canonical == "UFVS_STAGE4_ARI":
        candidates.append("PP_Stage IV ARI")
    elif canonical == "UFVS_USGS":
        candidates.extend(["PP_USGS", "PP_LSR/USGS only"])
    elif canonical == "UFVS_LSR_FLASH":
        candidates.extend(["PP_Flash LSR", "PP_LSR/USGS only"])
    elif canonical in {"UFVS_LSR_FLOOD", "UFVS_LSR_REGULAR"}:
        candidates.extend(["PP_Flood LSR", "PP_LSR/USGS only"])

    # Preserve order while removing duplicates.
    out = []
    seen = set()
    for c in candidates:
        if c and c not in seen:
            out.append(c)
            seen.add(c)
    return out


def _resolve_pp_truth_column_for_eval(df, pp_definition, verbose=True):
    """Resolve CAT_PP_DEFINITION / CSI_PP_DEFINITION aliases against actual PP_* columns."""
    pp_cols = [c for c in df.columns if str(c).startswith("PP_")]
    if not pp_cols:
        if verbose:
            print("No PP_* columns are available in this dataframe.")
        return None

    candidates = _pp_definition_candidates(pp_definition)
    for c in candidates:
        if c in df.columns:
            if verbose and str(pp_definition) != c and str(pp_definition) != c.replace("PP_", ""):
                print(f"Resolved PP definition {pp_definition!r} -> {c!r}")
            return c

    # Last chance: normalized matching against available PP columns, so things like
    # 'MRMS FFG', 'mrms_ffg', and 'MRMS>FFG' all work if a matching PP column exists.
    norm_lookup = {}
    for c in pp_cols:
        bare = str(c).replace("PP_", "", 1)
        norm_lookup[_norm_proxy_name(bare)] = c
        norm_lookup[_norm_proxy_name(c)] = c
    for c in candidates:
        key = _norm_proxy_name(str(c).replace("PP_", "", 1))
        if key in norm_lookup:
            resolved = norm_lookup[key]
            if verbose:
                print(f"Resolved PP definition {pp_definition!r} -> {resolved!r}")
            return resolved

    if verbose:
        print(
            f"Could not resolve PP definition {pp_definition!r}.\n"
            f"Tried candidates: {candidates}\n"
            f"Available PP definitions: {[c.replace('PP_', '', 1) for c in pp_cols]}\n"
            "Use one of the available PP definitions above, or use a raw/target option like "
            "MRMS_FFG in the *_PROXY_COL setting instead."
        )
    return None


# Override the older simple PP resolvers so case-plot and realtime checks understand aliases too.
def _truth_col_from_pp_definition_integrated(pp_definition):
    # Without a dataframe, return the first reasonable candidate.  Metric functions use
    # _resolve_pp_truth_column_for_eval(df, ...) and therefore choose the candidate that
    # actually exists in the selected dataframe.
    return _pp_definition_candidates(pp_definition)[0]


def _truth_col_from_pp_definition(pp_definition):
    return _pp_definition_candidates(pp_definition)[0]


# Extend the options display with PP aliases.
_display_available_truth_options_v9 = display_available_truth_options

def display_available_truth_options(df=None, prefer_realtime=False, radii=None, attempt_rebuild=False):
    opts = _display_available_truth_options_v9(df=df, prefer_realtime=prefer_realtime, radii=radii, attempt_rebuild=attempt_rebuild)
    if df is None:
        df, _ = choose_evaluation_dataframe(prefer_realtime=prefer_realtime)
    pp_cols = [c for c in df.columns if str(c).startswith("PP_")]
    print("\nAccepted aliases for *_PP_DEFINITION include:")
    alias_rows = []
    examples = [
        "Any flood proxy", "UFVS_ANY", "MRMS > FFG", "MRMS_FFG",
        "LSR/USGS only", "USGS", "FLASH_LSR", "STAGE4_FFG", "STAGE4_ARI",
    ]
    for ex in examples:
        resolved = _resolve_pp_truth_column_for_eval(df, ex, verbose=False)
        alias_rows.append({
            "example_input": ex,
            "resolved_PP_column": resolved if resolved is not None else "not available in this dataframe",
        })
    alias_df = pd.DataFrame(alias_rows)
    try:
        display(alias_df)
    except Exception:
        print(alias_df.to_string(index=False))
    return opts

print("V10 PP alias resolver active. Example: CAT_PP_DEFINITION='MRMS_FFG' resolves to PP_MRMS > FFG when that PP column is present.")


### 5e-2. Show available truth/proxy option strings

Run this cell to see what can be used in `*_PP_DEFINITION` and `*_PROXY_COL` settings.

Important distinction:

- `*_PP_DEFINITION` chooses an existing `PP_*` truth field. Aliases like `MRMS_FFG` now resolve to `PP_MRMS > FFG` when that PP column exists.
- `*_PROXY_COL` chooses raw/target truth, such as `MRMS_FFG`, `UFVS_ANY`, `STAGE4_FFG`, `USGS`, etc.


In [ ]:
# ---- Show selectable PP/proxy truth options ----
OPTIONS_PREFER_REALTIME = False   # False = historical/test-set dataframe; True = current realtime dataframe if available
OPTIONS_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
OPTIONS_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))  # four Day-2 radius members

OPTIONS_DF, OPTIONS_DF_LABEL = choose_evaluation_dataframe(prefer_realtime=OPTIONS_PREFER_REALTIME)
print(f"Using {OPTIONS_DF_LABEL} for option discovery")
print_available_forecasts_for_eval(OPTIONS_DF, OPTIONS_DF_LABEL)

# attempt_rebuild=False lists valid choices without immediately fetching all UFVS archive files.
# Set True if you want this cell itself to force-check/rebuild raw columns before the metric cells.
truth_options_table = display_available_truth_options(
    OPTIONS_DF,
    radii=OPTIONS_RADII_KM,
    attempt_rebuild=False,
)


### 5f. Run threshold-exceedance verification: ML/WPC/ensemble using Practically Perfect truth (all available radii)

In [ ]:
# ---- PP-truth categorical verification: all available radii on same plots ----
CAT_PREFER_REALTIME = False   # False = 2024-2025 historical/test set; True = current realtime dataframe if available
CAT_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
CAT_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))  # four Day-2 radius members
CAT_PP_DEFINITION = "Any flood proxy"  # examples: "Any flood proxy", "MRMS > FFG", "MRMS_FFG", "LSR/USGS only"
CAT_INCLUDE_ENSEMBLES = True

CAT_DF, CAT_DF_LABEL = choose_evaluation_dataframe(prefer_realtime=CAT_PREFER_REALTIME)
print(f"Using {CAT_DF_LABEL} for PP-truth categorical verification")
print_available_forecasts_for_eval(CAT_DF, CAT_DF_LABEL)

cat_pp_case, cat_pp_case_mean, cat_pp_pooled = compute_categorical_area_skill(
    CAT_DF,
    radii=CAT_RADII_KM,
    truth_mode="pp",
    pp_definition=CAT_PP_DEFINITION,
    include_members=True,
    include_ensembles=CAT_INCLUDE_ENSEMBLES,
    include_wpc=True,
)

print("Case-mean threshold-exceedance metrics using Practically Perfect truth")
display(cat_pp_case_mean.round(4))
print("Pooled threshold-exceedance metrics using Practically Perfect truth")
display(cat_pp_pooled.round(4))

for _metric in ["CSI", "POD", "FAR", "Bias"]:
    plot_categorical_skill_lines(
        cat_pp_case_mean,
        metric=_metric,
        title=f"{_metric} by threshold exceedance | PP truth = {CAT_PP_DEFINITION} | {CAT_DF_LABEL}",
    )


### 5g. Run threshold-exceedance verification: ML/WPC/ensemble using flood-proxy truth (all available radii)

In [ ]:
# ---- Flood-proxy-truth categorical verification: all available radii on same plots ----
PROXY_PREFER_REALTIME = False   # False = 2024-2025 historical/test set; True = current realtime dataframe if available
PROXY_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
PROXY_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))  # four Day-2 radius members
PROXY_TRUTH_COL = "UFVS_ANY"  # options: UFVS_ANY, STAGE4_FFG, STAGE4_ARI, USGS, FLASH_LSR, FLOOD_LSR, MRMS_FFG, MRMS_FFG_POINT
PROXY_INCLUDE_ENSEMBLES = True

PROXY_DF, PROXY_DF_LABEL = choose_evaluation_dataframe(prefer_realtime=PROXY_PREFER_REALTIME)
print(f"Using {PROXY_DF_LABEL} for proxy-truth categorical verification")
print_available_forecasts_for_eval(PROXY_DF, PROXY_DF_LABEL)

cat_proxy_case, cat_proxy_case_mean, cat_proxy_pooled = compute_categorical_area_skill(
    PROXY_DF,
    radii=PROXY_RADII_KM,
    truth_mode="proxy",
    proxy_col=PROXY_TRUTH_COL,
    proxy_truth_threshold=0.0,
    include_members=True,
    include_ensembles=PROXY_INCLUDE_ENSEMBLES,
    include_wpc=True,
    allow_pp_proxy_fallback=False,
)

print("Case-mean threshold-exceedance metrics using flood-proxy truth")
display(cat_proxy_case_mean.round(4))
print("Pooled threshold-exceedance metrics using flood-proxy truth")
display(cat_proxy_pooled.round(4))

for _metric in ["CSI", "POD", "FAR", "Bias"]:
    plot_categorical_skill_lines(
        cat_proxy_case_mean,
        metric=_metric,
        title=f"{_metric} by threshold exceedance | proxy truth = {PROXY_TRUTH_COL} | {PROXY_DF_LABEL}",
    )


### 5h. Flood-proxy fractional coverage by exclusive risk bin (all available radii)

Primary metric: `Fraction Of Bin Covered By Proxy = count(proxy & bin) / count(bin)`. `Proxy Capture Fraction = count(proxy & bin) / count(proxy)` is only a secondary diagnostic.


In [ ]:
# ---- Fractional coverage of selected flood proxy by exclusive risk bin: all available radii ----
# NOTE: This is intentionally exclusive-bin logic (5-15, 15-40, 40-70, >=70) because
# it diagnoses what fraction of each categorical risk area is covered by the selected proxy.
FRAC_PREFER_REALTIME = False   # False = 2024-2025 historical/test set; True = current realtime dataframe if available
FRAC_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
FRAC_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))  # four Day-2 radius members
FRAC_PROXY_COL = "UFVS_ANY"  # options: UFVS_ANY, STAGE4_FFG, STAGE4_ARI, USGS, FLASH_LSR, FLOOD_LSR, MRMS_FFG, MRMS_FFG_POINT
FRAC_INCLUDE_ENSEMBLES = True

FRAC_DF, FRAC_DF_LABEL = choose_evaluation_dataframe(prefer_realtime=FRAC_PREFER_REALTIME)
print(f"Using {FRAC_DF_LABEL} for fractional-coverage diagnostics")
print_available_forecasts_for_eval(FRAC_DF, FRAC_DF_LABEL)

frac_case, frac_summary = compute_flood_proxy_fractional_coverage_by_category(
    FRAC_DF,
    radii=FRAC_RADII_KM,
    proxy_col=FRAC_PROXY_COL,
    include_members=True,
    include_ensembles=FRAC_INCLUDE_ENSEMBLES,
    include_wpc=True,
    allow_pp_proxy_fallback=False,
)

print("Case-mean fractional coverage by exclusive risk bin")
display(frac_summary.round(4))

plot_fractional_coverage_lines(
    frac_summary,
    value_col="Fraction Of Bin Covered By Proxy",
    title=f"Fraction of each exclusive forecast bin covered by raw proxy | {FRAC_DF_LABEL}",
)
plot_fractional_coverage_lines(
    frac_summary,
    value_col="Forecast Area Fraction",
    title=f"Forecast area fraction by exclusive risk bin | {FRAC_DF_LABEL}",
)


### 5i. Realtime threshold-exceedance metrics and fractional coverage (all radius members + ensembles + WPC)

In [ ]:
# ---- Realtime threshold-exceedance verification and fractional coverage: all radius members / ensembles / WPC on same plots ----
if "df_realtime_viewer" not in globals():
    print("df_realtime_viewer is not defined yet. Run the multi-radius realtime cell first.")
else:
    RT_DF = df_realtime_viewer
    RT_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
    RT_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))
    RT_PP_DEFINITION = "Any flood proxy"
    RT_PROXY_COL = "UFVS_ANY"

    print_available_forecasts_for_eval(RT_DF, "realtime multi-radius df_realtime_viewer")

    if _truth_col_from_pp_definition_integrated(RT_PP_DEFINITION) in RT_DF.columns:
        rt_pp_case, rt_pp_case_mean, rt_pp_pooled = compute_categorical_area_skill(
            RT_DF,
            radii=RT_RADII_KM,
            truth_mode="pp",
            pp_definition=RT_PP_DEFINITION,
            include_members=True,
            include_ensembles=True,
            include_wpc=True,
        )
        print("Realtime threshold-exceedance metrics using PP truth")
        display(rt_pp_case_mean.round(4))
        for _metric in ["CSI", "POD", "FAR", "Bias"]:
            plot_categorical_skill_lines(
                rt_pp_case_mean,
                metric=_metric,
                title=f"Realtime {_metric} by threshold exceedance | PP truth = {RT_PP_DEFINITION}",
            )
    else:
        print(f"No realtime PP column {_truth_col_from_pp_definition_integrated(RT_PP_DEFINITION)} found; skipping PP-truth realtime metrics.")

    rt_proxy_case, rt_proxy_case_mean, rt_proxy_pooled = compute_categorical_area_skill(
        RT_DF,
        radii=RT_RADII_KM,
        truth_mode="proxy",
        proxy_col=RT_PROXY_COL,
        proxy_truth_threshold=0.0,
        include_members=True,
        include_ensembles=True,
        include_wpc=True,
        allow_pp_proxy_fallback=False,
    )
    print("Realtime threshold-exceedance metrics using flood-proxy truth")
    display(rt_proxy_case_mean.round(4))
    for _metric in ["CSI", "POD", "FAR", "Bias"]:
        plot_categorical_skill_lines(
            rt_proxy_case_mean,
            metric=_metric,
            title=f"Realtime {_metric} by threshold exceedance | proxy truth = {RT_PROXY_COL}",
        )

    rt_frac_case, rt_frac_summary = compute_flood_proxy_fractional_coverage_by_category(
        RT_DF,
        radii=RT_RADII_KM,
        proxy_col=RT_PROXY_COL,
        include_members=True,
        include_ensembles=True,
        include_wpc=True,
        allow_pp_proxy_fallback=False,
    )
    print("Realtime flood-proxy fractional coverage by exclusive forecast bin")
    display(rt_frac_summary.round(4))
    for _value_col in ["Fraction Of Bin Covered By Proxy", "Forecast Area Fraction", "Proxy Capture Fraction"]:
        plot_fractional_coverage_lines(
            rt_frac_summary,
            value_col=_value_col,
            title=f"Realtime {_value_col} by exclusive forecast bin",
        )


### 5j. Interactive predictor browser

Click through predictors and plot the selected predictor beside the ML forecast. Use `source='historical'` for the test-case master parquet or `source='realtime'` after the realtime feature cache has been built.

In [ ]:
# ---- Launch predictor browser ----
# Historical example:
# launch_predictor_browser(initial_date="20250629", initial_radius=40, source="historical")

# Realtime example after running the realtime cell:
# launch_predictor_browser(initial_date=REALTIME_DATE, initial_radius=40, source="realtime")

launch_predictor_browser(
    initial_date=REALTIME_DATE if "REALTIME_DATE" in globals() else "20250629",
    initial_radius=40,
    source="realtime" if "df_realtime_viewer" in globals() else "historical",
)


In [ ]:

# 5c. Optional realtime PP sanity check
# Run this after the realtime build to verify that the PP field is not saturating to High risk everywhere.
pp_cols = [c for c in df_realtime_viewer.columns if c.startswith("PP_")]
print("PP columns:", pp_cols)
for c in pp_cols:
    x = pd.to_numeric(df_realtime_viewer[c], errors="coerce").to_numpy(float)
    print(
        f"{c:24s} max={np.nanmax(x):.3f} "
        f"p99={np.nanpercentile(x, 99):.3f} "
        f">=0.05={np.nanmean(x >= 0.05):.3f} "
        f">=0.15={np.nanmean(x >= 0.15):.3f} "
        f">=0.40={np.nanmean(x >= 0.40):.3f} "
        f">=0.70={np.nanmean(x >= 0.70):.3f}"
    )


## 6. Adjustable practically-perfect expansion radius

In [ ]:
# ---- Custom PP / proxy expansion settings ----
EVENT_COL = None  # set this to a raw event/proxy column if present
EXPANSION_RADIUS_KM = 60

if EVENT_COL:
    df_radius_viewer_custom, CUSTOM_PP_COL = add_custom_pp_column(
        df_radius_viewer,
        event_col=EVENT_COL,
        expansion_radius_km=EXPANSION_RADIUS_KM,
    )
    print("Created:", CUSTOM_PP_COL)
    plot_case_ml_wpc_pp_proxy(CASE_DATE, radius_km=RADIUS_KM, pp_definition=PP_DEFINITION, proxy_col=CUSTOM_PP_COL)
else:
    candidate_cols = [c for c in df_radius_viewer.columns if not c.startswith("PP_") and any(s in c.lower() for s in ["proxy","lsr","usgs","ffg","flood","exceed"])]
    print("Set EVENT_COL to one of these candidate raw proxy/event columns if available:")
    print(candidate_cols[:80])

## 7. Reliability diagrams: all available radius tests

In [ ]:
# ---- Reliability diagram for all available radius tests / WPC / ensembles ----
RELIABILITY_PREFER_REALTIME = False   # False = 2024-2025 historical/test set; True = current realtime dataframe if available
RELIABILITY_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
RELIABILITY_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))
RELIABILITY_TRUTH_MODE = "pp"        # "pp" or "proxy"
RELIABILITY_PP_DEFINITION = "Any flood proxy"
RELIABILITY_PROXY_COL = "UFVS_ANY"  # options: UFVS_ANY, STAGE4_FFG, STAGE4_ARI, USGS, FLASH_LSR, FLOOD_LSR, MRMS_FFG, MRMS_FFG_POINT
RELIABILITY_EVENT_THRESHOLD = 0.05    # PP/probability-like truth converted to event for reliability
RELIABILITY_INCLUDE_ENSEMBLES = True
RELIABILITY_MIN_BIN_N = 1

REL_DF, REL_DF_LABEL = choose_evaluation_dataframe(prefer_realtime=RELIABILITY_PREFER_REALTIME)
print(f"Using {REL_DF_LABEL} for reliability")
print_available_forecasts_for_eval(REL_DF, REL_DF_LABEL)

rel_all = compute_reliability_all_sources(
    REL_DF,
    radii=RELIABILITY_RADII_KM,
    truth_mode=RELIABILITY_TRUTH_MODE,
    pp_definition=RELIABILITY_PP_DEFINITION,
    proxy_col=RELIABILITY_PROXY_COL,
    pp_event_threshold=RELIABILITY_EVENT_THRESHOLD,
    include_members=True,
    include_ensembles=RELIABILITY_INCLUDE_ENSEMBLES,
    include_wpc=True,
    allow_pp_proxy_fallback=False,
)

display(rel_all.round(4))
plot_reliability_all_sources(
    rel_all,
    min_bin_n=RELIABILITY_MIN_BIN_N,
    title=f"Reliability | truth={RELIABILITY_TRUTH_MODE}:{RELIABILITY_PP_DEFINITION if RELIABILITY_TRUTH_MODE == 'pp' else RELIABILITY_PROXY_COL} | {REL_DF_LABEL}",
)


## 8. Erickson-style fractional coverage by exclusive risk bin: all available radius tests

Primary metric: `Fraction Of Bin Covered By Proxy = count(proxy & bin) / count(bin)`.


In [ ]:
# This is the same exclusive-bin fractional-coverage diagnostic as section 5h, kept here as a standalone block.
ERICKSON_FRAC_PREFER_REALTIME = False
ERICKSON_FRAC_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
ERICKSON_FRAC_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))
ERICKSON_FRAC_PROXY_COL = "UFVS_ANY"  # options: UFVS_ANY, STAGE4_FFG, STAGE4_ARI, USGS, FLASH_LSR, FLOOD_LSR, MRMS_FFG, MRMS_FFG_POINT  # options: UFVS_ANY, STAGE4_FFG, STAGE4_ARI, USGS, FLASH_LSR, FLOOD_LSR, MRMS_FFG, MRMS_FFG_POINT

ERICKSON_DF, ERICKSON_DF_LABEL = choose_evaluation_dataframe(prefer_realtime=ERICKSON_FRAC_PREFER_REALTIME)
print(f"Using {ERICKSON_DF_LABEL} for Erickson-style fractional coverage")
print_available_forecasts_for_eval(ERICKSON_DF, ERICKSON_DF_LABEL)

fc_case, fc_summary = compute_flood_proxy_fractional_coverage_by_category(
    ERICKSON_DF,
    radii=ERICKSON_FRAC_RADII_KM,
    proxy_col=ERICKSON_FRAC_PROXY_COL,
    include_members=True,
    include_ensembles=True,
    include_wpc=True,
    allow_pp_proxy_fallback=False,
)

display(fc_summary.round(4))
plot_fractional_coverage_lines(fc_summary, value_col="Fraction Of Bin Covered By Proxy")
plot_fractional_coverage_lines(fc_summary, value_col="Forecast Area Fraction")
plot_fractional_coverage_lines(fc_summary, value_col="Proxy Capture Fraction")


## 9. Average CSI/POD/FAR/Bias using Practically Perfect truth: all available radius tests

# CSI and related metrics use cumulative threshold exceedances: >5%, >15%, >40%, >70%.
CSI_PP_PREFER_REALTIME = False
CSI_PP_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
CSI_PP_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))
CSI_PP_DEFINITION = "Any flood proxy"  # examples: "Any flood proxy", "MRMS > FFG", "MRMS_FFG", "LSR/USGS only"

CSI_PP_DF, CSI_PP_DF_LABEL = choose_evaluation_dataframe(prefer_realtime=CSI_PP_PREFER_REALTIME)
print(f"Using {CSI_PP_DF_LABEL} for PP-truth CSI")
print_available_forecasts_for_eval(CSI_PP_DF, CSI_PP_DF_LABEL)

csi_pp_case, csi_pp_summary, csi_pp_pooled = compute_categorical_area_skill(
    CSI_PP_DF,
    radii=CSI_PP_RADII_KM,
    truth_mode="pp",
    pp_definition=CSI_PP_DEFINITION,
    include_members=True,
    include_ensembles=True,
    include_wpc=True,
)

display(csi_pp_summary.round(4))
plot_categorical_skill_lines(csi_pp_summary, metric="CSI", title=f"CSI by threshold exceedance | PP truth={CSI_PP_DEFINITION} | {CSI_PP_DF_LABEL}")


## 10. Average CSI/POD/FAR/Bias using selected flood-proxy truth: all available radius tests

# CSI and related metrics use cumulative threshold exceedances: >5%, >15%, >40%, >70%.
CSI_PROXY_PREFER_REALTIME = False
CSI_PROXY_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
CSI_PROXY_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))
CSI_PROXY_COL = "UFVS_ANY"  # options: UFVS_ANY, STAGE4_FFG, STAGE4_ARI, USGS, FLASH_LSR, FLOOD_LSR, MRMS_FFG, MRMS_FFG_POINT

CSI_PROXY_DF, CSI_PROXY_DF_LABEL = choose_evaluation_dataframe(prefer_realtime=CSI_PROXY_PREFER_REALTIME)
print(f"Using {CSI_PROXY_DF_LABEL} for proxy-truth CSI")
print_available_forecasts_for_eval(CSI_PROXY_DF, CSI_PROXY_DF_LABEL)

csi_proxy_case, csi_proxy_summary, csi_proxy_pooled = compute_categorical_area_skill(
    CSI_PROXY_DF,
    radii=CSI_PROXY_RADII_KM,
    truth_mode="proxy",
    proxy_col=CSI_PROXY_COL,
    proxy_truth_threshold=0.0,
    include_members=True,
    include_ensembles=True,
    include_wpc=True,
    allow_pp_proxy_fallback=False,
)

display(csi_proxy_summary.round(4))
plot_categorical_skill_lines(csi_proxy_summary, metric="CSI", title=f"CSI by threshold exceedance | proxy={CSI_PROXY_COL} | {CSI_PROXY_DF_LABEL}")


## 11. Brier Score / Brier Skill Score by categorical risk area: all available radius tests

This block plots BS/BSS for all available ML radii plus WPC across the categorical exceedance areas:

`>5%`, `>15%`, `>40%`, `>70%`

Default behavior:

`BRIER_EVALUATION_REGION = "forecast_exceedance_area"`

For each threshold, the score is computed **only inside that source's forecast area exceeding the threshold**, while keeping the forecast value as the original continuous probability. This makes the plotted threshold categories meaningful for raw binary proxy truth.

When `BRIER_TRUTH_MODE = "proxy"`, the selected raw proxy truth is expanded by 40 km before scoring, matching the Erickson-style verification convention.



In [ ]:
# ---- Brier Score / Brier Skill Score by categorical forecast area for all available radius tests / WPC / ensembles ----
BRIER_PREFER_REALTIME = False   # False = 2024-2025 historical/test set; True = current realtime dataframe if available
BRIER_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
BRIER_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))
BRIER_TRUTH_MODE = "proxy"      # "pp" or "proxy"
BRIER_PP_DEFINITION = "Any flood proxy"   # used when BRIER_TRUTH_MODE == "pp"
BRIER_PROXY_COL = "UFVS_ANY"   # options: UFVS_ANY, STAGE4_FFG, STAGE4_ARI, USGS, FLASH_LSR, FLOOD_LSR, MRMS_FFG, MRMS_FFG_POINT
BRIER_PROXY_EXPANSION_RADIUS_KM = 40.0   # raw proxy truth expansion radius (Erickson-style verification)
BRIER_PROXY_THRESHOLD = 0.0
BRIER_INCLUDE_ENSEMBLES = True
BRIER_PLOT_SOURCE = "case_mean"  # "case_mean" or "pooled"

# Options:
#   "forecast_exceedance_area" = score only inside each source's p >= threshold area.
#   "full_domain"              = score entire grid/domain for each event threshold.
#   "deterministic_full_domain" = convert forecast to yes/no p >= threshold before scoring whole domain.
#
# For raw proxy truth, "full_domain" will be flat across thresholds because the binary proxy truth
# and continuous probability forecast do not change with the threshold. The default below avoids that.
BRIER_EVALUATION_REGION = "forecast_exceedance_area"


def _truth_binary_for_brier_threshold(
    sub,
    threshold,
    truth_mode="proxy",
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    proxy_truth_threshold=0.0,
    proxy_expand_radius_km=40.0,
):
    """Return (truth_col, truth_kind, binary_truth_array) for a single event threshold."""
    mode = str(truth_mode).lower().strip()

    if mode == "pp":
        truth_col = _resolve_pp_truth_column_for_eval(sub, pp_definition)
        y = _binary_from_col(sub, truth_col, threshold=float(threshold), mode=">=")
        return truth_col, "pp", y.astype(bool)

    if mode != "proxy":
        raise RuntimeError("truth_mode must be 'pp' or 'proxy'.")

    truth_col, truth_kind = _resolve_proxy_column(sub, proxy_col=proxy_col, allow_pp_fallback=False)
    y = _binary_from_col(sub, truth_col, threshold=float(proxy_truth_threshold), mode=">").astype(bool)

    # Expand raw proxy truth to the desired ROI radius for verification.
    should_expand = (
        proxy_expand_radius_km is not None
        and float(proxy_expand_radius_km) > 0
        and truth_kind in {"raw_ufvs", "selected_raw", "mrms_point"}
    )
    if should_expand:
        if "Lat" not in sub.columns or "Lon" not in sub.columns:
            raise RuntimeError(
                f"Cannot expand proxy truth {truth_col}; dataframe lacks Lat/Lon columns."
            )
        lat = pd.to_numeric(sub["Lat"], errors="coerce").to_numpy(float)
        lon = pd.to_numeric(sub["Lon"], errors="coerce").to_numpy(float)
        if np.isfinite(lat).any() and np.isfinite(lon).any():
            xyz = latlon_to_unit_xyz(lat, lon)
            tree = cKDTree(xyz)
            y = expand_binary_mask_radius_km(
                y,
                tree=tree,
                xyz=xyz,
                radius_km=float(proxy_expand_radius_km),
            )
    return truth_col, truth_kind, y.astype(bool)


def compute_brier_by_category_area_all_sources(
    df,
    radii=None,
    truth_mode="proxy",
    pp_definition="Any flood proxy",
    proxy_col="UFVS_ANY",
    proxy_truth_threshold=0.0,
    proxy_expand_radius_km=40.0,
    include_members=True,
    include_ensembles=True,
    include_wpc=True,
    thresholds=None,
    evaluation_region="forecast_exceedance_area",
):
    """Compute BS/BSS for all sources and categorical thresholds.

    evaluation_region:
      * forecast_exceedance_area:
          For each source and threshold, score only grid points where that forecast source has
          p >= threshold. Forecast values remain continuous probabilities.
      * full_domain:
          Score the whole domain using continuous probabilities. For raw proxy truth this will
          usually be identical across thresholds because the truth does not have 5/15/40/70 levels.
      * deterministic_full_domain:
          Convert forecast to yes/no p >= threshold, then score the whole domain.
    """
    if thresholds is None:
        thresholds = CATEGORICAL_THRESHOLDS

    region_mode = str(evaluation_region).lower().strip()
    if region_mode not in {"forecast_exceedance_area", "full_domain", "deterministic_full_domain"}:
        raise RuntimeError("evaluation_region must be 'forecast_exceedance_area', 'full_domain', or 'deterministic_full_domain'.")

    case_rows = []
    pooled_rows = []

    for source_label, fcst_col, sub, source_radius in _iter_eval_sources(
        df,
        radii=radii,
        include_members=include_members,
        include_ensembles=include_ensembles,
        include_wpc=include_wpc,
    ):
        pvals_all = pd.to_numeric(sub[fcst_col], errors="coerce").to_numpy(float)
        valid_fcst = np.isfinite(pvals_all)
        if not valid_fcst.any():
            print(f"Skipping Brier for {source_label}: no finite forecast values in {fcst_col}.")
            continue

        date_series_all = (
            sub["Date"].astype(str).str[:8].to_numpy()
            if "Date" in sub.columns else np.array(["ALL"] * len(sub), dtype=object)
        )

        for thr, thr_label in thresholds:
            try:
                truth_col, truth_kind, truth_yes_all = _truth_binary_for_brier_threshold(
                    sub,
                    threshold=float(thr),
                    truth_mode=truth_mode,
                    pp_definition=pp_definition,
                    proxy_col=proxy_col,
                    proxy_truth_threshold=proxy_truth_threshold,
                    proxy_expand_radius_km=proxy_expand_radius_km,
                )
            except Exception as exc:
                print(f"Skipping Brier for {source_label} at {thr_label}: {exc}")
                continue

            if region_mode == "forecast_exceedance_area":
                region = valid_fcst & (pvals_all >= float(thr))
            else:
                region = valid_fcst

            good = region & np.isfinite(truth_yes_all.astype(float))
            if not np.any(good):
                print(f"Skipping Brier for {source_label} at {thr_label}: no grid points in evaluation region.")
                continue

            p_cont = np.clip(pvals_all[good], 0.0, 1.0).astype(float)
            y = truth_yes_all[good].astype(float)
            dates = date_series_all[good]

            if region_mode == "deterministic_full_domain":
                p_score = (p_cont >= float(thr)).astype(float)
            else:
                p_score = p_cont

            clim = float(np.mean(y))
            bs = float(np.mean((p_score - y) ** 2))
            bs_ref = float(np.mean((clim - y) ** 2))
            bss = float(1.0 - bs / bs_ref) if bs_ref > 0 else np.nan

            pooled_rows.append({
                "Source": source_label,
                "Forecast Column": fcst_col,
                "Source Radius": source_radius,
                "Threshold": float(thr),
                "Threshold Label": thr_label,
                "Truth Mode": truth_mode,
                "Truth Column": truth_col,
                "Truth Kind": truth_kind,
                "Evaluation Region": region_mode,
                "Proxy Expansion Radius km": float(proxy_expand_radius_km) if str(truth_mode).lower() == "proxy" else np.nan,
                "N": int(len(p_score)),
                "Event Frequency": clim,
                "Forecast Area Fraction": float(np.mean(region & np.isfinite(truth_yes_all.astype(float)))),
                "Brier Score": bs,
                "Reference BS": bs_ref,
                "BSS": bss,
            })

            for d in sorted(pd.Series(dates).dropna().unique().tolist()):
                m = (dates == d)
                if not np.any(m):
                    continue
                yy = y[m]
                pp = p_score[m]
                bs_d = float(np.mean((pp - yy) ** 2))
                bsref_d = float(np.mean((clim - yy) ** 2))
                bss_d = float(1.0 - bs_d / bsref_d) if bsref_d > 0 else np.nan
                case_rows.append({
                    "Date": str(d),
                    "Source": source_label,
                    "Forecast Column": fcst_col,
                    "Source Radius": source_radius,
                    "Threshold": float(thr),
                    "Threshold Label": thr_label,
                    "Truth Mode": truth_mode,
                    "Truth Column": truth_col,
                    "Truth Kind": truth_kind,
                    "Evaluation Region": region_mode,
                    "Proxy Expansion Radius km": float(proxy_expand_radius_km) if str(truth_mode).lower() == "proxy" else np.nan,
                    "N": int(np.sum(m)),
                    "Event Frequency": clim,
                    "Brier Score": bs_d,
                    "Reference BS": bsref_d,
                    "BSS": bss_d,
                })

    case_table = pd.DataFrame(case_rows)
    pooled_table = pd.DataFrame(pooled_rows)
    if case_table.empty:
        return case_table, case_table.copy(), pooled_table

    group_cols = [
        "Source", "Forecast Column", "Source Radius", "Threshold", "Threshold Label",
        "Truth Mode", "Truth Column", "Truth Kind", "Evaluation Region", "Proxy Expansion Radius km"
    ]
    summary = (
        case_table.groupby(group_cols, as_index=False)[["Brier Score", "Reference BS", "BSS", "N", "Event Frequency"]]
        .mean(numeric_only=True)
    )
    if "Source" in summary.columns:
        summary = summary.sort_values(["Threshold", "Source"], key=lambda s: s.map(_source_sort_key) if s.name == "Source" else s)
    return case_table, summary, pooled_table


def plot_brier_threshold_lines(summary_df, metric="Brier Score", title=None):
    if summary_df is None or summary_df.empty:
        print(f"No {metric} data to plot.")
        return None
    order = [lab for _, lab in CATEGORICAL_THRESHOLDS]
    data = summary_df.copy()
    data = data[data["Threshold Label"].isin(order)].copy()
    if data.empty:
        print(f"No rows with recognized threshold labels for {metric}.")
        return None

    data["SourceSort"] = data["Source"].map(_source_sort_key)
    data = data.sort_values(["SourceSort", "Threshold"])
    fig, ax = plt.subplots(figsize=(11.5, 6.2))
    for source in data["Source"].drop_duplicates().tolist():
        sub = data[data["Source"] == source].copy()
        sub["Threshold Label"] = pd.Categorical(sub["Threshold Label"], categories=order, ordered=True)
        sub = sub.sort_values("Threshold Label")
        ax.plot(
            sub["Threshold Label"].astype(str),
            pd.to_numeric(sub[metric], errors="coerce"),
            marker="o",
            linewidth=2,
            label=str(source),
        )
    ax.set_xlabel("Categorical risk threshold")
    ax.set_ylabel(metric)
    ax.set_title(title or f"{metric}: all available radius tests")
    ax.grid(True, axis="y", alpha=0.3)
    ax.legend(loc="best", fontsize=9)
    plt.tight_layout()
    plt.show()
    plt.close(fig)
    return None


BRIER_DF, BRIER_DF_LABEL = choose_evaluation_dataframe(prefer_realtime=BRIER_PREFER_REALTIME)
print(f"Using {BRIER_DF_LABEL} for Brier/BSS by categorical risk area")
print_available_forecasts_for_eval(BRIER_DF, BRIER_DF_LABEL)
if str(BRIER_TRUTH_MODE).lower() == "proxy":
    print(
        f"Proxy-truth Brier verification will expand raw truth {BRIER_PROXY_COL!r} "
        f"to {float(BRIER_PROXY_EXPANSION_RADIUS_KM):.1f} km before scoring."
    )
print(f"Brier evaluation region mode: {BRIER_EVALUATION_REGION!r}")

bs_thr_case, bs_thr_case_mean, bs_thr_pooled = compute_brier_by_category_area_all_sources(
    BRIER_DF,
    radii=BRIER_RADII_KM,
    truth_mode=BRIER_TRUTH_MODE,
    pp_definition=BRIER_PP_DEFINITION,
    proxy_col=BRIER_PROXY_COL,
    proxy_truth_threshold=BRIER_PROXY_THRESHOLD,
    proxy_expand_radius_km=BRIER_PROXY_EXPANSION_RADIUS_KM,
    include_members=True,
    include_ensembles=BRIER_INCLUDE_ENSEMBLES,
    include_wpc=True,
    thresholds=CATEGORICAL_THRESHOLDS,
    evaluation_region=BRIER_EVALUATION_REGION,
)

print("Case-mean Brier/BSS by categorical risk area")
display(bs_thr_case_mean.round(4))
print("Pooled Brier/BSS by categorical risk area")
display(bs_thr_pooled.round(4))

BRIER_PLOT_DF = bs_thr_case_mean if str(BRIER_PLOT_SOURCE).lower() == "case_mean" else bs_thr_pooled
plot_brier_threshold_lines(
    BRIER_PLOT_DF,
    metric="Brier Score",
    title=(
        f"Brier Score by categorical risk area | truth={BRIER_TRUTH_MODE}:"
        f"{BRIER_PP_DEFINITION if str(BRIER_TRUTH_MODE).lower() == 'pp' else BRIER_PROXY_COL} | {BRIER_DF_LABEL}"
    ),
)
plot_brier_threshold_lines(
    BRIER_PLOT_DF,
    metric="BSS",
    title=(
        f"Brier Skill Score by categorical risk area | truth={BRIER_TRUTH_MODE}:"
        f"{BRIER_PP_DEFINITION if str(BRIER_TRUTH_MODE).lower() == 'pp' else BRIER_PROXY_COL} | {BRIER_DF_LABEL}"
    ),
)



## 12. Paper verification violin plots: ETS/POD/FAR and Brier Score

This section produces paper-style violin plots and saves every figure/table to disk.

Important category convention:

1. **Practically Perfect truth** uses **exclusive category matching**: `5–15%`, `15–40%`, `40–70%`, `≥70%`. This answers: if PP says a grid point is in the 15–40% category, did ML/WPC place it in the same 15–40% category?
2. **Raw flood-proxy truth** and **MRMS > FFG truth** use **cumulative forecast exceedance thresholds**: `>5%`, `>15%`, `>40%`, `>70%`. The raw proxy/MRMS event mask is expanded by 40 km before scoring.
3. Violin colors match the corresponding risk-category colors, and optional dashed/dotted horizontal WPC baseline lines show WPC mean/median for each threshold/category.



In [ ]:
# ======================================================================================
# Paper-style violin verification plots: ETS/POD/FAR and BS/BSS
# v18: PP uses exclusive categories; proxy/MRMS use cumulative >5/>15/>40/>70 thresholds.
# ======================================================================================

import os
import re
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.spatial import cKDTree

# -----------------------------
# User controls
# -----------------------------
PAPER_PREFER_REALTIME = False
PAPER_MODEL_LABELS = [spec["label"] for spec in MODEL_SPECS]
PAPER_RADII_KM = list(dict.fromkeys(spec["radius_km"] for spec in MODEL_SPECS))

# Truth options. UFVS proxies are raw point/grid proxies and are expanded before scoring.
PAPER_PROXY_OPTIONS = ["UFVS_ANY", "STAGE4_FFG", "STAGE4_ARI", "USGS", "FLASH_LSR", "FLOOD_LSR"]
PAPER_MRMS_TRUTH_OPTION = "MRMS_FFG_POINT"  # pointwise MRMS>FFG, then expanded to 40 km
PAPER_PROXY_EXPANSION_RADIUS_KM = 40.0
PAPER_PROXY_THRESHOLD = 0.0
PAPER_PP_DEFINITION = "Any flood proxy"

# Sources to include.
PAPER_INCLUDE_WPC = True
PAPER_INCLUDE_ENSEMBLE_MEAN = True
PAPER_INCLUDE_ENSEMBLE_MAX = True

# Brier-score region for raw proxy/MRMS truth:
#   "forecast_exceedance_area" = score continuous p only inside each source's p >= threshold area.
#   "full_domain"              = score continuous p over full domain; for binary proxy truth this is flat across thresholds.
#   "deterministic_full_domain" = score yes/no forecast over full domain.
PAPER_BS_REGION_MODE = "forecast_exceedance_area"
PAPER_BS_REFERENCE = "case_climatology"  # "case_climatology" or "pooled_climatology"

# Output and plot appearance controls.
PAPER_PLOT_DIR = os.path.join(PROJECT_DIR, "paper_verification_violin_plots_v33day2valid")
PAPER_SAVE_TABLES = True
PAPER_SHOW_PLOTS = True
PAPER_DPI = 220
PAPER_FIGSIZE_METRIC = (18, 7.8)
PAPER_VIOLIN_WIDTH = 0.13
PAPER_FONT = {
    "title": 16,
    "axis": 14,
    "tick": 11,
    "legend": 9,
    "annotation": 10,
}
PAPER_YLIMS = {
    "ETS": None,
    "POD": (0, 1),
    "FAR": (0, 1),
    "Brier Score": None,
    "BSS": None,
}

# WPC baseline horizontal lines on violin plots.
PAPER_SHOW_WPC_BASELINES = True
PAPER_WPC_BASELINE_STATS = ("mean", "median")  # any subset of ("mean", "median")
PAPER_WPC_MEAN_LINESTYLE = "--"
PAPER_WPC_MEDIAN_LINESTYLE = ":"
PAPER_WPC_BASELINE_ALPHA = 0.85
PAPER_WPC_BASELINE_LINEWIDTH = 1.6

# -----------------------------
# Category definitions and colors
# -----------------------------
PAPER_PP_EXCLUSIVE_CATEGORIES = [
    (0.05, 0.15, "5–15%"),
    (0.15, 0.40, "15–40%"),
    (0.40, 0.70, "40–70%"),
    (0.70, 1.01, "≥70%"),
]
PAPER_CUMULATIVE_THRESHOLDS = [
    (0.05, None, ">5%"),
    (0.15, None, ">15%"),
    (0.40, None, ">40%"),
    (0.70, None, ">70%"),
]

# Match the existing map risk colors where possible.
PAPER_BIN_COLORS = {
    "5–15%": "#3EE700",
    "15–40%": "#FFFB00",
    "40–70%": "#E74C3C",
    "≥70%": "#FF00FF",
    ">5%": "#3EE700",
    ">15%": "#FFFB00",
    ">40%": "#E74C3C",
    ">70%": "#FF00FF",
}

PAPER_ETS_METRICS = ["ETS", "POD", "FAR"]
PAPER_BS_METRICS = ["Brier Score", "BSS"]


def _paper_safe_name(s):
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", str(s)).strip("_")


def _paper_source_sort(label):
    s = str(label)
    m = re.search(r"ML r(\d+)", s)
    if m:
        return (0, int(m.group(1)), s)
    order = {"ML ens mean": 100, "ML ens max": 101, "WPC ERO": 200}
    return (1, order.get(s, 999), s)


def _paper_wide_from_long_radius_df(df, radii=None):
    """Convert historical long radius dataframe to one row per Date/Lat/Lon with ML_rXX columns."""
    if any(_wide_ml_member_metadata(c)[0] is not None for c in df.columns):
        out = df.copy()
    elif {"Date", "Lat", "Lon", "ML_Target_Radius_km", "ML_Forecast_Prob"}.issubset(df.columns):
        req = None if radii is None else [int(round(float(r))) for r in radii]
        work = df.copy()
        work["__RadiusInt"] = pd.to_numeric(work["ML_Target_Radius_km"], errors="coerce").round().astype("Int64")
        work["__ModelLabel"] = work["ML_Model_Label"].astype(str) if "ML_Model_Label" in work.columns else work["__RadiusInt"].map(lambda r: model_label_for_radius(r))
        if req is not None:
            work = work[work["__RadiusInt"].isin(req)].copy()
        keys = ["Date", "Lat", "Lon"]
        base_cols = [c for c in work.columns if c not in {"ML_Forecast_Prob", "ML_Target_Radius_km", "ML_Model_Label", "__RadiusInt", "__ModelLabel"}]
        base = work[base_cols].groupby(keys, as_index=False, sort=False).first()
        ml = (
            work.pivot_table(index=keys, columns="__ModelLabel", values="ML_Forecast_Prob", aggfunc="mean")
            .reset_index()
        )
        label_to_radius = {str(spec["label"]): int(spec["radius_km"]) for spec in MODEL_SPECS}
        ml.columns = [_model_prob_col(str(c), label_to_radius[str(c)]) if str(c) in label_to_radius else c for c in ml.columns]
        out = base.merge(ml, on=keys, how="left")
    else:
        out = df.copy()

    member_cols = [c for c in out.columns if _wide_ml_member_metadata(c)[0] is not None]
    if member_cols:
        arr = out[member_cols].apply(pd.to_numeric, errors="coerce")
        if PAPER_INCLUDE_ENSEMBLE_MEAN:
            out["ML_Radius_EnsMean"] = arr.mean(axis=1, skipna=True)
        if PAPER_INCLUDE_ENSEMBLE_MAX:
            out["ML_Radius_EnsMax"] = arr.max(axis=1, skipna=True)
    return out


def _paper_forecast_member_cols(df, radii=None):
    out = {}
    req = None if radii is None else [int(round(float(r))) for r in radii]
    for c in df.columns:
        model_label, r = _wide_ml_member_metadata(c)
        if model_label is None:
            continue
        if req is None or r in req:
            out[f"ML {model_label}"] = c
    return dict(sorted(out.items(), key=lambda kv: _paper_source_sort(kv[0])))


def _paper_source_columns(df, radii=None):
    cols = _paper_forecast_member_cols(df, radii=radii)
    if PAPER_INCLUDE_ENSEMBLE_MEAN and "ML_Radius_EnsMean" in df.columns:
        cols["ML ens mean"] = "ML_Radius_EnsMean"
    if PAPER_INCLUDE_ENSEMBLE_MAX and "ML_Radius_EnsMax" in df.columns:
        cols["ML ens max"] = "ML_Radius_EnsMax"
    if PAPER_INCLUDE_WPC and "WPC_ERO_Risk" in df.columns:
        cols["WPC ERO"] = "WPC_ERO_Risk"
    return dict(sorted(cols.items(), key=lambda kv: _paper_source_sort(kv[0])))


def _paper_exclusive_category_yes(values, category):
    lo, hi, label = category
    v = np.asarray(values, dtype=float)
    if hi is None or float(hi) >= 1.0:
        return np.isfinite(v) & (v >= float(lo)) & (v <= 1.0)
    return np.isfinite(v) & (v >= float(lo)) & (v < float(hi))


def _paper_cumulative_yes(values, threshold):
    lo, _, label = threshold
    v = np.asarray(values, dtype=float)
    return np.isfinite(v) & (v >= float(lo))


def _paper_contingency_stats(f_yes, t_yes):
    f = np.asarray(f_yes, dtype=bool)
    t = np.asarray(t_yes, dtype=bool)
    h = int(np.sum(f & t))
    m = int(np.sum((~f) & t))
    fa = int(np.sum(f & (~t)))
    cn = int(np.sum((~f) & (~t)))
    n = h + m + fa + cn
    pod = h / (h + m) if (h + m) else np.nan
    far = fa / (h + fa) if (h + fa) else np.nan
    csi = h / (h + m + fa) if (h + m + fa) else np.nan
    rand_hits = ((h + fa) * (h + m) / n) if n else np.nan
    ets_denom = h + m + fa - rand_hits if np.isfinite(rand_hits) else np.nan
    ets = (h - rand_hits) / ets_denom if np.isfinite(ets_denom) and ets_denom != 0 else np.nan
    bias = (h + fa) / (h + m) if (h + m) else np.nan
    return {
        "N": n,
        "Hits": h,
        "Misses": m,
        "FalseAlarms": fa,
        "CorrectNegatives": cn,
        "RandomHits": rand_hits,
        "ETS": ets,
        "CSI": csi,
        "POD": pod,
        "FAR": far,
        "BIAS": bias,
    }


_PAPER_EXPANDED_TRUTH_CACHE = {}


def _paper_expand_truth_by_date(df, raw_mask, radius_km=40.0):
    raw = np.asarray(raw_mask, dtype=bool)
    if radius_km is None or float(radius_km) <= 0:
        return raw.copy()
    if not {"Date", "Lat", "Lon"}.issubset(df.columns):
        raise RuntimeError("Cannot expand truth mask; dataframe lacks Date/Lat/Lon.")
    out = np.zeros(len(df), dtype=bool)
    dates = df["Date"].astype(str).str[:8]
    for _, idx in dates.groupby(dates, sort=False).groups.items():
        idx = np.asarray(list(idx), dtype=int)
        if len(idx) == 0:
            continue
        submask = raw[idx]
        if not submask.any():
            continue
        lat = pd.to_numeric(df.iloc[idx]["Lat"], errors="coerce").to_numpy(float)
        lon = pd.to_numeric(df.iloc[idx]["Lon"], errors="coerce").to_numpy(float)
        good = np.isfinite(lat) & np.isfinite(lon)
        if not good.any():
            continue
        idx_good = idx[good]
        xyz = latlon_to_unit_xyz(lat[good], lon[good])
        tree = cKDTree(xyz)
        expanded_good = expand_binary_mask_radius_km(submask[good], tree=tree, xyz=xyz, radius_km=float(radius_km))
        out[idx_good] = expanded_good
    return out


def _paper_pp_truth_mask(df, category, pp_definition="Any flood proxy"):
    pp_col = _resolve_pp_truth_column_for_eval(df, pp_definition)
    vals = pd.to_numeric(df[pp_col], errors="coerce").to_numpy(float)
    truth = _paper_exclusive_category_yes(vals, category)
    return pp_col, "pp_exclusive", truth


def _paper_expanded_proxy_truth_mask(df, proxy_option="UFVS_ANY", expand_radius_km=40.0):
    canonical = _canonical_proxy_column(proxy_option)
    cache_key = (id(df), canonical, float(expand_radius_km), float(PAPER_PROXY_THRESHOLD))
    if cache_key in _PAPER_EXPANDED_TRUTH_CACHE:
        truth_col, truth_kind, expanded = _PAPER_EXPANDED_TRUTH_CACHE[cache_key]
        return truth_col, truth_kind, expanded.copy()

    truth_col, truth_kind = _resolve_proxy_column(df, proxy_col=proxy_option, allow_pp_fallback=False)
    raw_vals = pd.to_numeric(df[truth_col], errors="coerce").to_numpy(float)
    raw = np.isfinite(raw_vals) & (raw_vals > float(PAPER_PROXY_THRESHOLD))
    do_expand = truth_kind in {"raw_ufvs", "selected_raw", "mrms_point"}
    expanded = _paper_expand_truth_by_date(df, raw, radius_km=expand_radius_km) if do_expand else raw
    _PAPER_EXPANDED_TRUTH_CACHE[cache_key] = (truth_col, truth_kind, expanded.astype(bool))
    return truth_col, truth_kind, expanded.astype(bool)


def compute_paper_pp_ets_rows(df, pp_definition="Any flood proxy", sources=None):
    """PP verification: exclusive category-to-category matching only."""
    if sources is None:
        sources = _paper_source_columns(df, radii=PAPER_RADII_KM)
    rows = []
    for category in PAPER_PP_EXCLUSIVE_CATEGORIES:
        lo, hi, cat_label = category
        truth_col, truth_kind, truth_all = _paper_pp_truth_mask(df, category, pp_definition=pp_definition)
        for source, fcst_col in sources.items():
            p_all = pd.to_numeric(df[fcst_col], errors="coerce").to_numpy(float)
            f_all = _paper_exclusive_category_yes(p_all, category)
            dates = df["Date"].astype(str).str[:8] if "Date" in df.columns else pd.Series("ALL", index=df.index)
            for date_value, idx in dates.groupby(dates, sort=False).groups.items():
                idx = np.asarray(list(idx), dtype=int)
                stats = _paper_contingency_stats(f_all[idx], truth_all[idx])
                rows.append({
                    "Date": str(date_value),
                    "Source": source,
                    "Forecast Column": fcst_col,
                    "Category": cat_label,
                    "Category Low": lo,
                    "Category High": hi,
                    "Category Type": "PP exclusive category",
                    "Truth Mode": "pp",
                    "Truth Option": pp_definition,
                    "Truth Column": truth_col,
                    "Truth Kind": truth_kind,
                    "Proxy Expansion Radius km": np.nan,
                    **stats,
                })
    return pd.DataFrame(rows)


def compute_paper_proxy_ets_rows(df, proxy_option="UFVS_ANY", sources=None):
    """Proxy/MRMS verification: cumulative forecast exceedance against 40-km-expanded raw truth."""
    if sources is None:
        sources = _paper_source_columns(df, radii=PAPER_RADII_KM)
    rows = []
    truth_col, truth_kind, truth_all = _paper_expanded_proxy_truth_mask(
        df, proxy_option=proxy_option, expand_radius_km=PAPER_PROXY_EXPANSION_RADIUS_KM
    )
    for threshold in PAPER_CUMULATIVE_THRESHOLDS:
        lo, hi, cat_label = threshold
        for source, fcst_col in sources.items():
            p_all = pd.to_numeric(df[fcst_col], errors="coerce").to_numpy(float)
            f_all = _paper_cumulative_yes(p_all, threshold)
            dates = df["Date"].astype(str).str[:8] if "Date" in df.columns else pd.Series("ALL", index=df.index)
            for date_value, idx in dates.groupby(dates, sort=False).groups.items():
                idx = np.asarray(list(idx), dtype=int)
                stats = _paper_contingency_stats(f_all[idx], truth_all[idx])
                rows.append({
                    "Date": str(date_value),
                    "Source": source,
                    "Forecast Column": fcst_col,
                    "Category": cat_label,
                    "Category Low": lo,
                    "Category High": hi,
                    "Category Type": "cumulative exceedance",
                    "Truth Mode": "proxy",
                    "Truth Option": proxy_option,
                    "Truth Column": truth_col,
                    "Truth Kind": truth_kind,
                    "Proxy Expansion Radius km": PAPER_PROXY_EXPANSION_RADIUS_KM,
                    **stats,
                })
    return pd.DataFrame(rows)


def compute_paper_bs_rows(df, proxy_option="UFVS_ANY", sources=None):
    """BS/BSS using cumulative thresholds against 40-km-expanded proxy/MRMS truth."""
    if sources is None:
        sources = _paper_source_columns(df, radii=PAPER_RADII_KM)
    rows = []
    truth_col, truth_kind, truth_all = _paper_expanded_proxy_truth_mask(
        df, proxy_option=proxy_option, expand_radius_km=PAPER_PROXY_EXPANSION_RADIUS_KM
    )
    for threshold in PAPER_CUMULATIVE_THRESHOLDS:
        lo, hi, cat_label = threshold
        for source, fcst_col in sources.items():
            p_all = pd.to_numeric(df[fcst_col], errors="coerce").to_numpy(float)
            valid = np.isfinite(p_all)
            fyes = _paper_cumulative_yes(p_all, threshold)
            if PAPER_BS_REGION_MODE == "forecast_exceedance_area":
                region = valid & fyes
                p_score_all = np.clip(p_all, 0.0, 1.0)
            elif PAPER_BS_REGION_MODE == "full_domain":
                region = valid
                p_score_all = np.clip(p_all, 0.0, 1.0)
            elif PAPER_BS_REGION_MODE == "deterministic_full_domain":
                region = valid
                p_score_all = fyes.astype(float)
            else:
                raise RuntimeError("PAPER_BS_REGION_MODE must be forecast_exceedance_area, full_domain, or deterministic_full_domain.")

            dates = df["Date"].astype(str).str[:8] if "Date" in df.columns else pd.Series("ALL", index=df.index)
            pooled_good = region & np.isfinite(truth_all.astype(float))
            pooled_clim = float(np.mean(truth_all[pooled_good])) if pooled_good.any() else np.nan
            for date_value, idx in dates.groupby(dates, sort=False).groups.items():
                idx = np.asarray(list(idx), dtype=int)
                good = region[idx] & np.isfinite(truth_all[idx].astype(float))
                if not np.any(good):
                    continue
                yy = truth_all[idx][good].astype(float)
                pp = p_score_all[idx][good].astype(float)
                clim = float(np.mean(yy)) if PAPER_BS_REFERENCE == "case_climatology" else pooled_clim
                bs = float(np.mean((pp - yy) ** 2))
                bs_ref = float(np.mean((clim - yy) ** 2)) if np.isfinite(clim) else np.nan
                bss = float(1.0 - bs / bs_ref) if np.isfinite(bs_ref) and bs_ref > 0 else np.nan
                rows.append({
                    "Date": str(date_value),
                    "Source": source,
                    "Forecast Column": fcst_col,
                    "Category": cat_label,
                    "Category Low": lo,
                    "Category High": hi,
                    "Category Type": "cumulative exceedance",
                    "Truth Option": proxy_option,
                    "Truth Column": truth_col,
                    "Truth Kind": truth_kind,
                    "Proxy Expansion Radius km": PAPER_PROXY_EXPANSION_RADIUS_KM,
                    "BS Region Mode": PAPER_BS_REGION_MODE,
                    "N": int(np.sum(good)),
                    "Event Frequency": float(np.mean(yy)),
                    "Brier Score": bs,
                    "Reference BS": bs_ref,
                    "BSS": bss,
                })
    return pd.DataFrame(rows)


def _paper_save_table(df, name):
    outdir = Path(PAPER_PLOT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    path = outdir / f"{_paper_safe_name(name)}.csv"
    df.to_csv(path, index=False)
    print("Saved table:", path)
    return path


def _paper_wpc_baseline_handles(ax, data, metric, categories, source_col="Source", category_col="Category"):
    """Add WPC mean/median baseline lines by category and return legend handles."""
    handles = []
    if not PAPER_SHOW_WPC_BASELINES or "WPC ERO" not in set(data[source_col].astype(str)):
        return handles
    wpc = data[data[source_col].astype(str) == "WPC ERO"].copy()
    if wpc.empty:
        return handles
    x0, x1 = ax.get_xlim()
    for cat in categories:
        vals = pd.to_numeric(wpc[wpc[category_col].astype(str) == str(cat)][metric], errors="coerce").dropna().to_numpy(float)
        if vals.size == 0:
            continue
        color = PAPER_BIN_COLORS.get(str(cat), "0.35")
        if "mean" in PAPER_WPC_BASELINE_STATS:
            mean = float(np.nanmean(vals))
            ax.axhline(mean, color=color, linestyle=PAPER_WPC_MEAN_LINESTYLE,
                       linewidth=PAPER_WPC_BASELINE_LINEWIDTH, alpha=PAPER_WPC_BASELINE_ALPHA, zorder=1)
            handles.append(plt.Line2D([0], [0], color=color, linestyle=PAPER_WPC_MEAN_LINESTYLE,
                                      linewidth=PAPER_WPC_BASELINE_LINEWIDTH,
                                      label=f"{cat} WPC mean"))
        if "median" in PAPER_WPC_BASELINE_STATS:
            median = float(np.nanmedian(vals))
            ax.axhline(median, color=color, linestyle=PAPER_WPC_MEDIAN_LINESTYLE,
                       linewidth=PAPER_WPC_BASELINE_LINEWIDTH, alpha=PAPER_WPC_BASELINE_ALPHA, zorder=1)
            handles.append(plt.Line2D([0], [0], color=color, linestyle=PAPER_WPC_MEDIAN_LINESTYLE,
                                      linewidth=PAPER_WPC_BASELINE_LINEWIDTH,
                                      label=f"{cat} WPC median"))
    return handles


def plot_grouped_violin(
    rows,
    metric,
    title,
    filename,
    category_order,
    source_col="Source",
    category_col="Category",
    figsize=None,
    ylim=None,
):
    if rows is None or len(rows) == 0:
        print(f"No rows to plot for {title}")
        return None
    data = rows.copy()
    data[metric] = pd.to_numeric(data[metric], errors="coerce")
    data = data[np.isfinite(data[metric])].copy()
    if data.empty:
        print(f"No finite {metric} values for {title}")
        return None

    categories = [c for c in category_order if c in set(data[category_col].astype(str))]
    if not categories:
        categories = sorted(data[category_col].astype(str).unique().tolist())
    sources = sorted(data[source_col].astype(str).unique().tolist(), key=_paper_source_sort)

    ncat = len(categories)
    base_x = np.arange(len(sources), dtype=float)
    offsets = np.array([0.0]) if ncat == 1 else np.linspace(-0.30, 0.30, ncat)
    width = float(PAPER_VIOLIN_WIDTH)

    fig, ax = plt.subplots(figsize=figsize or PAPER_FIGSIZE_METRIC)
    legend_handles = []

    for j, cat in enumerate(categories):
        vals_by_source, pos = [], []
        for i, src in enumerate(sources):
            vals = data[(data[source_col].astype(str) == src) & (data[category_col].astype(str) == cat)][metric].dropna().to_numpy(float)
            if vals.size == 0:
                continue
            vals_by_source.append(vals)
            pos.append(base_x[i] + offsets[j])
        if not vals_by_source:
            continue
        color = PAPER_BIN_COLORS.get(str(cat), plt.get_cmap("tab10")(j % 10))
        vp = ax.violinplot(vals_by_source, positions=pos, widths=width, showmeans=False, showmedians=False, showextrema=False)
        for body in vp["bodies"]:
            body.set_facecolor(color)
            body.set_edgecolor("black")
            body.set_alpha(0.50)
            body.set_linewidth(0.8)
        for vals, x in zip(vals_by_source, pos):
            mean = float(np.nanmean(vals))
            median = float(np.nanmedian(vals))
            ax.scatter([x], [mean], marker="s", s=28, color="black", zorder=4)
            ax.hlines(median, x - width * 0.38, x + width * 0.38, color="black", linewidth=2.0, zorder=5)
        legend_handles.append(plt.Line2D([0], [0], color=color, lw=8, alpha=0.50, label=str(cat)))

    ax.set_xticks(base_x)
    ax.set_xticklabels(sources, rotation=35, ha="right", fontsize=PAPER_FONT["tick"])
    ax.tick_params(axis="y", labelsize=PAPER_FONT["tick"])
    ax.set_ylabel(metric, fontsize=PAPER_FONT["axis"])
    ax.set_xlabel("Forecast source", fontsize=PAPER_FONT["axis"])
    ax.set_title(title, fontsize=PAPER_FONT["title"])
    ax.grid(True, axis="y", alpha=0.3)
    if ylim is not None:
        ax.set_ylim(*ylim)

    # Add WPC baseline lines after setting axis but before legend.
    wpc_handles = _paper_wpc_baseline_handles(ax, data, metric, categories, source_col=source_col, category_col=category_col)

    legend_handles.append(plt.Line2D([0], [0], marker="s", color="black", linestyle="None", label="Violin mean", markersize=6))
    legend_handles.append(plt.Line2D([0], [0], color="black", lw=2, label="Violin median"))
    legend_handles.extend(wpc_handles)
    ax.legend(handles=legend_handles, fontsize=PAPER_FONT["legend"], loc="best", ncol=2)
    plt.tight_layout()

    outdir = Path(PAPER_PLOT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    outpath = outdir / filename
    fig.savefig(outpath, dpi=int(PAPER_DPI), bbox_inches="tight")
    print("Saved:", outpath)
    if PAPER_SHOW_PLOTS:
        plt.show()
    plt.close(fig)
    return outpath


def _plot_metric_set(rows, metrics, title_prefix, file_prefix, category_order):
    for metric in metrics:
        plot_grouped_violin(
            rows,
            metric=metric,
            title=f"{title_prefix}: {metric}",
            filename=f"{_paper_safe_name(file_prefix)}_{_paper_safe_name(metric)}.png",
            category_order=category_order,
            ylim=PAPER_YLIMS.get(metric),
        )


def run_paper_verification_plot_suite():
    eval_df, eval_label = choose_evaluation_dataframe(prefer_realtime=PAPER_PREFER_REALTIME)
    print(f"Using {eval_label} for paper verification plots")
    print_available_forecasts_for_eval(eval_df, eval_label)
    wide = _paper_wide_from_long_radius_df(eval_df, radii=PAPER_RADII_KM)
    sources = _paper_source_columns(wide, radii=PAPER_RADII_KM)
    print("Sources included:", list(sources.keys()))
    Path(PAPER_PLOT_DIR).mkdir(parents=True, exist_ok=True)

    all_tables = {}

    # 1) PP truth: exclusive category matching.
    pp_ets = compute_paper_pp_ets_rows(wide, pp_definition=PAPER_PP_DEFINITION, sources=sources)
    all_tables["ets_pod_far_pp_exclusive"] = pp_ets
    if PAPER_SAVE_TABLES:
        _paper_save_table(pp_ets, "ets_pod_far_pp_exclusive")
    _plot_metric_set(
        pp_ets,
        PAPER_ETS_METRICS,
        title_prefix=f"PP truth exclusive category matching ({PAPER_PP_DEFINITION})",
        file_prefix="pp_exclusive_category_matching",
        category_order=[c[-1] for c in PAPER_PP_EXCLUSIVE_CATEGORIES],
    )

    # 2) Raw flood proxies: cumulative thresholds against expanded truth.
    proxy_ets_tables = []
    proxy_bs_tables = []
    for proxy in PAPER_PROXY_OPTIONS:
        print(f"\nProcessing expanded flood proxy: {proxy}")
        ets_rows = compute_paper_proxy_ets_rows(wide, proxy_option=proxy, sources=sources)
        proxy_ets_tables.append(ets_rows)
        if PAPER_SAVE_TABLES:
            _paper_save_table(ets_rows, f"ets_pod_far_expanded_proxy_{proxy}")
        _plot_metric_set(
            ets_rows,
            PAPER_ETS_METRICS,
            title_prefix=f"Expanded proxy truth {proxy}: cumulative thresholds",
            file_prefix=f"expanded_proxy_{proxy}_cumulative",
            category_order=[c[-1] for c in PAPER_CUMULATIVE_THRESHOLDS],
        )

        bs_rows = compute_paper_bs_rows(wide, proxy_option=proxy, sources=sources)
        proxy_bs_tables.append(bs_rows)
        if PAPER_SAVE_TABLES:
            _paper_save_table(bs_rows, f"brier_expanded_proxy_{proxy}")
        _plot_metric_set(
            bs_rows,
            PAPER_BS_METRICS,
            title_prefix=f"Expanded proxy truth {proxy}: Brier score cumulative thresholds",
            file_prefix=f"brier_expanded_proxy_{proxy}_cumulative",
            category_order=[c[-1] for c in PAPER_CUMULATIVE_THRESHOLDS],
        )

    if proxy_ets_tables:
        all_proxy_ets = pd.concat(proxy_ets_tables, ignore_index=True)
        all_tables["ets_pod_far_all_expanded_proxies"] = all_proxy_ets
        if PAPER_SAVE_TABLES:
            _paper_save_table(all_proxy_ets, "ets_pod_far_all_expanded_proxies")
    if proxy_bs_tables:
        all_proxy_bs = pd.concat(proxy_bs_tables, ignore_index=True)
        all_tables["brier_all_expanded_proxies"] = all_proxy_bs
        if PAPER_SAVE_TABLES:
            _paper_save_table(all_proxy_bs, "brier_all_expanded_proxies")

    # 3) MRMS > FFG truth: cumulative thresholds against expanded MRMS truth.
    print(f"\nProcessing MRMS truth: {PAPER_MRMS_TRUTH_OPTION}")
    mrms_ets = compute_paper_proxy_ets_rows(wide, proxy_option=PAPER_MRMS_TRUTH_OPTION, sources=sources)
    all_tables["ets_pod_far_mrms_ffg_expanded"] = mrms_ets
    if PAPER_SAVE_TABLES:
        _paper_save_table(mrms_ets, "ets_pod_far_mrms_ffg_expanded")
    _plot_metric_set(
        mrms_ets,
        PAPER_ETS_METRICS,
        title_prefix="Expanded MRMS > FFG truth: cumulative thresholds",
        file_prefix="mrms_ffg_expanded_cumulative",
        category_order=[c[-1] for c in PAPER_CUMULATIVE_THRESHOLDS],
    )

    mrms_bs = compute_paper_bs_rows(wide, proxy_option=PAPER_MRMS_TRUTH_OPTION, sources=sources)
    all_tables["brier_mrms_ffg_expanded"] = mrms_bs
    if PAPER_SAVE_TABLES:
        _paper_save_table(mrms_bs, "brier_mrms_ffg_expanded")
    _plot_metric_set(
        mrms_bs,
        PAPER_BS_METRICS,
        title_prefix="Expanded MRMS > FFG truth: Brier score cumulative thresholds",
        file_prefix="brier_mrms_ffg_expanded_cumulative",
        category_order=[c[-1] for c in PAPER_CUMULATIVE_THRESHOLDS],
    )

    print(f"\nDone. Plots/tables saved in: {PAPER_PLOT_DIR}")
    return all_tables


#paper_verification_tables = run_paper_verification_plot_suite()

# ======================================================================================
# SEABORN VIOLIN OVERRIDE
#
# Placement:
#   Put this AFTER the paper-verification helper functions are defined,
#   but BEFORE run_paper_verification_plot_suite() is called.
#
# If your current block defines functions and immediately runs:
#   paper_verification_tables = run_paper_verification_plot_suite()
#
# then either:
#   1. Comment out that run line, run definitions, run this cell, then run the suite.
#   2. Paste this block directly above that run line in the same cell.
# ======================================================================================

import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import seaborn as sns


# -----------------------------
# User controls
# -----------------------------
PAPER_SEABORN_STYLE = "whitegrid"
PAPER_VIOLIN_ALPHA = 0.72
PAPER_VIOLIN_LINEWIDTH = 0.75
PAPER_VIOLIN_CUT = 0
PAPER_VIOLIN_INNER = None
PAPER_VIOLIN_DENSITY_NORM = "width"

PAPER_WPC_BASELINE_ALPHA = 0.50
PAPER_WPC_BASELINE_LINEWIDTH = 1.7
PAPER_WPC_MEAN_LINESTYLE = "--"
PAPER_WPC_MEDIAN_LINESTYLE = ":"

PAPER_MEAN_MARKER_SIZE = 34
PAPER_MEDIAN_LINEWIDTH = 2.1
PAPER_LEGEND_NCOL = 2


def _pretty_truth_name_for_title(s):
    s = str(s)
    replacements = {
        "UFVS_ANY": "Any Flood Proxy",
        "STAGE4_FFG": "Stage IV > FFG",
        "STAGE4_ARI": "Stage IV ARI",
        "USGS": "USGS Streamflow",
        "FLASH_LSR": "Flash Flood LSR",
        "FLOOD_LSR": "Flood LSR",
        "MRMS_FFG_POINT": "MRMS > FFG",
        "MRMS_FFG": "MRMS > FFG",
        "PP truth": "Practically Perfect Truth",
        "proxy truth": "Flood-Proxy Truth",
        "Expanded proxy truth": "Expanded Flood-Proxy Truth",
        "UVFS": "UFVS",
        "UFVS": "UFVS",
    }
    for old, new in replacements.items():
        s = s.replace(old, new)
    s = s.replace("_", " ")
    s = re.sub(r"\s+", " ", s).strip()
    return s


def _pretty_source_name(s):
    s = str(s)
    s = re.sub(r"ML r(\d+)", r"ML r\1 km", s)
    s = s.replace("ML ens mean", "ML Ensemble Mean")
    s = s.replace("ML ens max", "ML Ensemble Max")
    s = s.replace("WPC ERO", "WPC ERO")
    return s


def _pretty_category_name(s):
    s = str(s)
    s = s.replace(">=", "≥")
    return s


def _category_color(cat):
    cat = str(cat)
    if "PAPER_BIN_COLORS" in globals() and cat in PAPER_BIN_COLORS:
        return PAPER_BIN_COLORS[cat]
    fallback = {
        "5–15%": "#3EE700",
        "15–40%": "#FFFB00",
        "40–70%": "#E74C3C",
        "≥70%": "#FF00FF",
        ">5%": "#3EE700",
        ">15%": "#FFFB00",
        ">40%": "#E74C3C",
        ">70%": "#FF00FF",
    }
    return fallback.get(cat, "0.55")


def _source_sort_for_plot(s):
    raw = str(s)
    if "_paper_source_sort" in globals():
        return _paper_source_sort(raw)
    if raw.startswith("ML r"):
        m = re.search(r"(\d+)", raw)
        return int(m.group(1)) if m else 999
    if "Ensemble Mean" in raw or "ens mean" in raw:
        return 500
    if "Ensemble Max" in raw or "ens max" in raw:
        return 501
    if "WPC" in raw:
        return 900
    return 800


def _apply_violin_alpha(ax, alpha=PAPER_VIOLIN_ALPHA):
    for coll in ax.collections:
        try:
            coll.set_alpha(alpha)
            coll.set_linewidth(PAPER_VIOLIN_LINEWIDTH)
        except Exception:
            pass


def _draw_mean_median_on_violins(ax, data, metric, source_order, category_order):
    n_hue = max(1, len(category_order))
    dodge_width = 0.8

    if n_hue == 1:
        offsets = [0.0]
    else:
        offsets = np.linspace(
            -dodge_width / 2 + dodge_width / (2 * n_hue),
            dodge_width / 2 - dodge_width / (2 * n_hue),
            n_hue,
        )

    half_bar = 0.030 if n_hue >= 4 else 0.050

    for i, src in enumerate(source_order):
        for j, cat in enumerate(category_order):
            vals = data[
                (data["__SourceLabel"] == src) &
                (data["__CategoryLabel"] == cat)
            ][metric].dropna().to_numpy(float)

            if vals.size == 0:
                continue

            x = i + offsets[j]
            mean_val = float(np.nanmean(vals))
            median_val = float(np.nanmedian(vals))

            ax.scatter(
                [x], [mean_val],
                marker="s",
                s=PAPER_MEAN_MARKER_SIZE,
                color="black",
                edgecolor="white",
                linewidth=0.45,
                zorder=8,
            )
            ax.hlines(
                median_val,
                x - half_bar,
                x + half_bar,
                color="black",
                linewidth=PAPER_MEDIAN_LINEWIDTH,
                zorder=9,
            )


def _draw_wpc_guides(ax, data, metric, category_order):
    handles = []

    if not globals().get("PAPER_SHOW_WPC_BASELINES", True):
        return handles

    wpc = data[data["Source"].astype(str).eq("WPC ERO")].copy()
    if wpc.empty:
        return handles

    stats_to_draw = globals().get("PAPER_WPC_BASELINE_STATS", ("mean", "median"))

    for cat in category_order:
        vals = pd.to_numeric(
            wpc[wpc["__CategoryLabel"] == cat][metric],
            errors="coerce",
        ).dropna().to_numpy(float)

        if vals.size == 0:
            continue

        color = _category_color(cat)

        if "mean" in stats_to_draw:
            y = float(np.nanmean(vals))
            ax.axhline(
                y,
                color=color,
                linestyle=PAPER_WPC_MEAN_LINESTYLE,
                linewidth=PAPER_WPC_BASELINE_LINEWIDTH,
                alpha=PAPER_WPC_BASELINE_ALPHA,
                zorder=1,
            )
            handles.append(
                plt.Line2D(
                    [0], [0],
                    color=color,
                    linestyle=PAPER_WPC_MEAN_LINESTYLE,
                    linewidth=PAPER_WPC_BASELINE_LINEWIDTH,
                    alpha=PAPER_WPC_BASELINE_ALPHA,
                    label=f"{cat} WPC mean",
                )
            )

        if "median" in stats_to_draw:
            y = float(np.nanmedian(vals))
            ax.axhline(
                y,
                color=color,
                linestyle=PAPER_WPC_MEDIAN_LINESTYLE,
                linewidth=PAPER_WPC_BASELINE_LINEWIDTH,
                alpha=PAPER_WPC_BASELINE_ALPHA,
                zorder=1,
            )
            handles.append(
                plt.Line2D(
                    [0], [0],
                    color=color,
                    linestyle=PAPER_WPC_MEDIAN_LINESTYLE,
                    linewidth=PAPER_WPC_BASELINE_LINEWIDTH,
                    alpha=PAPER_WPC_BASELINE_ALPHA,
                    label=f"{cat} WPC median",
                )
            )

    return handles


def plot_grouped_violin(
    rows,
    metric,
    title,
    filename,
    category_order,
    source_col="Source",
    category_col="Category",
    figsize=None,
    ylim=None,
):
    """
    Replacement for the original matplotlib violin function.
    Uses seaborn and writes the figure to PAPER_PLOT_DIR.
    """
    if rows is None or len(rows) == 0:
        print(f"No rows to plot for {title}")
        return None

    data = rows.copy()
    data[metric] = pd.to_numeric(data[metric], errors="coerce")
    data = data[np.isfinite(data[metric])].copy()

    if data.empty:
        print(f"No finite {metric} values for {title}")
        return None

    data["__SourceLabel"] = data[source_col].astype(str).map(_pretty_source_name)
    data["__CategoryLabel"] = data[category_col].astype(str).map(_pretty_category_name)

    category_order_pretty = [_pretty_category_name(c) for c in category_order]
    category_order_pretty = [c for c in category_order_pretty if c in set(data["__CategoryLabel"])]

    if not category_order_pretty:
        category_order_pretty = sorted(data["__CategoryLabel"].unique().tolist())

    source_order = sorted(
        data["__SourceLabel"].unique().tolist(),
        key=_source_sort_for_plot,
    )

    palette = {cat: _category_color(cat) for cat in category_order_pretty}

    sns.set_theme(style=PAPER_SEABORN_STYLE)

    fig, ax = plt.subplots(figsize=figsize or PAPER_FIGSIZE_METRIC)

    sns.violinplot(
        data=data,
        x="__SourceLabel",
        y=metric,
        hue="__CategoryLabel",
        order=source_order,
        hue_order=category_order_pretty,
        palette=palette,
        inner=PAPER_VIOLIN_INNER,
        cut=PAPER_VIOLIN_CUT,
        density_norm=PAPER_VIOLIN_DENSITY_NORM,
        linewidth=PAPER_VIOLIN_LINEWIDTH,
        dodge=True,
        ax=ax,
    )

    _apply_violin_alpha(ax)
    _draw_mean_median_on_violins(ax, data, metric, source_order, category_order_pretty)

    # Remove seaborn legend and rebuild a clean one.
    if ax.get_legend() is not None:
        ax.get_legend().remove()

    bin_handles = [
        plt.Line2D(
            [0], [0],
            color=_category_color(cat),
            linewidth=9,
            alpha=PAPER_VIOLIN_ALPHA,
            label=cat,
        )
        for cat in category_order_pretty
    ]

    stat_handles = [
        plt.Line2D([0], [0], marker="s", color="black", linestyle="None",
                   label="Mean", markersize=6),
        plt.Line2D([0], [0], color="black", linewidth=PAPER_MEDIAN_LINEWIDTH,
                   label="Median"),
    ]

    wpc_handles = _draw_wpc_guides(ax, data, metric, category_order_pretty)

    ax.legend(
        handles=bin_handles + stat_handles + wpc_handles,
        fontsize=PAPER_FONT["legend"],
        loc="best",
        ncol=PAPER_LEGEND_NCOL,
        frameon=True,
        framealpha=0.92,
    )

    ax.set_title(_pretty_truth_name_for_title(title), fontsize=PAPER_FONT["title"])
    ax.set_xlabel("Forecast source", fontsize=PAPER_FONT["axis"])
    ax.set_ylabel(metric, fontsize=PAPER_FONT["axis"])
    ax.tick_params(axis="x", labelrotation=35, labelsize=PAPER_FONT["tick"])
    ax.tick_params(axis="y", labelsize=PAPER_FONT["tick"])

    for lab in ax.get_xticklabels():
        lab.set_horizontalalignment("right")

    ax.grid(True, axis="y", alpha=0.25)

    if ylim is not None:
        ax.set_ylim(*ylim)

    plt.tight_layout()

    outdir = Path(PAPER_PLOT_DIR)
    outdir.mkdir(parents=True, exist_ok=True)
    outpath = outdir / filename
    fig.savefig(outpath, dpi=int(PAPER_DPI), bbox_inches="tight")
    print("Saved:", outpath)

    if globals().get("PAPER_SHOW_PLOTS", True):
        plt.show()

    plt.close(fig)
    return outpath


print("Seaborn plot_grouped_violin override loaded.")



In [ ]:
# ======================================================================================
# FULL PAPER VERIFICATION BLOCK: FOUR DAY-2 MODELS, NO PMM SOURCES
#
#   - Rebuilds UFVS_ANY from UFVS archive when needed
#   - Expands UFVS/MRMS event truth to 40 km
#   - Creates categorical BS plots:
#         Including Marginal: forecast >= 5%
#         Excluding Marginal: forecast >= 15%
#     using representative category probabilities for BOTH ML and WPC
#
#   - Creates ETS/POD/FAR plots using binary forecast exceedance thresholds
#   - This merged version evaluates ETS/POD/FAR only at forecast >=5%
#     and labels it as Marginal or greater
#
#   - Includes the four Day-2 ML radius tests, ML ensemble mean, ML ensemble max, and WPC ERO
#   - Uses seaborn violins
#   - Saves plots and CSVs
#   - No titles
#   - Legend outside plot
#   - Big font sizes
#   - Mean square gets red outline if better than WPC mean for same evaluation/threshold
#   - Strong dashed horizontal and vertical major gridlines
#
# Run this after your historical/test dataframe is loaded.
# Expected dataframe names:
#   df_radius_viewer, df_realtime_viewer, or choose_evaluation_dataframe(...)
# ======================================================================================

import os
import re
import io
import math
import urllib.request
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.spatial import cKDTree

try:
    import seaborn as sns
except Exception as exc:
    raise RuntimeError("This block requires seaborn. Install seaborn or import it before running.") from exc


# ======================================================================================
# USER CONTROLS
# ======================================================================================

PAPER_PREFER_REALTIME = False

# PMM is excluded at the model-specification, dataframe-column, metric-row,
# and final plotting stages.
def _paper_is_pmm_model(value):
    """Return True for any PMM or probability-matched-mean label/column."""
    if value is None:
        return False

    text = str(value)
    return bool(
        re.search(r"PMM", text, flags=re.IGNORECASE)
        or re.search(r"probability[ _-]*matched[ _-]*mean", text, flags=re.IGNORECASE)
    )


def _paper_spec_is_pmm(spec):
    """Inspect the entire model specification for PMM naming."""
    if not isinstance(spec, dict):
        return _paper_is_pmm_model(spec)

    if any(
        _paper_is_pmm_model(key) or _paper_is_pmm_model(value)
        for key, value in spec.items()
    ):
        return True

    try:
        generated_col = _model_prob_col(spec.get("label"), spec.get("radius_km"))
    except Exception:
        generated_col = None

    return _paper_is_pmm_model(generated_col)


def _paper_is_excluded_model(value):
    """Exclude PMM forecast sources."""
    return _paper_is_pmm_model(value)


def _paper_drop_excluded_source_rows(rows, context="metric rows"):
    """
    Final downstream safeguard. Remove PMM source rows even when they were
    created before this block or came from a previously prepared intermediate table.
    """
    if rows is None or len(rows) == 0 or "Source" not in rows.columns:
        return rows

    out = rows.copy()
    bad = out["Source"].astype(str).map(_paper_is_excluded_model)

    if bad.any():
        removed_sources = sorted(out.loc[bad, "Source"].astype(str).unique().tolist())
        print(f"Removing excluded PMM sources from {context}: {removed_sources}")
        out = out.loc[~bad].copy()

    surviving_bad = sorted(
        source
        for source in out["Source"].dropna().astype(str).unique().tolist()
        if _paper_is_excluded_model(source)
    )
    if surviving_bad:
        raise RuntimeError(
            f"Excluded sources survived final filtering in {context}: {surviving_bad}"
        )

    return out


def _paper_is_excluded_spec(spec):
    """Exclude an entire model specification when any field identifies PMM."""
    return _paper_spec_is_pmm(spec)


def _paper_spec_display_name(spec):
    if isinstance(spec, dict):
        return str(spec.get("label", spec))
    return str(spec)


# Retain the four trained Day-2 model specifications and exclude PMM.
PAPER_MODEL_SPECS = [
    spec
    for spec in MODEL_SPECS
    if not _paper_is_excluded_spec(spec)
]

PAPER_EXCLUDED_PMM_MODEL_LABELS = [
    _paper_spec_display_name(spec)
    for spec in MODEL_SPECS
    if _paper_spec_is_pmm(spec)
]

PAPER_MODEL_LABELS = [str(spec["label"]) for spec in PAPER_MODEL_SPECS]
PAPER_RADII_KM = list(
    dict.fromkeys(spec["radius_km"] for spec in PAPER_MODEL_SPECS)
)

PAPER_INCLUDE_ENSEMBLE_MEAN = True
PAPER_INCLUDE_ENSEMBLE_MAX = True
PAPER_INCLUDE_WPC = True


# Truth expansion for point/proxy event fields
PAPER_TRUTH_EXPANSION_RADIUS_KM = 40.0

# UFVS archive controls
PAPER_FORCE_UFVS_ARCHIVE_REBUILD = False
PAPER_UFVS_MAX_NEAREST_DIST_KM = 25.0
PAPER_UFVS_INCLUDE_REGULAR_FLOOD_LSR = False
PAPER_UFVS_BASE_URL = "https://ftp-wpc.ncep.noaa.gov/erickson/FFaIR/UFVS"

# UFVS proxies used to define Any Flood Proxy
PAPER_UFVS_PREFIX_MAP = {
    "ST4gFFG": "Stage IV > FFG",
    "ST4gARI": "Stage IV ARI",
    "USGS": "USGS",
    "LSRFLASH": "Flash Flood LSR",
}
if PAPER_UFVS_INCLUDE_REGULAR_FLOOD_LSR:
    PAPER_UFVS_PREFIX_MAP["LSRREG"] = "Flood LSR"

# BS representative probabilities.
# This makes BS categorical for BOTH ML and WPC.
# Change Slight from 0.25 to 0.275 if you want the literal midpoint of 15-40%.
BS_CATEGORY_REPRESENTATIVE_PROBS = {
    "Marginal": 0.10,   # 5-15%
    "Slight": 0.25,     # 15-40%
    "Moderate": 0.55,   # 40-70%
    "High": 0.85,       # >=70%
}

BS_EVAL_MODES = [
    {
        "Evaluation": "Including Marginal",
        "Threshold": 0.05,
        "Legend Label": "Including Marginal (>=5%)",
    },
    {
        "Evaluation": "Excluding Marginal",
        "Threshold": 0.15,
        "Legend Label": "Excluding Marginal (>=15%)",
    },
]

ETS_MARGINAL_OR_GREATER_THRESHOLD = 0.05
ETS_MARGINAL_OR_GREATER_LABEL = "Marginal or greater"

# Binary flood-proxy truths only get marginal-or-greater ETS/POD/FAR.
ETS_BINARY_PROXY_THRESHOLDS = [
    (ETS_MARGINAL_OR_GREATER_THRESHOLD, ETS_MARGINAL_OR_GREATER_LABEL),
]

# Practically Perfect fields retain categorical risk-threshold ETS/POD/FAR.
ETS_PP_CATEGORICAL_THRESHOLDS = [
    (0.05, ">=5%"),
    (0.15, ">=15%"),
    (0.40, ">=40%"),
    (0.70, ">=70%"),
]

# Backward-compatible default shown in setup prints; do not use directly for every truth.
ETS_THRESHOLDS = ETS_BINARY_PROXY_THRESHOLDS

# Truth datasets
BS_TRUTH_OPTIONS = [
    {
        "Truth Label": "Any Flood Proxy",
        "Truth Option": "UFVS_ANY",
        "Filename Stem": "bs_any_flood_proxy_include_exclude_marginal",
    },
    {
        "Truth Label": "MRMS > FFG",
        "Truth Option": "MRMS_FFG_POINT",
        "Filename Stem": "bs_mrms_ffg_include_exclude_marginal",
    },
]

ETS_TRUTH_OPTIONS = [
    {
        "Truth Label": "Any Flood Proxy",
        "Truth Option": "UFVS_ANY",
        "Filename Stem": "ets_any_flood_proxy",
    },
    {
        "Truth Label": "MRMS > FFG",
        "Truth Option": "MRMS_FFG_POINT",
        "Filename Stem": "ets_mrms_ffg",
    },
    # This is optional. It will be skipped if PP Any Flood Proxy is not found.
    {
        "Truth Label": "Practically Perfect: Any Flood Proxy",
        "Truth Option": "PP_ANY",
        "Filename Stem": "ets_pp_any_flood_proxy",
        "Optional": True,
    },
]

# Colorblind-friendly Okabe-Ito-style colors
BS_EVAL_PALETTE = {
    "Including Marginal": "#0072B2",   # blue
    "Excluding Marginal": "#D55E00",   # vermillion
}

ETS_THRESHOLD_PALETTE = {
    ETS_MARGINAL_OR_GREATER_LABEL: "#0072B2",
    ">=5%": "#0072B2",
    ">=15%": "#E69F00",
    ">=40%": "#009E73",
    ">=70%": "#CC79A7",
}

# Plot appearance
PAPER_FIGSIZE = (44, 22)
PAPER_DPI = 220
PAPER_SHOW_PLOTS = True

PAPER_FONT = {
    "axis": 50,
    "tick": 50,
    "legend": 50,
    "annotation": 50,
}

PAPER_VIOLIN_ALPHA = 0.78
PAPER_VIOLIN_LINEWIDTH = 2.5

PAPER_MEAN_MARKER_SIZE = 360
PAPER_MEAN_MARKER_FACE = "white"
PAPER_MEAN_MARKER_EDGE_DEFAULT = "black"
PAPER_MEAN_MARKER_EDGE_BETTER = "red"
PAPER_MEAN_MARKER_LINEWIDTH_DEFAULT = 3.0
PAPER_MEAN_MARKER_LINEWIDTH_BETTER = 6.0

PAPER_MEDIAN_LINEWIDTH = 5.0
PAPER_LEGEND_NCOL = 1
PAPER_ROTATE_XTICKS = 35

# Strong major gridlines on both axes.
PAPER_GRID_COLOR = "0.35"
PAPER_GRID_ALPHA = 0.40
PAPER_GRID_LINEWIDTH = 2.2
PAPER_GRID_LINESTYLE = "--"

if "PROJECT_DIR" in globals():
    PAPER_VERIF_OUTDIR = Path(PROJECT_DIR) / "paper_verification_bs_ets_final_v33day2valid"
else:
    PAPER_VERIF_OUTDIR = Path.cwd() / "paper_verification_bs_ets_final_v33day2valid"

PAPER_UFVS_CACHE_DIR = PAPER_VERIF_OUTDIR / "ufvs_archive_grid_cache"


# ======================================================================================
# DATAFRAME SELECTION AND FORECAST SOURCE PREP
# ======================================================================================

def _paper_choose_dataframe(prefer_realtime=False):
    if prefer_realtime and "df_realtime_viewer" in globals():
        return df_realtime_viewer.copy(), "realtime"

    if "choose_evaluation_dataframe" in globals():
        try:
            df, label = choose_evaluation_dataframe(prefer_realtime=prefer_realtime)
            return df.copy(), label
        except Exception as exc:
            print("choose_evaluation_dataframe failed; trying direct dataframe names:", exc)

    if "df_radius_viewer" in globals():
        return df_radius_viewer.copy(), "historical/test"

    if "df_realtime_viewer" in globals():
        return df_realtime_viewer.copy(), "realtime"

    raise RuntimeError(
        "No evaluation dataframe found. Expected df_radius_viewer, df_realtime_viewer, "
        "or choose_evaluation_dataframe(...)."
    )


def _paper_date8_array(df):
    if "Date" in df.columns:
        return df["Date"].astype(str).str[:8].to_numpy()

    if "ValidDate" in df.columns:
        return df["ValidDate"].astype(str).str[:8].to_numpy()

    if "REALTIME_DATE" in globals():
        return np.array([str(REALTIME_DATE)[:8]] * len(df), dtype=object)

    raise RuntimeError("Could not identify date column. Expected Date, ValidDate, or REALTIME_DATE.")


def _paper_source_sort_key(source):
    s = str(source)
    m = re.search(r"ML r(\d+)", s)
    if m:
        return int(m.group(1))
    if "Ensemble Mean" in s:
        return 500
    if "Ensemble Max" in s:
        return 501
    if "WPC" in s:
        return 900
    return 800


def _paper_pretty_source(source):
    s = str(source)
    s = re.sub(r"ML r(\d+)$", r"ML r\1 km", s)
    s = re.sub(r"ML r(\d+) km km", r"ML r\1 km", s)
    s = s.replace("ML ens mean", "ML Ensemble Mean")
    s = s.replace("ML ens max", "ML Ensemble Max")
    s = s.replace("ML_Radius_EnsMean", "ML Ensemble Mean")
    s = s.replace("ML_Radius_EnsMax", "ML Ensemble Max")
    return s


def _paper_radius_prob_col(radius_km):
    return f"ML_r{int(radius_km)}_Prob"


def _paper_column_metadata_values(col):
    """Return the raw column name plus every value returned by metadata parsing."""
    values = [col]
    try:
        metadata = _wide_ml_member_metadata(col)
    except Exception:
        metadata = None

    if metadata is None:
        return values

    if isinstance(metadata, (tuple, list)):
        values.extend(metadata)
    else:
        values.append(metadata)

    return values


def _paper_column_is_excluded_model(col):
    """Reject a wide forecast column if its name or parsed metadata identifies PMM."""
    return any(
        _paper_is_excluded_model(value)
        for value in _paper_column_metadata_values(col)
        if value is not None
    )


def _paper_active_ml_member_columns(df):
    """Return only non-PMM wide ML probability columns."""
    active = []

    for col in df.columns:
        try:
            metadata = _wide_ml_member_metadata(col)
            model_label = metadata[0]
        except Exception:
            continue

        if model_label is None:
            continue

        if _paper_column_is_excluded_model(col):
            continue

        active.append(col)

    return active


def _paper_excluded_ml_member_columns(df):
    """Return every parsed ML member column identified as PMM."""
    excluded = []

    for col in df.columns:
        try:
            metadata = _wide_ml_member_metadata(col)
            model_label = metadata[0]
        except Exception:
            continue

        if model_label is None:
            continue

        if _paper_column_is_excluded_model(col):
            excluded.append(col)

    return excluded


def _paper_drop_excluded_model_columns(df):
    """
    Physically drop PMM forecast columns before any ensemble calculation.

    This prevents previously loaded excluded columns from leaking into ensemble mean/max,
    even when MODEL_SPECS or metadata parsing uses an unexpected naming convention.
    """
    out = df.copy()

    excluded = set(_paper_excluded_ml_member_columns(out))

    # Additional raw-name safeguard for forecast/probability columns that the
    # notebook metadata helper does not recognize.
    for col in out.columns:
        col_text = str(col)
        looks_forecast_like = bool(
            re.search(r"ML|Prob|Forecast|Model|PMM", col_text, flags=re.IGNORECASE)
        )
        if looks_forecast_like and _paper_is_excluded_model(col_text):
            excluded.add(col)

    if excluded:
        excluded = sorted(excluded, key=str)
        print("Dropping excluded PMM forecast columns:", excluded)
        out = out.drop(columns=excluded, errors="ignore")

    return out


def _paper_find_radius_col(df):
    candidates = [
        "ML_Target_Radius_km",
        "Target_Radius_km",
        "Radius_km",
        "Radius km",
        "radius_km",
        "Radius",
    ]
    for c in candidates:
        if c in df.columns:
            return c
    return None


def _paper_ensure_wide_ml_radius_columns(df):
    """
    Supports either:
      1. wide dataframe with ML_r40_Prob, ML_r60_Prob, ...
      2. long dataframe with ML_Forecast_Prob and a radius column

    Returns a dataframe with ML_rXX_Prob columns when possible.
    """
    out = _paper_drop_excluded_model_columns(df)

    existing = _paper_active_ml_member_columns(out)
    if existing:
        return out

    if "ML_Forecast_Prob" not in out.columns:
        return out

    radius_col = _paper_find_radius_col(out)
    if radius_col is None:
        return out

    key_cols = [c for c in ["Date", "Lat", "Lon"] if c in out.columns]
    if len(key_cols) < 3:
        return out

    print(f"Converting long ML_Forecast_Prob/radius table to wide ML_rXX_Prob columns using {radius_col!r}.")

    model_col = "ML_Model_Label" if "ML_Model_Label" in out.columns else None
    base_cols = [c for c in out.columns if c not in ["ML_Forecast_Prob", radius_col, model_col]]
    base = out[base_cols].drop_duplicates(subset=key_cols).copy()

    tmp_cols = key_cols + [radius_col, "ML_Forecast_Prob"] + ([model_col] if model_col else [])
    tmp = out[tmp_cols].copy()
    tmp[radius_col] = pd.to_numeric(tmp[radius_col], errors="coerce")
    tmp["ML_Forecast_Prob"] = pd.to_numeric(tmp["ML_Forecast_Prob"], errors="coerce")
    tmp["__ModelLabel"] = (
        tmp[model_col].astype(str)
        if model_col
        else tmp[radius_col].map(lambda r: model_label_for_radius(r))
    )

    # Remove every PMM row before pivoting. Check the resolved model label,
    # the original model-label column, and any source/version-like columns available.
    excluded_row_mask = tmp["__ModelLabel"].map(_paper_is_excluded_model)

    if model_col and model_col in tmp.columns:
        excluded_row_mask |= tmp[model_col].map(_paper_is_excluded_model)

    tmp = tmp[~excluded_row_mask].copy()

    if tmp.empty:
        return base

    wide = tmp.pivot_table(
        index=key_cols,
        columns="__ModelLabel",
        values="ML_Forecast_Prob",
        aggfunc="first",
    ).reset_index()

    rename = {}
    for c in wide.columns:
        if c in key_cols:
            continue
        spec = next(
            (
                x
                for x in PAPER_MODEL_SPECS
                if str(x["label"]) == str(c)
            ),
            None,
        )
        if spec is not None:
            rename[c] = _model_prob_col(spec["label"], spec["radius_km"])

    wide = wide.rename(columns=rename)

    # Drop any unrenamed PMM columns as a second safeguard.
    excluded_wide_cols = [
        c
        for c in wide.columns
        if c not in key_cols and _paper_column_is_excluded_model(c)
    ]
    if excluded_wide_cols:
        wide = wide.drop(columns=excluded_wide_cols)

    merged = base.merge(wide, on=key_cols, how="left")
    return merged


def _paper_add_ensemble_columns(df):
    out = _paper_drop_excluded_model_columns(
        _paper_ensure_wide_ml_radius_columns(df)
    )

    # Never reuse an ensemble column that may have been calculated earlier with PMM.
    out = out.drop(
        columns=["ML_Radius_EnsMean", "ML_Radius_EnsMax", "ML_Radius_PMM"],
        errors="ignore",
    )

    member_cols = sorted(
        _paper_active_ml_member_columns(out),
        key=lambda c: (
            _wide_ml_member_metadata(c)[1],
            _wide_ml_member_metadata(c)[0],
        ),
    )

    if member_cols:
        arr = out[member_cols].apply(pd.to_numeric, errors="coerce").to_numpy(float)
        out["ML_Radius_EnsMean"] = np.nanmean(arr, axis=1).astype(np.float32)
        out["ML_Radius_EnsMax"] = np.nanmax(arr, axis=1).astype(np.float32)

    return out


def _paper_iter_forecast_sources(df):
    # Work from a physically cleaned dataframe so excluded columns cannot be selected.
    clean_df = _paper_drop_excluded_model_columns(df)
    sources = []

    for spec in PAPER_MODEL_SPECS:
        if _paper_is_excluded_spec(spec):
            continue

        r = int(spec["radius_km"])
        model_label = str(spec["label"])

        if r not in PAPER_RADII_KM:
            continue

        c = _model_prob_col(model_label, r)

        if _paper_is_excluded_model(model_label) or _paper_is_excluded_model(c):
            continue

        if c in clean_df.columns:
            sources.append({"Source": f"ML {model_label}", "Column": c})

    if PAPER_INCLUDE_ENSEMBLE_MEAN and "ML_Radius_EnsMean" in clean_df.columns:
        sources.append({"Source": "ML Ensemble Mean", "Column": "ML_Radius_EnsMean"})

    if PAPER_INCLUDE_ENSEMBLE_MAX and "ML_Radius_EnsMax" in clean_df.columns:
        sources.append({"Source": "ML Ensemble Max", "Column": "ML_Radius_EnsMax"})

    if PAPER_INCLUDE_WPC and "WPC_ERO_Risk" in clean_df.columns:
        sources.append({"Source": "WPC ERO", "Column": "WPC_ERO_Risk"})

    # Hard final source-list filter prevents PMM sources from entering verification.
    excluded_sources = [
        src for src in sources
        if _paper_is_excluded_model(src["Source"])
        or _paper_is_excluded_model(src["Column"])
    ]
    if excluded_sources:
        print("Removing excluded forecast sources before verification:", excluded_sources)
        sources = [
            src for src in sources
            if not _paper_is_excluded_model(src["Source"])
            and not _paper_is_excluded_model(src["Column"])
        ]

    # Final hard assertion: nothing labeled PMM may leave this function.
    bad_sources = [
        src
        for src in sources
        if _paper_is_excluded_model(src["Source"])
        or _paper_is_excluded_model(src["Column"])
    ]
    if bad_sources:
        raise RuntimeError(
            "Excluded PMM forecast sources survived source filtering: "
            f"{bad_sources}"
        )

    if not sources:
        raise RuntimeError(
            "No forecast source columns found after removing PMM models. "
            "Expected Day-2 ML probability columns and/or WPC_ERO_Risk."
        )

    return sources


# ======================================================================================
# GEOGRAPHIC / TRUTH HELPERS
# ======================================================================================

def _paper_latlon_to_unit_xyz(lat, lon):
    lat = np.deg2rad(np.asarray(lat, dtype=float))
    lon = np.deg2rad(np.asarray(lon, dtype=float))
    return np.column_stack([
        np.cos(lat) * np.cos(lon),
        np.cos(lat) * np.sin(lon),
        np.sin(lat),
    ])


def _paper_chord_radius_from_km(radius_km):
    earth_radius_km = 6371.0
    angular = float(radius_km) / earth_radius_km
    return 2.0 * np.sin(angular / 2.0)


def _paper_expand_grid_mask_radius_km(mask, lat, lon, radius_km):
    mask = np.asarray(mask, dtype=bool)
    if radius_km is None or float(radius_km) <= 0 or not np.any(mask):
        return mask.copy()

    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    good = np.isfinite(lat) & np.isfinite(lon)

    out = mask.copy()
    if not np.any(good):
        return out

    xyz = _paper_latlon_to_unit_xyz(lat, lon)
    tree = cKDTree(xyz)
    chord = _paper_chord_radius_from_km(radius_km)

    event_idx = np.where(mask & good)[0]
    for idx in event_idx:
        neigh = tree.query_ball_point(xyz[idx], r=chord)
        out[neigh] = True

    return out


def _paper_latlon_columns(points_df):
    lower = {str(c).lower(): c for c in points_df.columns}

    lat_col = None
    lon_col = None

    for name in ["lat", "latitude", "y"]:
        if name in lower:
            lat_col = lower[name]
            break

    for name in ["lon", "longitude", "x"]:
        if name in lower:
            lon_col = lower[name]
            break

    if lat_col is not None and lon_col is not None:
        return lat_col, lon_col

    # Headerless fallback: infer from first two numeric columns.
    numeric_cols = []
    for c in points_df.columns:
        vals = pd.to_numeric(points_df[c], errors="coerce")
        if vals.notna().sum() > 0:
            numeric_cols.append(c)

    if len(numeric_cols) < 2:
        raise RuntimeError(f"Could not infer lat/lon columns. Columns: {list(points_df.columns)}")

    c0, c1 = numeric_cols[:2]
    v0 = pd.to_numeric(points_df[c0], errors="coerce")
    v1 = pd.to_numeric(points_df[c1], errors="coerce")

    # Typical CONUS: lat ~ 20-55, lon ~ -130 to -60
    c0_lat_like = v0.between(-90, 90).mean() > 0.8 and v0.median() > 10
    c1_lon_like = v1.between(-180, 180).mean() > 0.8 and v1.median() < -40

    c0_lon_like = v0.between(-180, 180).mean() > 0.8 and v0.median() < -40
    c1_lat_like = v1.between(-90, 90).mean() > 0.8 and v1.median() > 10

    if c0_lat_like and c1_lon_like:
        return c0, c1

    if c0_lon_like and c1_lat_like:
        return c1, c0

    # Last fallback: assume lat, lon
    return c0, c1


def _paper_grid_signature(day_df):
    lat = pd.to_numeric(day_df["Lat"], errors="coerce")
    lon = pd.to_numeric(day_df["Lon"], errors="coerce")
    return (
        f"n{len(day_df)}_"
        f"lat{lat.min():.3f}_{lat.max():.3f}_"
        f"lon{lon.min():.3f}_{lon.max():.3f}"
    ).replace("-", "m").replace(".", "p")


# ======================================================================================
# UFVS ARCHIVE DOWNLOAD / REBUILD
# ======================================================================================

def _paper_read_url_text(url, timeout=60):
    with urllib.request.urlopen(url, timeout=timeout) as r:
        return r.read().decode("utf-8", errors="replace")


def _paper_parse_ufvs_points_from_text(text):
    """
    Robust parser for simple UFVS lat/lon text tables.
    Handles headered or headerless whitespace/comma-separated files.
    """
    if text is None or not str(text).strip():
        return pd.DataFrame()

    # Try headered table first
    try:
        df = pd.read_csv(io.StringIO(text), sep=r"\s+|,", engine="python", comment="#")
        if len(df.columns) >= 2 and len(df) > 0:
            return df
    except Exception:
        pass

    # Headerless fallback
    try:
        df = pd.read_csv(io.StringIO(text), sep=r"\s+|,", engine="python", comment="#", header=None)
        return df
    except Exception:
        return pd.DataFrame()


def _paper_fetch_ufvs_points_existing_helper(date8, prefix, force=False):
    """
    Prefer your notebook's existing fetch_ufvs_points if available.
    """
    if "fetch_ufvs_points" not in globals():
        return None

    pts = fetch_ufvs_points(str(date8), prefix, force=bool(force))

    if pts is None:
        return pd.DataFrame()

    if isinstance(pts, tuple):
        pts = pts[0]

    if not isinstance(pts, pd.DataFrame):
        pts = pd.DataFrame(pts)

    return pts.copy()


def _paper_fetch_ufvs_points_fallback(date8, prefix):
    """
    Fallback UFVS archive downloader.
    It scrapes the UFVS directory and chooses files containing both the prefix and date.
    """
    index_text = _paper_read_url_text(PAPER_UFVS_BASE_URL)
    hrefs = re.findall(r'href=["\']([^"\']+)["\']', index_text, flags=re.I)
    lookup_date8 = (
        day2_observation_date(date8)
        if "day2_observation_date" in globals()
        else str(date8)
    )

    candidates = []
    for href in hrefs:
        h = href.split("/")[-1]
        if prefix.lower() in h.lower() and str(lookup_date8) in h:
            candidates.append(href)

    if not candidates:
        # Sometimes dates may appear as YYYY-MM-DD
        d_alt = f"{str(lookup_date8)[:4]}-{str(lookup_date8)[4:6]}-{str(lookup_date8)[6:8]}"
        for href in hrefs:
            h = href.split("/")[-1]
            if prefix.lower() in h.lower() and d_alt in h:
                candidates.append(href)

    if not candidates:
        return pd.DataFrame()

    frames = []
    for href in candidates:
        if href.startswith("http"):
            url = href
        else:
            url = PAPER_UFVS_BASE_URL.rstrip("/") + "/" + href.lstrip("/")

        try:
            text = _paper_read_url_text(url)
            df = _paper_parse_ufvs_points_from_text(text)
            if len(df):
                frames.append(df)
        except Exception as exc:
            print(f"UFVS fallback download failed for {url}: {exc}")

    if not frames:
        return pd.DataFrame()

    return pd.concat(frames, ignore_index=True)


def _paper_fetch_ufvs_points(date8, prefix, force=False):
    """
    Fetch UFVS points for one date/prefix.
    Uses existing notebook helper if available; otherwise tries direct archive scraping.
    """
    pts = None

    try:
        pts = _paper_fetch_ufvs_points_existing_helper(date8, prefix, force=force)
    except Exception as exc:
        print(f"Existing fetch_ufvs_points failed for {date8} {prefix}: {exc}")
        pts = None

    if pts is not None and len(pts) > 0:
        return pts

    try:
        return _paper_fetch_ufvs_points_fallback(date8, prefix)
    except Exception as exc:
        print(f"Fallback UFVS fetch failed for {date8} {prefix}: {exc}")
        return pd.DataFrame()


def _paper_points_to_grid_mask_nearest(points, day_df, max_nearest_dist_km=25.0):
    """
    Map each UFVS point to nearest forecast grid point.
    """
    mask = np.zeros(len(day_df), dtype=bool)

    if points is None or len(points) == 0:
        return mask

    lat_col, lon_col = _paper_latlon_columns(points)

    p_lat = pd.to_numeric(points[lat_col], errors="coerce").to_numpy(float)
    p_lon = pd.to_numeric(points[lon_col], errors="coerce").to_numpy(float)
    p_good = np.isfinite(p_lat) & np.isfinite(p_lon)

    if not np.any(p_good):
        return mask

    g_lat = pd.to_numeric(day_df["Lat"], errors="coerce").to_numpy(float)
    g_lon = pd.to_numeric(day_df["Lon"], errors="coerce").to_numpy(float)
    g_good = np.isfinite(g_lat) & np.isfinite(g_lon)

    if not np.any(g_good):
        raise RuntimeError("Forecast grid has no finite Lat/Lon values for UFVS mapping.")

    grid_xyz = _paper_latlon_to_unit_xyz(g_lat, g_lon)
    point_xyz = _paper_latlon_to_unit_xyz(p_lat[p_good], p_lon[p_good])

    tree = cKDTree(grid_xyz)
    dist, idx = tree.query(point_xyz, k=1)

    max_chord = _paper_chord_radius_from_km(max_nearest_dist_km)
    keep = np.isfinite(dist) & (dist <= max_chord)

    if np.any(keep):
        mask[idx[keep]] = True

    return mask


def _paper_build_ufvs_any_raw_for_day(date8, day_df, force=False):
    """
    Build raw UFVS_ANY on the forecast grid for one date.
    This is nearest-grid mapping only. Expansion to 40 km happens later.
    """
    PAPER_UFVS_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    sig = _paper_grid_signature(day_df)
    cache_path = (
        PAPER_UFVS_CACHE_DIR
        / f"ufvs_any_raw_{date8}_{sig}_nearest{int(PAPER_UFVS_MAX_NEAREST_DIST_KM)}km.parquet"
    )

    if cache_path.exists() and not force:
        cached = pd.read_parquet(cache_path)
        if "UFVS_ANY_RAW" in cached.columns and len(cached) == len(day_df):
            return cached["UFVS_ANY_RAW"].astype(bool).to_numpy()

    any_mask = np.zeros(len(day_df), dtype=bool)
    counts = {}

    for prefix, label in PAPER_UFVS_PREFIX_MAP.items():
        pts = _paper_fetch_ufvs_points(date8, prefix, force=force)
        m = _paper_points_to_grid_mask_nearest(
            pts,
            day_df,
            max_nearest_dist_km=PAPER_UFVS_MAX_NEAREST_DIST_KM,
        )
        any_mask |= m
        counts[label] = int(np.sum(m))

    pd.DataFrame({
        "Date": [str(date8)] * len(day_df),
        "UFVS_ANY_RAW": any_mask.astype(np.int8),
    }).to_parquet(cache_path, index=False)

    print(
        f"UFVS_ANY raw built for {date8}: raw grid events={int(np.sum(any_mask))}; "
        f"source counts={counts}"
    )

    return any_mask


def _paper_build_ufvs_any_raw_for_dataframe(df, force=False):
    """
    Build raw UFVS_ANY mask for all dates in df.
    """
    if "Lat" not in df.columns or "Lon" not in df.columns:
        raise RuntimeError("UFVS rebuild requires Lat and Lon columns.")

    date_arr = _paper_date8_array(df)
    out = np.zeros(len(df), dtype=bool)

    for date8 in sorted(pd.Series(date_arr).dropna().astype(str).unique().tolist()):
        idx = np.where(date_arr == date8)[0]
        if len(idx) == 0:
            continue

        day_df = df.iloc[idx].copy().reset_index(drop=True)
        day_mask = _paper_build_ufvs_any_raw_for_day(
            date8,
            day_df,
            force=force,
        )

        if len(day_mask) != len(idx):
            raise RuntimeError(f"UFVS length mismatch for {date8}")

        out[idx] = day_mask

    return out


# ======================================================================================
# TRUTH FIELDS
# ======================================================================================

def _paper_find_pp_any_col(df):
    candidates = [
        "PP_Any flood proxy",
        "PP_Any Flood Proxy",
        "PP_ANY",
        "PP_UFVS_ANY",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    any_cols = [c for c in df.columns if str(c).startswith("PP_") and "any" in str(c).lower()]
    return any_cols[0] if any_cols else None


def _paper_find_mrms_ffg_col(df):
    candidates = [
        "MRMS_FFG_POINT",
        "Obs_Day2_MRMS_FFG_Exceeded_Point",
        "Target_Day2_MRMS_FFG_Exceeded_Point",
        "Obs_MRMS_FFG_Exceeded_Point",
        "Target_MRMS_FFG_Exceeded_Point",
        "Obs_MRMS_FFG",
        "MRMS_FFG",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    possible = [
        c for c in df.columns
        if "MRMS" in str(c) and "FFG" in str(c) and "Exceeded" in str(c)
    ]

    point_like = [c for c in possible if "Point" in str(c)]
    if point_like:
        return point_like[0]

    if possible:
        print("WARNING: using MRMS/FFG target-like column as truth:", possible[0])
        return possible[0]

    raise RuntimeError(
        "Could not find MRMS > FFG truth column. "
        f"Available MRMS/FFG columns: {[c for c in df.columns if 'MRMS' in str(c) or 'FFG' in str(c)][:80]}"
    )


def _paper_truth_binary(df, truth_option, expand_radius_km=40.0):
    """
    Returns:
      truth_col_name, expanded_binary_truth
    """
    opt = str(truth_option)

    if opt == "UFVS_ANY":
        if PAPER_FORCE_UFVS_ARCHIVE_REBUILD or "UFVS_ANY" not in df.columns:
            print("Building UFVS_ANY from archive...")
            y_raw = _paper_build_ufvs_any_raw_for_dataframe(
                df,
                force=PAPER_FORCE_UFVS_ARCHIVE_REBUILD,
            )
            truth_col = "UFVS_ANY_ARCHIVE_REBUILT"
        else:
            print("Using existing UFVS_ANY column.")
            y_raw = pd.to_numeric(df["UFVS_ANY"], errors="coerce").fillna(0).to_numpy(float) > 0
            truth_col = "UFVS_ANY"

    elif opt == "MRMS_FFG_POINT":
        truth_col = _paper_find_mrms_ffg_col(df)
        y_raw = pd.to_numeric(df[truth_col], errors="coerce").fillna(0).to_numpy(float) > 0

    elif opt == "PP_ANY":
        truth_col = _paper_find_pp_any_col(df)
        if truth_col is None:
            raise RuntimeError("No PP Any Flood Proxy column found.")
        # PP is probabilistic. For binary ETS thresholding, this function returns raw PP field later is handled separately.
        y_raw = pd.to_numeric(df[truth_col], errors="coerce").fillna(0).to_numpy(float) > 0
        # Do not expand PP. PP is already a gridded/smoothed probability-like field.
        print(f"Using PP truth column: {truth_col}")
        return truth_col, y_raw.astype(bool)

    else:
        if opt not in df.columns:
            raise RuntimeError(f"Truth option {opt!r} not found in dataframe.")
        truth_col = opt
        y_raw = pd.to_numeric(df[truth_col], errors="coerce").fillna(0).to_numpy(float) > 0

    if expand_radius_km is not None and float(expand_radius_km) > 0:
        lat = pd.to_numeric(df["Lat"], errors="coerce").to_numpy(float)
        lon = pd.to_numeric(df["Lon"], errors="coerce").to_numpy(float)
        y = _paper_expand_grid_mask_radius_km(y_raw, lat, lon, float(expand_radius_km))
    else:
        y = y_raw

    print(
        f"Truth ready: {truth_col}; raw events={int(np.sum(y_raw))}; "
        f"expanded events={int(np.sum(y))}; expansion={expand_radius_km} km"
    )

    return truth_col, y.astype(bool)


def _paper_pp_truth_binary_at_threshold(df, threshold):
    """
    Binary PP event at the given PP probability threshold.
    No 40-km expansion because PP is already gridded/smoothed.
    """
    pp_col = _paper_find_pp_any_col(df)
    if pp_col is None:
        raise RuntimeError("No PP Any Flood Proxy column found.")

    vals = pd.to_numeric(df[pp_col], errors="coerce").fillna(0).to_numpy(float)
    return pp_col, vals >= float(threshold)



# ======================================================================================
# BS CALCULATION
# ======================================================================================

def _bs_to_representative_category_probability(p):
    p = np.asarray(p, dtype=float)
    out = np.full(p.shape, np.nan, dtype=float)

    out[(p >= 0.05) & (p < 0.15)] = BS_CATEGORY_REPRESENTATIVE_PROBS["Marginal"]
    out[(p >= 0.15) & (p < 0.40)] = BS_CATEGORY_REPRESENTATIVE_PROBS["Slight"]
    out[(p >= 0.40) & (p < 0.70)] = BS_CATEGORY_REPRESENTATIVE_PROBS["Moderate"]
    out[p >= 0.70] = BS_CATEGORY_REPRESENTATIVE_PROBS["High"]

    return out


def compute_bs_include_exclude_marginal(df, truth_label, truth_option):
    df = _paper_add_ensemble_columns(df)

    truth_col, truth_yes = _paper_truth_binary(
        df,
        truth_option=truth_option,
        expand_radius_km=PAPER_TRUTH_EXPANSION_RADIUS_KM,
    )

    date_arr = _paper_date8_array(df)

    rows = []

    for src in _paper_iter_forecast_sources(df):
        source = src["Source"]
        col = src["Column"]

        raw_p = pd.to_numeric(df[col], errors="coerce").to_numpy(float)
        cat_p = _bs_to_representative_category_probability(raw_p)

        for mode in BS_EVAL_MODES:
            eval_name = mode["Evaluation"]
            threshold = float(mode["Threshold"])

            region = np.isfinite(raw_p) & np.isfinite(cat_p) & (raw_p >= threshold)

            if not np.any(region):
                continue

            for date8 in sorted(pd.Series(date_arr[region]).dropna().astype(str).unique().tolist()):
                dm = region & (date_arr == date8)
                if not np.any(dm):
                    continue

                f = cat_p[dm].astype(float)
                o = truth_yes[dm].astype(float)

                bs = float(np.mean((f - o) ** 2))

                rows.append({
                    "Date": str(date8),
                    "Truth": truth_label,
                    "Truth Column": truth_col,
                    "Source": source,
                    "Forecast Column": col,
                    "Evaluation": eval_name,
                    "Threshold": threshold,
                    "N": int(len(f)),
                    "Event Frequency": float(np.mean(o)),
                    "Mean Forecast Probability": float(np.mean(f)),
                    "Brier Score": bs,
                })

    out = pd.DataFrame(rows)
    out = _paper_drop_excluded_source_rows(
        out,
        context=f"Brier Score rows for {truth_label}",
    )
    if out.empty:
        raise RuntimeError(f"No BS rows created for truth={truth_label}")
    return out


# ======================================================================================
# ETS / POD / FAR CALCULATION
# ======================================================================================

def _contingency_stats(forecast_yes, truth_yes):
    f = np.asarray(forecast_yes, dtype=bool)
    o = np.asarray(truth_yes, dtype=bool)

    valid = np.isfinite(f.astype(float)) & np.isfinite(o.astype(float))
    f = f[valid]
    o = o[valid]

    n = int(len(f))
    if n == 0:
        return {
            "N": 0,
            "Hits": 0,
            "Misses": 0,
            "False Alarms": 0,
            "Correct Negatives": 0,
            "ETS": np.nan,
            "POD": np.nan,
            "FAR": np.nan,
            "Bias": np.nan,
        }

    hits = int(np.sum(f & o))
    misses = int(np.sum(~f & o))
    false_alarms = int(np.sum(f & ~o))
    correct_negatives = int(np.sum(~f & ~o))

    hits_random = ((hits + false_alarms) * (hits + misses)) / n
    ets_den = hits + misses + false_alarms - hits_random
    ets = (hits - hits_random) / ets_den if ets_den > 0 else np.nan

    pod = hits / (hits + misses) if (hits + misses) > 0 else np.nan
    far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else np.nan
    bias = (hits + false_alarms) / (hits + misses) if (hits + misses) > 0 else np.nan

    return {
        "N": n,
        "Hits": hits,
        "Misses": misses,
        "False Alarms": false_alarms,
        "Correct Negatives": correct_negatives,
        "ETS": float(ets),
        "POD": float(pod),
        "FAR": float(far),
        "Bias": float(bias),
    }



def _paper_ets_thresholds_for_truth(truth_label, truth_option):
    """
    Choose ETS/POD/FAR forecast thresholds by truth type.

    Binary proxy truths such as UFVS_ANY and MRMS_FFG_POINT are only evaluated
    at marginal-or-greater because the proxy observations are binary occurrence
    fields, not categorical-intensity fields.

    Practically Perfect truth (PP_ANY) retains the categorical risk thresholds
    so PP verification is computed at >=5%, >=15%, >=40%, and >=70%.
    """
    if str(truth_option) == "PP_ANY":
        return ETS_PP_CATEGORICAL_THRESHOLDS

    return ETS_BINARY_PROXY_THRESHOLDS


def compute_ets_pod_far(df, truth_label, truth_option):
    df = _paper_add_ensemble_columns(df)
    date_arr = _paper_date8_array(df)

    rows = []

    thresholds_this_truth = _paper_ets_thresholds_for_truth(
        truth_label=truth_label,
        truth_option=truth_option,
    )

    for threshold, threshold_label in thresholds_this_truth:
        if truth_option == "PP_ANY":
            truth_col, truth_yes = _paper_pp_truth_binary_at_threshold(df, threshold)
        else:
            truth_col, truth_yes = _paper_truth_binary(
                df,
                truth_option=truth_option,
                expand_radius_km=PAPER_TRUTH_EXPANSION_RADIUS_KM,
            )

        for src in _paper_iter_forecast_sources(df):
            source = src["Source"]
            col = src["Column"]

            raw_p = pd.to_numeric(df[col], errors="coerce").to_numpy(float)
            valid_p = np.isfinite(raw_p)

            for date8 in sorted(pd.Series(date_arr[valid_p]).dropna().astype(str).unique().tolist()):
                dm = valid_p & (date_arr == date8)
                if not np.any(dm):
                    continue

                forecast_yes = raw_p[dm] >= float(threshold)
                obs_yes = truth_yes[dm]

                stats = _contingency_stats(forecast_yes, obs_yes)

                row = {
                    "Date": str(date8),
                    "Truth": truth_label,
                    "Truth Column": truth_col,
                    "Source": source,
                    "Forecast Column": col,
                    "Threshold": float(threshold),
                    "Threshold Label": threshold_label,
                }
                row.update(stats)
                rows.append(row)

    out = pd.DataFrame(rows)
    out = _paper_drop_excluded_source_rows(
        out,
        context=f"ETS/POD/FAR rows for {truth_label}",
    )
    if out.empty:
        raise RuntimeError(f"No ETS/POD/FAR rows created for truth={truth_label}")
    return out


# ======================================================================================
# PLOTTING HELPERS
# ======================================================================================

def _plot_apply_violin_alpha(ax, alpha=PAPER_VIOLIN_ALPHA):
    for coll in ax.collections:
        try:
            coll.set_alpha(alpha)
            coll.set_linewidth(PAPER_VIOLIN_LINEWIDTH)
        except Exception:
            pass


def _plot_marker_positions(source_order, hue_order):
    n_hue = len(hue_order)
    dodge_width = 0.8

    if n_hue <= 1:
        offsets = [0.0]
    else:
        offsets = np.linspace(
            -dodge_width / 2 + dodge_width / (2 * n_hue),
            dodge_width / 2 - dodge_width / (2 * n_hue),
            n_hue,
        )

    positions = {}
    for i, src in enumerate(source_order):
        for j, hue in enumerate(hue_order):
            positions[(src, hue)] = i + offsets[j]

    return positions


def _metric_is_better(mean_val, wpc_mean, metric):
    if not np.isfinite(mean_val) or not np.isfinite(wpc_mean):
        return False

    metric = str(metric)

    if metric in ["Brier Score", "FAR"]:
        return mean_val < wpc_mean

    if metric in ["ETS", "POD", "Bias"]:
        return mean_val > wpc_mean

    return False


def _plot_violin_metric(
    rows,
    metric,
    hue_col,
    hue_order,
    palette,
    filename,
    y_label=None,
):
    plot_df = _paper_drop_excluded_source_rows(
        rows.copy(),
        context=f"plot input for {filename}",
    )
    plot_df["Source"] = plot_df["Source"].map(_paper_pretty_source)

    # Check again after display-name formatting, so labels such as
    # PMM sources can never reach the x axis.
    plot_df = _paper_drop_excluded_source_rows(
        plot_df,
        context=f"formatted plot input for {filename}",
    )
    plot_df[metric] = pd.to_numeric(plot_df[metric], errors="coerce")
    plot_df = plot_df[np.isfinite(plot_df[metric])].copy()

    if plot_df.empty:
        print(f"No finite values for {metric}; skipping {filename}")
        return None

    source_order = sorted(plot_df["Source"].unique().tolist(), key=_paper_source_sort_key)
    hue_order = [h for h in hue_order if h in set(plot_df[hue_col].astype(str))]

    plot_df[hue_col] = plot_df[hue_col].astype(str)

    # WPC means by hue category
    wpc_means = (
        plot_df[plot_df["Source"].eq("WPC ERO")]
        .groupby(hue_col)[metric]
        .mean()
        .to_dict()
    )

    sns.set_theme(style="whitegrid")
    fig, ax = plt.subplots(figsize=PAPER_FIGSIZE)

    violin_kwargs = dict(
        data=plot_df,
        x="Source",
        y=metric,
        hue=hue_col,
        order=source_order,
        hue_order=hue_order,
        palette=palette,
        inner=None,
        cut=0,
        linewidth=PAPER_VIOLIN_LINEWIDTH,
        dodge=True,
        ax=ax,
    )

    try:
        sns.violinplot(**violin_kwargs, density_norm="width")
    except TypeError:
        sns.violinplot(**violin_kwargs, scale="width")

    _plot_apply_violin_alpha(ax)

    if ax.get_legend() is not None:
        ax.get_legend().remove()

    positions = _plot_marker_positions(source_order, hue_order)
    half_width = 0.08

    for src in source_order:
        for hue in hue_order:
            vals = (
                plot_df[(plot_df["Source"] == src) & (plot_df[hue_col] == hue)]
                [metric]
                .dropna()
                .to_numpy(float)
            )

            if len(vals) == 0:
                continue

            mean_val = float(np.mean(vals))
            median_val = float(np.median(vals))
            x = positions[(src, hue)]

            wpc_mean = wpc_means.get(hue, np.nan)
            better = src != "WPC ERO" and _metric_is_better(mean_val, wpc_mean, metric)

            edge_color = PAPER_MEAN_MARKER_EDGE_BETTER if better else PAPER_MEAN_MARKER_EDGE_DEFAULT
            edge_width = PAPER_MEAN_MARKER_LINEWIDTH_BETTER if better else PAPER_MEAN_MARKER_LINEWIDTH_DEFAULT

            ax.scatter(
                [x],
                [mean_val],
                marker="s",
                s=PAPER_MEAN_MARKER_SIZE,
                facecolor=PAPER_MEAN_MARKER_FACE,
                edgecolor=edge_color,
                linewidth=edge_width,
                zorder=10,
            )

            ax.hlines(
                median_val,
                x - half_width,
                x + half_width,
                color="black",
                linewidth=PAPER_MEDIAN_LINEWIDTH,
                zorder=11,
            )

    # No titles
    ax.set_title("")
    ax.set_xlabel("")
    ax.set_ylabel(y_label or metric, fontsize=PAPER_FONT["axis"])

    ax.tick_params(axis="x", labelsize=PAPER_FONT["tick"], rotation=PAPER_ROTATE_XTICKS)
    ax.tick_params(axis="y", labelsize=PAPER_FONT["tick"])

    for tick in ax.get_xticklabels():
        tick.set_ha("right")

    # Strong major horizontal and vertical gridlines on every verification plot.
    ax.set_axisbelow(True)
    ax.grid(
        True,
        which="major",
        axis="both",
        color=PAPER_GRID_COLOR,
        linestyle=PAPER_GRID_LINESTYLE,
        linewidth=PAPER_GRID_LINEWIDTH,
        alpha=PAPER_GRID_ALPHA,
    )

    hue_handles = [
        plt.Line2D(
            [0],
            [0],
            color=palette[h],
            linewidth=18,
            alpha=PAPER_VIOLIN_ALPHA,
            label=h,
        )
        for h in hue_order
    ]

    marker_handles = [
        plt.Line2D(
            [0],
            [0],
            marker="s",
            linestyle="None",
            markerfacecolor=PAPER_MEAN_MARKER_FACE,
            markeredgecolor=PAPER_MEAN_MARKER_EDGE_DEFAULT,
            markeredgewidth=PAPER_MEAN_MARKER_LINEWIDTH_DEFAULT,
            color="black",
            markersize=18,
            label="Mean",
        ),
        plt.Line2D(
            [0],
            [0],
            marker="s",
            linestyle="None",
            markerfacecolor=PAPER_MEAN_MARKER_FACE,
            markeredgecolor=PAPER_MEAN_MARKER_EDGE_BETTER,
            markeredgewidth=PAPER_MEAN_MARKER_LINEWIDTH_BETTER,
            color="black",
            markersize=18,
            label="Mean better than WPC",
        ),
        plt.Line2D(
            [0],
            [0],
            color="black",
            linewidth=PAPER_MEDIAN_LINEWIDTH,
            label="Median",
        ),
    ]

    ax.legend(
        handles=hue_handles + marker_handles,
        fontsize=PAPER_FONT["legend"],
        loc="upper left",
        bbox_to_anchor=(1.01, 1.0),
        frameon=True,
        framealpha=0.95,
        ncol=PAPER_LEGEND_NCOL,
    )

    plt.tight_layout()

    PAPER_VERIF_OUTDIR.mkdir(parents=True, exist_ok=True)
    outpath = PAPER_VERIF_OUTDIR / filename
    fig.savefig(outpath, dpi=int(PAPER_DPI), bbox_inches="tight")
    print("Saved:", outpath)

    if PAPER_SHOW_PLOTS:
        plt.show()

    plt.close(fig)
    return outpath


# ======================================================================================
# RUN EVERYTHING
# ======================================================================================

def run_final_bs_ets_verification_plots():
    df, df_label = _paper_choose_dataframe(prefer_realtime=PAPER_PREFER_REALTIME)
    df = _paper_add_ensemble_columns(df)

    print("Using dataframe:", df_label)
    print("Rows:", len(df))
    print("Forecast sources:", [s["Source"] for s in _paper_iter_forecast_sources(df)])
    print("Excluded PMM model labels:", PAPER_EXCLUDED_PMM_MODEL_LABELS)
    print("Retained model labels:", PAPER_MODEL_LABELS)
    print("Retained ML member columns:", _paper_active_ml_member_columns(df))

    surviving_excluded_cols = _paper_excluded_ml_member_columns(df)
    if surviving_excluded_cols:
        raise RuntimeError(
            "PMM columns remain after dataframe cleaning: "
            f"{surviving_excluded_cols}"
        )
    print("Output directory:", PAPER_VERIF_OUTDIR)
    print("UFVS cache directory:", PAPER_UFVS_CACHE_DIR)
    print("Truth expansion radius km:", PAPER_TRUTH_EXPANSION_RADIUS_KM)
    print("BS representative probabilities:", BS_CATEGORY_REPRESENTATIVE_PROBS)
    print("ETS/POD/FAR binary-proxy threshold policy:", ETS_BINARY_PROXY_THRESHOLDS)
    print("ETS/POD/FAR PP categorical threshold policy:", ETS_PP_CATEGORICAL_THRESHOLDS)

    results = {
        "BS": {},
        "ETS": {},
    }

    # -----------------------------
    # BS plots
    # -----------------------------
    bs_hue_order = [m["Evaluation"] for m in BS_EVAL_MODES]

    for truth in BS_TRUTH_OPTIONS:
        truth_label = truth["Truth Label"]
        truth_option = truth["Truth Option"]
        stem = truth["Filename Stem"]

        print(f"\nComputing BS: {truth_label}")
        bs_rows = compute_bs_include_exclude_marginal(
            df,
            truth_label=truth_label,
            truth_option=truth_option,
        )

        results["BS"][truth_label] = bs_rows

        PAPER_VERIF_OUTDIR.mkdir(parents=True, exist_ok=True)
        csv_path = PAPER_VERIF_OUTDIR / f"{stem}.csv"
        bs_rows.to_csv(csv_path, index=False)
        print("Saved:", csv_path)

        display(
            bs_rows.groupby(["Truth", "Source", "Evaluation"], as_index=False)
            .agg(
                Mean_BS=("Brier Score", "mean"),
                Median_BS=("Brier Score", "median"),
                N_cases=("Date", "nunique"),
                Mean_N=("N", "mean"),
                Mean_Event_Frequency=("Event Frequency", "mean"),
            )
            .sort_values(["Truth", "Evaluation", "Source"], key=lambda s: s.map(_paper_source_sort_key) if s.name == "Source" else s)
            .round(4)
        )

        _plot_violin_metric(
            bs_rows,
            metric="Brier Score",
            hue_col="Evaluation",
            hue_order=bs_hue_order,
            palette=BS_EVAL_PALETTE,
            filename=f"{stem}.png",
            y_label="Brier Score",
        )

    # -----------------------------
    # ETS/POD/FAR plots
    # -----------------------------
    for truth in ETS_TRUTH_OPTIONS:
        truth_label = truth["Truth Label"]
        truth_option = truth["Truth Option"]
        stem = truth["Filename Stem"]
        optional = bool(truth.get("Optional", False))

        thresholds_this_truth = _paper_ets_thresholds_for_truth(
            truth_label=truth_label,
            truth_option=truth_option,
        )
        ets_hue_order = [label for _, label in thresholds_this_truth]

        print(f"\nComputing ETS/POD/FAR: {truth_label}")
        print("  ETS thresholds used:", thresholds_this_truth)

        try:
            ets_rows = compute_ets_pod_far(
                df,
                truth_label=truth_label,
                truth_option=truth_option,
            )
        except Exception as exc:
            if optional:
                print(f"Skipping optional truth {truth_label}: {exc}")
                continue
            raise

        results["ETS"][truth_label] = ets_rows

        csv_path = PAPER_VERIF_OUTDIR / f"{stem}_metrics.csv"
        ets_rows.to_csv(csv_path, index=False)
        print("Saved:", csv_path)

        display(
            ets_rows.groupby(["Truth", "Source", "Threshold Label"], as_index=False)
            .agg(
                Mean_ETS=("ETS", "mean"),
                Median_ETS=("ETS", "median"),
                Mean_POD=("POD", "mean"),
                Mean_FAR=("FAR", "mean"),
                N_cases=("Date", "nunique"),
                Mean_N=("N", "mean"),
            )
            .sort_values(["Truth", "Threshold Label", "Source"], key=lambda s: s.map(_paper_source_sort_key) if s.name == "Source" else s)
            .round(4)
        )

        for metric in ["ETS", "POD", "FAR"]:
            _plot_violin_metric(
                ets_rows,
                metric=metric,
                hue_col="Threshold Label",
                hue_order=ets_hue_order,
                palette=ETS_THRESHOLD_PALETTE,
                filename=f"{stem}_{metric.lower()}.png",
                y_label=metric,
            )

    return results



# ======================================================================================
# INTEGRATED MRMS TRUTH-FROM-MASTER PATCH
#
# This patch is inserted into the merged full block so MRMS > FFG truth is not looked
# for only as a dataframe column and is never pulled from PP_MRMS > FFG.
# ======================================================================================

# ======================================================================================
# PATCH: Load raw MRMS > FFG truth from model master parquet when absent from df
#
# Paste AFTER the full paper verification block, then rerun:
#   paper_verification_results = run_final_bs_ets_verification_plots()
#
# This prevents the MRMS BS/ETS section from accidentally using PP_MRMS > FFG.
# ======================================================================================

import os
import re
import glob
import numpy as np
import pandas as pd
from pathlib import Path

PAPER_MRMS_TRUTH_RADIUS_KM = 40
PAPER_MRMS_ALLOW_ALREADY_EXPANDED_TARGET = True

# If True, prints exactly which MRMS/FFG truth column was pulled.
PAPER_MRMS_DEBUG = True


def _paper_list_parquet_columns(path):
    try:
        import pyarrow.parquet as pq
        return list(pq.ParquetFile(path).schema.names)
    except Exception:
        # fallback: read zero rows if possible
        try:
            return list(pd.read_parquet(path).head(0).columns)
        except Exception:
            return []


def _paper_find_master_parquet_for_mrms(radius_km=40):
    """
    Find the master parquet that contains MRMS/FFG truth columns.

    Preferred path comes from find_artifacts_for_radius(radius_km), if defined.
    Fallback searches PROJECT_DIR recursively.
    """
    candidates = []

    if "find_artifacts_for_radius" in globals():
        try:
            art = find_artifacts_for_radius(radius_km)
            if isinstance(art, dict):
                for k in ["master_path", "master", "master_parquet", "master_file"]:
                    p = art.get(k)
                    if p and os.path.exists(p):
                        candidates.append(p)
        except Exception as exc:
            print("find_artifacts_for_radius failed while looking for MRMS truth:", exc)

    if "PROJECT_DIR" in globals():
        root = Path(PROJECT_DIR)
        patterns = [
            "**/*master*.parquet",
            "**/*pixel_domain*.parquet",
            "**/*forecasts*.parquet",
            "**/*v33*.parquet",
        ]
        for pat in patterns:
            candidates.extend([str(p) for p in root.glob(pat)])

    # Unique existing files
    seen = set()
    candidates = [p for p in candidates if os.path.exists(p) and not (p in seen or seen.add(p))]

    scored = []
    for p in candidates:
        cols = _paper_list_parquet_columns(p)
        has_keys = all(c in cols for c in ["Date", "Lat", "Lon"])
        mrms_cols = [c for c in cols if "MRMS" in str(c) and "FFG" in str(c)]
        target_cols = [c for c in mrms_cols if "Target" in str(c) or "Obs" in str(c) or "Exceeded" in str(c)]
        score = 0
        if has_keys:
            score += 10
        if mrms_cols:
            score += 20
        if target_cols:
            score += 20
        if f"R{int(radius_km)}km" in " ".join(map(str, cols)):
            score += 5
        if "master" in os.path.basename(p).lower():
            score += 5
        if score > 0:
            scored.append((score, p, mrms_cols, target_cols))

    if not scored:
        raise RuntimeError(
            "Could not find a master/parquet file containing MRMS/FFG truth columns. "
            "Check that PROJECT_DIR points to your v33 project directory and that the master parquet exists."
        )

    scored = sorted(scored, key=lambda x: x[0], reverse=True)

    if PAPER_MRMS_DEBUG:
        print("Top MRMS truth parquet candidates:")
        for score, p, mrms_cols, target_cols in scored[:5]:
            print(f"  score={score:02d} | {p}")
            print(f"    MRMS/FFG columns: {mrms_cols[:12]}")

    return scored[0][1]


def _paper_choose_mrms_truth_column_from_columns(cols, radius_km=40):
    """
    Prefer raw pointwise MRMS>FFG if available.
    If not, fall back to already-expanded target radius column.
    """
    cols = list(cols)

    exact_preferred = [
        "MRMS_FFG_POINT",
        "Obs_Day2_MRMS_FFG_Exceeded_Point",
        "Target_Day2_MRMS_FFG_Exceeded_Point",
        "Obs_MRMS_FFG_Exceeded_Point",
        "Target_MRMS_FFG_Exceeded_Point",
        "Obs_MRMS_FFG",
        "MRMS_FFG",
    ]

    for c in exact_preferred:
        if c in cols:
            return c, "point/raw"

    mrms_ffg_cols = [c for c in cols if "MRMS" in str(c) and "FFG" in str(c)]

    point_like = [
        c for c in mrms_ffg_cols
        if re.search(r"point", str(c), flags=re.I)
    ]
    if point_like:
        return point_like[0], "point/raw"

    obs_like = [
        c for c in mrms_ffg_cols
        if re.search(r"obs", str(c), flags=re.I)
    ]
    if obs_like:
        return obs_like[0], "obs/raw-ish"

    r_col_patterns = [
        f"Target_Day2_MRMS_FFG_Exceeded_R{int(radius_km)}km",
        f"Obs_Day2_MRMS_FFG_Exceeded_R{int(radius_km)}km",
        f"Obs_Day2_MRMS_FFG_Exceeded_R{int(radius_km)}km_Fraction",
        f"Target_MRMS_FFG_Exceeded_R{int(radius_km)}km",
        f"Obs_MRMS_FFG_Exceeded_R{int(radius_km)}km",
        f"MRMS_FFG_Exceeded_R{int(radius_km)}km",
    ]
    for c in r_col_patterns:
        if c in cols:
            return c, f"already-expanded-r{int(radius_km)}km"

    radius_like = [
        c for c in mrms_ffg_cols
        if f"R{int(radius_km)}km" in str(c)
    ]
    if radius_like:
        return radius_like[0], f"already-expanded-r{int(radius_km)}km"

    exceeded_like = [
        c for c in mrms_ffg_cols
        if re.search(r"exceed|target", str(c), flags=re.I)
    ]
    if exceeded_like:
        return exceeded_like[0], "fallback-target-like"

    if mrms_ffg_cols:
        return mrms_ffg_cols[0], "fallback-mrms-ffg-like"

    raise RuntimeError("No MRMS/FFG truth-like column found in candidate parquet.")


def _paper_merge_truth_column_from_parquet(df, parquet_path, truth_col):
    """
    Merge truth_col from parquet_path onto df using Date/Lat/Lon.

    First tries exact Date/Lat/Lon merge.
    If that leaves missing values, tries rounded Lat/Lon merge.
    """
    if not all(c in df.columns for c in ["Date", "Lat", "Lon"]):
        raise RuntimeError("Evaluation dataframe must have Date, Lat, and Lon to merge MRMS truth.")

    read_cols = ["Date", "Lat", "Lon", truth_col]
    truth_df = pd.read_parquet(parquet_path, columns=read_cols).copy()

    truth_df["Date"] = truth_df["Date"].astype(str).str[:8]
    out = df.copy()
    out["Date"] = out["Date"].astype(str).str[:8]

    # Avoid duplicate explosion
    truth_df = truth_df.drop_duplicates(subset=["Date", "Lat", "Lon"])

    merged = out.merge(
        truth_df,
        on=["Date", "Lat", "Lon"],
        how="left",
        suffixes=("", "_MRMS_FROM_MASTER"),
    )

    if truth_col in out.columns:
        merged_col = f"{truth_col}_MRMS_FROM_MASTER"
    else:
        merged_col = truth_col

    nonnull = merged[merged_col].notna().sum() if merged_col in merged.columns else 0

    if nonnull == len(out):
        return merged, merged_col

    # Rounded fallback
    if PAPER_MRMS_DEBUG:
        print(
            f"Exact MRMS merge filled {nonnull:,}/{len(out):,} rows. "
            "Trying rounded Lat/Lon merge fallback..."
        )

    out2 = out.copy()
    truth2 = truth_df.copy()

    out2["_DateKey"] = out2["Date"].astype(str).str[:8]
    truth2["_DateKey"] = truth2["Date"].astype(str).str[:8]

    for ndigits in [5, 4, 3]:
        out2["_LatKey"] = pd.to_numeric(out2["Lat"], errors="coerce").round(ndigits)
        out2["_LonKey"] = pd.to_numeric(out2["Lon"], errors="coerce").round(ndigits)
        truth2["_LatKey"] = pd.to_numeric(truth2["Lat"], errors="coerce").round(ndigits)
        truth2["_LonKey"] = pd.to_numeric(truth2["Lon"], errors="coerce").round(ndigits)

        truth2_small = (
            truth2[["_DateKey", "_LatKey", "_LonKey", truth_col]]
            .drop_duplicates(subset=["_DateKey", "_LatKey", "_LonKey"])
        )

        merged2 = out2.merge(
            truth2_small,
            on=["_DateKey", "_LatKey", "_LonKey"],
            how="left",
            suffixes=("", "_MRMS_FROM_MASTER"),
        )

        if truth_col in out2.columns:
            merged_col2 = f"{truth_col}_MRMS_FROM_MASTER"
        else:
            merged_col2 = truth_col

        nonnull2 = merged2[merged_col2].notna().sum() if merged_col2 in merged2.columns else 0

        if PAPER_MRMS_DEBUG:
            print(f"Rounded {ndigits} decimals MRMS merge filled {nonnull2:,}/{len(out):,} rows.")

        if nonnull2 > nonnull:
            drop_cols = [c for c in ["_DateKey", "_LatKey", "_LonKey"] if c in merged2.columns]
            merged2 = merged2.drop(columns=drop_cols)
            return merged2, merged_col2

    return merged, merged_col


def _paper_get_mrms_truth_from_master(df, radius_km=40):
    parquet_path = _paper_find_master_parquet_for_mrms(radius_km=radius_km)
    cols = _paper_list_parquet_columns(parquet_path)
    truth_col, truth_kind = _paper_choose_mrms_truth_column_from_columns(cols, radius_km=radius_km)

    if PAPER_MRMS_DEBUG:
        print("Using MRMS truth parquet:", parquet_path)
        print("Using MRMS truth column:", truth_col)
        print("MRMS truth kind:", truth_kind)

    merged, merged_col = _paper_merge_truth_column_from_parquet(df, parquet_path, truth_col)

    vals = pd.to_numeric(merged[merged_col], errors="coerce")
    filled = vals.notna().sum()

    if filled == 0:
        raise RuntimeError(
            f"MRMS truth merge found column {truth_col!r}, but no rows matched Date/Lat/Lon. "
            "This likely means the evaluation dataframe grid does not match the master parquet grid."
        )

    if filled < len(df):
        print(
            f"WARNING: MRMS truth merge filled only {filled:,}/{len(df):,} rows. "
            "Missing rows will be treated as no event after fillna(0)."
        )

    y_raw = vals.fillna(0).to_numpy(float) > 0

    return merged_col, truth_kind, y_raw


def _paper_find_mrms_ffg_col(df):
    """
    Override old function:
    Do NOT use PP_MRMS > FFG as raw MRMS truth.
    Prefer raw/target MRMS columns if already in df.
    Otherwise return None and _paper_truth_binary will pull from master parquet.
    """
    candidates = [
        "MRMS_FFG_POINT",
        "Obs_Day2_MRMS_FFG_Exceeded_Point",
        "Target_Day2_MRMS_FFG_Exceeded_Point",
        "Obs_MRMS_FFG_Exceeded_Point",
        "Target_MRMS_FFG_Exceeded_Point",
        "Obs_MRMS_FFG",
        "MRMS_FFG",
        f"Target_Day2_MRMS_FFG_Exceeded_R{int(PAPER_MRMS_TRUTH_RADIUS_KM)}km",
        f"Obs_Day2_MRMS_FFG_Exceeded_R{int(PAPER_MRMS_TRUTH_RADIUS_KM)}km",
        f"Obs_Day2_MRMS_FFG_Exceeded_R{int(PAPER_MRMS_TRUTH_RADIUS_KM)}km_Fraction",
        f"Target_MRMS_FFG_Exceeded_R{int(PAPER_MRMS_TRUTH_RADIUS_KM)}km",
        f"Obs_MRMS_FFG_Exceeded_R{int(PAPER_MRMS_TRUTH_RADIUS_KM)}km",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    possible = [
        c for c in df.columns
        if "MRMS" in str(c)
        and "FFG" in str(c)
        and not str(c).startswith("PP_")
    ]

    point_like = [c for c in possible if "Point" in str(c)]
    if point_like:
        return point_like[0]

    radius_like = [
        c for c in possible
        if f"R{int(PAPER_MRMS_TRUTH_RADIUS_KM)}km" in str(c)
    ]
    if radius_like:
        return radius_like[0]

    if possible:
        return possible[0]

    return None


def _paper_truth_binary(df, truth_option, expand_radius_km=40.0):
    """
    Full override of truth helper.

    UFVS_ANY:
      uses archive rebuild if needed.

    MRMS_FFG_POINT:
      uses raw MRMS/FFG column if present;
      otherwise loads/merges raw/target MRMS/FFG from master parquet.
      Does NOT use PP_MRMS > FFG.

    PP_ANY:
      uses PP Any only for PP-based ETS if requested.
    """
    opt = str(truth_option)

    if opt == "UFVS_ANY":
        if PAPER_FORCE_UFVS_ARCHIVE_REBUILD or "UFVS_ANY" not in df.columns:
            print("Building UFVS_ANY from archive...")
            y_raw = _paper_build_ufvs_any_raw_for_dataframe(
                df,
                force=PAPER_FORCE_UFVS_ARCHIVE_REBUILD,
            )
            truth_col = "UFVS_ANY_ARCHIVE_REBUILT"
            truth_kind = "raw-point/proxy"
        else:
            print("Using existing UFVS_ANY column.")
            y_raw = pd.to_numeric(df["UFVS_ANY"], errors="coerce").fillna(0).to_numpy(float) > 0
            truth_col = "UFVS_ANY"
            truth_kind = "raw-point/proxy"

    elif opt == "MRMS_FFG_POINT":
        truth_col = _paper_find_mrms_ffg_col(df)

        if truth_col is not None:
            print(f"Using existing raw MRMS/FFG column: {truth_col}")
            y_raw = pd.to_numeric(df[truth_col], errors="coerce").fillna(0).to_numpy(float) > 0
            truth_kind = "existing"
        else:
            print("No raw MRMS/FFG column found in dataframe. Loading from master parquet...")
            truth_col, truth_kind, y_raw = _paper_get_mrms_truth_from_master(
                df,
                radius_km=PAPER_MRMS_TRUTH_RADIUS_KM,
            )

    elif opt == "PP_ANY":
        truth_col = _paper_find_pp_any_col(df)
        if truth_col is None:
            raise RuntimeError("No PP Any Flood Proxy column found.")
        vals = pd.to_numeric(df[truth_col], errors="coerce").fillna(0).to_numpy(float)
        y_raw = vals > 0
        truth_kind = "pp-grid"
        print(f"Using PP truth column: {truth_col}")
        return truth_col, y_raw.astype(bool)

    else:
        if opt not in df.columns:
            raise RuntimeError(f"Truth option {opt!r} not found in dataframe.")
        truth_col = opt
        y_raw = pd.to_numeric(df[truth_col], errors="coerce").fillna(0).to_numpy(float) > 0
        truth_kind = "existing"

    # Expansion logic:
    # - raw point/proxy truth should be expanded to 40 km
    # - if the selected MRMS truth is already Target...R40km, do not expand again
    already_expanded = bool(re.search(rf"R{int(PAPER_MRMS_TRUTH_RADIUS_KM)}km", str(truth_col)))

    if already_expanded and opt == "MRMS_FFG_POINT":
        print(
            f"MRMS truth {truth_col!r} appears already radius-expanded; "
            "not applying an additional expansion."
        )
        y = y_raw

    elif expand_radius_km is not None and float(expand_radius_km) > 0:
        lat = pd.to_numeric(df["Lat"], errors="coerce").to_numpy(float)
        lon = pd.to_numeric(df["Lon"], errors="coerce").to_numpy(float)
        y = _paper_expand_grid_mask_radius_km(y_raw, lat, lon, float(expand_radius_km))

    else:
        y = y_raw

    print(
        f"Truth ready: {truth_col}; kind={truth_kind}; "
        f"raw events={int(np.sum(y_raw))}; final events={int(np.sum(y))}; "
        f"requested expansion={expand_radius_km} km"
    )

    return truth_col, y.astype(bool)


print("MRMS truth-from-master patch loaded.")
print("Now rerun: paper_verification_results = run_final_bs_ets_verification_plots()")
print("Full paper verification block loaded: four Day-2 ML models are active and PMM sources are excluded; stronger gridlines enabled.")
print("Run: paper_verification_results = run_final_bs_ets_verification_plots()")

paper_verification_results = run_final_bs_ets_verification_plots()

## 12. SHAP feature definitions and readable labels

This cell builds a feature-definition table directly from the trained model feature list. It also creates readable labels for SHAP plots so waterfall/dependence plots show meaningful feature names instead of unlabeled array columns.



In [ ]:
# ======================================================================================
# SHAP feature labels, definitions, and robust SHAP utilities
# ======================================================================================
# This cell intentionally overrides the older SHAP helper functions.  The previous version
# passed a bare NumPy matrix into SHAP, so waterfall plots did not retain feature names.
# Here we pass a DataFrame with readable feature labels to SHAP, but retain the original
# raw feature values for dependence plots and feature-definition tables.

import re
import textwrap

SHAP_FEATURE_LABEL_MAXLEN = 88

_FEATURE_STAT_SUFFIXES = {
    "Mean": "mean",
    "Min": "minimum",
    "Max": "maximum",
    "Std": "standard deviation",
    "STD": "standard deviation",
}

_FEATURE_VARIABLE_HINTS = [
    ("APCP", "Accumulated precipitation / QPF", "mm"),
    ("FFG", "Flash Flood Guidance", "mm"),
    ("PWAT", "Precipitable water", "mm"),
    ("CAPE", "Convective available potential energy", "J kg$^{-1}$"),
    ("CIN", "Convective inhibition", "J kg$^{-1}$"),
    ("MLCAPE", "Mixed-layer CAPE", "J kg$^{-1}$"),
    ("MLCIN", "Mixed-layer CIN", "J kg$^{-1}$"),
    ("SBCAPE", "Surface-based CAPE", "J kg$^{-1}$"),
    ("SBCIN", "Surface-based CIN", "J kg$^{-1}$"),
    ("DPT", "Dewpoint temperature", "K or °C, depending source"),
    ("TMP", "Temperature", "K or °C, depending source"),
    ("T2", "2-m temperature", "K or °C, depending source"),
    ("TD2", "2-m dewpoint", "K or °C, depending source"),
    ("RH", "Relative humidity", "%"),
    ("WIND", "Wind speed/vector-derived quantity", "m s$^{-1}$"),
    ("UGRD", "U-component wind", "m s$^{-1}$"),
    ("VGRD", "V-component wind", "m s$^{-1}$"),
    ("SHEAR", "Vertical wind shear magnitude", "m s$^{-1}$"),
    ("HGT", "Geopotential height", "m"),
    ("PBL", "Planetary boundary-layer height", "m"),
    ("SOIL", "Soil/land-surface state", "varies"),
    ("MRMS", "MRMS precipitation / MRMS>FFG verification-derived feature", "varies"),
    ("PrevDay", "Previous-day MRMS/FFG proxy history", "fraction/count/flag"),
    ("Ratio", "QPF-to-FFG ratio", "dimensionless"),
]


def _infer_outer_radius_stat(feature):
    """Return (base_name, stat_token, stat_description) for final spatial radius-stat suffix."""
    s = str(feature)
    for token, desc in sorted(_FEATURE_STAT_SUFFIXES.items(), key=lambda kv: -len(kv[0])):
        suffix = f"_{token}"
        if s.endswith(suffix):
            return s[:-len(suffix)], token, desc
    return s, None, None


def _infer_inner_stat_tokens(base):
    toks = re.split(r"[_\s]+", str(base))
    found = []
    for tok in toks:
        clean = tok.strip()
        if clean in _FEATURE_STAT_SUFFIXES:
            found.append((clean, _FEATURE_STAT_SUFFIXES[clean]))
    # Also catch strings like Max06h, Max12h, Min1h.
    for m in re.finditer(r"(Max|Min|Mean|Std|STD)(\d+)?h?", str(base)):
        found.append((m.group(0), _FEATURE_STAT_SUFFIXES.get(m.group(1), m.group(1).lower())))
    # Preserve order but remove duplicates.
    out = []
    seen = set()
    for x in found:
        if x[0] not in seen:
            out.append(x)
            seen.add(x[0])
    return out


def _infer_variable_family(feature):
    upper = str(feature).upper()
    # Prefer more specific tokens first.
    for token, family, units in sorted(_FEATURE_VARIABLE_HINTS, key=lambda x: -len(x[0])):
        if token.upper() in upper:
            return family, units
    return "Model-derived predictor", "unknown/varies"


def _infer_time_window(feature):
    s = str(feature)
    windows = []
    for pat in [
        r"(\d{1,2})_(\d{1,2})_(\d{1,2})_(\d{1,2})_(\d{1,2})h",
        r"(\d{1,2})_(\d{1,2})_(\d{1,2})h",
        r"(\d{1,2})h",
        r"f(\d{2,3})",
        r"F(\d{2,3})",
        r"(\d{2})Z",
    ]:
        for m in re.finditer(pat, s):
            windows.append(m.group(0))
    return ", ".join(dict.fromkeys(windows)) if windows else "not explicit in name"


def _pretty_base_quantity(base):
    s = str(base)
    replacements = {
        "APCP": "QPF/APCP",
        "RunTotal": "run-total accumulation",
        "Max06h": "maximum 6-h accumulation",
        "Max12h": "maximum 12-h accumulation",
        "Max24h": "maximum 24-h accumulation",
        "Guide_FFG": "FFG guidance",
        "PrevDay": "previous-day",
        "frac": "fraction",
        "point": "point",
        "rate": "rate",
    }
    out = s
    for k, v in replacements.items():
        out = out.replace(k, v)
    out = out.replace("_", " ")
    out = re.sub(r"\s+", " ", out).strip()
    return out


def _feature_aggregation_order_note(feature, base, outer_stat_token):
    """Explain the order of temporal/duration/window vs neighborhood statistics."""
    f = str(feature)
    b = str(base)
    notes = []
    has_temporal_summary = bool(re.search(r"_(Mean|Min|Max|Std)$", b)) or bool(re.search(r"Across_?6h12h24h_(Mean|Min|Max|Std)", b, flags=re.I))
    has_period_tag = bool(re.search(r"0[_-]?(06|6).*24h|0to24|6h12h24h|00_06|06_12|12_18|18_24", f, flags=re.I))
    if outer_stat_token is not None:
        notes.append(
            "Aggregation order: the base predictor is computed first at each grid point, "
            "then the final suffix is the neighborhood statistic used by the radius-sensitivity model."
        )
    if has_temporal_summary or has_period_tag:
        notes.append(
            "Temporal/duration tokens inside the base name describe summaries across forecast hours, "
            "accumulation windows, or duration-matched QPF/FFG ratios before the final neighborhood statistic is applied."
        )
    if re.search(r"Across_?6h12h24h_Std|6h12h24h_Std", f, flags=re.I):
        notes.append(
            "For QPF/FFG ratio features with Across_6h12h24h_Std, the inner Std is the spread across "
            "the duration-matched 6-h, 12-h, and 24-h QPF/FFG ratios at the grid point."
        )
    if re.search(r"RunTotal.*0.*06.*12.*18.*24.*_Std", f, flags=re.I):
        notes.append(
            "For run-total APCP features with 0/6/12/18/24h and Std, the inner Std summarizes the "
            "sequence of run-total values at those lead times at the grid point."
        )
    return " ".join(notes)


def describe_feature(feature, radius_km=None):
    """Return a dictionary with a readable definition for one feature name."""
    feature = str(feature)
    base, outer_stat_token, outer_stat_desc = _infer_outer_radius_stat(feature)
    family, units = _infer_variable_family(feature)
    time_window = _infer_time_window(feature)
    inner_stats = _infer_inner_stat_tokens(base)
    pretty_base = _pretty_base_quantity(base)

    if radius_km is None:
        radius_phrase = "the model target/neighborhood radius"
    else:
        radius_phrase = f"the {int(round(float(radius_km)))}-km neighborhood around the grid point"

    if outer_stat_desc is not None:
        definition = f"{outer_stat_desc.capitalize()} over {radius_phrase} of the base predictor: {pretty_base}."
        spatial_stat = outer_stat_desc
        aggregation_order = "base predictor first, then neighborhood statistic"
    else:
        definition = f"Base predictor: {pretty_base}."
        spatial_stat = "none/pointwise or pre-aggregated"
        aggregation_order = "no final neighborhood statistic detected in feature name"

    notes = []
    order_note = _feature_aggregation_order_note(feature, base, outer_stat_token)
    if order_note:
        notes.append(order_note)
    if inner_stats and outer_stat_desc is not None:
        inner_desc = ", ".join([f"{tok} ({desc})" for tok, desc in inner_stats])
        notes.append(
            "Nested statistic: the base quantity already contains "
            f"{inner_desc}; the final {outer_stat_token} is the neighborhood statistic. "
            "For example, a '...Std..._Mean' feature usually means a temporal/duration/window standard "
            "deviation was computed first at each grid point, then averaged over the neighborhood."
        )
    if "RATIO" in feature.upper() or ("FFG" in feature.upper() and "APCP" in feature.upper()):
        notes.append("This is likely a duration-matched QPF/APCP-to-FFG guidance ratio or related flood-guidance feature.")
    if "PREVDAY" in feature.upper() or "PREV_DAY" in feature.upper():
        notes.append("Previous-day hydrologic memory feature; verify exact construction in the training script if needed.")
    if time_window == "not explicit in name":
        notes.append("No explicit forecast-hour/time-window token was found in the feature name.")

    return {
        "Feature": feature,
        "Readable label": make_feature_label(feature, radius_km=radius_km, max_len=SHAP_FEATURE_LABEL_MAXLEN),
        "Family": family,
        "Units inferred": units,
        "Outer neighborhood statistic": spatial_stat,
        "Aggregation order": aggregation_order,
        "Time/window tokens": time_window,
        "Definition": definition,
        "Notes": " ".join(notes),
    }


def make_feature_label(feature, radius_km=None, max_len=88):
    """Short label for SHAP axes/feature names."""
    feature = str(feature)
    base, outer_stat_token, outer_stat_desc = _infer_outer_radius_stat(feature)
    pretty = _pretty_base_quantity(base)
    if outer_stat_token:
        label = f"{outer_stat_token}({pretty})"
        if radius_km is not None:
            label += f" within r{int(round(float(radius_km)))}km"
    else:
        label = pretty
    label = re.sub(r"\s+", " ", label).strip()
    if len(label) > int(max_len):
        label = label[: int(max_len) - 1] + "…"
    return label


def build_feature_definition_table(feature_names, radius_km=None):
    return pd.DataFrame([describe_feature(f, radius_km=radius_km) for f in feature_names])


def _display_feature_definition_table(feature_names, radius_km=None, max_rows=200):
    df_defs = build_feature_definition_table(feature_names, radius_km=radius_km)
    print(f"Feature definitions for {len(df_defs):,} predictors. Showing first {min(max_rows, len(df_defs)):,} rows.")
    display(df_defs.head(int(max_rows)))
    return df_defs


def load_shap_matrix(radius_km=40, sample_n=50000, random_state=42, model_label=None):
    """Load model, scaled feature matrix with readable labels, raw feature dataframe, and metadata."""
    art = find_artifacts_for_radius(radius_km, model_label=model_label or globals().get("SHAP_MODEL_LABEL"))
    if not os.path.exists(art["master_path"]):
        raise RuntimeError(f"SHAP requires the radius master parquet, but it was not found: {art['master_path']}")

    feats = _load_feature_names(art["features_path"])
    if art.get("feature_master_paths"):
        df = _sample_multiradius_feature_frame(art, feats, sample_n=sample_n, random_state=random_state)
    else:
        cols = ["Date", "Lat", "Lon"] + feats
        df = pd.read_parquet(art["master_path"], columns=cols)
        df["Date"] = df["Date"].astype(str).str[:8]
        df = df[df["Date"].str[:4].isin(TEST_YEARS)].copy()
        if sample_n and len(df) > int(sample_n):
            df = df.sample(int(sample_n), random_state=int(random_state))

    X_raw_df = df[feats].replace([np.inf, -np.inf], np.nan).fillna(0.0).astype(np.float32)
    scaler = joblib.load(art["scaler_path"])
    X_scaled = scaler.transform(X_raw_df.to_numpy(np.float32)).astype(np.float32)

    # Use readable but unique labels.  SHAP plots use these names; dependence tables still retain originals.
    labels = [make_feature_label(f, radius_km=radius_km, max_len=SHAP_FEATURE_LABEL_MAXLEN) for f in feats]
    seen = {}
    labels_unique = []
    for lab, feat in zip(labels, feats):
        if lab in seen:
            seen[lab] += 1
            lab2 = f"{lab} [{seen[lab]}]"
        else:
            seen[lab] = 1
            lab2 = lab
        labels_unique.append(lab2)

    X_scaled_df = pd.DataFrame(X_scaled, columns=labels_unique, index=df.index)
    model = joblib.load(art["model_path"])
    feature_defs = build_feature_definition_table(feats, radius_km=radius_km)
    feature_defs["SHAP label"] = labels_unique
    return model, X_scaled_df, X_raw_df, df[["Date", "Lat", "Lon"]].copy(), feats, labels_unique, feature_defs, art


def _shap_explainer_values(model, X_scaled_df):
    import shap
    explainer = shap.TreeExplainer(model)
    sv = explainer(X_scaled_df)
    # Guarantee feature names survive even for older SHAP versions.
    try:
        sv.feature_names = list(X_scaled_df.columns)
    except Exception:
        pass
    return sv


def plot_shap_waterfall(radius_km=40, row=0, sample_n=50000, max_display=25, random_state=42):
    """Waterfall plot with readable feature names and raw feature values attached."""
    import shap
    model, X_scaled_df, X_raw_df, meta_df, feats, labels, feature_defs, art = load_shap_matrix(
        radius_km=radius_km,
        sample_n=sample_n,
        random_state=random_state,
    )
    sv = _shap_explainer_values(model, X_scaled_df)
    row = int(row)
    if row < 0 or row >= len(X_scaled_df):
        raise RuntimeError(f"SHAP_ROW={row} is out of range for sample size {len(X_scaled_df)}.")

    # Rebuild the row explanation with raw values so the waterfall labels show interpretable values.
    row_exp = shap.Explanation(
        values=sv.values[row],
        base_values=sv.base_values[row] if np.ndim(sv.base_values) > 0 else sv.base_values,
        data=X_raw_df.iloc[row].to_numpy(float),
        feature_names=labels,
    )
    print("Waterfall row metadata:")
    display(meta_df.iloc[[row]].reset_index(drop=True))
    print("Top feature definitions for displayed row:")
    top_idx = np.argsort(np.abs(np.asarray(sv.values[row], dtype=float)))[::-1][: int(max_display)]
    display(feature_defs.iloc[top_idx][["Feature", "SHAP label", "Definition", "Notes"]].reset_index(drop=True))
    shap.plots.waterfall(row_exp, max_display=int(max_display), show=True)
    return sv, X_raw_df, feats, feature_defs


def shap_importance_table(radius_km=40, sample_n=50000, random_state=42):
    model, X_scaled_df, X_raw_df, meta_df, feats, labels, feature_defs, art = load_shap_matrix(
        radius_km=radius_km,
        sample_n=sample_n,
        random_state=random_state,
    )
    sv = _shap_explainer_values(model, X_scaled_df)
    mean_abs = np.abs(sv.values).mean(axis=0)
    imp = pd.DataFrame({
        "Feature": feats,
        "SHAP label": labels,
        "mean_abs_shap": mean_abs,
    }).sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
    imp = imp.merge(feature_defs, on=["Feature", "SHAP label"], how="left")
    return imp, sv, X_raw_df, meta_df, feats, labels, feature_defs


def plot_shap_dependence(
    radius_km=40,
    feature=None,
    sample_n=50000,
    random_state=42,
    color_by="auto",
    alpha=0.30,
    point_size=6,
):
    """Single dependence plot that uses raw feature values on x-axis and SHAP values on y-axis."""
    imp, sv, X_raw_df, meta_df, feats, labels, feature_defs = shap_importance_table(
        radius_km=radius_km,
        sample_n=sample_n,
        random_state=random_state,
    )
    if feature is None:
        feature = imp.iloc[0]["Feature"]

    # Accept original feature name or readable SHAP label.
    if feature in feats:
        i = feats.index(feature)
    elif feature in labels:
        i = labels.index(feature)
    else:
        matches = [j for j, f in enumerate(feats) if str(feature).lower() in f.lower()]
        if not matches:
            raise RuntimeError(f"Feature {feature!r} not found. Use display(shap_importance) for options.")
        i = matches[0]
    feat = feats[i]
    lab = labels[i]

    x = pd.to_numeric(X_raw_df[feat], errors="coerce").to_numpy(float)
    y = np.asarray(sv.values[:, i], dtype=float)
    good = np.isfinite(x) & np.isfinite(y)
    fig, ax = plt.subplots(figsize=(7.5, 5.2))
    ax.scatter(x[good], y[good], s=float(point_size), alpha=float(alpha))
    ax.axhline(0, linewidth=0.8)
    ax.set_xlabel(f"{lab}\noriginal: {feat}")
    ax.set_ylabel("SHAP value")
    ax.set_title(f"SHAP dependence: {lab}")
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()

    print("Feature definition:")
    display(feature_defs.iloc[[i]][["Feature", "SHAP label", "Family", "Units inferred", "Outer neighborhood statistic", "Time/window tokens", "Definition", "Notes"]].reset_index(drop=True))
    return fig, imp, sv


def plot_shap_dependence_for_all_predictors(radius_km=40, sample_n=50000, save_dir=None, show=False, top_n=None, random_state=42):
    """Save dependence plots for all predictors, or top_n most important predictors if top_n is set."""
    imp, sv, X_raw_df, meta_df, feats, labels, feature_defs = shap_importance_table(
        radius_km=radius_km,
        sample_n=sample_n,
        random_state=random_state,
    )
    display(imp.head(50))
    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
    rows = imp if top_n is None else imp.head(int(top_n))
    for _, row in rows.iterrows():
        feat = row["Feature"]
        lab = row["SHAP label"]
        i = feats.index(feat)
        x = pd.to_numeric(X_raw_df[feat], errors="coerce").to_numpy(float)
        y = np.asarray(sv.values[:, i], dtype=float)
        good = np.isfinite(x) & np.isfinite(y)
        fig, ax = plt.subplots(figsize=(7, 4.8))
        ax.scatter(x[good], y[good], s=5, alpha=0.25)
        ax.axhline(0, linewidth=0.8)
        ax.set_xlabel(f"{lab}\noriginal: {feat}")
        ax.set_ylabel("SHAP value")
        ax.set_title(f"SHAP dependence: {lab}")
        ax.grid(True, alpha=0.25)
        plt.tight_layout()
        if save_dir:
            safe = re.sub(r"[^A-Za-z0-9_.-]+", "_", str(feat))[:160]
            fig.savefig(os.path.join(save_dir, f"shap_dependence_{safe}.png"), dpi=150)
        if show:
            plt.show()
        else:
            plt.close(fig)
    return imp, sv


def launch_shap_dependence_browser(radius_km=40, sample_n=50000, random_state=42):
    """Interactive SHAP dependence browser for clicking through predictors."""
    if not HAS_WIDGETS:
        raise RuntimeError("ipywidgets is not available.")
    imp, sv, X_raw_df, meta_df, feats, labels, feature_defs = shap_importance_table(
        radius_km=radius_km,
        sample_n=sample_n,
        random_state=random_state,
    )
    options = [(f"{rank+1:03d}. {row['SHAP label']} | {row['Feature']}", row["Feature"]) for rank, row in imp.iterrows()]
    dd = widgets.Dropdown(options=options, value=imp.iloc[0]["Feature"], description="Feature", layout=widgets.Layout(width="95%"))
    out = widgets.Output()

    def _draw(change=None):
        with out:
            out.clear_output(wait=True)
            feat = dd.value
            i = feats.index(feat)
            x = pd.to_numeric(X_raw_df[feat], errors="coerce").to_numpy(float)
            y = np.asarray(sv.values[:, i], dtype=float)
            good = np.isfinite(x) & np.isfinite(y)
            fig, ax = plt.subplots(figsize=(7.5, 5.2))
            ax.scatter(x[good], y[good], s=6, alpha=0.30)
            ax.axhline(0, linewidth=0.8)
            ax.set_xlabel(f"{labels[i]}\noriginal: {feat}")
            ax.set_ylabel("SHAP value")
            ax.set_title(f"SHAP dependence: {labels[i]}")
            ax.grid(True, alpha=0.25)
            plt.tight_layout()
            plt.show()
            display(feature_defs.iloc[[i]][["Feature", "SHAP label", "Family", "Units inferred", "Definition", "Notes"]].reset_index(drop=True))

    dd.observe(_draw, names="value")
    display(dd, out)
    _draw()
    return imp, sv

print("SHAP feature-definition and labeled plotting utilities loaded.")



# ======================================================================================
# V16 GLOBAL SHAP SUMMARY OVERRIDES
# ======================================================================================
# A waterfall plot is a local explanation for one row.  For all test cases, use global
# SHAP summaries: mean(|SHAP|) bar plot and beeswarm across the sampled 2024-2025 points.


def _shap_values_with_raw_data(sv, X_raw_df, labels):
    """Return a SHAP Explanation with raw feature values attached for plotting/coloring."""
    import shap
    values = np.asarray(sv.values)
    base_values = sv.base_values
    try:
        # base_values can be scalar or per-row.  Preserve whatever SHAP returned.
        raw_exp = shap.Explanation(
            values=values,
            base_values=base_values,
            data=X_raw_df.to_numpy(float),
            feature_names=list(labels),
        )
        return raw_exp
    except Exception:
        return sv


def plot_shap_global_summary(
    radius_km=40,
    sample_n=50000,
    random_state=42,
    max_display=30,
    make_beeswarm=True,
    save_feature_table=True,
):
    """Global SHAP summary across all sampled test-case grid points.

    This is the correct plot for aggregate interpretation across the 2024-2025 test cases.
    It returns the importance table, the SHAP values, raw feature matrix, metadata, and definitions.
    """
    import shap
    imp, sv, X_raw_df, meta_df, feats, labels, feature_defs = shap_importance_table(
        radius_km=radius_km,
        sample_n=sample_n,
        random_state=random_state,
    )
    shap_exp_raw = _shap_values_with_raw_data(sv, X_raw_df, labels)

    n_cases = meta_df["Date"].astype(str).str[:8].nunique() if "Date" in meta_df.columns else np.nan
    print(
        f"Global SHAP sample: rows={len(meta_df):,}, unique dates={n_cases}, "
        f"radius={int(radius_km)} km, predictors={len(feats):,}"
    )
    if "Date" in meta_df.columns:
        print("Date range:", meta_df["Date"].astype(str).min(), "to", meta_df["Date"].astype(str).max())

    # Save the feature dictionary/definitions for writing and interpretation.
    if save_feature_table:
        out_csv = os.path.join(PROJECT_DIR, f"shap_feature_definitions_v33day2valid_r{int(radius_km)}km.csv")
        try:
            feature_defs.to_csv(out_csv, index=False)
            print(f"Saved SHAP feature-definition table: {out_csv}")
        except Exception as exc:
            print("Could not save SHAP feature-definition CSV:", exc)

    display_cols = ["Feature", "SHAP label", "mean_abs_shap", "Family", "Units inferred", "Definition", "Notes"]
    display(imp.head(int(max_display))[display_cols].reset_index(drop=True))

    # Custom global importance bar plot. This always shows readable feature labels.
    top = imp.head(int(max_display)).iloc[::-1].copy()
    fig, ax = plt.subplots(figsize=(11.5, max(6.0, 0.32 * len(top) + 1.2)))
    ax.barh(top["SHAP label"].astype(str), pd.to_numeric(top["mean_abs_shap"], errors="coerce"))
    ax.set_xlabel("mean(|SHAP value|) across sampled test points")
    ax.set_ylabel("Feature")
    ax.set_title(f"Global SHAP importance | r{int(radius_km)} km | sampled test grid points")
    ax.grid(True, axis="x", alpha=0.25)
    plt.tight_layout()
    plt.show()
    plt.close(fig)

    # Beeswarm shows distribution/sign across all sampled test points.  The custom Explanation
    # attaches raw feature values for coloring while keeping readable feature labels.
    if make_beeswarm:
        try:
            shap.plots.beeswarm(shap_exp_raw, max_display=int(max_display), show=True)
        except Exception as exc:
            print("SHAP beeswarm failed, but the global bar plot and tables above are still valid:", exc)

    return imp, shap_exp_raw, X_raw_df, meta_df, feats, labels, feature_defs


def plot_shap_waterfall_optional(radius_km=40, row=0, sample_n=50000, max_display=25, random_state=42):
    """Optional local SHAP waterfall for one sampled grid point.

    This is intentionally not the main SHAP product because waterfall plots are local,
    not aggregate across cases.
    """
    print(
        "NOTE: SHAP waterfall is a LOCAL explanation for one sampled grid point only. "
        "Use plot_shap_global_summary(...) for all test cases."
    )
    return plot_shap_waterfall(
        radius_km=radius_km,
        row=row,
        sample_n=sample_n,
        max_display=max_display,
        random_state=random_state,
    )

print("V16 global SHAP summary utilities loaded: use plot_shap_global_summary(...) for all sampled test cases.")




In [ ]:
paper_verification_results = run_final_bs_ets_verification_plots()

## 13. SHAP feature-definition table

Run this before the SHAP plots if you want a searchable table of model predictors and their inferred meanings.



In [ ]:
SHAP_RADIUS_KM = 100
SHAP_MODEL_LABEL = "r100km"  # trained Day-2 R100 target member
SHAP_SAMPLE_N = 50000
SHAP_RANDOM_STATE = 42
SHAP_FEATURE_DEFINITION_MAX_ROWS = 300

try:
    art_for_defs = find_artifacts_for_radius(SHAP_RADIUS_KM, model_label=SHAP_MODEL_LABEL)
    shap_feature_names_for_defs = _load_feature_names(art_for_defs["features_path"])
    shap_feature_definitions = _display_feature_definition_table(
        shap_feature_names_for_defs,
        radius_km=SHAP_RADIUS_KM,
        max_rows=SHAP_FEATURE_DEFINITION_MAX_ROWS,
    )
except Exception as exc:
    print("Could not build SHAP feature-definition table:", exc)



## 14. Global SHAP summary across all sampled test cases

A waterfall plot is a **single-row/local** explanation, so it is not the right primary plot for interpreting the model across all test cases. This section now computes global SHAP summaries across the sampled 2024-2025 test grid points:

- mean absolute SHAP importance table
- global SHAP bar plot
- SHAP beeswarm distribution plot
- feature-definition CSV/table

The optional local waterfall is left available only as a diagnostic and is turned off by default.



In [ ]:
# ======================================================================================
# PAPER-READY GLOBAL SHAP SUMMARY OVERRIDE
#
# Paste this BEFORE the cell that calls plot_shap_global_summary(...)
#
# Changes:
#   - font size 50 everywhere
#   - no plot titles
#   - wrapped y-axis labels so the figure is not extremely wide
#   - saves feature-definition table as before
#   - formats both the mean(|SHAP|) bar plot and optional beeswarm
# ======================================================================================

import os
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SHAP_PAPER_FONT_SIZE = 50
SHAP_PAPER_LABEL_WRAP_WIDTH = 26
SHAP_PAPER_BAR_FIGSIZE = (24, 20)
SHAP_PAPER_BEESWARM_FIGSIZE = (26, 20)
SHAP_PAPER_DPI = 300
SHAP_PAPER_SAVE_FIGS = True

if "PROJECT_DIR" in globals():
    SHAP_PAPER_OUTDIR = os.path.join(PROJECT_DIR, "paper_shap_figures_v33day2valid")
else:
    SHAP_PAPER_OUTDIR = os.path.join(os.getcwd(), "paper_shap_figures_v33day2valid")


def _wrap_shap_label(label, width=SHAP_PAPER_LABEL_WRAP_WIDTH):
    """
    Wrap long SHAP feature labels onto multiple lines.
    """
    label = str(label)

    # Prefer breaking after common separators.
    label = label.replace(" within ", "\nwithin ")
    label = label.replace(" of ", " of ")
    label = label.replace(" / ", "/")

    parts = []
    for chunk in label.split("\n"):
        wrapped = textwrap.wrap(
            chunk,
            width=int(width),
            break_long_words=False,
            break_on_hyphens=False,
        )
        if wrapped:
            parts.extend(wrapped)
        else:
            parts.append(chunk)

    return "\n".join(parts)


def _format_shap_axis_paper(ax):
    """
    Apply paper formatting to a matplotlib axis.
    """
    ax.set_title("")

    ax.tick_params(axis="both", labelsize=SHAP_PAPER_FONT_SIZE)

    ax.xaxis.label.set_size(SHAP_PAPER_FONT_SIZE)
    ax.yaxis.label.set_size(SHAP_PAPER_FONT_SIZE)

    for tick in ax.get_xticklabels():
        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)

    for tick in ax.get_yticklabels():
        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)

    ax.grid(True, axis="x", alpha=0.25)


def _shap_values_with_raw_data_paper(sv, X_raw_df, labels):
    """
    Return SHAP Explanation with raw feature values and wrapped feature labels.
    """
    import shap

    values = np.asarray(sv.values)
    base_values = sv.base_values
    wrapped_labels = [_wrap_shap_label(lab) for lab in labels]

    try:
        return shap.Explanation(
            values=values,
            base_values=base_values,
            data=X_raw_df.to_numpy(float),
            feature_names=wrapped_labels,
        )
    except Exception:
        try:
            sv.feature_names = wrapped_labels
        except Exception:
            pass
        return sv


def plot_shap_global_summary(
    radius_km=40,
    sample_n=50000,
    random_state=42,
    max_display=30,
    make_beeswarm=True,
    save_feature_table=True,
):
    """
    Paper-ready global SHAP summary across sampled test-case grid points.

    Produces:
      1. mean(|SHAP|) bar plot with wrapped feature labels
      2. optional SHAP beeswarm with wrapped feature labels

    This intentionally removes titles and uses large font sizes.
    """
    import shap

    os.makedirs(SHAP_PAPER_OUTDIR, exist_ok=True)

    imp, sv, X_raw_df, meta_df, feats, labels, feature_defs = shap_importance_table(
        radius_km=radius_km,
        sample_n=sample_n,
        random_state=random_state,
    )

    wrapped_labels = [_wrap_shap_label(lab) for lab in labels]

    shap_exp_raw = _shap_values_with_raw_data_paper(
        sv,
        X_raw_df,
        labels,
    )

    n_cases = meta_df["Date"].astype(str).str[:8].nunique() if "Date" in meta_df.columns else np.nan

    print(
        f"Global SHAP sample: rows={len(meta_df):,}, unique dates={n_cases}, "
        f"radius={int(radius_km)} km, predictors={len(feats):,}"
    )

    if "Date" in meta_df.columns:
        print("Date range:", meta_df["Date"].astype(str).min(), "to", meta_df["Date"].astype(str).max())

    # Save feature dictionary/definitions
    if save_feature_table:
        out_csv = os.path.join(
            PROJECT_DIR if "PROJECT_DIR" in globals() else os.getcwd(),
            f"shap_feature_definitions_v33day2valid_r{int(radius_km)}km.csv",
        )
        try:
            feature_defs.to_csv(out_csv, index=False)
            print(f"Saved SHAP feature-definition table: {out_csv}")
        except Exception as exc:
            print("Could not save SHAP feature-definition CSV:", exc)

    display_cols = [
        "Feature",
        "SHAP label",
        "mean_abs_shap",
        "Family",
        "Units inferred",
        "Definition",
        "Notes",
    ]

    display(
        imp.head(int(max_display))[display_cols]
        .reset_index(drop=True)
    )

    # ----------------------------------------------------------------------------------
    # Mean absolute SHAP bar plot
    # ----------------------------------------------------------------------------------
    top = imp.head(int(max_display)).iloc[::-1].copy()
    top["Wrapped SHAP label"] = top["SHAP label"].map(_wrap_shap_label)

    with plt.rc_context({
        "font.size": SHAP_PAPER_FONT_SIZE,
        "axes.labelsize": SHAP_PAPER_FONT_SIZE,
        "xtick.labelsize": SHAP_PAPER_FONT_SIZE,
        "ytick.labelsize": SHAP_PAPER_FONT_SIZE,
    }):
        fig, ax = plt.subplots(figsize=SHAP_PAPER_BAR_FIGSIZE)

        ax.barh(
            top["Wrapped SHAP label"].astype(str),
            pd.to_numeric(top["mean_abs_shap"], errors="coerce"),
        )

        ax.set_xlabel("mean(|SHAP value|)")
        ax.set_ylabel("")
        ax.set_title("")

        _format_shap_axis_paper(ax)

        plt.tight_layout()

        if SHAP_PAPER_SAVE_FIGS:
            out_png = os.path.join(
                SHAP_PAPER_OUTDIR,
                f"global_shap_importance_r{int(radius_km)}km.png",
            )
            out_pdf = os.path.join(
                SHAP_PAPER_OUTDIR,
                f"global_shap_importance_r{int(radius_km)}km.pdf",
            )

            fig.savefig(out_png, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")
            fig.savefig(out_pdf, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")

            print("Saved:", out_png)
            print("Saved:", out_pdf)

        plt.show()
        plt.close(fig)

    # ----------------------------------------------------------------------------------
    # Beeswarm plot
    # ----------------------------------------------------------------------------------
    if make_beeswarm:
        try:
            with plt.rc_context({
                "font.size": SHAP_PAPER_FONT_SIZE,
                "axes.labelsize": SHAP_PAPER_FONT_SIZE,
                "xtick.labelsize": SHAP_PAPER_FONT_SIZE,
                "ytick.labelsize": SHAP_PAPER_FONT_SIZE,
            }):
                shap.plots.beeswarm(
                    shap_exp_raw,
                    max_display=int(max_display),
                    show=False,
                )

                fig = plt.gcf()
                fig.set_size_inches(SHAP_PAPER_BEESWARM_FIGSIZE)

                for ax in fig.axes:
                    ax.set_title("")
                    ax.tick_params(axis="both", labelsize=SHAP_PAPER_FONT_SIZE)

                    ax.xaxis.label.set_size(SHAP_PAPER_FONT_SIZE)
                    ax.yaxis.label.set_size(SHAP_PAPER_FONT_SIZE)

                    for tick in ax.get_xticklabels():
                        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)

                    for tick in ax.get_yticklabels():
                        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)

                    # Remove default title if SHAP inserted one
                    ax.title.set_text("")

                plt.tight_layout()

                if SHAP_PAPER_SAVE_FIGS:
                    out_png = os.path.join(
                        SHAP_PAPER_OUTDIR,
                        f"global_shap_beeswarm_r{int(radius_km)}km.png",
                    )
                    out_pdf = os.path.join(
                        SHAP_PAPER_OUTDIR,
                        f"global_shap_beeswarm_r{int(radius_km)}km.pdf",
                    )

                    fig.savefig(out_png, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")
                    fig.savefig(out_pdf, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")

                    print("Saved:", out_png)
                    print("Saved:", out_pdf)

                plt.show()
                plt.close(fig)

        except Exception as exc:
            print("SHAP beeswarm failed, but the global bar plot and tables above are still valid:", exc)

    return imp, shap_exp_raw, X_raw_df, meta_df, feats, wrapped_labels, feature_defs


print("Paper-ready plot_shap_global_summary override loaded.")

In [ ]:
# ======================================================================================
# PATCH: PAPER SHAP PLOTS WITH DYNAMIC HEIGHT + DYNAMIC BAR SPACING
#
# Paste AFTER the previous paper SHAP override and BEFORE rerunning plot_shap_global_summary.
#
# Fixes:
#   - labels no longer overlap after wrapping
#   - bar spacing increases for multi-line labels
#   - figure size automatically grows based on wrapped label line count
#   - title removed
#   - font size 50 everywhere
# ======================================================================================

import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SHAP_PAPER_FONT_SIZE = 50

# Smaller width = more line breaks. Increase if too many lines.
SHAP_PAPER_LABEL_WRAP_WIDTH = 24

# Dynamic figure sizing controls
SHAP_PAPER_BAR_WIDTH = 34
SHAP_PAPER_HEIGHT_PER_LABEL_LINE = 1.15
SHAP_PAPER_MIN_BAR_HEIGHT = 22
SHAP_PAPER_EXTRA_VERTICAL_SPACE = 3.5

SHAP_PAPER_BEESWARM_WIDTH = 36
SHAP_PAPER_MIN_BEESWARM_HEIGHT = 24
SHAP_PAPER_BEESWARM_HEIGHT_PER_LABEL_LINE = 1.45
SHAP_PAPER_BEESWARM_EXTRA_VERTICAL_SPACE = 5.0

SHAP_PAPER_DPI = 300
SHAP_PAPER_SAVE_FIGS = True

if "PROJECT_DIR" in globals():
    SHAP_PAPER_OUTDIR = os.path.join(PROJECT_DIR, "paper_shap_figures_v33day2valid")
else:
    SHAP_PAPER_OUTDIR = os.path.join(os.getcwd(), "paper_shap_figures_v33day2valid")


def _wrap_shap_label(label, width=SHAP_PAPER_LABEL_WRAP_WIDTH):
    label = str(label)

    # Add intentional breaks at common semantic separators.
    label = label.replace(" within ", "\nwithin ")
    label = label.replace(" over ", "\nover ")
    label = label.replace(" radius ", "\nradius ")
    label = label.replace(" neighborhood ", "\nneighborhood ")

    wrapped_lines = []

    for chunk in label.split("\n"):
        lines = textwrap.wrap(
            chunk,
            width=int(width),
            break_long_words=False,
            break_on_hyphens=False,
        )
        if lines:
            wrapped_lines.extend(lines)
        else:
            wrapped_lines.append(chunk)

    return "\n".join(wrapped_lines)


def _count_label_lines(label):
    return max(1, str(label).count("\n") + 1)


def _dynamic_y_positions_for_wrapped_labels(labels):
    """
    Create unevenly spaced y positions so multi-line labels get extra vertical room.
    """
    line_counts = np.array([_count_label_lines(lab) for lab in labels], dtype=float)

    # Each label gets baseline room plus extra room for additional wrapped lines.
    spacing = 1.0 + 0.48 * np.maximum(line_counts - 1.0, 0.0)

    y = np.zeros(len(labels), dtype=float)
    for i in range(1, len(labels)):
        y[i] = y[i - 1] + spacing[i - 1]

    return y, spacing, line_counts


def _dynamic_fig_height_from_labels(labels, min_height, per_line, extra):
    total_lines = sum(_count_label_lines(lab) for lab in labels)
    return max(float(min_height), float(extra) + float(per_line) * float(total_lines))


def _paper_format_axis(ax):
    ax.set_title("")
    ax.tick_params(axis="both", labelsize=SHAP_PAPER_FONT_SIZE)
    ax.xaxis.label.set_size(SHAP_PAPER_FONT_SIZE)
    ax.yaxis.label.set_size(SHAP_PAPER_FONT_SIZE)

    for tick in ax.get_xticklabels():
        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)

    for tick in ax.get_yticklabels():
        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)
        tick.set_linespacing(0.95)

    ax.grid(True, axis="x", alpha=0.25)
    ax.grid(False, axis="y")


def _shap_values_with_wrapped_feature_names(sv, X_raw_df, labels):
    import shap

    wrapped_labels = [_wrap_shap_label(lab) for lab in labels]

    try:
        return shap.Explanation(
            values=np.asarray(sv.values),
            base_values=sv.base_values,
            data=X_raw_df.to_numpy(float),
            feature_names=wrapped_labels,
        )
    except Exception:
        try:
            sv.feature_names = wrapped_labels
        except Exception:
            pass
        return sv


def plot_shap_global_summary(
    radius_km=40,
    sample_n=50000,
    random_state=42,
    max_display=30,
    make_beeswarm=True,
    save_feature_table=True,
):
    import shap

    os.makedirs(SHAP_PAPER_OUTDIR, exist_ok=True)

    imp, sv, X_raw_df, meta_df, feats, labels, feature_defs = shap_importance_table(
        radius_km=radius_km,
        sample_n=sample_n,
        random_state=random_state,
    )

    n_cases = meta_df["Date"].astype(str).str[:8].nunique() if "Date" in meta_df.columns else np.nan
    print(
        f"Global SHAP sample: rows={len(meta_df):,}, unique dates={n_cases}, "
        f"radius={int(radius_km)} km, predictors={len(feats):,}"
    )

    if "Date" in meta_df.columns:
        print("Date range:", meta_df["Date"].astype(str).min(), "to", meta_df["Date"].astype(str).max())

    if save_feature_table:
        out_csv = os.path.join(
            PROJECT_DIR if "PROJECT_DIR" in globals() else os.getcwd(),
            f"shap_feature_definitions_v33day2valid_r{int(radius_km)}km.csv",
        )
        try:
            feature_defs.to_csv(out_csv, index=False)
            print(f"Saved SHAP feature-definition table: {out_csv}")
        except Exception as exc:
            print("Could not save SHAP feature-definition CSV:", exc)

    display_cols = [
        "Feature",
        "SHAP label",
        "mean_abs_shap",
        "Family",
        "Units inferred",
        "Definition",
        "Notes",
    ]

    display(
        imp.head(int(max_display))[display_cols]
        .reset_index(drop=True)
    )

    # ==================================================================================
    # Mean absolute SHAP bar plot with dynamic y spacing
    # ==================================================================================
    top = imp.head(int(max_display)).iloc[::-1].copy()
    top["Wrapped SHAP label"] = top["SHAP label"].map(_wrap_shap_label)

    labels_wrapped = top["Wrapped SHAP label"].astype(str).tolist()
    values = pd.to_numeric(top["mean_abs_shap"], errors="coerce").to_numpy(float)

    y_pos, y_spacing, line_counts = _dynamic_y_positions_for_wrapped_labels(labels_wrapped)

    fig_height = _dynamic_fig_height_from_labels(
        labels_wrapped,
        min_height=SHAP_PAPER_MIN_BAR_HEIGHT,
        per_line=SHAP_PAPER_HEIGHT_PER_LABEL_LINE,
        extra=SHAP_PAPER_EXTRA_VERTICAL_SPACE,
    )

    with plt.rc_context({
        "font.size": SHAP_PAPER_FONT_SIZE,
        "axes.labelsize": SHAP_PAPER_FONT_SIZE,
        "xtick.labelsize": SHAP_PAPER_FONT_SIZE,
        "ytick.labelsize": SHAP_PAPER_FONT_SIZE,
    }):
        fig, ax = plt.subplots(figsize=(SHAP_PAPER_BAR_WIDTH, fig_height))

        # Bar height scales with available label spacing.
        bar_heights = np.minimum(0.82, 0.72 * y_spacing)

        ax.barh(
            y_pos,
            values,
            height=bar_heights,
        )

        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels_wrapped)

        ax.set_xlabel("mean(|SHAP value|)")
        ax.set_ylabel("")
        ax.set_title("")

        # Add a bit of vertical padding so top/bottom labels do not clip.
        ax.set_ylim(y_pos[0] - 0.8, y_pos[-1] + 0.8)

        _paper_format_axis(ax)

        # More left margin for large multi-line labels.
        fig.subplots_adjust(left=0.43, right=0.98, bottom=0.12, top=0.98)

        if SHAP_PAPER_SAVE_FIGS:
            out_png = os.path.join(
                SHAP_PAPER_OUTDIR,
                f"global_shap_importance_r{int(radius_km)}km.png",
            )
            out_pdf = os.path.join(
                SHAP_PAPER_OUTDIR,
                f"global_shap_importance_r{int(radius_km)}km.pdf",
            )

            fig.savefig(out_png, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")
            fig.savefig(out_pdf, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")

            print("Saved:", out_png)
            print("Saved:", out_pdf)

        plt.show()
        plt.close(fig)

    # ==================================================================================
    # Beeswarm plot with larger dynamic height
    # ==================================================================================
    shap_exp_raw = _shap_values_with_wrapped_feature_names(
        sv,
        X_raw_df,
        labels,
    )

    if make_beeswarm:
        try:
            # Use top labels to estimate required figure height.
            beeswarm_labels = [_wrap_shap_label(lab) for lab in imp.head(int(max_display))["SHAP label"].tolist()]

            beeswarm_height = _dynamic_fig_height_from_labels(
                beeswarm_labels,
                min_height=SHAP_PAPER_MIN_BEESWARM_HEIGHT,
                per_line=SHAP_PAPER_BEESWARM_HEIGHT_PER_LABEL_LINE,
                extra=SHAP_PAPER_BEESWARM_EXTRA_VERTICAL_SPACE,
            )

            with plt.rc_context({
                "font.size": SHAP_PAPER_FONT_SIZE,
                "axes.labelsize": SHAP_PAPER_FONT_SIZE,
                "xtick.labelsize": SHAP_PAPER_FONT_SIZE,
                "ytick.labelsize": SHAP_PAPER_FONT_SIZE,
            }):
                shap.plots.beeswarm(
                    shap_exp_raw,
                    max_display=int(max_display),
                    show=False,
                )

                fig = plt.gcf()
                fig.set_size_inches(SHAP_PAPER_BEESWARM_WIDTH, beeswarm_height)

                for ax in fig.axes:
                    ax.set_title("")
                    ax.tick_params(axis="both", labelsize=SHAP_PAPER_FONT_SIZE)
                    ax.xaxis.label.set_size(SHAP_PAPER_FONT_SIZE)
                    ax.yaxis.label.set_size(SHAP_PAPER_FONT_SIZE)

                    for tick in ax.get_xticklabels():
                        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)

                    for tick in ax.get_yticklabels():
                        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)
                        tick.set_linespacing(0.95)

                fig.subplots_adjust(left=0.43, right=0.97, bottom=0.12, top=0.98)

                if SHAP_PAPER_SAVE_FIGS:
                    out_png = os.path.join(
                        SHAP_PAPER_OUTDIR,
                        f"global_shap_beeswarm_r{int(radius_km)}km.png",
                    )
                    out_pdf = os.path.join(
                        SHAP_PAPER_OUTDIR,
                        f"global_shap_beeswarm_r{int(radius_km)}km.pdf",
                    )

                    fig.savefig(out_png, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")
                    fig.savefig(out_pdf, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")

                    print("Saved:", out_png)
                    print("Saved:", out_pdf)

                plt.show()
                plt.close(fig)

        except Exception as exc:
            print("SHAP beeswarm failed, but the global bar plot and tables above are still valid:", exc)

    wrapped_labels_return = [_wrap_shap_label(lab) for lab in labels]

    return imp, shap_exp_raw, X_raw_df, meta_df, feats, wrapped_labels_return, feature_defs


print("Dynamic-spacing paper SHAP override loaded.")

In [ ]:
# ======================================================================================
# PATCH: SHAP PAPER FIGURE FONT 40 + LARGER FIGURE + NO TRUNCATED LABELS
#
# Paste AFTER the previous dynamic-spacing SHAP override, then rerun:
#   plot_shap_global_summary(...)
#
# Fixes:
#   - font size reduced from 50 to 40
#   - figure size increased slightly
#   - SHAP labels rebuilt from original feature names without truncating to "..."
# ======================================================================================

import os
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------------------
# Updated controls
# --------------------------------------------------------------------------------------
SHAP_PAPER_FONT_SIZE = 40

# Smaller value = more wrapping. Larger value = fewer lines.
SHAP_PAPER_LABEL_WRAP_WIDTH = 28

# Increase plot size slightly compared to previous version
SHAP_PAPER_BAR_WIDTH = 38
SHAP_PAPER_MIN_BAR_HEIGHT = 26
SHAP_PAPER_HEIGHT_PER_LABEL_LINE = 1.45
SHAP_PAPER_EXTRA_VERTICAL_SPACE = 5.0

SHAP_PAPER_BEESWARM_WIDTH = 40
SHAP_PAPER_MIN_BEESWARM_HEIGHT = 28
SHAP_PAPER_BEESWARM_HEIGHT_PER_LABEL_LINE = 1.55
SHAP_PAPER_BEESWARM_EXTRA_VERTICAL_SPACE = 6.0

SHAP_PAPER_DPI = 300
SHAP_PAPER_SAVE_FIGS = True

# Set this high so labels are not truncated with "..."
SHAP_PAPER_LABEL_MAXLEN = 1000

# Also override the global label max if your earlier label function uses it
SHAP_FEATURE_LABEL_MAXLEN = SHAP_PAPER_LABEL_MAXLEN

if "PROJECT_DIR" in globals():
    SHAP_PAPER_OUTDIR = os.path.join(PROJECT_DIR, "paper_shap_figures_v33day2valid")
else:
    SHAP_PAPER_OUTDIR = os.path.join(os.getcwd(), "paper_shap_figures_v33day2valid")


def _make_untruncated_shap_label(feature, radius_km=None):
    """
    Rebuild readable SHAP labels from original feature names without truncation.

    This avoids labels ending in '...' or '…' from earlier make_feature_label calls.
    """
    feature = str(feature)

    if "make_feature_label" in globals():
        try:
            return make_feature_label(
                feature,
                radius_km=radius_km,
                max_len=SHAP_PAPER_LABEL_MAXLEN,
            )
        except TypeError:
            # If older make_feature_label does not accept max_len
            lab = make_feature_label(feature, radius_km=radius_km)
            return str(lab).replace("...", "").replace("…", "")

    # Fallback if make_feature_label is not available
    label = feature.replace("_", " ")
    if radius_km is not None and f"r{int(radius_km)}" not in label.lower():
        label = f"{label} within r{int(radius_km)} km"

    return label.replace("...", "").replace("…", "")


def _wrap_shap_label(label, width=SHAP_PAPER_LABEL_WRAP_WIDTH):
    """
    Wrap long labels without truncating.
    """
    label = str(label).replace("...", "").replace("…", "")

    # Add useful semantic breaks before regular wrapping.
    replacements = {
        " within ": "\nwithin ",
        " over ": "\nover ",
        " radius ": "\nradius ",
        " neighborhood ": "\nneighborhood ",
        " of the ": " of the ",
        " Accumulated ": "\nAccumulated ",
        " Flash Flood Guidance": "\nFlash Flood Guidance",
        " QPF/APCP": "\nQPF/APCP",
    }

    for old, new in replacements.items():
        label = label.replace(old, new)

    wrapped_lines = []

    for chunk in label.split("\n"):
        lines = textwrap.wrap(
            chunk,
            width=int(width),
            break_long_words=False,
            break_on_hyphens=False,
        )
        if lines:
            wrapped_lines.extend(lines)
        else:
            wrapped_lines.append(chunk)

    return "\n".join(wrapped_lines)


def _count_label_lines(label):
    return max(1, str(label).count("\n") + 1)


def _dynamic_y_positions_for_wrapped_labels(labels):
    """
    Uneven y spacing so multi-line labels get more vertical room.
    """
    line_counts = np.array([_count_label_lines(lab) for lab in labels], dtype=float)

    # Extra spacing for additional wrapped lines
    spacing = 0.50 + 0.30 * np.maximum(line_counts - 1.0, 0.0)

    y = np.zeros(len(labels), dtype=float)
    for i in range(1, len(labels)):
        y[i] = y[i - 1] + spacing[i - 1]

    return y, spacing, line_counts


def _dynamic_fig_height_from_labels(labels, min_height, per_line, extra):
    total_lines = sum(_count_label_lines(lab) for lab in labels)
    return max(float(min_height), float(extra) + float(per_line) * float(total_lines))


def _paper_format_axis(ax):
    ax.set_title("")
    ax.tick_params(axis="both", labelsize=SHAP_PAPER_FONT_SIZE)
    ax.xaxis.label.set_size(SHAP_PAPER_FONT_SIZE)
    ax.yaxis.label.set_size(SHAP_PAPER_FONT_SIZE)

    for tick in ax.get_xticklabels():
        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)

    for tick in ax.get_yticklabels():
        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)
        tick.set_linespacing(0.95)

    ax.grid(True, axis="x", alpha=0.25)
    ax.grid(False, axis="y")


def _shap_values_with_wrapped_feature_names(sv, X_raw_df, labels):
    import shap

    wrapped_labels = [_wrap_shap_label(lab) for lab in labels]

    try:
        return shap.Explanation(
            values=np.asarray(sv.values),
            base_values=sv.base_values,
            data=X_raw_df.to_numpy(float),
            feature_names=wrapped_labels,
        )
    except Exception:
        try:
            sv.feature_names = wrapped_labels
        except Exception:
            pass
        return sv


def plot_shap_global_summary(
    radius_km=40,
    sample_n=50000,
    random_state=42,
    max_display=30,
    make_beeswarm=True,
    save_feature_table=True,
):
    import shap

    os.makedirs(SHAP_PAPER_OUTDIR, exist_ok=True)

    imp, sv, X_raw_df, meta_df, feats, old_labels, feature_defs = shap_importance_table(
        radius_km=radius_km,
        sample_n=sample_n,
        random_state=random_state,
    )

    # ------------------------------------------------------------------
    # Rebuild untruncated readable labels from original feature names.
    # This is the key fix for labels ending in "r100..." or similar.
    # ------------------------------------------------------------------
    labels = [
        _make_untruncated_shap_label(feat, radius_km=radius_km)
        for feat in feats
    ]

    # Preserve uniqueness if two readable labels collide
    seen = {}
    labels_unique = []
    for lab in labels:
        if lab in seen:
            seen[lab] += 1
            labels_unique.append(f"{lab} [{seen[lab]}]")
        else:
            seen[lab] = 1
            labels_unique.append(lab)

    # Replace labels in importance table
    imp = imp.copy()
    label_map = dict(zip(feats, labels_unique))
    imp["SHAP label"] = imp["Feature"].map(label_map).fillna(imp["SHAP label"].astype(str))

    feature_defs = feature_defs.copy()
    if "Feature" in feature_defs.columns:
        feature_defs["SHAP label"] = feature_defs["Feature"].map(label_map).fillna(
            feature_defs.get("SHAP label", "").astype(str)
        )

    n_cases = meta_df["Date"].astype(str).str[:8].nunique() if "Date" in meta_df.columns else np.nan
    print(
        f"Global SHAP sample: rows={len(meta_df):,}, unique dates={n_cases}, "
        f"radius={int(radius_km)} km, predictors={len(feats):,}"
    )

    if "Date" in meta_df.columns:
        print("Date range:", meta_df["Date"].astype(str).min(), "to", meta_df["Date"].astype(str).max())

    if save_feature_table:
        out_csv = os.path.join(
            PROJECT_DIR if "PROJECT_DIR" in globals() else os.getcwd(),
            f"shap_feature_definitions_v33day2valid_r{int(radius_km)}km.csv",
        )
        try:
            feature_defs.to_csv(out_csv, index=False)
            print(f"Saved SHAP feature-definition table: {out_csv}")
        except Exception as exc:
            print("Could not save SHAP feature-definition CSV:", exc)

    display_cols = [
        "Feature",
        "SHAP label",
        "mean_abs_shap",
        "Family",
        "Units inferred",
        "Definition",
        "Notes",
    ]

    display(
        imp.head(int(max_display))[display_cols]
        .reset_index(drop=True)
    )

    # ==================================================================================
    # Mean absolute SHAP bar plot
    # ==================================================================================
    top = imp.head(int(max_display)).iloc[::-1].copy()
    top["Wrapped SHAP label"] = top["SHAP label"].map(_wrap_shap_label)

    labels_wrapped = top["Wrapped SHAP label"].astype(str).tolist()
    values = pd.to_numeric(top["mean_abs_shap"], errors="coerce").to_numpy(float)

    y_pos, y_spacing, line_counts = _dynamic_y_positions_for_wrapped_labels(labels_wrapped)

    fig_height = _dynamic_fig_height_from_labels(
        labels_wrapped,
        min_height=SHAP_PAPER_MIN_BAR_HEIGHT,
        per_line=SHAP_PAPER_HEIGHT_PER_LABEL_LINE,
        extra=SHAP_PAPER_EXTRA_VERTICAL_SPACE,
    )

    with plt.rc_context({
        "font.size": SHAP_PAPER_FONT_SIZE,
        "axes.labelsize": SHAP_PAPER_FONT_SIZE,
        "xtick.labelsize": SHAP_PAPER_FONT_SIZE,
        "ytick.labelsize": SHAP_PAPER_FONT_SIZE,
    }):
        fig, ax = plt.subplots(figsize=(SHAP_PAPER_BAR_WIDTH, fig_height))

        bar_heights = np.minimum(0.82, 0.72 * y_spacing)

        ax.barh(
            y_pos,
            values,
            height=bar_heights,
        )

        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels_wrapped)

        ax.set_xlabel("mean(|SHAP value|)")
        ax.set_ylabel("")
        ax.set_title("")

        ax.set_ylim(y_pos[0] - 0.9, y_pos[-1] + 0.9)

        _paper_format_axis(ax)

        # Reduced font allows slightly smaller left margin than before
        fig.subplots_adjust(left=0.38, right=0.98, bottom=0.12, top=0.98)

        if SHAP_PAPER_SAVE_FIGS:
            out_png = os.path.join(
                SHAP_PAPER_OUTDIR,
                f"global_shap_importance_r{int(radius_km)}km.png",
            )
            out_pdf = os.path.join(
                SHAP_PAPER_OUTDIR,
                f"global_shap_importance_r{int(radius_km)}km.pdf",
            )

            fig.savefig(out_png, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")
            fig.savefig(out_pdf, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")

            print("Saved:", out_png)
            print("Saved:", out_pdf)

        plt.show()
        plt.close(fig)

    # ==================================================================================
    # Beeswarm plot
    # ==================================================================================
    shap_exp_raw = _shap_values_with_wrapped_feature_names(
        sv,
        X_raw_df,
        labels_unique,
    )

    if make_beeswarm:
        try:
            beeswarm_labels = [
                _wrap_shap_label(lab)
                for lab in imp.head(int(max_display))["SHAP label"].tolist()
            ]

            beeswarm_height = _dynamic_fig_height_from_labels(
                beeswarm_labels,
                min_height=SHAP_PAPER_MIN_BEESWARM_HEIGHT,
                per_line=SHAP_PAPER_BEESWARM_HEIGHT_PER_LABEL_LINE,
                extra=SHAP_PAPER_BEESWARM_EXTRA_VERTICAL_SPACE,
            )

            with plt.rc_context({
                "font.size": SHAP_PAPER_FONT_SIZE,
                "axes.labelsize": SHAP_PAPER_FONT_SIZE,
                "xtick.labelsize": SHAP_PAPER_FONT_SIZE,
                "ytick.labelsize": SHAP_PAPER_FONT_SIZE,
            }):
                shap.plots.beeswarm(
                    shap_exp_raw,
                    max_display=int(max_display),
                    show=False,
                )

                fig = plt.gcf()
                fig.set_size_inches(SHAP_PAPER_BEESWARM_WIDTH, beeswarm_height)

                for ax in fig.axes:
                    ax.set_title("")
                    ax.tick_params(axis="both", labelsize=SHAP_PAPER_FONT_SIZE)
                    ax.xaxis.label.set_size(SHAP_PAPER_FONT_SIZE)
                    ax.yaxis.label.set_size(SHAP_PAPER_FONT_SIZE)

                    for tick in ax.get_xticklabels():
                        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)

                    for tick in ax.get_yticklabels():
                        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)
                        tick.set_linespacing(0.95)

                fig.subplots_adjust(left=0.38, right=0.97, bottom=0.12, top=0.98)

                if SHAP_PAPER_SAVE_FIGS:
                    out_png = os.path.join(
                        SHAP_PAPER_OUTDIR,
                        f"global_shap_beeswarm_r{int(radius_km)}km.png",
                    )
                    out_pdf = os.path.join(
                        SHAP_PAPER_OUTDIR,
                        f"global_shap_beeswarm_r{int(radius_km)}km.pdf",
                    )

                    fig.savefig(out_png, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")
                    fig.savefig(out_pdf, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")

                    print("Saved:", out_png)
                    print("Saved:", out_pdf)

                plt.show()
                plt.close(fig)

        except Exception as exc:
            print("SHAP beeswarm failed, but the global bar plot and tables above are still valid:", exc)

    wrapped_labels_return = [_wrap_shap_label(lab) for lab in labels_unique]

    return imp, shap_exp_raw, X_raw_df, meta_df, feats, wrapped_labels_return, feature_defs


print("Updated SHAP paper override loaded: font=40, larger figure, untruncated labels.")

In [ ]:
# ======================================================================================
# PAPER-READY GLOBAL SHAP SUMMARY OVERRIDE
#
# Full replacement block.
#
# Changes:
#   - font size 50 everywhere
#   - no plot titles
#   - targeted plain-English feature labels
#   - fixes duplicate "6-h 6-h" wording
#   - uses "Std" instead of "SD"
#   - uses "within 100km" instead of "within r100km"
#   - removes duplicate-label suffixes such as [2]
#   - dynamic figure height and dynamic bar spacing
#   - feature-definition table includes original SHAP label + simplified paper label
#   - saves mean(|SHAP|) bar plot and optional beeswarm
#
# Examples:
#   Max(Forecast QPF/APCP Max 6h Window 0to24h to Guidance FFG 06h Ratio R100km)
#   within r100km
#
# becomes:
#   Max 6-h QPF: 6-h FFG ratio within 100km
#
#   Std(Guide FFG Min 1 3 6 12 24h mm R100km) within r100km
#
# becomes:
#   Std of 1-, 3-, 6-, 12-, and 24-h FFGs within 100km
#
#   Max(APCP RunTotal 0 06 12 18 24h Std R100km) within r100km
#
# becomes:
#   Max Std between 0-6, 0-12, 0-18, and 0-24 h QPFs within 100km
# ======================================================================================

import os
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# ======================================================================================
# PAPER CONTROLS
# ======================================================================================

SHAP_PAPER_FONT_SIZE = 50

# Smaller = more wrapping. Larger = fewer line breaks.
SHAP_PAPER_LABEL_WRAP_WIDTH = 38

# Dynamic figure sizing controls
SHAP_PAPER_BAR_WIDTH = 34
SHAP_PAPER_MIN_BAR_HEIGHT = 22
SHAP_PAPER_HEIGHT_PER_LABEL_LINE = 1.25
SHAP_PAPER_EXTRA_VERTICAL_SPACE = 4.5

SHAP_PAPER_BEESWARM_WIDTH = 36
SHAP_PAPER_MIN_BEESWARM_HEIGHT = 24
SHAP_PAPER_BEESWARM_HEIGHT_PER_LABEL_LINE = 1.45
SHAP_PAPER_BEESWARM_EXTRA_VERTICAL_SPACE = 5.5

SHAP_PAPER_DPI = 300
SHAP_PAPER_SAVE_FIGS = True

# Set high so no upstream readable labels get truncated with "..."
SHAP_PAPER_LABEL_MAXLEN = 1000
SHAP_FEATURE_LABEL_MAXLEN = SHAP_PAPER_LABEL_MAXLEN

if "PROJECT_DIR" in globals():
    SHAP_PAPER_OUTDIR = os.path.join(PROJECT_DIR, "paper_shap_figures_v33day2valid")
else:
    SHAP_PAPER_OUTDIR = os.path.join(os.getcwd(), "paper_shap_figures_v33day2valid")


# ======================================================================================
# FEATURE-LABEL HELPERS
# ======================================================================================

def _plain_radius_text(radius_km=None, text=None):
    """
    Return radius text as '100km' rather than 'r100km'.
    """
    if radius_km is not None:
        return f"{int(radius_km)}km"

    if text is not None:
        m = re.search(r"\b[rR]\s*(\d+)\s*km\b", str(text))
        if m:
            return f"{int(m.group(1))}km"

        m = re.search(r"\bR\s*(\d+)\b", str(text))
        if m:
            return f"{int(m.group(1))}km"

    return None


def _normalize_radius_tokens(text):
    """
    Normalize radius tokens internally to r100km first.
    Final labels later convert to '100km'.
    """
    text = str(text)

    text = re.sub(
        r"\bR\s*(\d+)\s*km\b",
        lambda m: f"r{int(m.group(1))}km",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\bR\s*(\d+)\b",
        lambda m: f"r{int(m.group(1))}km",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\br\s*(\d+)\s*km\b",
        lambda m: f"r{int(m.group(1))}km",
        text,
        flags=re.IGNORECASE,
    )
    text = re.sub(
        r"\bradius\s*(\d+)\s*km\b",
        lambda m: f"r{int(m.group(1))}km",
        text,
        flags=re.IGNORECASE,
    )

    return text


def _extract_outer_stat_radius(label, radius_km=None):
    """
    Extract labels of form:
      Max(something) within r100km

    Returns:
      outer_stat, inner_label, radius_text_plain
    """
    label = str(label).strip().replace("...", "").replace("…", "")
    label = _normalize_radius_tokens(label)

    radius_text = _plain_radius_text(radius_km=radius_km, text=label)

    # Remove trailing "within r100km" if present.
    label = re.sub(r"\s+within\s+r\d+km\s*$", "", label, flags=re.IGNORECASE).strip()

    outer_stat = None
    inner = label

    m_outer = re.match(
        r"^\s*([A-Za-z0-9_ ]+?)\s*\((.*)\)\s*$",
        label,
        flags=re.IGNORECASE,
    )

    if m_outer:
        possible_stat = m_outer.group(1).strip()
        possible_inner = m_outer.group(2).strip()

        stat_map = {
            "mean": "Mean",
            "avg": "Mean",
            "average": "Mean",
            "max": "Max",
            "maximum": "Max",
            "min": "Min",
            "minimum": "Min",
            "std": "Std",
            "stdev": "Std",
            "standard deviation": "Std",
            "standard_deviation": "Std",
            "median": "Median",
            "p10": "P10",
            "p25": "P25",
            "p50": "P50",
            "p75": "P75",
            "p90": "P90",
            "p95": "P95",
            "p99": "P99",
        }

        if possible_stat.lower() in stat_map:
            outer_stat = stat_map[possible_stat.lower()]
            inner = possible_inner

    return outer_stat, inner, radius_text


def _rebuild_untruncated_label_from_feature(feature, radius_km=None):
    """
    Use existing make_feature_label if available, but prevent truncation.
    """
    feature = str(feature)

    if "make_feature_label" in globals():
        try:
            return make_feature_label(
                feature,
                radius_km=radius_km,
                max_len=SHAP_PAPER_LABEL_MAXLEN,
            )
        except TypeError:
            try:
                lab = make_feature_label(feature, radius_km=radius_km)
                return str(lab).replace("...", "").replace("…", "")
            except Exception:
                pass
        except Exception:
            pass

    label = feature.replace("_", " ")
    label = _normalize_radius_tokens(label)

    if radius_km is not None and not re.search(r"\br\d+km\b", label, flags=re.IGNORECASE):
        label = f"{label} within r{int(radius_km)}km"

    return label.replace("...", "").replace("…", "")


def _normalize_text_for_matching(text):
    """
    Lowercase, remove punctuation-like separators, and normalize times so pattern matching
    is robust to underscores, spaces, and mixed notation.
    """
    s = str(text).replace("...", "").replace("…", "")
    s = _normalize_radius_tokens(s)

    s = s.replace("_", " ")
    s = s.replace("/", " ")
    s = s.replace("-", " ")
    s = s.replace("–", " ")
    s = s.replace(":", " ")
    s = s.replace(",", " ")
    s = s.replace("(", " ")
    s = s.replace(")", " ")

    s = re.sub(r"\b0\s*to\s*24\s*h?\b", "0 24", s, flags=re.IGNORECASE)
    s = re.sub(r"\b0to24h\b", "0 24", s, flags=re.IGNORECASE)
    s = re.sub(r"\b06h\b", "6", s, flags=re.IGNORECASE)
    s = re.sub(r"\b6h\b", "6", s, flags=re.IGNORECASE)
    s = re.sub(r"\b03h\b", "3", s, flags=re.IGNORECASE)
    s = re.sub(r"\b3h\b", "3", s, flags=re.IGNORECASE)
    s = re.sub(r"\b01h\b", "1", s, flags=re.IGNORECASE)
    s = re.sub(r"\b1h\b", "1", s, flags=re.IGNORECASE)
    s = re.sub(r"\b12h\b", "12", s, flags=re.IGNORECASE)
    s = re.sub(r"\b24h\b", "24", s, flags=re.IGNORECASE)

    s = re.sub(r"\s+", " ", s).strip().lower()

    return s


def _append_radius(label, radius_text):
    """
    Append 'within 100km' exactly once.
    """
    label = str(label).strip()

    label = re.sub(r"\s+within\s+r?\d+km\s*$", "", label, flags=re.IGNORECASE).strip()
    label = re.sub(r"\s+within\s+\d+\s*km\s*$", "", label, flags=re.IGNORECASE).strip()

    if radius_text is not None:
        label = f"{label} within {radius_text}"

    return label


def _detect_qpf_ffg_ratio_label(inner, outer_stat=None, radius_text=None):
    """
    Targeted QPF:FFG ratio labels.

    Examples:
      Max(Forecast QPF/APCP Max 6h Window 0to24h to Guidance FFG 06h Ratio R100km)
      -> Max 6-h QPF: 6-h FFG ratio within 100km

      Std(Forecast QPF/APCP Max 6h Window 0to24h to Guidance FFG 06h Ratio R100km)
      -> Std 6-h QPF: 6-h FFG ratio within 100km
    """
    raw = str(inner)
    s = _normalize_text_for_matching(raw)

    has_qpf = ("qpf" in s) or ("apcp" in s)
    has_ffg = "ffg" in s
    has_ratio = "ratio" in s

    if not (has_qpf and has_ffg and has_ratio):
        return None

    stat = outer_stat

    if stat is None:
        if re.search(r"\bstd\b|\bstdev\b|standard deviation", s):
            stat = "Std"
        elif re.search(r"\bmax\b|\bmaximum\b", s):
            stat = "Max"
        elif re.search(r"\bmean\b|\bavg\b|\baverage\b", s):
            stat = "Mean"
        elif re.search(r"\bmin\b|\bminimum\b", s):
            stat = "Min"

    if stat is None:
        stat = ""

    # Match the QPF/FFG duration. For the known important predictors this is 6-h.
    if re.search(r"\b6\b", s):
        duration = "6-h"
    elif re.search(r"\b3\b", s):
        duration = "3-h"
    elif re.search(r"\b1\b", s):
        duration = "1-h"
    elif re.search(r"\b12\b", s):
        duration = "12-h"
    elif re.search(r"\b24\b", s):
        duration = "24-h"
    else:
        duration = ""

    if duration:
        label = f"{stat} {duration} QPF: {duration} FFG ratio".strip()
    else:
        label = f"{stat} QPF: FFG ratio".strip()

    # Remove accidental duplicate duration phrases, e.g. "Max 6-h 6-h QPF".
    label = re.sub(r"\b(1|3|6|12|24)-h\s+\1-h\s+", r"\1-h ", label)
    label = re.sub(r"\s+", " ", label).strip()

    return _append_radius(label, radius_text)


def _detect_multiduration_ffg_label(inner, outer_stat=None, radius_text=None):
    """
    Targeted labels for features summarizing 1-, 3-, 6-, 12-, and 24-h FFGs.

    Examples:
      Std(...) -> Std of 1-, 3-, 6-, 12-, and 24-h FFGs within 100km
      Min(...) -> Min of 1-, 3-, 6-, 12-, and 24-h FFGs within 100km
    """
    s = _normalize_text_for_matching(inner)

    has_ffg = "ffg" in s
    has_multi_durations = all(tok in s.split() for tok in ["1", "3", "6", "12", "24"])

    if not (has_ffg and has_multi_durations):
        return None

    stat = outer_stat

    if stat is None:
        if re.search(r"\bstd\b|\bstdev\b|standard deviation", s):
            stat = "Std"
        elif re.search(r"\bmin\b|\bminimum\b", s):
            stat = "Min"
        elif re.search(r"\bmax\b|\bmaximum\b", s):
            stat = "Max"
        elif re.search(r"\bmean\b|\bavg\b|\baverage\b", s):
            stat = "Mean"

    if stat is None:
        stat = "Summary"

    label = f"{stat} of 1-, 3-, 6-, 12-, and 24-h FFGs"

    return _append_radius(label, radius_text)


def _detect_qpf_window_std_label(inner, outer_stat=None, radius_text=None):
    """
    Targeted labels for QPF run-total window Std features.

    Examples:
      Max(APCP RunTotal 0 06 12 18 24h Std R100km)
      -> Max Std between 0-6, 0-12, 0-18, and 0-24 h QPFs within 100km

      Mean(APCP RunTotal 0 06 12 18 24h Std R100km)
      -> Mean Std between 0-6, 0-12, 0-18, and 0-24 h QPFs within 100km
    """
    s = _normalize_text_for_matching(inner)

    has_qpf = ("qpf" in s) or ("apcp" in s)
    has_runtotal = ("runtotal" in s) or ("run total" in s) or ("run" in s and "total" in s)
    has_std = re.search(r"\bstd\b|\bstdev\b|standard deviation", s) is not None

    # These can appear as 0 06 12 18 24 or 0 6 12 18 24 after normalization.
    has_windows = all(tok in s.split() for tok in ["0", "6", "12", "18", "24"])

    if not (has_qpf and has_std and has_windows):
        return None

    stat = outer_stat

    if stat is None:
        if re.search(r"\bmax\b|\bmaximum\b", s):
            stat = "Max"
        elif re.search(r"\bmean\b|\bavg\b|\baverage\b", s):
            stat = "Mean"
        elif re.search(r"\bmin\b|\bminimum\b", s):
            stat = "Min"
        else:
            stat = ""

    label = f"{stat} Std between 0-6, 0-12, 0-18, and 0-24 h QPFs".strip()

    return _append_radius(label, radius_text)


def _detect_qpf_amount_label(inner, outer_stat=None, radius_text=None):
    """
    Shorten QPF amount features that are not ratios or window-Std features.
    """
    s = _normalize_text_for_matching(inner)

    has_qpf = ("qpf" in s) or ("apcp" in s)
    has_ffg = "ffg" in s
    has_ratio = "ratio" in s
    has_std_windows = (
        ("std" in s)
        and all(tok in s.split() for tok in ["0", "6", "12", "18", "24"])
    )

    if not has_qpf or has_ffg or has_ratio or has_std_windows:
        return None

    stat = outer_stat

    if stat is None:
        if re.search(r"\bstd\b|\bstdev\b|standard deviation", s):
            stat = "Std"
        elif re.search(r"\bmax\b|\bmaximum\b", s):
            stat = "Max"
        elif re.search(r"\bmean\b|\bavg\b|\baverage\b", s):
            stat = "Mean"
        elif re.search(r"\bmin\b|\bminimum\b", s):
            stat = "Min"

    if re.search(r"\b6\b", s):
        base = "6-h QPF"
    elif re.search(r"\b3\b", s):
        base = "3-h QPF"
    elif re.search(r"\b1\b", s):
        base = "1-h QPF"
    elif re.search(r"\b12\b", s):
        base = "12-h QPF"
    elif re.search(r"\b24\b", s):
        base = "24-h QPF"
    else:
        base = "QPF"

    label = f"{stat} {base}".strip() if stat else base

    label = re.sub(r"\b(1|3|6|12|24)-h\s+\1-h\s+", r"\1-h ", label)
    label = re.sub(r"\s+", " ", label).strip()

    return _append_radius(label, radius_text)


def _detect_prior_day_label(inner, outer_stat=None, radius_text=None):
    """
    Shorten prior-day MRMS/FFG exceedance labels.
    """
    s = _normalize_text_for_matching(inner)

    if not (("prev" in s or "prior" in s or "previous" in s) and ("ffg" in s or "mrms" in s)):
        return None

    stat = outer_stat

    if stat is None:
        if re.search(r"\bstd\b|\bstdev\b|standard deviation", s):
            stat = "Std"
        elif re.search(r"\bmax\b|\bmaximum\b", s):
            stat = "Max"
        elif re.search(r"\bmean\b|\bavg\b|\baverage\b", s):
            stat = "Mean"
        elif re.search(r"\bmin\b|\bminimum\b", s):
            stat = "Min"

    if "frac" in s or "fraction" in s:
        base = "prior-day MRMS>FFG fraction"
    elif "rate" in s:
        base = "prior-day MRMS>FFG rate"
    else:
        base = "prior-day MRMS>FFG"

    label = f"{stat} {base}".strip() if stat else base

    return _append_radius(label, radius_text)


def _detect_environment_label(inner, outer_stat=None, radius_text=None):
    """
    Shorten common environmental predictors.
    """
    s = _normalize_text_for_matching(inner)

    env_terms = [
        ("mucape", "MUCAPE"),
        ("mlcape", "MLCAPE"),
        ("sbcape", "SBCAPE"),
        ("cape", "CAPE"),
        ("cin", "CIN"),
        ("pwat", "PWAT"),
        ("precipitable water", "PWAT"),
        ("ivt", "IVT"),
        ("bulk shear", "bulk shear"),
        ("shear", "shear"),
        ("srh", "SRH"),
        ("helicity", "SRH"),
        ("dewpoint", "dewpoint"),
        ("temperature", "temperature"),
        ("theta e", "theta-e"),
        ("thetae", "theta-e"),
        ("relative humidity", "RH"),
        ("humidity", "RH"),
        ("wind", "wind"),
        ("u wind", "u-wind"),
        ("v wind", "v-wind"),
    ]

    matched = None

    for key, nice in env_terms:
        if key in s:
            matched = nice
            break

    if matched is None:
        return None

    stat = outer_stat

    if stat is None:
        if re.search(r"\bstd\b|\bstdev\b|standard deviation", s):
            stat = "Std"
        elif re.search(r"\bmax\b|\bmaximum\b", s):
            stat = "Max"
        elif re.search(r"\bmean\b|\bavg\b|\baverage\b", s):
            stat = "Mean"
        elif re.search(r"\bmin\b|\bminimum\b", s):
            stat = "Min"

    label = f"{stat} {matched}".strip() if stat else matched

    return _append_radius(label, radius_text)


def _fallback_plain_label(inner, outer_stat=None, radius_text=None):
    """
    General fallback if no targeted meteorological simplifier catches the feature.
    """
    s = str(inner).replace("...", "").replace("…", "")
    s = _normalize_radius_tokens(s)
    s = s.replace("_", " ")

    replacements = [
        (r"\bForecast\b", ""),
        (r"\bGuidance\b", ""),
        (r"\bQPF/APCP\b", "QPF"),
        (r"\bAPCP\b", "QPF"),
        (r"\bFlash Flood Guidance\b", "FFG"),
        (r"\bFFG 06h\b", "6-h FFG"),
        (r"\bFFG 6h\b", "6-h FFG"),
        (r"\b06h\b", "6-h"),
        (r"\b6h\b", "6-h"),
        (r"\b03h\b", "3-h"),
        (r"\b3h\b", "3-h"),
        (r"\b01h\b", "1-h"),
        (r"\b1h\b", "1-h"),
        (r"\b12h\b", "12-h"),
        (r"\b24h\b", "24-h"),
        (r"\bStd\b", "Std"),
        (r"\bstandard deviation\b", "Std"),
        (r"\bWindow\b", ""),
        (r"\bFeature\b", ""),
        (r"\s+to\s+", ": "),
        (r"\s+Ratio\b", " ratio"),
    ]

    for pattern, repl in replacements:
        s = re.sub(pattern, repl, s, flags=re.IGNORECASE)

    s = re.sub(r"\br\d+km\b", "", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s).strip(" -_:,")

    if outer_stat and not s.lower().startswith(outer_stat.lower() + " "):
        s = f"{outer_stat} {s}"

    s = re.sub(r"\bSD\b", "Std", s)
    s = re.sub(r"\bStd Std\b", "Std", s)
    s = re.sub(r"\bMax Max\b", "Max", s)
    s = re.sub(r"\bMean Mean\b", "Mean", s)
    s = re.sub(r"\bMin Min\b", "Min", s)
    s = re.sub(r"\b(1|3|6|12|24)-h\s+\1-h\s+", r"\1-h ", s)
    s = re.sub(r"\s+", " ", s).strip()

    return _append_radius(s, radius_text)


def make_plain_english_shap_label(feature, old_label=None, radius_km=None):
    """
    Build a short, plain-English, paper-friendly feature label.

    Uses targeted meteorological simplifiers before falling back to a generic cleanup.
    """
    if old_label is None or str(old_label).strip() == "":
        raw_label = _rebuild_untruncated_label_from_feature(feature, radius_km=radius_km)
    else:
        raw_label = str(old_label).replace("...", "").replace("…", "")

    raw_label = _normalize_radius_tokens(raw_label)

    outer_stat, inner, radius_text = _extract_outer_stat_radius(raw_label, radius_km=radius_km)

    # Use original feature text too, because sometimes the readable label has already
    # removed important details.
    combined_inner = f"{inner} {feature}"

    simplified = (
        _detect_qpf_ffg_ratio_label(combined_inner, outer_stat=outer_stat, radius_text=radius_text)
        or _detect_multiduration_ffg_label(combined_inner, outer_stat=outer_stat, radius_text=radius_text)
        or _detect_qpf_window_std_label(combined_inner, outer_stat=outer_stat, radius_text=radius_text)
        or _detect_qpf_amount_label(combined_inner, outer_stat=outer_stat, radius_text=radius_text)
        or _detect_prior_day_label(combined_inner, outer_stat=outer_stat, radius_text=radius_text)
        or _detect_environment_label(combined_inner, outer_stat=outer_stat, radius_text=radius_text)
        or _fallback_plain_label(inner, outer_stat=outer_stat, radius_text=radius_text)
    )

    # Final cleanup.
    simplified = str(simplified).replace("...", "").replace("…", "")
    simplified = simplified.replace("SD", "Std")
    simplified = re.sub(r"\bStd Std\b", "Std", simplified)
    simplified = re.sub(r"\bMax Max\b", "Max", simplified)
    simplified = re.sub(r"\bMean Mean\b", "Mean", simplified)
    simplified = re.sub(r"\bMin Min\b", "Min", simplified)

    # Remove duplicate duration, especially "Max 6-h 6-h QPF".
    simplified = re.sub(r"\b(1|3|6|12|24)-h\s+\1-h\s+", r"\1-h ", simplified)

    # Normalize QPF:FFG spacing.
    simplified = re.sub(r"QPF\s*:\s*", "QPF: ", simplified)
    simplified = re.sub(r"\s+ratio", " ratio", simplified)

    # Normalize radius.
    simplified = re.sub(r"\bwithin\s+r?(\d+)km\b", r"within \1km", simplified, flags=re.IGNORECASE)
    simplified = re.sub(r"\bwithin\s+(\d+)\s+km\b", r"within \1km", simplified, flags=re.IGNORECASE)

    simplified = re.sub(r"\s+", " ", simplified).strip()

    return simplified


def _wrap_shap_label(label, width=SHAP_PAPER_LABEL_WRAP_WIDTH):
    """
    Wrap plain-English labels. Labels are intentionally concise before this step.
    """
    label = str(label).replace("...", "").replace("…", "")
    label = re.sub(r"\bwithin\s+r?(\d+)km\b", r"within \1km", label, flags=re.IGNORECASE)

    # Intentional semantic breaks.
    label = label.replace(" within ", "\nwithin ")
    label = label.replace(": ", ":\n")

    wrapped_lines = []

    for chunk in label.split("\n"):
        lines = textwrap.wrap(
            chunk,
            width=int(width),
            break_long_words=False,
            break_on_hyphens=False,
        )
        if lines:
            wrapped_lines.extend(lines)
        else:
            wrapped_lines.append(chunk)

    return "\n".join(wrapped_lines)


def _count_label_lines(label):
    return max(1, str(label).count("\n") + 1)


def _dynamic_y_positions_for_wrapped_labels(labels):
    """
    Create uneven y positions so multi-line labels get extra vertical room.
    """
    line_counts = np.array([_count_label_lines(lab) for lab in labels], dtype=float)

    # Baseline spacing plus extra for additional wrapped lines.
    spacing = 0.70 + 0.42 * np.maximum(line_counts - 1.0, 0.0)

    y = np.zeros(len(labels), dtype=float)
    for i in range(1, len(labels)):
        y[i] = y[i - 1] + spacing[i - 1]

    return y, spacing, line_counts


def _dynamic_fig_height_from_labels(labels, min_height, per_line, extra):
    total_lines = sum(_count_label_lines(lab) for lab in labels)
    return max(float(min_height), float(extra) + float(per_line) * float(total_lines))


def _paper_format_axis(ax):
    ax.set_title("")
    ax.tick_params(axis="both", labelsize=SHAP_PAPER_FONT_SIZE)
    ax.xaxis.label.set_size(SHAP_PAPER_FONT_SIZE)
    ax.yaxis.label.set_size(SHAP_PAPER_FONT_SIZE)

    for tick in ax.get_xticklabels():
        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)

    for tick in ax.get_yticklabels():
        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)
        tick.set_linespacing(0.92)

    ax.grid(True, axis="x", alpha=0.25)
    ax.grid(False, axis="y")


def _shap_values_with_wrapped_feature_names(sv, X_raw_df, labels):
    """
    Return SHAP Explanation with raw feature values and paper-ready feature names.
    """
    import shap

    wrapped_labels = [_wrap_shap_label(lab) for lab in labels]

    try:
        return shap.Explanation(
            values=np.asarray(sv.values),
            base_values=sv.base_values,
            data=X_raw_df.to_numpy(float),
            feature_names=wrapped_labels,
        )
    except Exception:
        try:
            sv.feature_names = wrapped_labels
        except Exception:
            pass
        return sv


def _make_labels_without_suffixes(labels):
    """
    Keep labels exactly as written. Do not append [2], [3], etc.
    """
    return [str(lab) for lab in labels]


# ======================================================================================
# MAIN OVERRIDE
# ======================================================================================

def plot_shap_global_summary(
    radius_km=40,
    sample_n=50000,
    random_state=42,
    max_display=30,
    make_beeswarm=True,
    save_feature_table=True,
):
    """
    Paper-ready global SHAP summary across sampled test-case grid points.

    Produces:
      1. mean absolute SHAP bar plot with short plain-English feature labels
      2. optional SHAP beeswarm with the same simplified labels
      3. feature-definition CSV with original and simplified labels
    """
    import shap

    os.makedirs(SHAP_PAPER_OUTDIR, exist_ok=True)

    imp, sv, X_raw_df, meta_df, feats, old_labels, feature_defs = shap_importance_table(
        radius_km=radius_km,
        sample_n=sample_n,
        random_state=random_state,
    )

    # ------------------------------------------------------------------
    # Rebuild untruncated labels from original feature names, then simplify.
    # ------------------------------------------------------------------
    untruncated_labels = [
        _rebuild_untruncated_label_from_feature(feat, radius_km=radius_km)
        for feat in feats
    ]

    plain_labels = [
        make_plain_english_shap_label(
            feature=feat,
            old_label=untruncated_label,
            radius_km=radius_km,
        )
        for feat, untruncated_label in zip(feats, untruncated_labels)
    ]

    # Do NOT append [2] or similar duplicate suffixes.
    plain_labels_final = _make_labels_without_suffixes(plain_labels)

    label_map = dict(zip(feats, plain_labels_final))
    original_label_map = dict(zip(feats, untruncated_labels))

    imp = imp.copy()

    if "Feature" not in imp.columns:
        raise RuntimeError("Expected SHAP importance table to contain a 'Feature' column.")

    if "SHAP label" not in imp.columns:
        imp["SHAP label"] = imp["Feature"].astype(str)

    imp["Original SHAP label"] = imp["Feature"].map(original_label_map).fillna(imp["SHAP label"].astype(str))
    imp["Paper SHAP label"] = imp["Feature"].map(label_map).fillna(imp["Original SHAP label"].astype(str))
    imp["SHAP label"] = imp["Paper SHAP label"]

    feature_defs = feature_defs.copy()

    if "Feature" in feature_defs.columns:
        feature_defs["Original SHAP label"] = feature_defs["Feature"].map(original_label_map)

        if "SHAP label" in feature_defs.columns:
            feature_defs["Original SHAP label"] = feature_defs["Original SHAP label"].fillna(
                feature_defs["SHAP label"].astype(str)
            )

        feature_defs["Paper SHAP label"] = feature_defs["Feature"].map(label_map)

        if "SHAP label" in feature_defs.columns:
            feature_defs["Paper SHAP label"] = feature_defs["Paper SHAP label"].fillna(
                feature_defs["SHAP label"].astype(str)
            )

        feature_defs["SHAP label"] = feature_defs["Paper SHAP label"]

    n_cases = meta_df["Date"].astype(str).str[:8].nunique() if "Date" in meta_df.columns else np.nan

    print(
        f"Global SHAP sample: rows={len(meta_df):,}, unique dates={n_cases}, "
        f"radius={int(radius_km)} km, predictors={len(feats):,}"
    )

    if "Date" in meta_df.columns:
        print("Date range:", meta_df["Date"].astype(str).min(), "to", meta_df["Date"].astype(str).max())

    if save_feature_table:
        out_csv = os.path.join(
            PROJECT_DIR if "PROJECT_DIR" in globals() else os.getcwd(),
            f"shap_feature_definitions_v33day2valid_r{int(radius_km)}km.csv",
        )
        try:
            feature_defs.to_csv(out_csv, index=False)
            print(f"Saved SHAP feature-definition table: {out_csv}")
        except Exception as exc:
            print("Could not save SHAP feature-definition CSV:", exc)

    display_cols = [
        "Feature",
        "Original SHAP label",
        "Paper SHAP label",
        "mean_abs_shap",
        "Family",
        "Units inferred",
        "Definition",
        "Notes",
    ]
    display_cols = [c for c in display_cols if c in imp.columns]

    display(
        imp.head(int(max_display))[display_cols]
        .reset_index(drop=True)
    )

    # ==================================================================================
    # Mean absolute SHAP bar plot
    # ==================================================================================
    top = imp.head(int(max_display)).iloc[::-1].copy()
    top["Wrapped SHAP label"] = top["Paper SHAP label"].map(_wrap_shap_label)

    labels_wrapped = top["Wrapped SHAP label"].astype(str).tolist()
    values = pd.to_numeric(top["mean_abs_shap"], errors="coerce").to_numpy(float)

    y_pos, y_spacing, line_counts = _dynamic_y_positions_for_wrapped_labels(labels_wrapped)

    fig_height = _dynamic_fig_height_from_labels(
        labels_wrapped,
        min_height=SHAP_PAPER_MIN_BAR_HEIGHT,
        per_line=SHAP_PAPER_HEIGHT_PER_LABEL_LINE,
        extra=SHAP_PAPER_EXTRA_VERTICAL_SPACE,
    )

    with plt.rc_context({
        "font.size": SHAP_PAPER_FONT_SIZE,
        "axes.labelsize": SHAP_PAPER_FONT_SIZE,
        "xtick.labelsize": SHAP_PAPER_FONT_SIZE,
        "ytick.labelsize": SHAP_PAPER_FONT_SIZE,
    }):
        fig, ax = plt.subplots(figsize=(SHAP_PAPER_BAR_WIDTH, fig_height))

        bar_heights = np.minimum(0.86, 0.72 * y_spacing)

        ax.barh(
            y_pos,
            values,
            height=bar_heights,
        )

        ax.set_yticks(y_pos)
        ax.set_yticklabels(labels_wrapped)

        ax.set_xlabel("Mean absolute SHAP value")
        ax.set_ylabel("")
        ax.set_title("")

        ax.set_ylim(y_pos[0] - 0.85, y_pos[-1] + 0.85)

        _paper_format_axis(ax)

        fig.subplots_adjust(left=0.34, right=0.98, bottom=0.12, top=0.98)

        if SHAP_PAPER_SAVE_FIGS:
            out_png = os.path.join(
                SHAP_PAPER_OUTDIR,
                f"global_shap_importance_r{int(radius_km)}km.png",
            )
            out_pdf = os.path.join(
                SHAP_PAPER_OUTDIR,
                f"global_shap_importance_r{int(radius_km)}km.pdf",
            )

            fig.savefig(out_png, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")
            fig.savefig(out_pdf, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")

            print("Saved:", out_png)
            print("Saved:", out_pdf)

        plt.show()
        plt.close(fig)

    # ==================================================================================
    # Beeswarm plot
    # ==================================================================================
    shap_exp_raw = _shap_values_with_wrapped_feature_names(
        sv,
        X_raw_df,
        plain_labels_final,
    )

    if make_beeswarm:
        try:
            beeswarm_labels = [
                _wrap_shap_label(lab)
                for lab in imp.head(int(max_display))["Paper SHAP label"].tolist()
            ]

            beeswarm_height = _dynamic_fig_height_from_labels(
                beeswarm_labels,
                min_height=SHAP_PAPER_MIN_BEESWARM_HEIGHT,
                per_line=SHAP_PAPER_BEESWARM_HEIGHT_PER_LABEL_LINE,
                extra=SHAP_PAPER_BEESWARM_EXTRA_VERTICAL_SPACE,
            )

            with plt.rc_context({
                "font.size": SHAP_PAPER_FONT_SIZE,
                "axes.labelsize": SHAP_PAPER_FONT_SIZE,
                "xtick.labelsize": SHAP_PAPER_FONT_SIZE,
                "ytick.labelsize": SHAP_PAPER_FONT_SIZE,
            }):
                shap.plots.beeswarm(
                    shap_exp_raw,
                    max_display=int(max_display),
                    show=False,
                )

                fig = plt.gcf()
                fig.set_size_inches(SHAP_PAPER_BEESWARM_WIDTH, beeswarm_height)

                for ax in fig.axes:
                    ax.set_title("")
                    ax.tick_params(axis="both", labelsize=SHAP_PAPER_FONT_SIZE)
                    ax.xaxis.label.set_size(SHAP_PAPER_FONT_SIZE)
                    ax.yaxis.label.set_size(SHAP_PAPER_FONT_SIZE)

                    for tick in ax.get_xticklabels():
                        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)

                    for tick in ax.get_yticklabels():
                        tick.set_fontsize(SHAP_PAPER_FONT_SIZE)
                        tick.set_linespacing(0.92)

                fig.subplots_adjust(left=0.34, right=0.97, bottom=0.12, top=0.98)

                if SHAP_PAPER_SAVE_FIGS:
                    out_png = os.path.join(
                        SHAP_PAPER_OUTDIR,
                        f"global_shap_beeswarm_r{int(radius_km)}km.png",
                    )
                    out_pdf = os.path.join(
                        SHAP_PAPER_OUTDIR,
                        f"global_shap_beeswarm_r{int(radius_km)}km.pdf",
                    )

                    fig.savefig(out_png, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")
                    fig.savefig(out_pdf, dpi=int(SHAP_PAPER_DPI), bbox_inches="tight")

                    print("Saved:", out_png)
                    print("Saved:", out_pdf)

                plt.show()
                plt.close(fig)

        except Exception as exc:
            print("SHAP beeswarm failed, but the global bar plot and tables above are still valid:", exc)

    wrapped_labels_return = [_wrap_shap_label(lab) for lab in plain_labels_final]

    return imp, shap_exp_raw, X_raw_df, meta_df, feats, wrapped_labels_return, feature_defs


print("Paper-ready SHAP override loaded: font=50, targeted plain-English labels, no duplicate suffixes.")


# ======================================================================================
# RUN GLOBAL SHAP SUMMARY
# ======================================================================================

SHAP_GLOBAL_MAX_DISPLAY = 10
SHAP_MAKE_BEESWARM = True
SHAP_SAVE_FEATURE_TABLE = True

# Optional local diagnostic only. Leave False for the all-test-case interpretation.
SHAP_RUN_LOCAL_WATERFALL = False
SHAP_ROW = 0
SHAP_WATERFALL_MAX_DISPLAY = 10

try:
    shap_importance_global, shap_values_global, shap_df_raw_global, shap_meta_global, shap_features_global, shap_labels_global, shap_feature_definitions = plot_shap_global_summary(
        radius_km=SHAP_RADIUS_KM,
        sample_n=SHAP_SAMPLE_N,
        random_state=SHAP_RANDOM_STATE,
        max_display=SHAP_GLOBAL_MAX_DISPLAY,
        make_beeswarm=SHAP_MAKE_BEESWARM,
        save_feature_table=SHAP_SAVE_FEATURE_TABLE,
    )
except Exception as exc:
    print("Global SHAP summary not run:", exc)

if SHAP_RUN_LOCAL_WATERFALL:
    try:
        shap_waterfall_values, shap_waterfall_raw, shap_waterfall_features, shap_waterfall_feature_definitions = plot_shap_waterfall_optional(
            radius_km=SHAP_RADIUS_KM,
            row=SHAP_ROW,
            sample_n=SHAP_SAMPLE_N,
            max_display=SHAP_WATERFALL_MAX_DISPLAY,
            random_state=SHAP_RANDOM_STATE,
        )
    except Exception as exc:
        print("Optional local SHAP waterfall not run:", exc)

## 15. SHAP dependence plots across sampled test cases

The dependence plot uses the same sampled 2024-2025 test grid points as the global SHAP summary. The x-axis uses the original/raw feature values, while the y-axis uses SHAP values for that feature.

Use the browser to click through predictors interactively. You can also save dependence plots for every predictor, but that can take a while for large feature sets.



In [ ]:
# ======================================================================================
# PAPER-READY SHAP DEPENDENCE SUBPLOTS:
# MOST IMPORTANT PREDICTOR + BOTTOM 3 OF THE TOP 10
#
# This creates a 2x2 figure with:
#   - rank 1 feature by mean(|SHAP|)
#   - ranks 8, 9, and 10 by mean(|SHAP|)
#
# Formatting:
#   - font size 50 everywhere
#   - no subplot titles
#   - NO feature-name x-axis labels under subplots
#   - subplot labels a), b), c), d)
#   - saved as PNG and PDF
# ======================================================================================

import os
import string
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# --------------------------------------------------------------------------------------
# Controls
# --------------------------------------------------------------------------------------
SHAP_DEP_FONT_SIZE = 50

SHAP_DEP_POINT_SIZE = 18
SHAP_DEP_ALPHA = 0.22

# Slightly shorter height since feature-name labels are removed.
SHAP_DEP_FIGSIZE = (44, 34)
SHAP_DEP_DPI = 300

SHAP_DEP_SAVE_DIR = os.path.join(PROJECT_DIR, f"shap_dependence_v33day2valid_r{int(SHAP_RADIUS_KM)}")
os.makedirs(SHAP_DEP_SAVE_DIR, exist_ok=True)

# Ranks to plot: 1st, then 8th-10th from top-10 importance list
SHAP_DEP_SELECTED_RANKS = [1, 8, 9, 10]

# Set True if you want a generic x-axis label like "Feature value" on bottom row only.
# Default False removes x-axis labels completely.
SHAP_DEP_USE_GENERIC_XLABEL = False
SHAP_DEP_GENERIC_XLABEL = "Feature value"

# Set True if you only want "SHAP value" on the left column.
# Default True makes the figure less repetitive.
SHAP_DEP_YLABEL_LEFT_ONLY = True


# --------------------------------------------------------------------------------------
# Helper functions
# --------------------------------------------------------------------------------------
def _dep_get_shap_values_2d(sv):
    vals = np.asarray(sv.values)

    if vals.ndim == 2:
        return vals

    if vals.ndim == 3:
        if vals.shape[-1] == 2:
            return vals[:, :, 1]
        if vals.shape[0] == 2:
            return vals[1, :, :]
        return np.squeeze(vals)

    vals = np.squeeze(vals)

    if vals.ndim != 2:
        raise RuntimeError(f"Could not interpret SHAP values with shape {np.asarray(sv.values).shape}")

    return vals


def _dep_format_axis(ax):
    ax.tick_params(axis="both", labelsize=SHAP_DEP_FONT_SIZE)

    ax.xaxis.label.set_size(SHAP_DEP_FONT_SIZE)
    ax.yaxis.label.set_size(SHAP_DEP_FONT_SIZE)

    for tick in ax.get_xticklabels():
        tick.set_fontsize(SHAP_DEP_FONT_SIZE)

    for tick in ax.get_yticklabels():
        tick.set_fontsize(SHAP_DEP_FONT_SIZE)

    ax.grid(True, alpha=0.25)
    ax.set_title("")


def _dep_add_panel_label(ax, label):
    ax.text(
        0.025,
        0.965,
        label,
        transform=ax.transAxes,
        ha="left",
        va="top",
        fontsize=SHAP_DEP_FONT_SIZE,
        fontweight="bold",
        bbox=dict(
            facecolor="white",
            edgecolor="none",
            alpha=0.80,
            boxstyle="round,pad=0.16",
        ),
        zorder=20,
    )


# --------------------------------------------------------------------------------------
# Load SHAP values and importance table
# --------------------------------------------------------------------------------------
shap_importance_dep, shap_values_dep, shap_df_raw_dep, shap_meta_dep, shap_features_dep, shap_labels_dep, shap_feature_defs_dep = shap_importance_table(
    radius_km=SHAP_RADIUS_KM,
    sample_n=SHAP_SAMPLE_N,
    random_state=SHAP_RANDOM_STATE,
)

shap_vals_2d = _dep_get_shap_values_2d(shap_values_dep)

top10_df = shap_importance_dep.head(10).copy().reset_index(drop=True)
top10_df["Rank"] = np.arange(1, len(top10_df) + 1)

selected_df = top10_df[top10_df["Rank"].isin(SHAP_DEP_SELECTED_RANKS)].copy()

print("Selected dependence features:")
display(
    selected_df[["Rank", "Feature", "SHAP label", "mean_abs_shap"]]
    .reset_index(drop=True)
)


# --------------------------------------------------------------------------------------
# Create 2x2 subplot figure
# --------------------------------------------------------------------------------------
n_panels = len(selected_df)
ncols = 2
nrows = 2

with plt.rc_context({
    "font.size": SHAP_DEP_FONT_SIZE,
    "axes.labelsize": SHAP_DEP_FONT_SIZE,
    "xtick.labelsize": SHAP_DEP_FONT_SIZE,
    "ytick.labelsize": SHAP_DEP_FONT_SIZE,
}):
    fig, axes = plt.subplots(
        nrows=nrows,
        ncols=ncols,
        figsize=SHAP_DEP_FIGSIZE,
        squeeze=False,
    )

    axes_flat = axes.ravel()
    panel_letters = [f"{x})" for x in string.ascii_lowercase]

    for panel_idx, (_, row) in enumerate(selected_df.iterrows()):
        ax = axes_flat[panel_idx]

        feature = row["Feature"]

        if feature not in shap_features_dep:
            matches = [
                i for i, f in enumerate(shap_features_dep)
                if str(feature).lower() in str(f).lower()
            ]
            if not matches:
                raise RuntimeError(f"Feature {feature!r} not found in SHAP feature list.")
            feat_idx = matches[0]
            feature = shap_features_dep[feat_idx]
        else:
            feat_idx = shap_features_dep.index(feature)

        x = pd.to_numeric(shap_df_raw_dep[feature], errors="coerce").to_numpy(float)
        y = np.asarray(shap_vals_2d[:, feat_idx], dtype=float)

        good = np.isfinite(x) & np.isfinite(y)

        ax.scatter(
            x[good],
            y[good],
            s=SHAP_DEP_POINT_SIZE,
            alpha=SHAP_DEP_ALPHA,
            linewidths=0,
            rasterized=True,
        )

        ax.axhline(0.0, color="black", linewidth=2.0, alpha=0.75)

        # ------------------------------------------------------------------
        # No feature-name labels under each subplot.
        # Optionally add a generic x-label only on the bottom row.
        # ------------------------------------------------------------------
        row_idx = panel_idx // ncols
        col_idx = panel_idx % ncols

        if SHAP_DEP_USE_GENERIC_XLABEL and row_idx == nrows - 1:
            ax.set_xlabel(SHAP_DEP_GENERIC_XLABEL)
        else:
            ax.set_xlabel("")

        if SHAP_DEP_YLABEL_LEFT_ONLY:
            if col_idx == 0:
                ax.set_ylabel("SHAP value")
            else:
                ax.set_ylabel("")
        else:
            ax.set_ylabel("SHAP value")

        _dep_format_axis(ax)
        _dep_add_panel_label(ax, panel_letters[panel_idx])

    for ax in axes_flat[n_panels:]:
        ax.set_visible(False)

    fig.subplots_adjust(
        left=0.10,
        right=0.98,
        bottom=0.07,
        top=0.98,
        wspace=0.28,
        hspace=0.28,
    )

    out_png = os.path.join(
        SHAP_DEP_SAVE_DIR,
        f"selected_shap_dependence_rank1_rank8to10_r{int(SHAP_RADIUS_KM)}km.png",
    )
    out_pdf = os.path.join(
        SHAP_DEP_SAVE_DIR,
        f"selected_shap_dependence_rank1_rank8to10_r{int(SHAP_RADIUS_KM)}km.pdf",
    )

    fig.savefig(out_png, dpi=int(SHAP_DEP_DPI), bbox_inches="tight")
    fig.savefig(out_pdf, dpi=int(SHAP_DEP_DPI), bbox_inches="tight")

    print("Saved:", out_png)
    print("Saved:", out_pdf)

    plt.show()
    plt.close(fig)

### Testing something that might explain SHAP

In [ ]:
# ======================================================================================
# MLCAPE FEATURE VS 100-km MRMS/FFG EXCEEDANCE TARGET
#
# Goal:
#   For the training set, compare the binary occurrence of any MRMS > FFG within 100 km
#   against the selected MLCAPE feature:
#
#   Max(MLCAPE 0h 6h 12h 18h 24h Mean) within r100km
#
# Notes:
#   - This uses the same master parquet and feature list as the SHAP workflow.
#   - If the target column is fractional coverage instead of binary, this treats any value > 0
#     as "at least one grid point within 100 km exceeded FFG."
# ======================================================================================

import os
import re
import textwrap
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    import pyarrow.parquet as pq
except Exception:
    pq = None

# --------------------------------------------------------------------------------------
# Controls
# --------------------------------------------------------------------------------------
RADIUS_KM = 100
MLCAPE_THRESHOLD = 1300

POINT_SAMPLE_N = 300000   # scatter sample for plotting only; set None to plot all points
RANDOM_STATE = 42

FIGSIZE = (20, 14)
FONT_SIZE = 34
DPI = 300

SAVE_FIG = True

MLCAPE_FEATURE_OVERRIDE = None
TARGET_COL_OVERRIDE = None


# --------------------------------------------------------------------------------------
# Required helpers from your existing workflow
# --------------------------------------------------------------------------------------
if "find_artifacts_for_radius" not in globals():
    raise RuntimeError("find_artifacts_for_radius(...) is not defined in this notebook/session.")

if "_load_feature_names" not in globals():
    raise RuntimeError("_load_feature_names(...) is not defined in this notebook/session.")

art = find_artifacts_for_radius(RADIUS_KM)
master_path = art["master_path"]
features_path = art["features_path"]

if not os.path.exists(master_path):
    raise FileNotFoundError(f"Master parquet not found: {master_path}")

if not os.path.exists(features_path):
    raise FileNotFoundError(f"Feature list not found: {features_path}")

feats = _load_feature_names(features_path)

print("Using master parquet:", master_path)
print("Using feature list:", features_path)
print("Number of model features:", len(feats))


# --------------------------------------------------------------------------------------
# Get column names without loading full parquet
# --------------------------------------------------------------------------------------
if pq is not None:
    all_cols = pq.read_schema(master_path).names
else:
    all_cols = pd.read_parquet(master_path).columns.tolist()


# --------------------------------------------------------------------------------------
# Training years
# --------------------------------------------------------------------------------------
if "TRAIN_YEARS" in globals():
    train_years = [str(y) for y in TRAIN_YEARS]
else:
    train_years = ["2018", "2019", "2020", "2021", "2022", "2023"]

print("Training years used:", train_years)


# --------------------------------------------------------------------------------------
# Find selected MLCAPE feature
# --------------------------------------------------------------------------------------
def _score_mlcape_feature(c):
    cu = str(c).upper()
    score = 0

    if "MLCAPE" in cu:
        score += 30
    if "MEAN" in cu:
        score += 10
    if "MAX" in cu:
        score += 10

    # Prefer features mentioning the intended times
    for tok in ["0", "06", "6", "12", "18", "24"]:
        if re.search(rf"(^|[_A-Z]){tok}H?($|[_A-Z])", cu) or f"{tok}H" in cu:
            score += 2

    # Prefer r100 / 100km if embedded in name
    if "100" in cu or "R100" in cu or "100KM" in cu:
        score += 8

    return score


if MLCAPE_FEATURE_OVERRIDE is not None:
    mlcape_col = MLCAPE_FEATURE_OVERRIDE
else:
    mlcape_candidates = [
        c for c in feats
        if "MLCAPE" in str(c).upper()
        and "MEAN" in str(c).upper()
        and "MAX" in str(c).upper()
    ]

    mlcape_candidates = sorted(
        mlcape_candidates,
        key=_score_mlcape_feature,
        reverse=True,
    )

    print("\nMLCAPE candidates:")
    for c in mlcape_candidates[:25]:
        print("  ", c)

    if not mlcape_candidates:
        raise RuntimeError("No MLCAPE feature candidate found.")

    mlcape_col = mlcape_candidates[0]

print("\nUsing MLCAPE feature:", mlcape_col)


# --------------------------------------------------------------------------------------
# Find 100-km target column: MRMS exceeding FFG within 100 km
# --------------------------------------------------------------------------------------
def _score_target_col(c):
    cu = str(c).upper()
    score = 0

    if "TARGET" in cu:
        score += 30
    if "MRMS" in cu:
        score += 20
    if "FFG" in cu:
        score += 20
    if "EXCEED" in cu or "EXCEEDED" in cu or "EXCEEDS" in cu:
        score += 15
    if "100KM" in cu or "R100" in cu or "R_100" in cu or "100" in cu:
        score += 25
    if "FRACTION" in cu or "FRAC" in cu:
        score += 4
    if "POINT" in cu:
        score -= 5

    # Avoid forecast/verification products, if present
    for bad in ["PRED", "PROB", "ML_", "WPC", "PP_", "ERO", "LSR", "USGS"]:
        if bad in cu:
            score -= 20

    return score


if TARGET_COL_OVERRIDE is not None:
    target_col = TARGET_COL_OVERRIDE
else:
    target_candidates = [
        c for c in all_cols
        if (
            "FFG" in str(c).upper()
            and ("TARGET" in str(c).upper() or "EXCEED" in str(c).upper() or "FRAC" in str(c).upper())
            and ("100" in str(c).upper() or "R100" in str(c).upper() or "100KM" in str(c).upper())
        )
    ]

    target_candidates = sorted(
        target_candidates,
        key=_score_target_col,
        reverse=True,
    )

    print("\n100-km MRMS/FFG target candidates:")
    for c in target_candidates[:50]:
        print("  ", c)

    if not target_candidates:
        print("\nAll FFG-related columns:")
        for c in all_cols:
            if "FFG" in str(c).upper():
                print("  ", c)
        raise RuntimeError("No 100-km MRMS/FFG target candidate found. Set TARGET_COL_OVERRIDE manually.")

    target_col = target_candidates[0]

print("\nUsing target column:", target_col)


# --------------------------------------------------------------------------------------
# Load data
# --------------------------------------------------------------------------------------
cols_to_load = ["Date", mlcape_col, target_col]
cols_to_load = list(dict.fromkeys(cols_to_load))

df = pd.read_parquet(master_path, columns=cols_to_load)

df["Date"] = df["Date"].astype(str).str[:8]
df = df[df["Date"].str[:4].isin(train_years)].copy()

df[mlcape_col] = pd.to_numeric(df[mlcape_col], errors="coerce")
df[target_col] = pd.to_numeric(df[target_col], errors="coerce")

df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=[mlcape_col, target_col]).copy()

# Binary target:
# If target is already 0/1, this preserves it.
# If target is fractional coverage, this means "any MRMS > FFG within 100 km."
df["MRMS_FFG_any_r100"] = (df[target_col] > 0).astype(int)

print("\nRows after training-year filter and NaN removal:", len(df))
print("Target values summary:")
display(df[target_col].describe())


# --------------------------------------------------------------------------------------
# Summary table for >1000 vs <=1000 MLCAPE
# --------------------------------------------------------------------------------------
df["MLCAPE_group"] = np.where(
    df[mlcape_col] > MLCAPE_THRESHOLD,
    f"> {MLCAPE_THRESHOLD:.0f} J kg$^{{-1}}$",
    f"<= {MLCAPE_THRESHOLD:.0f} J kg$^{{-1}}$",
)

summary = (
    df.groupby("MLCAPE_group")
    .agg(
        total_gridpoint_cases=("MRMS_FFG_any_r100", "size"),
        mrms_ffg_exceedance_count=("MRMS_FFG_any_r100", "sum"),
        no_mrms_ffg_exceedance_count=("MRMS_FFG_any_r100", lambda x: int((x == 0).sum())),
        mean_binary_exceedance=("MRMS_FFG_any_r100", "mean"),
        mean_feature_value=(mlcape_col, "mean"),
        median_feature_value=(mlcape_col, "median"),
    )
    .reset_index()
)

summary["mrms_ffg_exceedance_frequency_pct"] = (
    100.0 * summary["mean_binary_exceedance"]
)

summary = summary[
    [
        "MLCAPE_group",
        "total_gridpoint_cases",
        "mrms_ffg_exceedance_count",
        "no_mrms_ffg_exceedance_count",
        "mrms_ffg_exceedance_frequency_pct",
        "mean_feature_value",
        "median_feature_value",
    ]
]

display(summary)


# --------------------------------------------------------------------------------------
# Binned event frequency for plotting
# --------------------------------------------------------------------------------------
plot_df = df[[mlcape_col, "MRMS_FFG_any_r100"]].copy()
plot_df = plot_df.sort_values(mlcape_col)

# Quantile bins are useful because the feature distribution is probably uneven.
plot_df["feature_bin"] = pd.qcut(
    plot_df[mlcape_col],
    q=30,
    duplicates="drop",
)

binned = (
    plot_df.groupby("feature_bin", observed=True)
    .agg(
        bin_x_mean=(mlcape_col, "mean"),
        bin_x_min=(mlcape_col, "min"),
        bin_x_max=(mlcape_col, "max"),
        exceedance_frequency=("MRMS_FFG_any_r100", "mean"),
        n=("MRMS_FFG_any_r100", "size"),
    )
    .reset_index(drop=True)
)

# Optional scatter sample for readability
if POINT_SAMPLE_N is not None and len(plot_df) > POINT_SAMPLE_N:
    plot_sample = plot_df.sample(
        n=int(POINT_SAMPLE_N),
        random_state=RANDOM_STATE,
    ).copy()
else:
    plot_sample = plot_df.copy()

rng = np.random.default_rng(RANDOM_STATE)
plot_sample["y_jitter"] = (
    plot_sample["MRMS_FFG_any_r100"].astype(float)
    + rng.normal(0.0, 0.035, size=len(plot_sample))
)


# --------------------------------------------------------------------------------------
# Readable x-axis label
# --------------------------------------------------------------------------------------
if "make_feature_label" in globals():
    try:
        x_label = make_feature_label(mlcape_col, radius_km=RADIUS_KM, max_len=1000)
    except Exception:
        x_label = mlcape_col
else:
    x_label = mlcape_col

x_label = str(x_label).replace("...", "").replace("…", "")
x_label_wrapped = "\n".join(
    textwrap.wrap(
        x_label,
        width=45,
        break_long_words=False,
        break_on_hyphens=False,
    )
)


# --------------------------------------------------------------------------------------
# Plot
# --------------------------------------------------------------------------------------
plt.rcParams.update({
    "font.size": FONT_SIZE,
    "axes.labelsize": FONT_SIZE,
    "xtick.labelsize": FONT_SIZE * 0.85,
    "ytick.labelsize": FONT_SIZE * 0.85,
})

fig, ax = plt.subplots(figsize=FIGSIZE)

ax.scatter(
    plot_sample[mlcape_col],
    plot_sample["y_jitter"],
    s=6,
    alpha=0.08,
    linewidths=0,
    rasterized=True,
    label="Training grid-point samples",
)

ax.plot(
    binned["bin_x_mean"],
    binned["exceedance_frequency"],
    linewidth=4,
    marker="o",
    markersize=9,
    label="Binned exceedance frequency",
)

ax.axvline(
    MLCAPE_THRESHOLD,
    linewidth=3,
    linestyle="--",
    label=f"{MLCAPE_THRESHOLD:.0f} J kg$^{{-1}}$ threshold",
)

ax.set_xlabel(x_label_wrapped)
ax.set_ylabel("Any MRMS > FFG within 100 km")

ax.set_ylim(-0.15, 1.15)
ax.set_yticks([0, 1])
ax.set_yticklabels(["No exceedance", "Exceedance"])

ax.grid(True, alpha=0.25)
ax.legend(
    loc="best",
    fontsize=FONT_SIZE * 0.62,
    frameon=True,
)

fig.tight_layout()

if SAVE_FIG:
    out_dir = os.path.join(PROJECT_DIR, f"mlcape_target_diagnostics_v33day2valid_r{int(RADIUS_KM)}")
    os.makedirs(out_dir, exist_ok=True)

    out_png = os.path.join(
        out_dir,
        f"mlcape_vs_mrms_ffg_any_r{int(RADIUS_KM)}km_training.png",
    )
    out_pdf = os.path.join(
        out_dir,
        f"mlcape_vs_mrms_ffg_any_r{int(RADIUS_KM)}km_training.pdf",
    )

    fig.savefig(out_png, dpi=DPI, bbox_inches="tight")
    fig.savefig(out_pdf, dpi=DPI, bbox_inches="tight")

    print("Saved:", out_png)
    print("Saved:", out_pdf)

plt.show()
plt.close(fig)

### ?? heatmaps, centroids, and area calculations

In [ ]:
# ======================================================================================
# RISK-AREA OCCURRENCE HEATMAPS + COMMON-CASE CATEGORICAL-COM DISPLACEMENT MARKERS
# + LOCALIZED PROBABILITY-MATCHED MEAN OF ML CONFIGS
#
# FULL REPLACEMENT CELL
#
# Run this AFTER the v33 viewer block has created:
#   df_radius_viewer
#
# Case matching:
#   For Marginal/Slight/Moderate:
#       valid case for a method = PP area > 0 AND WPC area > 0 AND method area > 0
#
#   For High:
#       if WPC issued no High risk anywhere, WPC is not required for non-WPC methods.
#
# Heatmap:
#   Uses all available cases for spatial occurrence accumulation.
#
# Tables/markers:
#   Use the common-case matched subset.
#
# New ML reporting method:
#   ML Local PMM 100km
#
#   For each date, the ML radius probability fields are combined using a localized
#   probability-matched mean over a 100-km ROI. The local PMM preserves the spatial
#   rank pattern of the ML ensemble-mean field, while drawing values from the local
#   pooled probability distribution across ML radius configurations.
#
# Plotted markers:
#   PP marker:
#       common-case mean PP categorical COM.
#
#   Forecast markers:
#       displacement-error markers, not raw mean forecast COMs.
#       Each forecast marker is shifted from the PP marker in the method's mean
#       signed displacement direction by the method's mean case-by-case COM
#       displacement error.
#
# Colorbar label:
#   "Case-occurrences"
# ======================================================================================

import os
import re
import math
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, ListedColormap, BoundaryNorm
from matplotlib.lines import Line2D

try:
    from IPython.display import display
except Exception:
    display = print

try:
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature
    HAS_CARTOPY_RISK_HEATMAP = True
except Exception:
    HAS_CARTOPY_RISK_HEATMAP = False

try:
    from scipy.spatial import cKDTree
    HAS_SCIPY_RISK_HEATMAP = True
except Exception:
    HAS_SCIPY_RISK_HEATMAP = False


# ======================================================================================
# USER CONTROLS
# ======================================================================================

HEATMAP_TRUTH_DEFINITION = "Any flood proxy"

HEATMAP_THRESHOLDS = [
    (0.05, "Marginal or greater"),
    (0.15, "Slight or greater"),
    (0.40, "Moderate or greater"),
    (0.70, "High risk"),
]

# Categorical midpoint / representative masses used for COM for ML, WPC, and PP.
RISK_CATEGORY_MASS = {
    "Marginal": 0.100,
    "Slight":   0.275,
    "Moderate": 0.550,
    "High":     0.850,
}

# ML heatmap counting mode:
#
#   "mean_radii_per_date"      <-- recommended
#       each date contributes the average binary occurrence across radii.
#       This makes ML comparable to WPC/PP because each case contributes at most 1.
#
#   "any_radius_per_date"
#       each date contributes 1 if any radius covers the pixel.
#
#   "date_radius_occurrences"
#       each date-radius configuration is one occurrence.
#       This is not recommended for the 3-column comparison.
#
HEATMAP_ML_COMBINE_MODE = "mean_radii_per_date"
# HEATMAP_ML_COMBINE_MODE = "any_radius_per_date"
# HEATMAP_ML_COMBINE_MODE = "date_radius_occurrences"

# Add localized probability-matched mean of the ML radius configurations to reporting.
HEATMAP_ADD_ML_LOCAL_PMM = True
HEATMAP_LOCAL_PMM_RADIUS_KM = 100.0
HEATMAP_LOCAL_PMM_METHOD_NAME = f"ML Local PMM {int(HEATMAP_LOCAL_PMM_RADIUS_KM)}km"

# Default: raw forecast/risk areas.
# Set True only if you want to expand each threshold mask before accumulating
# heatmaps, areas, and centroids.
HEATMAP_APPLY_VERIFY_ROI_EXPANSION = False
HEATMAP_VERIFY_ROI_KM = 40.0

# If you know the exact grid-cell area, set it here.
HEATMAP_CELL_AREA_KM2_OVERRIDE = None

# Plot style
HEATMAP_FONT_SIZE = 40
HEATMAP_PANEL_TITLE_SIZE = 40
HEATMAP_ROW_LABEL_SIZE = 40
HEATMAP_COLORBAR_FONT_SIZE = 34
HEATMAP_LEGEND_FONT_SIZE = 31

HEATMAP_FIGSIZE = (38, 39)
HEATMAP_DPI = 300
HEATMAP_POINT_SIZE = 13
HEATMAP_POINT_ALPHA = 0.95
HEATMAP_CENTROID_BASE_SIZE = 260

HEATMAP_BASELINE_BIN_COUNT = 5
HEATMAP_COUNT_LABEL_DECIMALS = 1

HEATMAP_PLOT_ALL_METHOD_CENTROIDS_ON_EACH_PANEL = True

HEATMAP_SAVE_FIG = True
HEATMAP_SAVE_TABLES = True

if "PROJECT_DIR" in globals():
    HEATMAP_OUTDIR = Path(PROJECT_DIR) / "risk_area_heatmaps_centroids_v33day2valid"
else:
    HEATMAP_OUTDIR = Path.cwd() / "risk_area_heatmaps_centroids_v33day2valid"

if "DEFAULT_EXTENT" in globals():
    HEATMAP_EXTENT = DEFAULT_EXTENT
else:
    HEATMAP_EXTENT = [-105.0, -80.5, 30.0, 50.0]

if "WPC_COL" not in globals():
    WPC_COL = "WPC_ERO_Risk"


plt.rcParams.update({
    "font.size": HEATMAP_FONT_SIZE,
    "axes.titlesize": HEATMAP_PANEL_TITLE_SIZE,
    "axes.labelsize": HEATMAP_FONT_SIZE,
    "xtick.labelsize": HEATMAP_COLORBAR_FONT_SIZE,
    "ytick.labelsize": HEATMAP_COLORBAR_FONT_SIZE,
    "legend.fontsize": HEATMAP_LEGEND_FONT_SIZE,
    "figure.titlesize": HEATMAP_FONT_SIZE,
})


# ======================================================================================
# INPUT VALIDATION
# ======================================================================================

if "df_radius_viewer" not in globals():
    raise RuntimeError("df_radius_viewer is not defined. Run the v33 viewer/metric block first.")

if WPC_COL not in df_radius_viewer.columns:
    raise RuntimeError(f"Missing WPC column {WPC_COL!r} in df_radius_viewer.")

required_base_cols = ["Date", "Lat", "Lon", "ML_Target_Radius_km", "ML_Forecast_Prob"]
missing_base_cols = [c for c in required_base_cols if c not in df_radius_viewer.columns]
if missing_base_cols:
    raise RuntimeError(f"df_radius_viewer is missing required columns: {missing_base_cols}")

if HEATMAP_ML_COMBINE_MODE not in {
    "mean_radii_per_date",
    "any_radius_per_date",
    "date_radius_occurrences",
}:
    raise ValueError(
        "HEATMAP_ML_COMBINE_MODE must be one of: "
        "'mean_radii_per_date', 'any_radius_per_date', 'date_radius_occurrences'."
    )

if HEATMAP_ADD_ML_LOCAL_PMM and not HAS_SCIPY_RISK_HEATMAP:
    raise RuntimeError("scipy.spatial.cKDTree is required for localized PMM.")


# ======================================================================================
# COLORS
# ======================================================================================

HEATMAP_THRESHOLD_COLOR_CONFIG = {
    "Marginal or greater": {
        "light": "#FFFFFF",
        "dark": "#006B2E",
        "name": "white_to_dark_green",
    },
    "Slight or greater": {
        "light": "#FFFFFF",
        "dark": "#A65F00",
        "name": "white_to_dark_amber",
    },
    "Moderate or greater": {
        "light": "#FFFFFF",
        "dark": "#8B0000",
        "name": "white_to_dark_red",
    },
    "High risk": {
        "light": "#FFFFFF",
        "dark": "#8A005D",
        "name": "white_to_dark_pink",
    },
}


def _risk_heatmap_continuous_cmap_for_threshold(threshold_label):
    cfg = HEATMAP_THRESHOLD_COLOR_CONFIG.get(
        str(threshold_label),
        {"light": "#FFFFFF", "dark": "#333333", "name": "white_to_dark_gray"},
    )
    return LinearSegmentedColormap.from_list(
        cfg["name"],
        [cfg["light"], cfg["dark"]],
        N=256,
    )


def _risk_heatmap_binned_cmap_norm_labels(threshold_label, pp_max_count, row_max_count):
    pp_max = float(pp_max_count) if np.isfinite(pp_max_count) else 0.0
    row_max = float(row_max_count) if np.isfinite(row_max_count) else 0.0

    if pp_max <= 0:
        baseline_top = max(1.0, row_max)
    else:
        baseline_top = pp_max

    top_upper = max(row_max, baseline_top)

    if top_upper <= baseline_top:
        top_upper = baseline_top + 1.0

    base_bounds = np.linspace(
        0.0,
        baseline_top,
        int(HEATMAP_BASELINE_BIN_COUNT) + 1,
    )

    bounds = np.concatenate([base_bounds, [top_upper]])

    bounds = np.asarray(bounds, dtype=float)
    for i in range(1, len(bounds)):
        if bounds[i] <= bounds[i - 1]:
            bounds[i] = bounds[i - 1] + 1.0e-6

    n_bins = len(bounds) - 1

    continuous = _risk_heatmap_continuous_cmap_for_threshold(threshold_label)
    color_positions = np.linspace(0.05, 1.0, n_bins)
    colors = [continuous(x) for x in color_positions]

    cmap = ListedColormap(colors)
    norm = BoundaryNorm(bounds, cmap.N, clip=True)

    labels = []
    decimals = int(HEATMAP_COUNT_LABEL_DECIMALS)

    for i in range(n_bins):
        lo = bounds[i]
        hi = bounds[i + 1]

        if i == n_bins - 1:
            labels.append(f">{baseline_top:.{decimals}f}")
        else:
            labels.append(f"{lo:.{decimals}f}–{hi:.{decimals}f}")

    ticks = 0.5 * (bounds[:-1] + bounds[1:])

    return cmap, norm, bounds, ticks, labels, baseline_top, top_upper


# ======================================================================================
# COLUMN DISCOVERY
# ======================================================================================

def _risk_heatmap_find_pp_col(df, truth_definition="Any flood proxy"):
    candidates = [
        f"PP_{truth_definition}",
        "PP_Any flood proxy",
        "PP_Any Flood Proxy",
        "PP_ANY",
        "PP_UFVS_ANY",
    ]

    for c in candidates:
        if c in df.columns:
            return c

    any_cols = [
        c for c in df.columns
        if str(c).startswith("PP_") and "any" in str(c).lower()
    ]

    if any_cols:
        return any_cols[0]

    raise RuntimeError(
        "Could not find PP Any Flood Proxy column. "
        f"Available PP columns: {[c for c in df.columns if str(c).startswith('PP_')]}"
    )


HEATMAP_PP_COL = _risk_heatmap_find_pp_col(df_radius_viewer, HEATMAP_TRUTH_DEFINITION)

print("Using PP column:", HEATMAP_PP_COL)
print("Using WPC column:", WPC_COL)
print("ML combine mode:", HEATMAP_ML_COMBINE_MODE)
print("Add localized ML PMM:", HEATMAP_ADD_ML_LOCAL_PMM)
print("Localized ML PMM ROI:", HEATMAP_LOCAL_PMM_RADIUS_KM, "km")
print("Localized ML PMM method name:", HEATMAP_LOCAL_PMM_METHOD_NAME)
print("Categorical COM masses:", RISK_CATEGORY_MASS)
print("Output directory:", HEATMAP_OUTDIR)


# ======================================================================================
# GENERAL HELPERS
# ======================================================================================

def _risk_heatmap_keyed(df):
    out = df.copy()
    out["Date"] = out["Date"].astype(str).str.slice(0, 8)
    out["Lat"] = pd.to_numeric(out["Lat"], errors="coerce")
    out["Lon"] = pd.to_numeric(out["Lon"], errors="coerce")
    out["ML_Target_Radius_km"] = pd.to_numeric(out["ML_Target_Radius_km"], errors="coerce")
    out["__RadiusInt"] = out["ML_Target_Radius_km"].round().astype("Int64")
    out["__ModelLabel"] = out["ML_Model_Label"].astype(str) if "ML_Model_Label" in out.columns else out["__RadiusInt"].map(lambda r: model_label_for_radius(r))
    out["__LatKey"] = out["Lat"].round(5)
    out["__LonKey"] = out["Lon"].round(5)
    return out


def _risk_heatmap_normalize_probability_values(series, assume_category=False):
    """
    Convert common risk/probability encodings to 0-1.

    assume_category=True:
      0,1,2,3,4 maps to none/marginal/slight/moderate/high.

    assume_category=False:
      0-1 values are treated as probabilities. This avoids converting binary
      0/1 arrays into categorical marginal-only values.
    """
    s = pd.Series(series)
    vals = pd.to_numeric(s, errors="coerce").to_numpy(float)

    if np.isfinite(vals).any():
        finite = vals[np.isfinite(vals)]
        vmax = float(np.nanmax(finite))
        unique = set(np.unique(finite).astype(float).tolist())

        if assume_category and vmax <= 4.0 and unique.issubset({0.0, 1.0, 2.0, 3.0, 4.0}):
            mapped = np.full(vals.shape, np.nan, dtype=float)
            mapped[vals == 0] = 0.00
            mapped[vals == 1] = 0.05
            mapped[vals == 2] = 0.15
            mapped[vals == 3] = 0.40
            mapped[vals == 4] = 0.70
            return mapped

        if vmax > 1.0:
            return vals / 100.0

        return vals

    lower = s.astype(str).str.lower().str.strip()
    mapped = np.full(len(s), np.nan, dtype=float)

    mapped[lower.str.contains("marginal", na=False)] = 0.05
    mapped[lower.str.contains("slight", na=False)] = 0.15
    mapped[lower.str.contains("moderate", na=False)] = 0.40
    mapped[lower.str.contains("high", na=False)] = 0.70
    mapped[lower.str.contains("none|no risk", na=False)] = 0.00

    return mapped


def _risk_heatmap_categorical_mass(values):
    values = np.asarray(values, dtype=float)
    mass = np.zeros(values.shape, dtype=float)

    good = np.isfinite(values)

    mass[good & (values >= 0.05) & (values < 0.15)] = float(RISK_CATEGORY_MASS["Marginal"])
    mass[good & (values >= 0.15) & (values < 0.40)] = float(RISK_CATEGORY_MASS["Slight"])
    mass[good & (values >= 0.40) & (values < 0.70)] = float(RISK_CATEGORY_MASS["Moderate"])
    mass[good & (values >= 0.70)] = float(RISK_CATEGORY_MASS["High"])

    return mass


def _risk_heatmap_latlon_to_unit_xyz(lat, lon):
    lat_rad = np.deg2rad(np.asarray(lat, dtype=float))
    lon_rad = np.deg2rad(np.asarray(lon, dtype=float))
    cos_lat = np.cos(lat_rad)

    return np.column_stack([
        cos_lat * np.cos(lon_rad),
        cos_lat * np.sin(lon_rad),
        np.sin(lat_rad),
    ]).astype(np.float64)


def _risk_heatmap_km_to_unit_sphere_chord_radius(radius_km):
    earth_radius_km = 6371.0
    angular_radius = float(radius_km) / earth_radius_km
    return 2.0 * np.sin(angular_radius / 2.0)


def _risk_heatmap_haversine_km(lon1, lat1, lon2, lat2):
    vals = [lon1, lat1, lon2, lat2]
    if not all(np.isfinite(vals)):
        return np.nan

    r = 6371.0

    lon1 = np.deg2rad(float(lon1))
    lat1 = np.deg2rad(float(lat1))
    lon2 = np.deg2rad(float(lon2))
    lat2 = np.deg2rad(float(lat2))

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = (
        np.sin(dlat / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0) ** 2
    )
    c = 2.0 * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))

    return float(r * c)


def _risk_heatmap_signed_dxdy_km(from_lon, from_lat, to_lon, to_lat):
    """
    Approximate local east/north components from point A to point B.

    Returns:
      dx_km: positive eastward
      dy_km: positive northward
    """
    vals = [from_lon, from_lat, to_lon, to_lat]
    if not all(np.isfinite(vals)):
        return np.nan, np.nan

    mean_lat_rad = np.deg2rad((float(from_lat) + float(to_lat)) / 2.0)

    dx_km = 111.32 * np.cos(mean_lat_rad) * (float(to_lon) - float(from_lon))
    dy_km = 110.574 * (float(to_lat) - float(from_lat))

    return float(dx_km), float(dy_km)


def _risk_heatmap_shift_lonlat_by_dxdy(lon0, lat0, dx_km, dy_km):
    """
    Shift lon/lat by approximate local east/north km components.
    """
    vals = [lon0, lat0, dx_km, dy_km]
    if not all(np.isfinite(vals)):
        return np.nan, np.nan

    lat_new = float(lat0) + float(dy_km) / 110.574

    cos_lat = np.cos(np.deg2rad(float(lat0)))
    if abs(cos_lat) < 1.0e-6:
        return np.nan, np.nan

    lon_new = float(lon0) + float(dx_km) / (111.32 * cos_lat)

    return float(lon_new), float(lat_new)


def _risk_heatmap_build_neighbor_indices(lat, lon, radius_km):
    """
    Precompute neighbor lists within radius_km on the reference grid.
    """
    if not HAS_SCIPY_RISK_HEATMAP:
        raise RuntimeError("scipy.spatial.cKDTree is required for neighbor lookup.")

    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)

    xyz = _risk_heatmap_latlon_to_unit_xyz(lat, lon)
    tree = cKDTree(xyz)
    chord_radius = _risk_heatmap_km_to_unit_sphere_chord_radius(radius_km)

    neighbor_lists = tree.query_ball_point(xyz, r=chord_radius)
    neighbor_lists = [np.asarray(idx, dtype=int) for idx in neighbor_lists]

    return neighbor_lists


def _risk_heatmap_local_probability_matched_mean(ml_stack, local_neighbor_indices):
    """
    Localized probability-matched mean across ML radius configurations.

    ml_stack:
      shape = (n_configs, n_grid)

    For each grid point i:
      1. Compute the ML ensemble mean field.
      2. In the local ROI around i, get the distribution of local ensemble-mean values.
      3. In the same local ROI, pool probabilities from all ML configs.
      4. Match the rank/quantile of the target point's ensemble mean to the pooled
         local ML probability distribution.

    This keeps the spatial pattern/rank of the ensemble mean while restoring local
    distributional sharpness from the member/configuration probabilities.
    """
    ml_stack = np.asarray(ml_stack, dtype=float)

    if ml_stack.ndim != 2:
        raise ValueError(f"Expected ml_stack to be 2D, got shape {ml_stack.shape}")

    n_configs, n_grid = ml_stack.shape

    with np.errstate(invalid="ignore"):
        ens_mean = np.nanmean(ml_stack, axis=0)

    out = np.full(n_grid, np.nan, dtype=float)

    for i in range(n_grid):
        target_mean = ens_mean[i]

        if not np.isfinite(target_mean):
            continue

        neigh = local_neighbor_indices[i]

        if neigh.size == 0:
            out[i] = target_mean
            continue

        local_mean = ens_mean[neigh]
        local_mean = local_mean[np.isfinite(local_mean)]

        pooled = ml_stack[:, neigh].reshape(-1)
        pooled = pooled[np.isfinite(pooled)]

        if pooled.size == 0:
            out[i] = target_mean
            continue

        if local_mean.size <= 1:
            out[i] = float(np.nanmedian(pooled))
            continue

        sorted_local_mean = np.sort(local_mean)
        sorted_pooled = np.sort(pooled)

        rank = np.searchsorted(sorted_local_mean, target_mean, side="left")
        q = rank / max(local_mean.size - 1, 1)
        q = float(np.clip(q, 0.0, 1.0))

        pooled_idx = int(np.round(q * (sorted_pooled.size - 1)))
        pooled_idx = int(np.clip(pooled_idx, 0, sorted_pooled.size - 1))

        out[i] = sorted_pooled[pooled_idx]

    return np.clip(out, 0.0, 1.0)


def _risk_heatmap_expand_mask(mask, lat, lon, radius_km):
    mask = np.asarray(mask, dtype=bool)

    if not HEATMAP_APPLY_VERIFY_ROI_EXPANSION:
        return mask

    if not HAS_SCIPY_RISK_HEATMAP:
        raise RuntimeError("scipy.spatial.cKDTree is required for ROI expansion.")

    if radius_km is None or float(radius_km) <= 0:
        return mask

    if not np.any(mask):
        return np.zeros_like(mask, dtype=bool)

    if np.all(mask):
        return np.ones_like(mask, dtype=bool)

    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)

    xyz = _risk_heatmap_latlon_to_unit_xyz(lat, lon)
    tree = cKDTree(xyz)
    chord_radius = _risk_heatmap_km_to_unit_sphere_chord_radius(radius_km)

    out = np.zeros_like(mask, dtype=bool)
    yes_idx = np.flatnonzero(mask)

    for idx in yes_idx:
        neigh = tree.query_ball_point(xyz[idx], r=chord_radius)
        out[np.asarray(neigh, dtype=int)] = True

    return out


def _risk_heatmap_threshold_mask(values, threshold, lat, lon):
    values = np.asarray(values, dtype=float)
    raw = np.isfinite(values) & (values >= float(threshold))

    return _risk_heatmap_expand_mask(
        raw,
        lat=lat,
        lon=lon,
        radius_km=HEATMAP_VERIFY_ROI_KM,
    )


def _risk_heatmap_estimate_cell_area_km2(lat, lon):
    if HEATMAP_CELL_AREA_KM2_OVERRIDE is not None:
        return float(HEATMAP_CELL_AREA_KM2_OVERRIDE)

    if not HAS_SCIPY_RISK_HEATMAP:
        print("WARNING: scipy unavailable; using 1.0 km^2 placeholder cell area.")
        return 1.0

    lat = np.asarray(lat, dtype=float)
    lon = np.asarray(lon, dtype=float)
    good = np.isfinite(lat) & np.isfinite(lon)

    xyz = _risk_heatmap_latlon_to_unit_xyz(lat[good], lon[good])
    tree = cKDTree(xyz)

    dist, _ = tree.query(xyz, k=2)
    nn_chord = dist[:, 1]

    nn_angle = 2.0 * np.arcsin(np.clip(nn_chord / 2.0, 0.0, 1.0))
    nn_km = 6371.0 * nn_angle

    dx = float(np.nanmedian(nn_km))
    area = dx ** 2

    print(f"Estimated representative grid spacing: {dx:.2f} km")
    print(f"Estimated representative grid-cell area: {area:.2f} km^2")
    print("Set HEATMAP_CELL_AREA_KM2_OVERRIDE if you want to force a known value.")

    return area


def _risk_heatmap_make_reference_grid(dfk, dates, radii):
    first_date = str(dates[0])
    first_radius = int(radii[0])

    ref = dfk[
        (dfk["Date"].astype(str) == first_date)
        & (dfk["__RadiusInt"].eq(first_radius))
    ][["Lat", "Lon", "__LatKey", "__LonKey"]].copy()

    ref = (
        ref
        .dropna(subset=["Lat", "Lon"])
        .drop_duplicates(subset=["__LatKey", "__LonKey"])
        .sort_values(["__LatKey", "__LonKey"])
        .reset_index(drop=True)
    )

    if ref.empty:
        raise RuntimeError("Reference grid is empty.")

    return ref


def _risk_heatmap_align_values_to_ref(ref, sub, value_col, assume_category=False):
    small = sub[["__LatKey", "__LonKey", value_col]].copy()
    small = small.drop_duplicates(subset=["__LatKey", "__LonKey"])

    aligned = ref[["__LatKey", "__LonKey"]].merge(
        small,
        on=["__LatKey", "__LonKey"],
        how="left",
    )

    return _risk_heatmap_normalize_probability_values(
        aligned[value_col],
        assume_category=assume_category,
    )


def _risk_heatmap_area_and_categorical_com(values, threshold, lon, lat, cell_area_km2):
    values = np.asarray(values, dtype=float)
    lon = np.asarray(lon, dtype=float)
    lat = np.asarray(lat, dtype=float)

    mask = _risk_heatmap_threshold_mask(
        values,
        threshold=threshold,
        lat=lat,
        lon=lon,
    )

    good_area = mask & np.isfinite(lon) & np.isfinite(lat)
    area_km2 = float(np.sum(good_area) * cell_area_km2)

    if area_km2 <= 0:
        return np.nan, np.nan, 0.0

    mass = _risk_heatmap_categorical_mass(values)
    mass = np.where(mask & np.isfinite(lon) & np.isfinite(lat), mass, 0.0)

    mass_sum = float(np.nansum(mass))

    if mass_sum <= 0:
        com_lon = float(np.mean(lon[good_area]))
        com_lat = float(np.mean(lat[good_area]))
        return com_lon, com_lat, area_km2

    com_lon = float(np.nansum(lon * mass) / mass_sum)
    com_lat = float(np.nansum(lat * mass) / mass_sum)

    return com_lon, com_lat, area_km2


def _risk_heatmap_method_sort_key(method):
    s = str(method)

    m = re.search(r"ML r(\d+)", s)
    if m:
        return (0, int(m.group(1)))

    if s == "ML Ensemble Mean":
        return (1, 0)

    if s.startswith("ML Local PMM"):
        return (1, 1)

    if s == "WPC ERO":
        return (2, 0)

    if s == "Practically Perfect":
        return (3, 0)

    return (9, s)


def _risk_heatmap_setup_map_axis(ax):
    if HAS_CARTOPY_RISK_HEATMAP:
        ax.set_extent(HEATMAP_EXTENT, crs=ccrs.PlateCarree())

        try:
            ax.add_feature(cfeature.COASTLINE.with_scale("50m"), linewidth=1.2, edgecolor="0.25")
        except Exception:
            try:
                ax.coastlines(resolution="50m", linewidth=1.2)
            except Exception:
                pass

        try:
            ax.add_feature(cfeature.BORDERS.with_scale("50m"), linewidth=1.0, edgecolor="0.25")
        except Exception:
            pass

        try:
            ax.add_feature(cfeature.STATES.with_scale("50m"), linewidth=0.8, edgecolor="0.30")
        except Exception:
            pass

    else:
        ax.set_xlim(HEATMAP_EXTENT[0], HEATMAP_EXTENT[1])
        ax.set_ylim(HEATMAP_EXTENT[2], HEATMAP_EXTENT[3])
        ax.grid(True, alpha=0.25)

    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(labelsize=HEATMAP_COLORBAR_FONT_SIZE)


def _risk_heatmap_centroid_style(method):
    method = str(method)

    base_styles = {
        "ML Ensemble Mean": {
            "marker": "*",
            "color": "black",
            "edgecolor": "white",
            "linewidth": 2.6,
            "s": 430,
            "label": "ML Ensemble Mean",
        },
        HEATMAP_LOCAL_PMM_METHOD_NAME: {
            "marker": "H",
            "color": "#CC79A7",
            "edgecolor": "black",
            "linewidth": 2.2,
            "s": 380,
            "label": HEATMAP_LOCAL_PMM_METHOD_NAME,
        },
        "WPC ERO": {
            "marker": "X",
            "color": "#D34737",
            "edgecolor": "black",
            "linewidth": 2.2,
            "s": 360,
            "label": "WPC ERO",
        },
        "Practically Perfect": {
            "marker": "P",
            "color": "#0072B2",
            "edgecolor": "black",
            "linewidth": 2.2,
            "s": 360,
            "label": "Practically Perfect",
        },
    }

    if method in base_styles:
        return base_styles[method].copy()

    if method.startswith("ML Local PMM"):
        return {
            "marker": "H",
            "color": "#CC79A7",
            "edgecolor": "black",
            "linewidth": 2.2,
            "s": 380,
            "label": method,
        }

    ml_markers = ["o", "^", "s", "D", "v", "<", ">"]
    ml_colors = ["#00A087", "#4DBBD5", "#E64B35", "#3C5488", "#F39B7F", "#8491B4", "#91D1C2"]

    m = re.search(r"ML r(\d+)", method)
    if m:
        radius = int(m.group(1))

        if "MODEL_SPECS" in globals():
            model_order = [str(x["label"]) for x in MODEL_SPECS]
        else:
            model_order = sorted(
                df_radius_viewer["ML_Model_Label"].dropna().astype(str).unique().tolist()
            )

        method_label = method.replace("ML ", "").replace(" km", "")
        idx = model_order.index(method_label) if method_label in model_order else 0

        return {
            "marker": ml_markers[idx % len(ml_markers)],
            "color": ml_colors[idx % len(ml_colors)],
            "edgecolor": "black",
            "linewidth": 1.8,
            "s": HEATMAP_CENTROID_BASE_SIZE,
            "label": method,
        }

    return {
        "marker": "o",
        "color": "gray",
        "edgecolor": "black",
        "linewidth": 1.8,
        "s": HEATMAP_CENTROID_BASE_SIZE,
        "label": method,
    }


# ======================================================================================
# MAIN HEATMAP / DAILY AREA / DAILY CATEGORICAL-COM COMPUTATION
# ======================================================================================

def compute_risk_heatmaps_centroids_area_displacement(df):
    dfk = _risk_heatmap_keyed(df)

    radii = sorted(dfk["__RadiusInt"].dropna().astype(int).unique().tolist())
    models = dfk[["__ModelLabel", "__RadiusInt"]].drop_duplicates().to_dict("records")
    dates = sorted(dfk["Date"].dropna().astype(str).unique().tolist())

    if not models:
        raise RuntimeError("No ML model labels found.")
    if not dates:
        raise RuntimeError("No Date values found.")

    ref = _risk_heatmap_make_reference_grid(dfk, dates, radii)

    lon = ref["Lon"].to_numpy(float)
    lat = ref["Lat"].to_numpy(float)
    n_grid = len(ref)

    cell_area_km2 = _risk_heatmap_estimate_cell_area_km2(lat, lon)
    domain_area_km2 = float(n_grid * cell_area_km2)

    local_pmm_neighbors = None
    if HEATMAP_ADD_ML_LOCAL_PMM:
        print(f"Precomputing localized PMM neighbors within {HEATMAP_LOCAL_PMM_RADIUS_KM:g} km...")
        local_pmm_neighbors = _risk_heatmap_build_neighbor_indices(
            lat=lat,
            lon=lon,
            radius_km=HEATMAP_LOCAL_PMM_RADIUS_KM,
        )
        neighbor_counts = np.array([len(x) for x in local_pmm_neighbors], dtype=float)
        print(
            "Localized PMM neighbor count:",
            f"median={np.nanmedian(neighbor_counts):.0f},",
            f"min={np.nanmin(neighbor_counts):.0f},",
            f"max={np.nanmax(neighbor_counts):.0f}",
        )

    print(f"Dates: {len(dates)}")
    print(f"Models: {[x['__ModelLabel'] for x in models]}")
    print(f"Reference grid points: {n_grid:,}")
    print(f"Cell area used: {cell_area_km2:.2f} km^2")
    print(f"Approximate domain area used: {domain_area_km2:,.1f} km^2")
    print(f"Apply {HEATMAP_VERIFY_ROI_KM:g}-km expansion:", HEATMAP_APPLY_VERIFY_ROI_EXPANSION)

    panel_names = ["ML", "WPC ERO", "Practically Perfect"]

    heat_count = {}
    heat_den = {}

    for _, threshold_label in HEATMAP_THRESHOLDS:
        for panel in panel_names:
            heat_count[(threshold_label, panel)] = np.zeros(n_grid, dtype=float)
            heat_den[(threshold_label, panel)] = 0

    area_daily_rows = []
    centroid_daily_rows = []

    for i, date in enumerate(dates, start=1):
        base = dfk[
            (dfk["Date"].astype(str) == str(date))
            & (dfk["__ModelLabel"].astype(str) == str(models[0]["__ModelLabel"]))
        ].copy()

        if base.empty:
            print(f"Skipping {date}: no base rows.")
            continue

        wpc_vals = _risk_heatmap_align_values_to_ref(
            ref,
            base,
            WPC_COL,
            assume_category=True,
        )

        pp_vals = _risk_heatmap_align_values_to_ref(
            ref,
            base,
            HEATMAP_PP_COL,
            assume_category=True,
        )

        ml_vals_by_model = {}

        for model_spec in models:
            model_label = str(model_spec["__ModelLabel"])
            radius = int(model_spec["__RadiusInt"])
            sub = dfk[
                (dfk["Date"].astype(str) == str(date))
                & (dfk["__RadiusInt"].eq(int(radius)))
                & (dfk["__ModelLabel"].astype(str) == model_label)
            ].copy()

            if sub.empty:
                raise RuntimeError(f"Date {date} model {model_label} is missing from df_radius_viewer.")

            vals = _risk_heatmap_align_values_to_ref(
                ref,
                sub,
                "ML_Forecast_Prob",
                assume_category=False,
            )
            ml_vals_by_model[model_label] = vals

        ml_stack = np.vstack([ml_vals_by_model[str(spec["__ModelLabel"])] for spec in models])

        with np.errstate(invalid="ignore"):
            ml_ensemble_mean_vals = np.nanmean(ml_stack, axis=0)

        method_values = {
            **{f"ML {str(spec['__ModelLabel'])}": ml_vals_by_model[str(spec["__ModelLabel"])] for spec in models},
            "ML Ensemble Mean": ml_ensemble_mean_vals,
        }

        if HEATMAP_ADD_ML_LOCAL_PMM:
            ml_local_pmm_vals = _risk_heatmap_local_probability_matched_mean(
                ml_stack=ml_stack,
                local_neighbor_indices=local_pmm_neighbors,
            )
            method_values[HEATMAP_LOCAL_PMM_METHOD_NAME] = ml_local_pmm_vals

        method_values["WPC ERO"] = wpc_vals
        method_values["Practically Perfect"] = pp_vals

        for threshold, threshold_label in HEATMAP_THRESHOLDS:
            ml_masks_this_date = []

            for model_spec in models:
                model_label = str(model_spec["__ModelLabel"])
                ml_mask = _risk_heatmap_threshold_mask(
                    ml_vals_by_model[model_label],
                    threshold=threshold,
                    lat=lat,
                    lon=lon,
                )
                ml_masks_this_date.append(ml_mask.astype(float))

                if HEATMAP_ML_COMBINE_MODE == "date_radius_occurrences":
                    heat_count[(threshold_label, "ML")] += ml_mask.astype(float)
                    heat_den[(threshold_label, "ML")] += 1

            if HEATMAP_ML_COMBINE_MODE == "any_radius_per_date":
                ml_any_mask = np.any(np.vstack(ml_masks_this_date).astype(bool), axis=0)
                heat_count[(threshold_label, "ML")] += ml_any_mask.astype(float)
                heat_den[(threshold_label, "ML")] += 1

            elif HEATMAP_ML_COMBINE_MODE == "mean_radii_per_date":
                ml_mean_mask = np.nanmean(np.vstack(ml_masks_this_date), axis=0)
                heat_count[(threshold_label, "ML")] += ml_mean_mask.astype(float)
                heat_den[(threshold_label, "ML")] += 1

            wpc_mask = _risk_heatmap_threshold_mask(
                wpc_vals,
                threshold=threshold,
                lat=lat,
                lon=lon,
            )
            pp_mask = _risk_heatmap_threshold_mask(
                pp_vals,
                threshold=threshold,
                lat=lat,
                lon=lon,
            )

            heat_count[(threshold_label, "WPC ERO")] += wpc_mask.astype(float)
            heat_den[(threshold_label, "WPC ERO")] += 1

            heat_count[(threshold_label, "Practically Perfect")] += pp_mask.astype(float)
            heat_den[(threshold_label, "Practically Perfect")] += 1

            for method, vals in method_values.items():
                com_lon, com_lat, area_km2 = _risk_heatmap_area_and_categorical_com(
                    vals,
                    threshold=threshold,
                    lon=lon,
                    lat=lat,
                    cell_area_km2=cell_area_km2,
                )

                area_daily_rows.append({
                    "Date": str(date),
                    "Risk Threshold": threshold_label,
                    "Threshold": float(threshold),
                    "Method": method,
                    "Area km^2": float(area_km2),
                    "Area Fraction": float(area_km2 / domain_area_km2) if domain_area_km2 > 0 else np.nan,
                })

                centroid_daily_rows.append({
                    "Date": str(date),
                    "Risk Threshold": threshold_label,
                    "Threshold": float(threshold),
                    "Method": method,
                    "COM Lon": com_lon,
                    "COM Lat": com_lat,
                    "Area km^2": float(area_km2),
                })

        if i % 5 == 0 or i == len(dates):
            print(f"  {i:03d}/{len(dates):03d} {date}: accumulated")

    df_area_daily = pd.DataFrame(area_daily_rows)
    df_centroid_daily = pd.DataFrame(centroid_daily_rows)

    return {
        "reference_grid": ref,
        "lon": lon,
        "lat": lat,
        "cell_area_km2": cell_area_km2,
        "domain_area_km2": domain_area_km2,
        "dates": dates,
        "radii": radii,
        "heat_count": heat_count,
        "heat_den": heat_den,
        "area_daily": df_area_daily,
        "centroid_daily": df_centroid_daily,
    }


# ======================================================================================
# COMMON-CASE SUMMARIES + DISPLACEMENT-PLOT MARKERS
# ======================================================================================

def build_common_case_risk_summaries(results):
    df_area_daily = results["area_daily"].copy()
    df_centroid_daily = results["centroid_daily"].copy()

    key_cols = ["Date", "Risk Threshold", "Threshold"]

    area_pivot = (
        df_area_daily
        .pivot_table(
            index=key_cols,
            columns="Method",
            values="Area km^2",
            aggfunc="first",
        )
        .reset_index()
    )
    area_pivot.columns.name = None

    methods = sorted(
        df_area_daily["Method"].dropna().astype(str).unique().tolist(),
        key=_risk_heatmap_method_sort_key,
    )

    if "Practically Perfect" not in methods:
        raise RuntimeError("Practically Perfect is missing from the area table.")

    if "WPC ERO" not in methods:
        raise RuntimeError("WPC ERO is missing from the area table.")

    threshold_order = {label: i for i, (_, label) in enumerate(HEATMAP_THRESHOLDS)}

    policy_rows = []

    for threshold, threshold_label in HEATMAP_THRESHOLDS:
        sub = area_pivot[area_pivot["Risk Threshold"].eq(threshold_label)].copy()

        pp_nonzero = int(np.sum(pd.to_numeric(sub["Practically Perfect"], errors="coerce").fillna(0) > 0))
        wpc_nonzero = int(np.sum(pd.to_numeric(sub["WPC ERO"], errors="coerce").fillna(0) > 0))

        require_wpc = True
        exception_used = False

        if str(threshold_label).lower() == "high risk" and wpc_nonzero == 0:
            require_wpc = False
            exception_used = True

        policy_rows.append({
            "Risk Threshold": threshold_label,
            "Threshold": float(threshold),
            "Total Cases": int(len(sub)),
            "PP Nonzero Cases": pp_nonzero,
            "WPC Nonzero Cases": wpc_nonzero,
            "Require WPC For Non-WPC Methods": bool(require_wpc),
            "High-Risk WPC-Absent Exception Used": bool(exception_used),
        })

    df_common_case_policy = pd.DataFrame(policy_rows)

    def _valid_case_rows_for_method(threshold_label, method):
        sub = area_pivot[area_pivot["Risk Threshold"].eq(threshold_label)].copy()

        if method not in sub.columns:
            return sub.iloc[0:0].copy()

        pp_area = pd.to_numeric(sub["Practically Perfect"], errors="coerce").fillna(0.0)
        wpc_area = pd.to_numeric(sub["WPC ERO"], errors="coerce").fillna(0.0)
        method_area = pd.to_numeric(sub[method], errors="coerce").fillna(0.0)

        policy = df_common_case_policy[df_common_case_policy["Risk Threshold"].eq(threshold_label)].iloc[0]
        require_wpc_for_non_wpc = bool(policy["Require WPC For Non-WPC Methods"])

        valid = (pp_area > 0) & (method_area > 0)

        if method == "WPC ERO":
            valid = valid & (wpc_area > 0)
        else:
            if require_wpc_for_non_wpc:
                valid = valid & (wpc_area > 0)

        return sub.loc[valid].copy()

    # ------------------------------------------------------------------
    # Area summaries: common-case matched
    # ------------------------------------------------------------------
    area_summary_rows = []

    for threshold, threshold_label in HEATMAP_THRESHOLDS:
        sub_all = area_pivot[area_pivot["Risk Threshold"].eq(threshold_label)].copy()

        pp_total_nonzero = int(np.sum(pd.to_numeric(sub_all["Practically Perfect"], errors="coerce").fillna(0) > 0))
        wpc_total_nonzero = int(np.sum(pd.to_numeric(sub_all["WPC ERO"], errors="coerce").fillna(0) > 0))

        for method in methods:
            valid_rows = _valid_case_rows_for_method(threshold_label, method)

            method_nonzero_cases = (
                int(np.sum(pd.to_numeric(sub_all[method], errors="coerce").fillna(0) > 0))
                if method in sub_all.columns
                else 0
            )

            if len(valid_rows) == 0:
                area_summary_rows.append({
                    "Risk Threshold": threshold_label,
                    "Threshold": float(threshold),
                    "Method": method,
                    "Total Cases": int(len(sub_all)),
                    "PP Nonzero Cases": pp_total_nonzero,
                    "WPC Nonzero Cases": wpc_total_nonzero,
                    "Method Nonzero Cases": method_nonzero_cases,
                    "N Common Cases": 0,
                    "Mean Area km^2": np.nan,
                    "Median Area km^2": np.nan,
                    "Max Area km^2": np.nan,
                    "Mean Area Fraction": np.nan,
                    "Matched PP Mean Area km^2": np.nan,
                    "Matched PP Median Area km^2": np.nan,
                    "Matched PP Mean Area Fraction": np.nan,
                    "Area Error vs Matched PP km^2": np.nan,
                    "Absolute Area Error vs Matched PP km^2": np.nan,
                    "Area Ratio vs Matched PP": np.nan,
                    "Percent Area Error vs Matched PP": np.nan,
                })
                continue

            method_area = pd.to_numeric(valid_rows[method], errors="coerce").to_numpy(float)
            pp_area = pd.to_numeric(valid_rows["Practically Perfect"], errors="coerce").to_numpy(float)

            method_area_fraction = method_area / float(results["domain_area_km2"])
            pp_area_fraction = pp_area / float(results["domain_area_km2"])

            mean_method_area = float(np.nanmean(method_area))
            mean_pp_area = float(np.nanmean(pp_area))
            area_error = mean_method_area - mean_pp_area

            area_summary_rows.append({
                "Risk Threshold": threshold_label,
                "Threshold": float(threshold),
                "Method": method,
                "Total Cases": int(len(sub_all)),
                "PP Nonzero Cases": pp_total_nonzero,
                "WPC Nonzero Cases": wpc_total_nonzero,
                "Method Nonzero Cases": method_nonzero_cases,
                "N Common Cases": int(len(valid_rows)),
                "Mean Area km^2": mean_method_area,
                "Median Area km^2": float(np.nanmedian(method_area)),
                "Max Area km^2": float(np.nanmax(method_area)),
                "Mean Area Fraction": float(np.nanmean(method_area_fraction)),
                "Matched PP Mean Area km^2": mean_pp_area,
                "Matched PP Median Area km^2": float(np.nanmedian(pp_area)),
                "Matched PP Mean Area Fraction": float(np.nanmean(pp_area_fraction)),
                "Area Error vs Matched PP km^2": area_error,
                "Absolute Area Error vs Matched PP km^2": abs(area_error),
                "Area Ratio vs Matched PP": mean_method_area / mean_pp_area if mean_pp_area > 0 else np.nan,
                "Percent Area Error vs Matched PP": 100.0 * area_error / mean_pp_area if mean_pp_area > 0 else np.nan,
            })

    df_area_summary_common = pd.DataFrame(area_summary_rows)
    df_area_error_vs_pp_common = df_area_summary_common.copy()

    # ------------------------------------------------------------------
    # COM/displacement summaries: common-case matched
    # ------------------------------------------------------------------
    centroid_summary_rows = []
    displacement_daily_rows = []

    pp_centroids = (
        df_centroid_daily[df_centroid_daily["Method"].eq("Practically Perfect")]
        [key_cols + ["COM Lon", "COM Lat", "Area km^2"]]
        .rename(columns={
            "COM Lon": "Matched PP COM Lon",
            "COM Lat": "Matched PP COM Lat",
            "Area km^2": "Matched PP Area km^2",
        })
    )

    for threshold, threshold_label in HEATMAP_THRESHOLDS:
        for method in methods:
            valid_rows = _valid_case_rows_for_method(threshold_label, method)
            valid_keys = valid_rows[key_cols].copy()

            empty_summary = {
                "Risk Threshold": threshold_label,
                "Threshold": float(threshold),
                "Method": method,
                "N Common Cases": 0,
                "Mean Forecast COM Lon": np.nan,
                "Mean Forecast COM Lat": np.nan,
                "Matched PP Mean COM Lon": np.nan,
                "Matched PP Mean COM Lat": np.nan,
                "Mean Signed Dx km": np.nan,
                "Mean Signed Dy km": np.nan,
                "Mean Signed Vector Magnitude km": np.nan,
                "Mean Displacement Error vs PP km": np.nan,
                "Median Displacement Error vs PP km": np.nan,
                "Directional Cancellation Ratio": np.nan,
                "Plot Anchor PP COM Lon": np.nan,
                "Plot Anchor PP COM Lat": np.nan,
                "Displacement Plot Lon": np.nan,
                "Displacement Plot Lat": np.nan,
                "Displacement Plot Dx km": np.nan,
                "Displacement Plot Dy km": np.nan,
                "Total Accumulated Area km^2": 0.0,
                "Matched PP Total Accumulated Area km^2": 0.0,
            }

            if valid_keys.empty:
                centroid_summary_rows.append(empty_summary.copy())
                continue

            method_centroids = (
                df_centroid_daily[
                    (df_centroid_daily["Risk Threshold"].eq(threshold_label))
                    & (df_centroid_daily["Method"].eq(method))
                ]
                [key_cols + ["COM Lon", "COM Lat", "Area km^2"]]
                .rename(columns={
                    "COM Lon": "Method COM Lon",
                    "COM Lat": "Method COM Lat",
                    "Area km^2": "Method Area km^2",
                })
            )

            paired = (
                valid_keys
                .merge(method_centroids, on=key_cols, how="left")
                .merge(pp_centroids, on=key_cols, how="left")
            )

            paired["Method Area km^2"] = pd.to_numeric(paired["Method Area km^2"], errors="coerce").fillna(0.0)
            paired["Matched PP Area km^2"] = pd.to_numeric(paired["Matched PP Area km^2"], errors="coerce").fillna(0.0)

            good = (
                (paired["Method Area km^2"] > 0)
                & (paired["Matched PP Area km^2"] > 0)
                & np.isfinite(pd.to_numeric(paired["Method COM Lon"], errors="coerce"))
                & np.isfinite(pd.to_numeric(paired["Method COM Lat"], errors="coerce"))
                & np.isfinite(pd.to_numeric(paired["Matched PP COM Lon"], errors="coerce"))
                & np.isfinite(pd.to_numeric(paired["Matched PP COM Lat"], errors="coerce"))
            )

            paired = paired.loc[good].copy()

            if paired.empty:
                centroid_summary_rows.append(empty_summary.copy())
                continue

            disp_km = []
            dx_km = []
            dy_km = []

            for _, row in paired.iterrows():
                d = _risk_heatmap_haversine_km(
                    row["Matched PP COM Lon"],
                    row["Matched PP COM Lat"],
                    row["Method COM Lon"],
                    row["Method COM Lat"],
                )

                dx, dy = _risk_heatmap_signed_dxdy_km(
                    row["Matched PP COM Lon"],
                    row["Matched PP COM Lat"],
                    row["Method COM Lon"],
                    row["Method COM Lat"],
                )

                disp_km.append(d)
                dx_km.append(dx)
                dy_km.append(dy)

                displacement_daily_rows.append({
                    "Date": row["Date"],
                    "Risk Threshold": threshold_label,
                    "Threshold": float(threshold),
                    "Method": method,
                    "Method COM Lon": row["Method COM Lon"],
                    "Method COM Lat": row["Method COM Lat"],
                    "Matched PP COM Lon": row["Matched PP COM Lon"],
                    "Matched PP COM Lat": row["Matched PP COM Lat"],
                    "Method Area km^2": row["Method Area km^2"],
                    "Matched PP Area km^2": row["Matched PP Area km^2"],
                    "Signed Dx km": dx,
                    "Signed Dy km": dy,
                    "Displacement Error vs PP km": d,
                })

            paired["Displacement Error vs PP km"] = disp_km
            paired["Signed Dx km"] = dx_km
            paired["Signed Dy km"] = dy_km

            method_lon = pd.to_numeric(paired["Method COM Lon"], errors="coerce").to_numpy(float)
            method_lat = pd.to_numeric(paired["Method COM Lat"], errors="coerce").to_numpy(float)
            pp_lon = pd.to_numeric(paired["Matched PP COM Lon"], errors="coerce").to_numpy(float)
            pp_lat = pd.to_numeric(paired["Matched PP COM Lat"], errors="coerce").to_numpy(float)

            mean_forecast_lon = float(np.nanmean(method_lon))
            mean_forecast_lat = float(np.nanmean(method_lat))
            mean_pp_lon = float(np.nanmean(pp_lon))
            mean_pp_lat = float(np.nanmean(pp_lat))

            mean_disp = float(np.nanmean(paired["Displacement Error vs PP km"]))
            med_disp = float(np.nanmedian(paired["Displacement Error vs PP km"]))

            mean_dx = float(np.nanmean(paired["Signed Dx km"]))
            mean_dy = float(np.nanmean(paired["Signed Dy km"]))

            signed_mag = float(np.sqrt(mean_dx ** 2 + mean_dy ** 2))

            if np.isfinite(mean_disp) and mean_disp > 0 and np.isfinite(signed_mag) and signed_mag > 1.0e-6:
                unit_dx = mean_dx / signed_mag
                unit_dy = mean_dy / signed_mag

                plot_dx = unit_dx * mean_disp
                plot_dy = unit_dy * mean_disp
                cancellation_ratio = signed_mag / mean_disp
            else:
                plot_dx = 0.0
                plot_dy = 0.0
                cancellation_ratio = np.nan if not np.isfinite(mean_disp) or mean_disp <= 0 else 0.0

            plot_lon, plot_lat = _risk_heatmap_shift_lonlat_by_dxdy(
                mean_pp_lon,
                mean_pp_lat,
                plot_dx,
                plot_dy,
            )

            if method == "Practically Perfect":
                plot_lon = mean_pp_lon
                plot_lat = mean_pp_lat
                plot_dx = 0.0
                plot_dy = 0.0
                signed_mag = 0.0
                cancellation_ratio = 1.0

            centroid_summary_rows.append({
                "Risk Threshold": threshold_label,
                "Threshold": float(threshold),
                "Method": method,
                "N Common Cases": int(len(paired)),
                "Mean Forecast COM Lon": mean_forecast_lon,
                "Mean Forecast COM Lat": mean_forecast_lat,
                "Matched PP Mean COM Lon": mean_pp_lon,
                "Matched PP Mean COM Lat": mean_pp_lat,
                "Mean Signed Dx km": mean_dx,
                "Mean Signed Dy km": mean_dy,
                "Mean Signed Vector Magnitude km": signed_mag,
                "Mean Displacement Error vs PP km": mean_disp,
                "Median Displacement Error vs PP km": med_disp,
                "Directional Cancellation Ratio": cancellation_ratio,
                "Plot Anchor PP COM Lon": mean_pp_lon,
                "Plot Anchor PP COM Lat": mean_pp_lat,
                "Displacement Plot Lon": plot_lon,
                "Displacement Plot Lat": plot_lat,
                "Displacement Plot Dx km": plot_dx,
                "Displacement Plot Dy km": plot_dy,
                "Total Accumulated Area km^2": float(np.nansum(paired["Method Area km^2"])),
                "Matched PP Total Accumulated Area km^2": float(np.nansum(paired["Matched PP Area km^2"])),
            })

    df_centroid_summary_common = pd.DataFrame(centroid_summary_rows)

    # ------------------------------------------------------------------
    # Re-anchor all forecast displacement markers to the displayed PP marker
    # for that threshold, so plotted distance from PP visually matches the
    # mean displacement error in the table.
    # ------------------------------------------------------------------
    for threshold, threshold_label in HEATMAP_THRESHOLDS:
        row_sel = df_centroid_summary_common["Risk Threshold"].eq(threshold_label)

        pp_rows = df_centroid_summary_common[
            row_sel
            & df_centroid_summary_common["Method"].eq("Practically Perfect")
            & (df_centroid_summary_common["N Common Cases"] > 0)
        ]

        if not pp_rows.empty:
            pp_anchor_lon = float(pp_rows.iloc[0]["Mean Forecast COM Lon"])
            pp_anchor_lat = float(pp_rows.iloc[0]["Mean Forecast COM Lat"])
        else:
            pp_candidates = df_centroid_summary_common[
                row_sel
                & np.isfinite(pd.to_numeric(df_centroid_summary_common["Matched PP Mean COM Lon"], errors="coerce"))
                & np.isfinite(pd.to_numeric(df_centroid_summary_common["Matched PP Mean COM Lat"], errors="coerce"))
            ]

            if pp_candidates.empty:
                pp_anchor_lon = np.nan
                pp_anchor_lat = np.nan
            else:
                pp_anchor_lon = float(np.nanmean(pp_candidates["Matched PP Mean COM Lon"]))
                pp_anchor_lat = float(np.nanmean(pp_candidates["Matched PP Mean COM Lat"]))

        for idx in df_centroid_summary_common[row_sel].index:
            method = df_centroid_summary_common.at[idx, "Method"]
            n_common = df_centroid_summary_common.at[idx, "N Common Cases"]

            df_centroid_summary_common.at[idx, "Plot Anchor PP COM Lon"] = pp_anchor_lon
            df_centroid_summary_common.at[idx, "Plot Anchor PP COM Lat"] = pp_anchor_lat

            if not np.isfinite(pp_anchor_lon) or not np.isfinite(pp_anchor_lat) or n_common <= 0:
                df_centroid_summary_common.at[idx, "Displacement Plot Lon"] = np.nan
                df_centroid_summary_common.at[idx, "Displacement Plot Lat"] = np.nan
                continue

            if method == "Practically Perfect":
                df_centroid_summary_common.at[idx, "Displacement Plot Lon"] = pp_anchor_lon
                df_centroid_summary_common.at[idx, "Displacement Plot Lat"] = pp_anchor_lat
                df_centroid_summary_common.at[idx, "Displacement Plot Dx km"] = 0.0
                df_centroid_summary_common.at[idx, "Displacement Plot Dy km"] = 0.0
                continue

            mean_dx = df_centroid_summary_common.at[idx, "Mean Signed Dx km"]
            mean_dy = df_centroid_summary_common.at[idx, "Mean Signed Dy km"]
            mean_disp = df_centroid_summary_common.at[idx, "Mean Displacement Error vs PP km"]

            if not np.isfinite(mean_dx) or not np.isfinite(mean_dy) or not np.isfinite(mean_disp) or mean_disp <= 0:
                plot_dx = 0.0
                plot_dy = 0.0
            else:
                signed_mag = float(np.sqrt(mean_dx ** 2 + mean_dy ** 2))

                if signed_mag <= 1.0e-6:
                    plot_dx = 0.0
                    plot_dy = 0.0
                else:
                    plot_dx = mean_disp * mean_dx / signed_mag
                    plot_dy = mean_disp * mean_dy / signed_mag

            plot_lon, plot_lat = _risk_heatmap_shift_lonlat_by_dxdy(
                pp_anchor_lon,
                pp_anchor_lat,
                plot_dx,
                plot_dy,
            )

            df_centroid_summary_common.at[idx, "Displacement Plot Dx km"] = plot_dx
            df_centroid_summary_common.at[idx, "Displacement Plot Dy km"] = plot_dy
            df_centroid_summary_common.at[idx, "Displacement Plot Lon"] = plot_lon
            df_centroid_summary_common.at[idx, "Displacement Plot Lat"] = plot_lat

    df_displacement_daily_common = pd.DataFrame(displacement_daily_rows)

    df_displacement_summary_common = df_centroid_summary_common[
        [
            "Risk Threshold",
            "Threshold",
            "Method",
            "N Common Cases",
            "Mean Displacement Error vs PP km",
            "Median Displacement Error vs PP km",
            "Mean Signed Dx km",
            "Mean Signed Dy km",
            "Mean Signed Vector Magnitude km",
            "Directional Cancellation Ratio",
            "Plot Anchor PP COM Lon",
            "Plot Anchor PP COM Lat",
            "Displacement Plot Lon",
            "Displacement Plot Lat",
            "Displacement Plot Dx km",
            "Displacement Plot Dy km",
            "Mean Forecast COM Lon",
            "Mean Forecast COM Lat",
            "Matched PP Mean COM Lon",
            "Matched PP Mean COM Lat",
        ]
    ].copy()

    # ------------------------------------------------------------------
    # Sort output tables
    # ------------------------------------------------------------------
    tables_to_sort = [
        df_area_summary_common,
        df_area_error_vs_pp_common,
        df_centroid_summary_common,
        df_displacement_summary_common,
    ]

    threshold_order = {label: i for i, (_, label) in enumerate(HEATMAP_THRESHOLDS)}

    for table in tables_to_sort:
        table["__thr_order"] = table["Risk Threshold"].map(threshold_order)
        table["__method_order"] = table["Method"].map(_risk_heatmap_method_sort_key)

    df_area_summary_common = (
        df_area_summary_common
        .sort_values(["__thr_order", "__method_order"])
        .drop(columns=["__thr_order", "__method_order"])
        .reset_index(drop=True)
    )

    df_area_error_vs_pp_common = (
        df_area_error_vs_pp_common
        .sort_values(["__thr_order", "__method_order"])
        .drop(columns=["__thr_order", "__method_order"])
        .reset_index(drop=True)
    )

    df_centroid_summary_common = (
        df_centroid_summary_common
        .sort_values(["__thr_order", "__method_order"])
        .drop(columns=["__thr_order", "__method_order"])
        .reset_index(drop=True)
    )

    df_displacement_summary_common = (
        df_displacement_summary_common
        .sort_values(["__thr_order", "__method_order"])
        .drop(columns=["__thr_order", "__method_order"])
        .reset_index(drop=True)
    )

    df_common_case_policy["__thr_order"] = df_common_case_policy["Risk Threshold"].map(threshold_order)
    df_common_case_policy = (
        df_common_case_policy
        .sort_values(["__thr_order"])
        .drop(columns=["__thr_order"])
        .reset_index(drop=True)
    )

    if not df_displacement_daily_common.empty:
        df_displacement_daily_common["__thr_order"] = df_displacement_daily_common["Risk Threshold"].map(threshold_order)
        df_displacement_daily_common["__method_order"] = df_displacement_daily_common["Method"].map(_risk_heatmap_method_sort_key)
        df_displacement_daily_common = (
            df_displacement_daily_common
            .sort_values(["Date", "__thr_order", "__method_order"])
            .drop(columns=["__thr_order", "__method_order"])
            .reset_index(drop=True)
        )

    return {
        "area_summary_common": df_area_summary_common,
        "area_error_vs_pp_common": df_area_error_vs_pp_common,
        "centroid_summary_common": df_centroid_summary_common,
        "displacement_daily_common": df_displacement_daily_common,
        "displacement_summary_common": df_displacement_summary_common,
        "common_case_policy": df_common_case_policy,
    }


# ======================================================================================
# PLOTTING
# ======================================================================================

def plot_risk_area_heatmap_panels(results):
    lon = results["lon"]
    lat = results["lat"]
    heat_count = results["heat_count"]
    df_centroids = results["centroid_summary"].copy()

    panel_cols = ["ML", "WPC ERO", "Practically Perfect"]

    panel_titles = {
        "ML": "ML combined radii",
        "WPC ERO": "WPC ERO",
        "Practically Perfect": "Practically Perfect",
    }

    nrows = len(HEATMAP_THRESHOLDS)
    ncols = len(panel_cols)

    projection = ccrs.PlateCarree() if HAS_CARTOPY_RISK_HEATMAP else None
    subplot_kw = {"projection": projection} if HAS_CARTOPY_RISK_HEATMAP else {}

    fig, axes = plt.subplots(
        nrows,
        ncols,
        figsize=HEATMAP_FIGSIZE,
        subplot_kw=subplot_kw,
        constrained_layout=False,
    )

    fig.subplots_adjust(
        left=0.075,
        right=0.89,
        top=0.955,
        bottom=0.085,
        wspace=0.075,
        hspace=0.10,
    )

    axes = np.asarray(axes)
    transform = ccrs.PlateCarree() if HAS_CARTOPY_RISK_HEATMAP else None

    for row_i, (threshold, threshold_label) in enumerate(HEATMAP_THRESHOLDS):
        row_arrays = [heat_count[(threshold_label, panel)] for panel in panel_cols]

        pp_arr = heat_count[(threshold_label, "Practically Perfect")]
        pp_positive = pp_arr[np.isfinite(pp_arr) & (pp_arr > 0)]
        pp_max = float(np.nanmax(pp_positive)) if pp_positive.size else 0.0

        row_positive_parts = [
            arr[np.isfinite(arr) & (arr > 0)]
            for arr in row_arrays
            if np.any(np.isfinite(arr) & (arr > 0))
        ]

        if row_positive_parts:
            row_max = float(np.nanmax(np.concatenate(row_positive_parts)))
        else:
            row_max = 0.0

        row_cmap, row_norm, row_bounds, row_ticks, row_ticklabels, pp_baseline_top, row_top = (
            _risk_heatmap_binned_cmap_norm_labels(
                threshold_label=threshold_label,
                pp_max_count=pp_max,
                row_max_count=row_max,
            )
        )

        last_scatter = None

        for col_i, panel in enumerate(panel_cols):
            ax = axes[row_i, col_i]
            _risk_heatmap_setup_map_axis(ax)

            count_arr = heat_count[(threshold_label, panel)]

            good = (
                np.isfinite(lon)
                & np.isfinite(lat)
                & np.isfinite(count_arr)
            )

            scatter_kwargs = dict(
                c=count_arr[good],
                s=HEATMAP_POINT_SIZE,
                cmap=row_cmap,
                norm=row_norm,
                alpha=HEATMAP_POINT_ALPHA,
                linewidths=0.0,
                marker="s",
                rasterized=True,
            )

            if transform is not None:
                scatter_kwargs["transform"] = transform

            last_scatter = ax.scatter(lon[good], lat[good], **scatter_kwargs)

            if HEATMAP_PLOT_ALL_METHOD_CENTROIDS_ON_EACH_PANEL:
                cent_sub = df_centroids[df_centroids["Risk Threshold"].eq(threshold_label)].copy()
            else:
                if panel == "ML":
                    cent_sub = df_centroids[
                        df_centroids["Risk Threshold"].eq(threshold_label)
                        & df_centroids["Method"].astype(str).str.startswith("ML")
                    ].copy()
                else:
                    cent_sub = df_centroids[
                        df_centroids["Risk Threshold"].eq(threshold_label)
                        & df_centroids["Method"].eq(panel)
                    ].copy()

            for _, crow in cent_sub.iterrows():
                cx = crow["Displacement Plot Lon"]
                cy = crow["Displacement Plot Lat"]

                if not np.isfinite(cx) or not np.isfinite(cy):
                    continue

                style = _risk_heatmap_centroid_style(crow["Method"])

                centroid_kwargs = dict(
                    marker=style["marker"],
                    c=style["color"],
                    edgecolors=style["edgecolor"],
                    linewidths=style["linewidth"],
                    s=style["s"],
                    zorder=100,
                )

                if transform is not None:
                    centroid_kwargs["transform"] = transform

                ax.scatter([cx], [cy], **centroid_kwargs)

            if row_i == 0:
                ax.set_title(panel_titles[panel], fontsize=HEATMAP_PANEL_TITLE_SIZE, pad=16)

            if col_i == 0:
                ax.text(
                    -0.075,
                    0.5,
                    threshold_label,
                    transform=ax.transAxes,
                    ha="right",
                    va="center",
                    rotation=90,
                    fontsize=HEATMAP_ROW_LABEL_SIZE,
                    fontweight="bold",
                )

        cb = fig.colorbar(
            last_scatter,
            ax=axes[row_i, :],
            orientation="vertical",
            shrink=0.80,
            pad=0.013,
            boundaries=row_bounds,
            ticks=row_ticks,
        )
        cb.ax.set_yticklabels(row_ticklabels)
        cb.set_label(
            "Case-occurrences",
            fontsize=HEATMAP_COLORBAR_FONT_SIZE,
            labelpad=12,
        )
        cb.ax.tick_params(labelsize=HEATMAP_COLORBAR_FONT_SIZE)

    methods_for_legend = sorted(
        df_centroids["Method"].dropna().astype(str).unique().tolist(),
        key=_risk_heatmap_method_sort_key,
    )

    legend_handles = []
    seen = set()

    for method in methods_for_legend:
        style = _risk_heatmap_centroid_style(method)
        label = style["label"]

        if label in seen:
            continue

        seen.add(label)

        legend_handles.append(
            Line2D(
                [0],
                [0],
                marker=style["marker"],
                linestyle="None",
                markerfacecolor=style["color"],
                markeredgecolor=style["edgecolor"],
                markeredgewidth=style["linewidth"],
                markersize=23 if style["marker"] != "*" else 30,
                label=label,
            )
        )

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        ncol=min(4, max(1, len(legend_handles))),
        fontsize=HEATMAP_LEGEND_FONT_SIZE,
        frameon=True,
        bbox_to_anchor=(0.5, 0.018),
        borderaxespad=0.0,
        handletextpad=0.6,
        columnspacing=1.1,
    )

    if HEATMAP_SAVE_FIG:
        HEATMAP_OUTDIR.mkdir(parents=True, exist_ok=True)

        roi_tag = "roi_expanded" if HEATMAP_APPLY_VERIFY_ROI_EXPANSION else "raw"
        ml_tag = str(HEATMAP_ML_COMBINE_MODE)
        pmm_tag = f"_with_local_pmm_{int(HEATMAP_LOCAL_PMM_RADIUS_KM)}km" if HEATMAP_ADD_ML_LOCAL_PMM else ""

        png_path = HEATMAP_OUTDIR / f"risk_area_occurrence_heatmaps_common_case_displacement_markers_pp_binned_{ml_tag}_{roi_tag}{pmm_tag}.png"
        pdf_path = HEATMAP_OUTDIR / f"risk_area_occurrence_heatmaps_common_case_displacement_markers_pp_binned_{ml_tag}_{roi_tag}{pmm_tag}.pdf"

        fig.savefig(png_path, dpi=int(HEATMAP_DPI), bbox_inches="tight")
        fig.savefig(pdf_path, dpi=int(HEATMAP_DPI), bbox_inches="tight")

        print("Saved:", png_path)
        print("Saved:", pdf_path)

    plt.show()

    return fig


# ======================================================================================
# RUN EVERYTHING
# ======================================================================================

risk_heatmap_results = compute_risk_heatmaps_centroids_area_displacement(df_radius_viewer)

common_case_results = build_common_case_risk_summaries(risk_heatmap_results)

df_risk_area_daily = risk_heatmap_results["area_daily"]
df_risk_centroid_daily = risk_heatmap_results["centroid_daily"]

df_risk_area_summary_common = common_case_results["area_summary_common"]
df_risk_area_error_vs_pp_common = common_case_results["area_error_vs_pp_common"]
df_risk_centroid_summary_common = common_case_results["centroid_summary_common"]
df_risk_displacement_daily_common = common_case_results["displacement_daily_common"]
df_risk_displacement_summary_common = common_case_results["displacement_summary_common"]
df_risk_common_case_policy = common_case_results["common_case_policy"]

risk_heatmap_results["centroid_summary"] = df_risk_centroid_summary_common.copy()


print("\n" + "=" * 120)
print("COMMON-CASE POLICY BY THRESHOLD")
print("=" * 120)
display(df_risk_common_case_policy)


print("\n" + "=" * 120)
print("COMMON-CASE AVERAGE RISK AREAS")
print("=" * 120)
display(
    df_risk_area_summary_common
    .round({
        "Mean Area km^2": 1,
        "Median Area km^2": 1,
        "Max Area km^2": 1,
        "Mean Area Fraction": 5,
        "Matched PP Mean Area km^2": 1,
        "Matched PP Median Area km^2": 1,
        "Matched PP Mean Area Fraction": 5,
        "Area Error vs Matched PP km^2": 1,
        "Absolute Area Error vs Matched PP km^2": 1,
        "Area Ratio vs Matched PP": 3,
        "Percent Area Error vs Matched PP": 1,
    })
)


print("\n" + "=" * 120)
print("COMMON-CASE AREA ERROR RELATIVE TO MATCHED PRACTICALLY PERFECT")
print("=" * 120)
display(
    df_risk_area_error_vs_pp_common[
        [
            "Risk Threshold",
            "Method",
            "N Common Cases",
            "Mean Area km^2",
            "Matched PP Mean Area km^2",
            "Area Error vs Matched PP km^2",
            "Absolute Area Error vs Matched PP km^2",
            "Area Ratio vs Matched PP",
            "Percent Area Error vs Matched PP",
        ]
    ]
    .round({
        "Mean Area km^2": 1,
        "Matched PP Mean Area km^2": 1,
        "Area Error vs Matched PP km^2": 1,
        "Absolute Area Error vs Matched PP km^2": 1,
        "Area Ratio vs Matched PP": 3,
        "Percent Area Error vs Matched PP": 1,
    })
)


print("\n" + "=" * 120)
print("COMMON-CASE CATEGORICAL COM + DISPLACEMENT-PLOT MARKERS")
print("=" * 120)
display(
    df_risk_centroid_summary_common
    .round({
        "Mean Forecast COM Lon": 3,
        "Mean Forecast COM Lat": 3,
        "Matched PP Mean COM Lon": 3,
        "Matched PP Mean COM Lat": 3,
        "Mean Signed Dx km": 1,
        "Mean Signed Dy km": 1,
        "Mean Signed Vector Magnitude km": 1,
        "Mean Displacement Error vs PP km": 1,
        "Median Displacement Error vs PP km": 1,
        "Directional Cancellation Ratio": 3,
        "Plot Anchor PP COM Lon": 3,
        "Plot Anchor PP COM Lat": 3,
        "Displacement Plot Lon": 3,
        "Displacement Plot Lat": 3,
        "Displacement Plot Dx km": 1,
        "Displacement Plot Dy km": 1,
        "Total Accumulated Area km^2": 1,
        "Matched PP Total Accumulated Area km^2": 1,
    })
)


print("\n" + "=" * 120)
print("COMMON-CASE AVERAGE CATEGORICAL-COM DISPLACEMENT ERROR RELATIVE TO MATCHED PRACTICALLY PERFECT")
print("=" * 120)
display(
    df_risk_displacement_summary_common
    .round({
        "Mean Displacement Error vs PP km": 1,
        "Median Displacement Error vs PP km": 1,
        "Mean Signed Dx km": 1,
        "Mean Signed Dy km": 1,
        "Mean Signed Vector Magnitude km": 1,
        "Directional Cancellation Ratio": 3,
        "Plot Anchor PP COM Lon": 3,
        "Plot Anchor PP COM Lat": 3,
        "Displacement Plot Lon": 3,
        "Displacement Plot Lat": 3,
        "Displacement Plot Dx km": 1,
        "Displacement Plot Dy km": 1,
        "Mean Forecast COM Lon": 3,
        "Mean Forecast COM Lat": 3,
        "Matched PP Mean COM Lon": 3,
        "Matched PP Mean COM Lat": 3,
    })
)


if HEATMAP_SAVE_TABLES:
    HEATMAP_OUTDIR.mkdir(parents=True, exist_ok=True)

    pmm_tag = f"_with_local_pmm_{int(HEATMAP_LOCAL_PMM_RADIUS_KM)}km" if HEATMAP_ADD_ML_LOCAL_PMM else ""

    df_risk_common_case_policy.to_csv(
        HEATMAP_OUTDIR / f"risk_common_case_policy{pmm_tag}.csv",
        index=False,
    )

    df_risk_area_daily.to_csv(
        HEATMAP_OUTDIR / f"risk_area_daily_all_cases{pmm_tag}.csv",
        index=False,
    )

    df_risk_centroid_daily.to_csv(
        HEATMAP_OUTDIR / f"risk_categorical_com_daily_all_cases{pmm_tag}.csv",
        index=False,
    )

    df_risk_area_summary_common.to_csv(
        HEATMAP_OUTDIR / f"risk_area_summary_common_case{pmm_tag}.csv",
        index=False,
    )

    df_risk_area_error_vs_pp_common.to_csv(
        HEATMAP_OUTDIR / f"risk_area_error_vs_matched_pp_common_case{pmm_tag}.csv",
        index=False,
    )

    df_risk_centroid_summary_common.to_csv(
        HEATMAP_OUTDIR / f"risk_categorical_com_displacement_marker_summary_common_case{pmm_tag}.csv",
        index=False,
    )

    df_risk_displacement_daily_common.to_csv(
        HEATMAP_OUTDIR / f"risk_categorical_com_displacement_daily_common_case{pmm_tag}.csv",
        index=False,
    )

    df_risk_displacement_summary_common.to_csv(
        HEATMAP_OUTDIR / f"risk_categorical_com_displacement_summary_common_case{pmm_tag}.csv",
        index=False,
    )

    print("\nSaved common-case tables to:", HEATMAP_OUTDIR)


fig_risk_area_heatmaps = plot_risk_area_heatmap_panels(risk_heatmap_results)


print("\nCreated objects:")
print("  risk_heatmap_results")
print("  common_case_results")
print("  df_risk_common_case_policy")
print("  df_risk_area_daily")
print("  df_risk_centroid_daily")
print("  df_risk_area_summary_common")
print("  df_risk_area_error_vs_pp_common")
print("  df_risk_centroid_summary_common")
print("  df_risk_displacement_daily_common")
print("  df_risk_displacement_summary_common")
print("  fig_risk_area_heatmaps")

In [ ]:
!pip install netcdf4

In [ ]:
# Day-2 verification invariants
print('Verification baselines: Practically Perfect and raw UFVS proxies expanded 40 km.')
print('WPC Day-2 is radius-independent; R40/R60/R75/R100 identify ML targets only.')
if 'df_case_metrics' in globals():
    assert_wpc_radius_independent(df_case_metrics)
